# 🧱 Grandpa's Block Lab: one notebook, any lesson

Upload a **story file** (`.json`) and this notebook makes the whole Short from it. Grandpa (the same voxel
character every time) builds, points, highlights, grows, cracks or collapses things while he explains, in his own
voice (offline Kokoro, `am_santa`), with his mouth moving.

You get a 1080×1920, 30 fps MP4 with lesson graphics, captions, procedural sound and −14 LUFS audio, checked automatically.

**How to run**
1. Choose *Runtime → Change runtime type → GPU*.
2. Choose *Runtime → Run all*.
3. In section 2, click **Choose files** and upload your story. If you don't upload one, the example set in `EXAMPLE` is used.

Example stories included: `giants` ("Why giants can't exist"), `odd_squares` ("Odd numbers make squares"), `stone_arch` ("Why stone arches stand"), `gears` ("How gears trade speed for force"), `eiffel_air` ("The Eiffel Tower is mostly air", in Paris), two looping Shorts: `trebuchet` and `lock_key` ("How a lock and key work"), and two widescreen builds: `eiffel_build_wide` (12 s) and `eiffel_full_build` (scaffold, ladder, hoist and hammer, ~75 s sped-up preview).

In [ ]:
# ---- settings ----
EXAMPLE = "lock_key"          # used when you don't upload a story: giants, odd_squares, stone_arch, gears, eiffel_air, eiffel_build_wide, eiffel_full_build, eiffel_rivets, eiffel_lessons_short, car_engine, trebuchet, lock_key
USE_GPU = True
QUICK_DRAFT = False           # True: fast low-res 3D test (graphics + captions still sharp at 1080x1920)
USE_DRIVE = False             # True: keep the engine, stories and MP4s in Google Drive (MyDrive/block_lab)
SAVE_FRAMES_TO_DRIVE = False  # True: keep rendered frames in Drive (resume after a disconnect)

import os
if USE_DRIVE or SAVE_FRAMES_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/block_lab" if USE_DRIVE else "/content/block_lab"
os.makedirs(WORK + "/stories", exist_ok=True)
os.chdir(WORK)
print("engine folder:", WORK)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "No GPU runtime: rendering will use the CPU"

## 1. The engine (same every time) and the example stories

In [ ]:
%%writefile voxelkit.py
"""Shared voxel kit: materials, box meshes, rig math and the voxel grandpa (armature + rigid-skinned mesh, jaw bone).

Extracted from the earlier grandpa shorts so every lesson uses the identical character.
"""
import math

import bpy
import bmesh
from mathutils import Matrix, Vector

L_UP, L_FORE, L_THIGH, L_SHIN = 0.20, 0.225, 0.22, 0.24
BONE_ORDER = ("hips", "spine", "head", "jaw", "upper_arm.R", "forearm.R", "upper_arm.L", "forearm.L",
              "thigh.R", "shin.R", "thigh.L", "shin.L")


def cl(x, a=0.0, b=1.0):
    return a if x < a else b if x > b else x


def smooth(x):
    x = cl(x)
    return x * x * (3 - 2 * x)


def smoother(x):
    x = cl(x)
    return x ** 3 * (x * (6 * x - 15) + 10)


def mat(name, color, rough=0.8, metal=0.0):
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    p = m.node_tree.nodes["Principled BSDF"]
    p.inputs["Base Color"].default_value = (*color, 1)
    p.inputs["Roughness"].default_value = rough
    p.inputs["Metallic"].default_value = metal
    return m


def pixel_material(name, c_dark, c_light, px, per_object=False, voxel=None, rough=0.9, metal=0.0, bump=0.12):
    """Minecraft-style pixel texture: position snapped to a texel grid -> white noise -> colour ramp."""
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    N, Lk = m.node_tree.nodes, m.node_tree.links
    p = N["Principled BSDF"]
    p.inputs["Roughness"].default_value = rough
    p.inputs["Metallic"].default_value = metal
    tc = N.new("ShaderNodeTexCoord")
    geo = N.new("ShaderNodeNewGeometry")
    inset = N.new("ShaderNodeVectorMath"); inset.operation = "SCALE"; inset.inputs[3].default_value = -px * 0.45
    Lk.new(tc.outputs["Normal"] if per_object else geo.outputs["Normal"], inset.inputs[0])
    src = tc.outputs["Object"] if per_object else geo.outputs["Position"]
    add = N.new("ShaderNodeVectorMath"); add.operation = "ADD"
    Lk.new(src, add.inputs[0]); Lk.new(inset.outputs[0], add.inputs[1])
    snap = N.new("ShaderNodeVectorMath"); snap.operation = "SNAP"; snap.inputs[1].default_value = (px, px, px)
    Lk.new(add.outputs[0], snap.inputs[0])
    wn = N.new("ShaderNodeTexWhiteNoise"); wn.noise_dimensions = "3D"
    Lk.new(snap.outputs[0], wn.inputs["Vector"])
    fac = wn.outputs["Value"]
    if voxel:
        sv = N.new("ShaderNodeVectorMath"); sv.operation = "SNAP"; sv.inputs[1].default_value = (voxel, voxel, voxel)
        Lk.new(add.outputs[0], sv.inputs[0])
        wv = N.new("ShaderNodeTexWhiteNoise"); wv.noise_dimensions = "3D"
        Lk.new(sv.outputs[0], wv.inputs["Vector"])
        mix = N.new("ShaderNodeMix"); mix.data_type = "FLOAT"; mix.inputs["Factor"].default_value = 0.4
        Lk.new(fac, mix.inputs["A"]); Lk.new(wv.outputs["Value"], mix.inputs["B"])
        fac = mix.outputs["Result"]
    if per_object:
        oi = N.new("ShaderNodeObjectInfo")
        mix = N.new("ShaderNodeMix"); mix.data_type = "FLOAT"; mix.inputs["Factor"].default_value = 0.25
        Lk.new(fac, mix.inputs["A"]); Lk.new(oi.outputs["Random"], mix.inputs["B"])
        fac = mix.outputs["Result"]
    ramp = N.new("ShaderNodeValToRGB")
    ramp.color_ramp.elements[0].color = (*c_dark, 1)
    ramp.color_ramp.elements[1].color = (*c_light, 1)
    Lk.new(fac, ramp.inputs["Fac"])
    Lk.new(ramp.outputs["Color"], p.inputs["Base Color"])
    bp = N.new("ShaderNodeBump"); bp.inputs["Strength"].default_value = bump
    Lk.new(wn.outputs["Value"], bp.inputs["Height"])
    Lk.new(bp.outputs["Normal"], p.inputs["Normal"])
    return m


def emission_material(name, color, strength):
    m = bpy.data.materials.new(name)
    m.use_nodes = True
    N, Lk = m.node_tree.nodes, m.node_tree.links
    N.remove(N["Principled BSDF"])
    em = N.new("ShaderNodeEmission")
    em.inputs["Color"].default_value = (*color, 1)
    em.inputs["Strength"].default_value = strength
    Lk.new(em.outputs[0], N["Material Output"].inputs["Surface"])
    return m


def add_box(bm, center, size, mi=0):
    r = bmesh.ops.create_cube(bm, size=1.0)
    for v in r["verts"]:
        v.co = Vector((v.co.x * size[0] + center[0], v.co.y * size[1] + center[1], v.co.z * size[2] + center[2]))
    for f in {f for v in r["verts"] for f in v.link_faces}:
        f.material_index = mi
    return r["verts"]


def mesh_obj(name, boxes, mats, parent=None, coll=None):
    """boxes: list of (center, size, material index)."""
    bm = bmesh.new()
    for c, s, mi in boxes:
        add_box(bm, c, s, mi)
    me = bpy.data.meshes.new(name)
    bm.to_mesh(me); bm.free()
    for m in mats:
        me.materials.append(m)
    o = bpy.data.objects.new(name, me)
    (coll or bpy.context.scene.collection).objects.link(o)
    if parent:
        o.parent = parent
    return o


def look_quat(eye, target):
    return (Vector(target) - Vector(eye)).to_track_quat("-Z", "Y")


def fcurves_of(o):
    ad = o.animation_data
    if not ad or not ad.action:
        return []
    act = ad.action
    if hasattr(act, "fcurves") and len(getattr(act, "fcurves", [])):
        return list(act.fcurves)
    out = []
    for layer in getattr(act, "layers", []):
        for strip in layer.strips:
            for cb in strip.channelbags:
                out.extend(cb.fcurves)
    return out


def set_interp(o, kind, frames=None):
    for fc in fcurves_of(o):
        for kp in fc.keyframe_points:
            if frames is None or int(round(kp.co.x)) in frames:
                kp.interpolation = kind


def rx(a):
    return Matrix.Rotation(a, 3, "X")


def ry(a):
    return Matrix.Rotation(a, 3, "Y")


def rz(a):
    return Matrix.Rotation(a, 3, "Z")


def two_bone(S, T, l1, l2, pole):
    d = T - S
    dist = d.length
    u = d.normalized() if dist > 1e-6 else Vector((0, 0, -1))
    dc = min(max(dist, abs(l1 - l2) + 1e-3), l1 + l2 - 1e-4)
    a = (l1 * l1 + dc * dc - l2 * l2) / (2 * dc)
    h = math.sqrt(max(0.0, l1 * l1 - a * a))
    n = pole - u * pole.dot(u)
    n = n.normalized() if n.length > 1e-6 else Vector((0, 1, 0))
    return S + u * a + n * h, S + u * dc, dist / (l1 + l2)


def aim(Qparent, rest_dir, d):
    cur = Qparent @ rest_dir
    return cur.rotation_difference(d.normalized()).to_matrix() @ Qparent


def bump(t, a, b, e=0.35):
    e = min(e, (b - a) * 0.3)
    return smooth((t - a) / e) * (1 - smooth((t - (b - e)) / e))


SHIRTS = {"rust": (0.62, 0.30, 0.20), "red": (0.72, 0.12, 0.12), "blue": (0.20, 0.38, 0.68), "green": (0.22, 0.50, 0.25),
          "yellow": (0.90, 0.72, 0.20), "purple": (0.45, 0.25, 0.60), "white": (0.90, 0.90, 0.88), "striped": (0.16, 0.24, 0.48)}


def build_grandpa(sc, parent, outfit=None):
    """outfit: {"hat": cap|hardhat|santa|beret|straw|beanie|none, "shirt": colour name, "extras": [goggles, scarf, vest]}"""
    outfit = dict(dict(hat="cap", shirt="rust", extras=[]), **(outfit or {}))
    arm_data = bpy.data.armatures.new("GrandpaRig")
    arm = bpy.data.objects.new("Grandpa", arm_data)
    sc.collection.objects.link(arm)
    arm.parent = parent
    bpy.context.view_layer.objects.active = arm
    bpy.ops.object.mode_set(mode="EDIT")
    BONES = [
        ("hips", (0, 0, 0.46), (0, 0, 0.56), None), ("spine", (0, 0, 0.46), (0, 0, 0.82), "hips"),
        ("head", (0, 0, 0.82), (0, 0, 1.06), "spine"), ("jaw", (0, -0.1, 0.86), (0, -0.1, 0.82), "head"),
        ("upper_arm.R", (-0.19, 0, 0.77), (-0.19, 0, 0.57), "spine"), ("forearm.R", (-0.19, 0, 0.57), (-0.19, 0, 0.37), "upper_arm.R"),
        ("upper_arm.L", (0.19, 0, 0.77), (0.19, 0, 0.57), "spine"), ("forearm.L", (0.19, 0, 0.57), (0.19, 0, 0.37), "upper_arm.L"),
        ("thigh.R", (-0.07, 0, 0.46), (-0.07, 0, 0.24), "hips"), ("shin.R", (-0.07, 0, 0.24), (-0.07, 0, 0.0), "thigh.R"),
        ("thigh.L", (0.07, 0, 0.46), (0.07, 0, 0.24), "hips"), ("shin.L", (0.07, 0, 0.24), (0.07, 0, 0.0), "thigh.L"),
    ]
    for name, h, t, par in BONES:
        eb = arm_data.edit_bones.new(name)
        eb.head, eb.tail, eb.roll = h, t, 0.0
        if par:
            eb.parent = arm_data.edit_bones[par]
            eb.use_connect = False
    bpy.ops.object.mode_set(mode="OBJECT")
    C = dict(skin=(0.86, 0.64, 0.50), skin2=(0.80, 0.52, 0.44), beard=(0.82, 0.82, 0.80), brow=(0.70, 0.70, 0.68),
             eye=(0.08, 0.06, 0.05), shirt=(0.62, 0.30, 0.20), denim=(0.20, 0.30, 0.46), boot=(0.28, 0.17, 0.10),
             belt=(0.18, 0.11, 0.07), cap=(0.36, 0.29, 0.20), cheek=(0.92, 0.50, 0.46), hair=(0.74, 0.74, 0.72),
             pouch=(0.48, 0.30, 0.15), smile=(0.45, 0.20, 0.18), white=(0.95, 0.95, 0.95),
             hard=(0.96, 0.74, 0.08), santa=(0.74, 0.06, 0.07), beret=(0.10, 0.10, 0.12), straw=(0.86, 0.73, 0.43),
             ribbon=(0.55, 0.12, 0.10), beanie=(0.18, 0.36, 0.70), beanie2=(0.28, 0.48, 0.82), scarf=(0.78, 0.14, 0.16),
             vest=(0.98, 0.45, 0.05), stripe=(0.85, 0.87, 0.88), lens=(0.35, 0.70, 0.85), frame=(0.12, 0.12, 0.13))
    C["shirt"] = SHIRTS.get(outfit["shirt"], SHIRTS["rust"])
    slot = {k: i for i, k in enumerate(C)}
    bm = bmesh.new()
    groups = {}

    def part(group, center, size, color):
        groups.setdefault(group, []).extend(add_box(bm, center, size, slot[color]))

    for s, x in (("R", -0.07), ("L", 0.07)):
        part("shin." + s, (x, -0.02, 0.04), (0.13, 0.17, 0.08), "boot")
        part("shin." + s, (x, 0, 0.165), (0.115, 0.115, 0.17), "denim")
        part("thigh." + s, (x, 0, 0.35), (0.12, 0.12, 0.22), "denim")
    part("hips", (0, 0, 0.51), (0.28, 0.15, 0.10), "denim")
    part("hips", (0, 0, 0.553), (0.29, 0.16, 0.026), "belt")
    part("hips", (-0.155, -0.01, 0.505), (0.05, 0.08, 0.075), "pouch")
    part("spine", (0, 0, 0.69), (0.28, 0.15, 0.26), "shirt")
    part("spine", (0, -0.078, 0.645), (0.16, 0.012, 0.15), "denim")
    for x in (-0.055, 0.055):
        part("spine", (x, -0.078, 0.765), (0.03, 0.012, 0.09), "denim")
        part("spine", (x, 0.078, 0.70), (0.03, 0.012, 0.22), "denim")
    part("head", (0, 0, 0.94), (0.24, 0.24, 0.24), "skin")
    part("jaw", (0, -0.125, 0.865), (0.244, 0.03, 0.09), "beard")
    part("head", (0, -0.1225, 0.8825), (0.10, 0.006, 0.045), "eye")
    part("head", (0, -0.13, 0.912), (0.13, 0.03, 0.03), "beard")
    for x in (-0.121, 0.121):
        part("head", (x, -0.03, 0.88), (0.012, 0.16, 0.12), "beard")
    part("jaw", (0, -0.1405, 0.889), (0.06, 0.004, 0.012), "smile")
    part("head", (0, -0.132, 0.935), (0.045, 0.03, 0.05), "skin2")
    for x in (-0.06, 0.06):
        part("head", (x, -0.1215, 0.972), (0.04, 0.006, 0.04), "white")
        part("head", (x, -0.1245, 0.968), (0.022, 0.006, 0.03), "eye")
        part("head", (x, -0.124, 1.008), (0.062, 0.01, 0.02), "brow")
        part("head", (x * 1.42, -0.1215, 0.93), (0.035, 0.004, 0.022), "cheek")
        part("head", (math.copysign(0.13, x), 0, 0.95), (0.02, 0.045, 0.055), "skin2")
        part("head", (math.copysign(0.1215, x), 0.06, 0.995), (0.012, 0.11, 0.07), "hair")
    part("head", (0, 0.1215, 0.99), (0.244, 0.012, 0.11), "hair")
    hat = outfit["hat"]
    if hat == "cap":
        part("head", (0, 0.012, 1.078), (0.25, 0.25, 0.036), "cap")
        part("head", (0, -0.142, 1.064), (0.2, 0.06, 0.014), "cap")
    elif hat == "hardhat":
        part("head", (0, 0, 1.066), (0.31, 0.31, 0.014), "hard")
        part("head", (0, 0, 1.105), (0.26, 0.26, 0.07), "hard")
        part("head", (0, 0, 1.15), (0.18, 0.18, 0.03), "hard")
        part("head", (0, -0.16, 1.07), (0.24, 0.05, 0.016), "hard")
    elif hat == "santa":
        part("head", (0, 0, 1.075), (0.264, 0.264, 0.04), "white")
        part("head", (0, 0.01, 1.12), (0.22, 0.22, 0.06), "santa")
        part("head", (0, 0.05, 1.165), (0.15, 0.15, 0.05), "santa")
        part("head", (0, 0.10, 1.195), (0.09, 0.09, 0.04), "santa")
        part("head", (0, 0.145, 1.20), (0.06, 0.06, 0.06), "white")
    elif hat == "beret":
        part("head", (0.02, 0, 1.075), (0.27, 0.27, 0.03), "beret")
        part("head", (0.03, 0, 1.097), (0.2, 0.2, 0.02), "beret")
        part("head", (0.03, 0, 1.112), (0.02, 0.02, 0.02), "beret")
    elif hat == "straw":
        part("head", (0, 0, 1.068), (0.42, 0.42, 0.014), "straw")
        part("head", (0, 0, 1.11), (0.24, 0.24, 0.08), "straw")
        part("head", (0, 0, 1.085), (0.246, 0.246, 0.022), "ribbon")
    elif hat == "beanie":
        part("head", (0, 0, 1.085), (0.255, 0.255, 0.06), "beanie")
        part("head", (0, 0, 1.06), (0.26, 0.26, 0.03), "beanie2")
        part("head", (0, 0, 1.135), (0.06, 0.06, 0.045), "white")
    else:                                                    # bare head: short grey hair on top
        part("head", (0, 0.01, 1.066), (0.236, 0.21, 0.014), "hair")
    extras = outfit["extras"]
    if "goggles" in extras:
        part("head", (0, -0.1235, 1.045), (0.21, 0.008, 0.014), "frame")
        for x in (-0.055, 0.055):
            part("head", (x, -0.127, 1.045), (0.06, 0.01, 0.04), "frame")
            part("head", (x, -0.1325, 1.045), (0.045, 0.004, 0.028), "lens")
    if "scarf" in extras:
        part("spine", (0, 0, 0.835), (0.27, 0.17, 0.05), "scarf")
        part("spine", (0.07, -0.088, 0.76), (0.06, 0.02, 0.12), "scarf")
    if "vest" in extras:
        part("spine", (0, 0.081, 0.69), (0.292, 0.012, 0.25), "vest")
        part("spine", (0, 0.088, 0.66), (0.292, 0.003, 0.025), "stripe")
        for x in (-1, 1):
            part("spine", (0.1435 * x, 0, 0.69), (0.012, 0.164, 0.25), "vest")
            part("spine", (0.105 * x, -0.0835, 0.69), (0.065, 0.012, 0.25), "vest")
            part("spine", (0.105 * x, -0.0905, 0.66), (0.065, 0.003, 0.025), "stripe")
    if "hammer" in extras:                                   # held in the right hand, head forward
        part("forearm.R", (-0.19, -0.06, 0.335), (0.024, 0.13, 0.024), "pouch")
        part("forearm.R", (-0.19, -0.13, 0.335), (0.036, 0.04, 0.08), "frame")
    if outfit["shirt"] == "striped":
        for z in (0.61, 0.67, 0.73, 0.79):
            part("spine", (0, 0, z), (0.284, 0.154, 0.02), "stripe")
    for s, x in (("R", -0.19), ("L", 0.19)):
        part("upper_arm." + s, (x, 0, 0.67), (0.10, 0.10, 0.20), "shirt")
        if outfit["shirt"] == "striped":
            for z in (0.62, 0.68, 0.74):
                part("upper_arm." + s, (x, 0, z), (0.104, 0.104, 0.02), "stripe")
        part("forearm." + s, (x, 0, 0.535), (0.104, 0.104, 0.07), "shirt")
        part("forearm." + s, (x, 0, 0.45), (0.09, 0.09, 0.12), "skin")
        part("forearm." + s, (x, 0, 0.345), (0.085, 0.085, 0.07), "skin")
    bm.verts.index_update()
    gidx = {g: [v.index for v in vs] for g, vs in groups.items()}
    gm = bpy.data.meshes.new("GrandpaMesh")
    bm.to_mesh(gm); bm.free()
    for k, c in C.items():
        gm.materials.append(mat("G_" + k, c, rough=0.95 if k not in ("eye", "white") else 0.35))
    gob = bpy.data.objects.new("GrandpaMesh", gm)
    sc.collection.objects.link(gob)
    for g, ids in gidx.items():
        gob.vertex_groups.new(name=g).add(ids, 1.0, "REPLACE")
    gob.parent = arm
    gob.modifiers.new("Rig", "ARMATURE").object = arm
    return arm

In [ ]:
%%writefile bl_common.py
"""Block Lab engine -- shared, pure-Python core: story loading, time references, structures and the full plan.

A story is a JSON file (see stories/). Structures: "cells" | "copy_of"+"scale" | "box" [nx,ny,nz] | "gnomon" k |
"pyramid" [9,7,5,...]; "centered": false puts cell (0,0) at "at" (so separate structures can interlock).
Times anywhere in a story can be:
    1.3                         seconds
    ["line_id", 0.62]           62 % of the way through that line
    ["line_id", "start", -0.25] start (or "end") of a line, plus/minus seconds
    "end"                       end of the film
Everything the scene, overlay and audio need is written once to build/plan.json.
"""
import json
import math
import os
import random
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
BUILD = os.path.join(HERE, "build")
FPS, B, STRIDE = 30, 0.1, 0.2
W_PX, H_PX = 1080, 1920


def story_path(argv=None):
    argv = sys.argv if argv is None else argv
    if "--story" in argv:
        return os.path.abspath(argv[argv.index("--story") + 1])
    if os.environ.get("STORY"):
        return os.path.abspath(os.environ["STORY"])
    p = os.path.join(BUILD, "story.json")
    if os.path.exists(p):
        return p
    raise SystemExit("no story: pass --story path/to/story.json (or set STORY)")


def load_story(path=None):
    path = path or story_path()
    s = json.load(open(path))
    s.setdefault("slug", os.path.splitext(os.path.basename(path))[0])
    s.setdefault("voice", "am_santa"); s.setdefault("speed", 0.95)
    s.setdefault("start", 0.3); s.setdefault("tail", 1.6)
    s.setdefault("channel_tag", "GRANDPA'S BLOCK LAB")
    return s


WORLD_OPTIONS = {                                  # story "world": {...}; anything left out keeps today's look
    "ground": ("desert", "grass", "snow", "beach", "stone", "paris"),
    "time": ("day", "morning", "sunset", "night"),
    "weather": ("clear", "clouds", "rain", "snow"),
    "animals": ("birds", "seagulls", "pigeons", "cat"),
}
OUTFIT_OPTIONS = {                                 # story "grandpa": {"outfit": {...}}
    "hat": ("cap", "hardhat", "santa", "beret", "straw", "beanie", "none"),
    "shirt": ("rust", "red", "blue", "green", "yellow", "purple", "white", "striped"),
    "extras": ("goggles", "scarf", "vest", "hammer"),
}


def world_settings(story):
    w = dict(ground="desert", time="day", weather="clear", animals=[])
    w.update(story.get("world", {}))
    o = dict(hat="cap", shirt="rust", extras=[])
    o.update(story.get("grandpa", {}).get("outfit", {}))
    for opts, d, what in ((WORLD_OPTIONS, w, "world"), (OUTFIT_OPTIONS, o, "outfit")):
        for k, v in d.items():
            if k not in opts:
                raise SystemExit("story %s: unknown setting %r (use %s)" % (what, k, ", ".join(opts)))
            for item in (v if isinstance(v, list) else [v]):
                if item not in opts[k]:
                    raise SystemExit("story %s %s: unknown %r (use %s)" % (what, k, item, ", ".join(opts[k])))
    return w, o


def cl(x, a=0.0, b=1.0):
    return a if x < a else b if x > b else x


def smooth(x):
    x = cl(x)
    return x * x * (3 - 2 * x)


def smoother(x):
    x = cl(x)
    return x ** 3 * (x * (6 * x - 15) + 10)


def motion_angle(a, t):
    """Degrees of a spin (eased from 0 to 'degrees', or linear with 'ease': false) or a swing (sine, faded in and out)."""
    if a["type"] == "spin":
        u = cl((t - a["t0"]) / max(1e-6, a["t1"] - a["t0"]))
        return a["degrees"] * (smoother(u) if a.get("ease", True) else u)
    if t <= a["t0"] or t >= a["t1"]:
        return 0.0
    fade = smooth((t - a["t0"]) / 0.4) * (1 - smooth((t - (a["t1"] - 0.4)) / 0.4))
    return a["degrees"] * fade * math.sin(2 * math.pi * (t - a["t0"]) / a.get("period", 1.6))


def fly_offset(a, t):
    """'fly': a thrown object on a true arc. Offset of the structure's root at time t:
    origin + v0*tau + 1/2*g*tau^2 (down) - anchor, with tau = tau0 + time since the throw (held after it lands)."""
    tau = a.get("tau0", 0.0) + min(max(t - a["t0"], 0.0), a["t1"] - a["t0"])
    v, g = a["v0"], a.get("g", 9.81)
    p = (v[0] * tau, v[1] * tau, v[2] * tau - 0.5 * g * tau * tau)
    o, an = a.get("origin", [0, 0, 0]), a.get("anchor", [0, 0, 0])
    return tuple(o[i] + p[i] - an[i] for i in range(3))


def fr(t):
    return int(round(t * FPS))


class Clock:
    """Resolves story time references once the narration timings exist."""

    def __init__(self, lines, film):
        self.L = {l["id"]: l for l in lines}
        self.film = film

    def __call__(self, ref):
        if ref is None:
            return None
        if isinstance(ref, (int, float)):
            return float(ref)
        if ref == "end":
            return self.film
        lid, where = ref[0], ref[1]
        off = ref[2] if len(ref) > 2 else 0.0
        ln = self.L[lid]
        if where == "start":
            t = ln["start"]
        elif where == "end":
            t = ln["end"]
        else:
            t = ln["start"] + (ln["end"] - ln["start"]) * float(where)
        return t + off


# ---------------------------------------------------------------- structures
def _part_fn(expr):
    code = compile(expr, "<part>", "eval")
    return lambda c: bool(eval(code, {"__builtins__": {}, "abs": abs, "min": min, "max": max}, {"x": c[0], "y": c[1], "z": c[2]}))


def structure_cells(name, structs, cache=None):
    cache = {} if cache is None else cache
    if name in cache:
        return cache[name]
    s = structs[name]
    if "cells" in s:
        cells = [tuple(c) for c in s["cells"]]
    elif "copy_of" in s:
        k = int(s.get("scale", 1))
        cells = [(k * x + dx, k * y + dy, k * z + dz) for x, y, z in structure_cells(s["copy_of"], structs, cache)
                 for dx in range(k) for dy in range(k) for dz in range(k)]
    elif "box" in s:
        nx, ny, nz = s["box"]
        cells = [(x, y, z) for z in range(nz) for y in range(ny) for x in range(nx)]
    elif "gnomon" in s:                                     # L-shaped row that turns a k x k square into (k+1) x (k+1)
        k = int(s["gnomon"])
        cells = [(x, y, 0) for x in range(k + 1) for y in range(k + 1) if max(x, y) == k]
    elif "pyramid" in s:                                    # e.g. [9, 7, 5, 3, 1]
        big = s["pyramid"][0]
        cells = [(x + (big - n) // 2, y + (big - n) // 2, z) for z, n in enumerate(s["pyramid"]) for y in range(n) for x in range(n)]
    else:
        raise ValueError("structure %s needs cells, copy_of, box or pyramid" % name)
    cache[name] = sorted(set(cells), key=lambda c: (c[2], c[1], c[0]))
    return cache[name]


def local_pos(cell, cells, centered=True):
    """Block centre relative to the structure's 'at' point: bbox-centred, or (centered=false) cell (0,0) at 'at'."""
    if not centered:
        return (cell[0] * B, cell[1] * B, (cell[2] + 0.5) * B)
    xs, ys = [c[0] for c in cells], [c[1] for c in cells]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    return ((cell[0] - cx) * B, (cell[1] - cy) * B, (cell[2] + 0.5) * B)


def extent(cells):
    xs, ys, zs = [c[0] for c in cells], [c[1] for c in cells], [c[2] for c in cells]
    return ((max(xs) - min(xs) + 1) * B, (max(ys) - min(ys) + 1) * B, (max(zs) + 1) * B)


# ---------------------------------------------------------------- plan
def make_plan(story=None):
    story = story or load_story()
    D = json.load(open(os.path.join(BUILD, "dialogue.json")))
    lines, film, nf = D["lines"], D["film"], D["nf"]
    T = Clock(lines, film)
    structs = story["structures"]
    cache = {}
    S = {}
    for name, s in structs.items():
        cells = structure_cells(name, structs, cache)
        parts = {pn: [c for c in cells if _part_fn(expr)(c)] for pn, expr in s.get("parts", {}).items()}
        S[name] = dict(name=name, cells=cells, at=list(s.get("at", [0, 0])), material=s.get("material", "sandstone"),
                       eyes=bool(s.get("eyes", False)), stand=s.get("stand", "right"), parts=parts,
                       size=extent(cells), expect=s.get("expect_blocks"), prebuilt=bool(s.get("prebuilt", False)),
                       centered=bool(s.get("centered", True)), block_shape=list(s.get("block_shape", [1, 1, 1])))
    acts = []
    for a in story.get("actions", []):
        a = dict(a)
        a["t0"] = T(a.get("when", 0.0))
        a["t1"] = T(a.get("until")) if a.get("until") is not None else a["t0"]
        if a.get("cool_from") is not None:
            a["c0"], a["c1"] = T(a["cool_from"]), T(a.get("cool_until", a["cool_from"]))
        acts.append(a)
    acts.sort(key=lambda a: a["t0"])

    # ---- blocks + build swings
    blocks, swings = [], []
    built = set()
    for a in acts:
        if a["type"] != "build":
            continue
        st = S[a["target"]]
        built.add(st["name"])
        order = a.get("order", "z")
        key = {"x": lambda c: c[0], "y": lambda c: c[1], "z": lambda c: c[2]}[order]
        groups = {}
        for c in st["cells"]:
            groups.setdefault(key(c), []).append(c)
        gk = sorted(groups)
        T_sw = (a["t1"] - a["t0"]) / max(1, len(gk))
        for gi, g in enumerate(gk):
            cs = sorted(groups[g], key=lambda c: (c[2], c[1], c[0]))
            s0 = a["t0"] + gi * T_sw
            sw = dict(s=s0, e=s0 + T_sw, target=st["name"], ids=[])
            for q, c in enumerate(cs):
                ta = s0 + T_sw * (0.2 + 0.6 * (q / max(1, len(cs) - 1)))
                b = dict(id=len(blocks), struct=st["name"], cell=list(c), pos=local_pos(c, st["cells"], st["centered"]),
                         appear=fr(ta), settle=fr(ta) + 2, prebuilt=False)
                blocks.append(b); sw["ids"].append(b["id"])
            swings.append(sw)
    for name, st in S.items():                                  # structures that are simply there from the start
        if name not in built:
            for c in st["cells"]:
                blocks.append(dict(id=len(blocks), struct=name, cell=list(c), pos=local_pos(c, st["cells"], st["centered"]),
                                   appear=0, settle=0, prebuilt=True))
    for a in acts:                                              # 'remove': take a built structure apart, top first
        if a["type"] != "remove":
            continue
        mine = [b for b in blocks if b["struct"] == a["target"]]
        groups = {}
        for b in mine:
            groups.setdefault(-b["cell"][2], []).append(b)
        gk = sorted(groups)
        T_sw = (a["t1"] - a["t0"]) / max(1, len(gk))
        for gi, g in enumerate(gk):
            s0 = a["t0"] + gi * T_sw
            sw = dict(s=s0, e=s0 + T_sw, target=a["target"], ids=[])
            for q, b in enumerate(groups[g]):
                b["vanish"] = fr(s0 + T_sw * (0.2 + 0.6 * (q / max(1, len(groups[g]) - 1))))
                sw["ids"].append(b["id"])
            if not a.get("quiet"):
                swings.append(sw)

    # ---- grandpa's walk: stand beside whatever he builds, walk where the story says
    def stand_point(st):
        w, d, _ = st["size"]
        x, y = st["at"]
        if not st["centered"]:                                   # 'at' is the corner: use the footprint centre
            xs, ys = [c[0] for c in st["cells"]], [c[1] for c in st["cells"]]
            x += (min(xs) + max(xs)) / 2 * B
            y += (min(ys) + max(ys)) / 2 * B
        return {"right": (x + w / 2 + 0.28, y + 0.18), "left": (x - w / 2 - 0.28, y + 0.18),
                "behind": (x + 0.12, y + d / 2 + 0.36), "front": (x + 0.15, y - d / 2 - 0.38)}[st["stand"]]

    xyz = lambda p: (p[0], p[1], p[2] if len(p) > 2 else 0.0)     # positions may give a height (scaffold, ladder)
    keys = [(0.0,) + xyz(story.get("grandpa", {}).get("start", [0.0, 0.5]))]
    for a in sorted(acts, key=lambda a: a["t0"]):
        if a["type"] in ("build", "remove") and not a.get("quiet"):     # quiet: happens on its own, grandpa stays put
            p = xyz(a.get("stand_at") or stand_point(S[a["target"]]))  # optional [x, y(, z)] override per build
            keys.append((a["t0"] - 0.05,) + p)
            keys.append((a["t1"],) + p)
        elif a["type"] == "move":
            last = keys[-1]
            keys.append((a["t0"],) + last[1:])
            for k, wp in enumerate(a.get("via", [])):                 # optional waypoints, evenly timed
                keys.append((a["t0"] + (a["t1"] - a["t0"]) * (k + 1) / (len(a["via"]) + 1),) + xyz(wp))
            keys.append((a["t1"],) + xyz(a["to"]))
    keys.sort(key=lambda k: k[0])
    keys.append((film + 1,) + keys[-1][1:])
    path = _root_path(keys, nf)
    phase, amp, steps = _gait(path, nf)

    melt = {}
    for a in acts:
        if a["type"] == "melt":
            melt[a["target"]] = melt_targets([b for b in blocks if b["struct"] == a["target"]], S[a["target"]]["cells"])
    world, outfit = world_settings(story)
    plan = dict(slug=story["slug"], title=story.get("title", story["slug"]), fps=FPS, nf=nf, film=film, block=B,
                world=world, outfit=outfit, melt=melt, loop=bool(story.get("loop", False)),
                w=W_PX if story.get("format", "vertical") == "vertical" else H_PX,
                h=H_PX if story.get("format", "vertical") == "vertical" else W_PX, voice=D["voice"], lines=lines, mouth=D["mouth"], channel_tag=story["channel_tag"],
                structures=S, blocks=blocks, swings=swings, actions=acts, path=path, phase=phase, amp=amp, steps=steps,
                camera=[dict(k, t=T(k["at"])) for k in story.get("camera", [])],
                cards=[_resolve_card(c, T) for c in story.get("cards", [])])
    return plan


def melt_targets(blocks, cells):
    """'melt': the structure's blocks settle into a flat slab over its footprint, bottom layer first.
    Lowest blocks take the lowest layer; within a layer each block takes the nearest free cell. -> {block id: cell}"""
    xs, ys = [c[0] for c in cells], [c[1] for c in cells]
    slab = [(x, y) for y in range(min(ys), max(ys) + 1) for x in range(min(xs), max(xs) + 1)]
    out, layer, free = {}, 0, list(slab)
    for b in sorted(blocks, key=lambda b: (b["cell"][2], b["cell"][0], b["cell"][1])):
        if not free:
            layer, free = layer + 1, list(slab)
        bx, by = b["cell"][0], b["cell"][1]
        k = min(range(len(free)), key=lambda i: (free[i][0] - bx) ** 2 + (free[i][1] - by) ** 2)
        x, y = free.pop(k)
        out[str(b["id"])] = [x, y, layer]
    return out


def _resolve_card(c, T):
    c = dict(c)
    c["t0"] = T(c.get("when", 0.0))
    c["t1"] = T(c.get("until", "end"))
    if "rows" in c:
        c["rows"] = [dict(r, t=T(r.get("at", c.get("when", 0.0)))) for r in c["rows"]]
    return c


def _root_path(keys, nf):
    def lin(t):
        if t <= keys[0][0]:
            return keys[0][1:]
        for k0, k1 in zip(keys, keys[1:]):
            if t <= k1[0]:
                u = (t - k0[0]) / max(1e-9, k1[0] - k0[0])
                return tuple(a + (b - a) * u for a, b in zip(k0[1:], k1[1:]))
        return keys[-1][1:]
    sub, sig = 4, 0.15 * FPS * 4
    raw = [lin((n / sub - 30) / FPS) for n in range((nf + 60) * sub)]
    rad = int(3 * sig)
    w = [math.exp(-0.5 * (q / sig) ** 2) for q in range(-rad, rad + 1)]
    out = []
    for f in range(nf):
        c = (f + 30) * sub
        sx = sy = sz = sw = 0.0
        for q, wq in zip(range(-rad, rad + 1), w):
            n = min(len(raw) - 1, max(0, c + q))
            sx += raw[n][0] * wq; sy += raw[n][1] * wq; sz += raw[n][2] * wq; sw += wq
        out.append((sx / sw, sy / sw, sz / sw))
    return out


def _gait(path, nf):
    phase, amp, steps = [], [], []
    ph = 0.0
    for f in range(nf):
        p0, p1 = path[max(0, f - 1)], path[f]
        ds = math.hypot(p1[0] - p0[0], p1[1] - p0[1])
        new = ph + ds / (2 * STRIDE)
        speed = ds * FPS
        a = cl(speed / 0.12)
        if a > 0.25:
            for foot, off in (("R", 0.5), ("L", 0.0)):
                if math.floor(new + off - 0.5) > math.floor(ph + off - 0.5):
                    steps.append(dict(frame=f, foot=foot, speed=round(speed, 3)))
        ph = new
        phase.append(ph); amp.append(a)
    sm = [sum(amp[max(0, f - 5):f + 6]) / len(amp[max(0, f - 5):f + 6]) for f in range(nf)]
    return phase, sm, steps


def save_plan(plan):
    os.makedirs(BUILD, exist_ok=True)
    json.dump(plan, open(os.path.join(BUILD, "plan.json"), "w"))


def load_plan():
    return json.load(open(os.path.join(BUILD, "plan.json")))


if __name__ == "__main__":
    st = load_story()
    p = make_plan(st)
    save_plan(p)
    print("plan: %s  %.2f s  %d frames" % (p["slug"], p["film"], p["nf"]))
    for n, s in p["structures"].items():
        n_b = sum(1 for b in p["blocks"] if b["struct"] == n)
        print("  %-8s %3d blocks  parts %s" % (n, n_b, {k: len(v) for k, v in s["parts"].items()}))
    print("  actions:", ", ".join("%s@%.2f" % (a["type"], a["t0"]) for a in p["actions"]))

In [ ]:
%%writefile bl_world.py
"""Block Lab: the world around the lesson -- ground style, sky / time of day, weather and a few block animals.

Story:  "world": {"ground": "grass", "time": "sunset", "weather": "clouds", "animals": ["birds", "cat"]}
Everything is voxels. Every loop (rain, snow, birds, wings, cat tail) is cyclic F-curve animation, so any frame
renders the same whatever order the frames are rendered in (resumable renders stay seamless).
"""
import math
import random

import bpy
import bmesh
from mathutils import Vector

import voxelkit as VK

rnd = random.Random(23)
V, R0 = 0.2, 2.7                                   # island voxel size and radius (m)
PROP_SPOTS = ((-2.0, 0.9), (1.9, -1.2), (-1.6, -1.7))

GROUND = {   # top, side and base texel colours (dark, light), plus the props that dress the island
    "desert": (((0.78, 0.60, 0.36), (0.95, 0.82, 0.56)), ((0.62, 0.45, 0.26), (0.80, 0.64, 0.40)),
               ((0.42, 0.30, 0.20), (0.58, 0.44, 0.30)), "cacti"),
    "grass": (((0.22, 0.45, 0.15), (0.40, 0.66, 0.26)), ((0.45, 0.31, 0.18), (0.60, 0.43, 0.26)),
              ((0.40, 0.40, 0.42), (0.55, 0.55, 0.57)), "trees"),
    "snow": (((0.80, 0.85, 0.92), (0.97, 0.98, 1.00)), ((0.50, 0.40, 0.32), (0.66, 0.56, 0.46)),
             ((0.40, 0.40, 0.44), (0.56, 0.56, 0.60)), "pines"),
    "beach": (((0.86, 0.74, 0.50), (0.98, 0.90, 0.68)), ((0.74, 0.62, 0.40), (0.88, 0.78, 0.56)),
              ((0.50, 0.42, 0.30), (0.66, 0.56, 0.42)), "palms"),
    "stone": (((0.46, 0.46, 0.48), (0.66, 0.66, 0.68)), ((0.36, 0.36, 0.38), (0.52, 0.52, 0.54)),
              ((0.30, 0.28, 0.26), (0.44, 0.42, 0.40)), "lamps"),
    "paris": (((0.55, 0.48, 0.37), (0.70, 0.63, 0.50)), ((0.45, 0.40, 0.33), (0.58, 0.53, 0.45)),
              ((0.40, 0.38, 0.35), (0.55, 0.53, 0.50)), "paris"),          # Champ de Mars gravel in a cobbled square
}
TIME = {     # light travel direction, sun energy / colour, sky horizon -> top, sky strength, exposure
    "day": dict(sun=(0.55, 0.62, -0.62), energy=4.2, color=(1.0, 0.90, 0.76), horizon=(0.96, 0.78, 0.56),
                top=(0.36, 0.60, 0.92), strength=0.8, exposure=-0.35),
    "morning": dict(sun=(-0.70, 0.55, -0.45), energy=3.4, color=(1.0, 0.86, 0.72), horizon=(0.98, 0.84, 0.70),
                    top=(0.48, 0.68, 0.95), strength=0.8, exposure=-0.3),
    "sunset": dict(sun=(0.75, 0.55, -0.22), energy=3.2, color=(1.0, 0.60, 0.36), horizon=(1.0, 0.56, 0.32),
                   top=(0.32, 0.32, 0.58), strength=0.75, exposure=-0.2),
    "night": dict(sun=(0.35, 0.55, -0.75), energy=0.45, color=(0.55, 0.65, 1.0), horizon=(0.07, 0.09, 0.18),
                  top=(0.01, 0.015, 0.04), strength=0.8, exposure=0.25),
}
OVERCAST = {"rain": (0.35, (0.50, 0.53, 0.58), 0.6), "snow": (0.6, (0.80, 0.83, 0.88), 0.5)}   # sun x, grey, mix


def _mix(a, b, k):
    return tuple(x + (y - x) * k for x, y in zip(a, b))


def _loop(o, path, index, v0, v1, frames):
    """Key o.<path>[index] from v0 (frame 0) to v1 (frame 'frames'), linear, repeating forever."""
    vec = getattr(o, path)
    vec[index] = v0; o.keyframe_insert(path, index=index, frame=0)
    vec[index] = v1; o.keyframe_insert(path, index=index, frame=frames)
    for fc in VK.fcurves_of(o):
        if fc.data_path == path and fc.array_index == index:
            for kp in fc.keyframe_points:
                kp.interpolation = "LINEAR"
            fc.modifiers.new("CYCLES")


def _wave(o, path, index, amp, period_frames, phase=0.0):
    """Add a sine (amp * sin) on top of o.<path>[index]; needs no keys of its own."""
    o.keyframe_insert(path, index=index, frame=0)
    for fc in VK.fcurves_of(o):
        if fc.data_path == path and fc.array_index == index:
            m = fc.modifiers.new("FNGENERATOR")
            m.function_type, m.use_additive = "SIN", True
            m.amplitude, m.phase_multiplier, m.phase_offset = amp, 2 * math.pi / period_frames, phase


def _no_shadow(o):
    o.visible_shadow = False
    return o


# ------------------------------------------------------------------ ground + props
def build_ground(sc, ground):
    top, side, base, props = GROUND[ground]
    mats = [VK.pixel_material("Ground", *top, V / 16, voxel=V), VK.pixel_material("GroundSide", *side, V / 16, voxel=V),
            VK.pixel_material("Bedrock", *base, V / 16, voxel=V)]
    bm = bmesh.new()
    for z0, r, mi in ((-0.2, R0, 0), (-0.4, R0 - 0.2, 1), (-0.6, R0 - 0.6, 2), (-0.8, R0 - 1.2, 2)):
        n = int(r / V) + 1
        for ix in range(-n, n):
            for iy in range(-n, n):
                cx, cy = (ix + 0.5) * V, (iy + 0.5) * V
                if math.hypot(cx, cy) <= r:
                    VK.add_box(bm, (cx, cy, z0 + V / 2), (V, V, V), mi)
    bmesh.ops.remove_doubles(bm, verts=bm.verts, dist=1e-5)
    me = bpy.data.meshes.new("Island")
    bm.to_mesh(me); bm.free()
    for m in mats:
        me.materials.append(m)
    sc.collection.objects.link(bpy.data.objects.new("Island", me))
    globals()["_props_" + props](sc)
    if ground == "beach":
        sea = VK.mat("Sea", (0.04, 0.30, 0.42), rough=0.08)
        VK.mesh_obj("Sea", [((0, 0, -0.13), (80, 80, 0.02), 0)], [sea], coll=sc.collection)


def _props_cacti(sc):
    cact = VK.pixel_material("Cactus", (0.16, 0.36, 0.14), (0.30, 0.55, 0.22), 0.1 / 8)
    boxes = []
    for (cx, cy), h in zip(PROP_SPOTS, (0.35, 0.28, 0.22)):
        boxes += [((cx, cy, h / 2), (0.1, 0.1, h), 0), ((cx + 0.08, cy, h * 0.55), (0.08, 0.06, 0.06), 0),
                  ((cx + 0.1, cy, h * 0.72), (0.05, 0.05, h * 0.3), 0)]
    VK.mesh_obj("Cacti", boxes, [cact], coll=sc.collection)


def _props_trees(sc):
    wood = VK.pixel_material("Trunk", (0.30, 0.19, 0.10), (0.46, 0.31, 0.17), 0.1 / 8)
    leaf = VK.pixel_material("Leaves", (0.12, 0.34, 0.10), (0.30, 0.56, 0.20), 0.1 / 8)
    petal = VK.mat("Petals", (0.95, 0.85, 0.20), rough=0.7)
    petal2 = VK.mat("Petals2", (0.90, 0.25, 0.30), rough=0.7)
    boxes = []
    for (cx, cy), h in zip(PROP_SPOTS, (0.45, 0.38, 0.32)):
        boxes += [((cx, cy, h / 2), (0.1, 0.1, h), 0), ((cx, cy, h + 0.08), (0.36, 0.36, 0.2), 1),
                  ((cx, cy, h + 0.24), (0.22, 0.22, 0.14), 1)]
    for k in range(26):
        a, r = rnd.uniform(0, 2 * math.pi), rnd.uniform(1.7, 2.5)
        boxes.append(((r * math.cos(a), r * math.sin(a), 0.02), (0.04, 0.04, 0.04), 2 + k % 2))
    VK.mesh_obj("Trees", boxes, [wood, leaf, petal, petal2], coll=sc.collection)


def _props_pines(sc):
    wood = VK.pixel_material("Trunk", (0.30, 0.19, 0.10), (0.46, 0.31, 0.17), 0.1 / 8)
    pine = VK.pixel_material("Pine", (0.06, 0.24, 0.12), (0.14, 0.38, 0.20), 0.1 / 8)
    snow = VK.mat("SnowCap", (0.95, 0.97, 1.0), rough=0.9)
    boxes = []
    for (cx, cy), s in zip(PROP_SPOTS, (1.0, 0.85, 0.7)):
        boxes.append(((cx, cy, 0.06 * s), (0.08, 0.08, 0.12 * s), 0))
        z = 0.12 * s
        for w in (0.40, 0.30, 0.20, 0.10):
            boxes += [((cx, cy, z + 0.05 * s), (w * s, w * s, 0.1 * s), 1), ((cx, cy, z + 0.1 * s + 0.006), (w * s * 0.8, w * s * 0.8, 0.012), 2)]
            z += 0.1 * s
    VK.mesh_obj("Pines", boxes, [wood, pine, snow], coll=sc.collection)


def _props_palms(sc):
    trunk = VK.pixel_material("PalmTrunk", (0.45, 0.33, 0.20), (0.62, 0.48, 0.30), 0.1 / 8)
    frond = VK.pixel_material("Frond", (0.14, 0.40, 0.12), (0.28, 0.60, 0.22), 0.1 / 8)
    nut = VK.mat("Coconut", (0.30, 0.20, 0.10), rough=0.8)
    boxes = []
    for (cx, cy), h in zip(PROP_SPOTS, (0.7, 0.6, 0.5)):
        n = int(h / 0.1)
        for k in range(n):
            boxes.append(((cx + 0.012 * k * k / n, cy, 0.05 + 0.1 * k), (0.08, 0.08, 0.1), 0))
        tx, tz = cx + 0.012 * n, 0.1 * n + 0.02
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            boxes += [((tx + 0.13 * dx, cy + 0.13 * dy, tz), (0.18 if dx else 0.07, 0.07 if dx else 0.18, 0.03), 1),
                      ((tx + 0.26 * dx, cy + 0.26 * dy, tz - 0.05), (0.1 if dx else 0.06, 0.06 if dx else 0.1, 0.03), 1)]
        boxes += [((tx + 0.04, cy - 0.04, tz - 0.06), (0.05, 0.05, 0.05), 2), ((tx - 0.04, cy + 0.03, tz - 0.06), (0.05, 0.05, 0.05), 2)]
    VK.mesh_obj("Palms", boxes, [trunk, frond, nut], coll=sc.collection)


def _props_paris(sc):
    """A cobbled square and a street of block Paris apartment buildings: cream stone, blue-grey mansard roofs,
    black balconies, chimney pots, a flag. Plus lamps, plane trees and a cafe table. Kept low and far back so the
    tower (in the story) stays the giant of the skyline."""
    cob = VK.pixel_material("Cobbles", (0.27, 0.27, 0.29), (0.43, 0.42, 0.41), 0.15, voxel=0.15)
    VK.mesh_obj("Square", [((0, 0, -0.015), (60, 60, 0.02), 0)], [cob], coll=sc.collection)
    stone = VK.pixel_material("Facade", (0.80, 0.74, 0.62), (0.93, 0.88, 0.77), 0.05)
    win, iron = VK.mat("Window", (0.10, 0.12, 0.16), 0.2), VK.mat("Balcony", (0.06, 0.06, 0.07), 0.5, 0.4)
    roof = VK.pixel_material("Zinc", (0.32, 0.36, 0.42), (0.46, 0.50, 0.57), 0.05)
    pot, door = VK.mat("ChimneyPot", (0.62, 0.32, 0.22), 0.8), VK.mat("Door", (0.18, 0.22, 0.30), 0.6)
    boxes = []
    x = -11.0
    while x < 11.0:
        w, floors = rnd.uniform(2.2, 3.4), rnd.choice((5, 5, 6))
        fh, y0, d = 0.36, rnd.uniform(9.0, 9.6), 1.6
        h = floors * fh
        cx = x + w / 2
        boxes.append(((cx, y0, h / 2), (w, d, h), 0))                                   # stone body
        fy = y0 - d / 2 - 0.006
        for f in range(floors):
            for k in range(int(w / 0.42)):
                wx = x + 0.25 + k * 0.42
                if wx > x + w - 0.2:
                    break
                if f == 0 and k == int(w / 0.42) // 2:
                    boxes.append(((wx, fy, 0.16), (0.2, 0.012, 0.3), 5))                # door
                else:
                    boxes.append(((wx, fy, f * fh + 0.2), (0.16, 0.012, 0.22), 1))     # window
            if f in (1, floors - 1):
                boxes.append(((cx, fy - 0.04, f * fh + 0.08), (w - 0.1, 0.08, 0.02), 2))   # balcony
                boxes.append(((cx, fy - 0.08, f * fh + 0.13), (w - 0.1, 0.01, 0.1), 2))
        boxes += [((cx, y0, h + 0.12), (w, d - 0.3, 0.24), 3), ((cx, y0, h + 0.34), (w - 0.2, d - 0.8, 0.2), 3)]   # mansard
        for k in range(2):
            boxes.append(((x + 0.4 + k * (w - 0.8), y0 + 0.2, h + 0.55), (0.18, 0.14, 0.22), 4))
        x += w + rnd.uniform(0.0, 0.6)
    VK.mesh_obj("ParisStreet", boxes, [stone, win, iron, roof, pot, door], coll=sc.collection)
    blue, white, red = VK.mat("FlagBlue", (0.0, 0.14, 0.52), 0.7), VK.mat("FlagWhite", (0.95, 0.95, 0.95), 0.7), VK.mat("FlagRed", (0.85, 0.08, 0.12), 0.7)
    VK.mesh_obj("Flag", [((2.0, 9.2, 2.6), (0.03, 0.03, 1.0), 3), ((2.17, 9.2, 2.95), (0.3, 0.01, 0.4), 0),
                         ((2.47, 9.2, 2.95), (0.3, 0.01, 0.4), 1), ((2.77, 9.2, 2.95), (0.3, 0.01, 0.4), 2)],
                [blue, white, red, iron], coll=sc.collection)
    # lamps and plane trees along the square, a cafe table with a striped umbrella on the island's edge
    glow = VK.emission_material("LampGlow", (1.0, 0.82, 0.55), 4.0)
    trunk = VK.pixel_material("PlaneTrunk", (0.42, 0.40, 0.33), (0.62, 0.60, 0.50), 0.05)
    leaf = VK.pixel_material("PlaneLeaves", (0.18, 0.38, 0.14), (0.34, 0.56, 0.24), 0.08)
    cloth_r, cloth_w = VK.mat("Awning", (0.80, 0.10, 0.12), 0.8), VK.mat("AwningW", (0.95, 0.94, 0.90), 0.8)
    st = []
    for lx in (-6.0, -3.2, 3.2, 6.0):
        st += [((lx, 4.2, 0.45), (0.05, 0.05, 0.9), 0), ((lx, 4.2, 0.95), (0.11, 0.11, 0.12), 1), ((lx, 4.2, 1.03), (0.14, 0.14, 0.03), 0)]
    for tx in (-7.5, -4.6, 4.6, 7.5):
        st += [((tx, 6.0, 0.4), (0.12, 0.12, 0.8), 2), ((tx, 6.0, 1.05), (0.9, 0.9, 0.55), 3), ((tx, 6.0, 1.4), (0.6, 0.6, 0.3), 3)]
    cx, cy = -2.1, 1.3
    st += [((cx, cy, 0.25), (0.04, 0.04, 0.5), 0), ((cx, cy, 0.5), (0.3, 0.3, 0.03), 0), ((cx, cy, 0.95), (0.03, 0.03, 0.9), 0)]
    for k in range(6):
        st.append(((cx - 0.375 + k * 0.15, cy, 1.38), (0.15, 0.75, 0.05), 4 + k % 2))
    for dx in (-0.32, 0.32):
        st += [((cx + dx, cy, 0.2), (0.16, 0.16, 0.03), 0), ((cx + dx, cy, 0.1), (0.03, 0.03, 0.2), 0), ((cx + dx * 1.3, cy, 0.32), (0.03, 0.16, 0.22), 0)]
    VK.mesh_obj("ParisSquare", st, [iron, glow, trunk, leaf, cloth_r, cloth_w], coll=sc.collection)


def _props_lamps(sc):
    iron = VK.mat("Iron", (0.08, 0.08, 0.09), rough=0.5, metal=0.6)
    glow = VK.emission_material("LampGlow", (1.0, 0.82, 0.55), 4.0)
    boxes = []
    for cx, cy in PROP_SPOTS:
        boxes += [((cx, cy, 0.30), (0.04, 0.04, 0.6), 0), ((cx, cy, 0.02), (0.1, 0.1, 0.04), 0),
                  ((cx, cy, 0.66), (0.09, 0.09, 0.1), 1), ((cx, cy, 0.725), (0.11, 0.11, 0.02), 0)]
    VK.mesh_obj("Lamps", boxes, [iron, glow], coll=sc.collection)


# ------------------------------------------------------------------ sky + light
def build_sky(sc, time, weather):
    T = dict(TIME[time])
    if weather in OVERCAST:
        k_sun, grey, k = OVERCAST[weather]
        if time == "night":
            grey = tuple(g * 0.15 for g in grey)
        T["energy"] *= k_sun
        T["horizon"], T["top"] = _mix(T["horizon"], grey, k), _mix(T["top"], grey, k)
    sun_d = bpy.data.lights.new("Sun", "SUN")
    sun_d.energy, sun_d.angle, sun_d.color = T["energy"], math.radians(4.0), T["color"]
    sun = bpy.data.objects.new("Sun", sun_d)
    sun.rotation_mode = "QUATERNION"
    sun.rotation_quaternion = Vector(T["sun"]).to_track_quat("-Z", "Y")
    sc.collection.objects.link(sun)
    w = bpy.data.worlds.new("Sky")
    sc.world = w
    w.use_nodes = True
    N, Lk = w.node_tree.nodes, w.node_tree.links
    tc = N.new("ShaderNodeTexCoord")
    sep = N.new("ShaderNodeSeparateXYZ"); Lk.new(tc.outputs["Generated"], sep.inputs[0])
    ramp = N.new("ShaderNodeValToRGB")
    ramp.color_ramp.elements[0].position, ramp.color_ramp.elements[0].color = 0.0, (*T["horizon"], 1)
    ramp.color_ramp.elements[1].position, ramp.color_ramp.elements[1].color = 0.55, (*T["top"], 1)
    Lk.new(sep.outputs["Z"], ramp.inputs["Fac"])
    Lk.new(ramp.outputs["Color"], N["Background"].inputs["Color"])
    N["Background"].inputs["Strength"].default_value = T["strength"]
    if time == "night":
        lamp = bpy.data.lights.new("Lantern", "POINT")              # warm fill so grandpa reads at night
        lamp.energy, lamp.color, lamp.shadow_soft_size = 75, (1.0, 0.76, 0.48), 0.4
        o = bpy.data.objects.new("Lantern", lamp); o.location = (0.0, -0.45, 1.9)
        sc.collection.objects.link(o)
        if weather not in OVERCAST:
            star = VK.emission_material("Stars", (1.0, 0.97, 0.9), 6.0)
            boxes = []
            for _ in range(420):
                a, e = rnd.uniform(0, 2 * math.pi), math.asin(rnd.uniform(0.12, 1.0))
                d = Vector((math.cos(e) * math.cos(a), math.cos(e) * math.sin(a), math.sin(e))) * 45
                s = rnd.choice((0.08, 0.1, 0.14))
                boxes.append((tuple(d), (s, s, s), 0))
            _no_shadow(VK.mesh_obj("Stars", boxes, [star], coll=sc.collection))
            moon = VK.emission_material("Moon", (0.92, 0.94, 1.0), 3.0)
            _no_shadow(VK.mesh_obj("Moon", [((9, 38, 15), (2.6, 0.2, 2.6), 0), ((9, 38.1, 15), (3.2, 0.2, 1.8), 0),
                                            ((9, 38.1, 15), (1.8, 0.2, 3.2), 0)], [moon], coll=sc.collection))
    return T["exposure"]


# ------------------------------------------------------------------ weather
def build_clouds(sc, nf, dark):
    m = VK.mat("Cloud", (0.55, 0.57, 0.60) if dark else (0.97, 0.97, 0.98), rough=1.0)
    for k in range(8):
        a = rnd.uniform(0.15 * math.pi, 0.85 * math.pi)             # mostly behind the island, where the camera looks
        r = rnd.uniform(6, 12)
        c = Vector((r * math.cos(a), r * math.sin(a), rnd.uniform(3.5, 6.5)))
        boxes = []
        for _ in range(rnd.randint(4, 7)):
            s = Vector((rnd.uniform(0.5, 1.1), rnd.uniform(0.4, 0.8), rnd.uniform(0.25, 0.45)))
            boxes.append(((rnd.uniform(-0.8, 0.8), rnd.uniform(-0.3, 0.3), rnd.uniform(-0.1, 0.15)), tuple(s), 0))
        o = _no_shadow(VK.mesh_obj("Cloud_%d" % k, boxes, [m], coll=sc.collection))
        o.location = c
        _loop(o, "location", 0, c.x, c.x + 0.25, 30 * 30)              # slow drift, 0.25 m every 30 s


def build_precip(sc, kind, fps):
    """Rain streaks or snowflakes: one mesh, tiled twice in height, sliding down one tile per cycle forever."""
    H = 6.0
    if kind == "rain":
        n, size, speed = 1300, (0.008, 0.008, 0.16), 6.0
        m = VK.mat("Raindrop", (0.72, 0.78, 0.88), rough=0.1)
        m.node_tree.nodes["Principled BSDF"].inputs["Alpha"].default_value = 0.4
    else:
        n, size, speed = 900, (0.025, 0.025, 0.025), 0.7
        m = VK.mat("Snowflake", (0.97, 0.98, 1.0), rough=1.0)
    boxes = []
    for _ in range(n):
        x, y, z = rnd.uniform(-4, 4), rnd.uniform(-1.2, 5), rnd.uniform(0, H)     # none right at the lens
        boxes += [((x, y, z), size, 0), ((x, y, z + H), size, 0)]
    o = _no_shadow(VK.mesh_obj(kind.capitalize(), boxes, [m], coll=sc.collection))
    _loop(o, "location", 2, 0.0, -H, round(H / speed * fps))
    if kind == "snow":
        _wave(o, "location", 0, 0.08, 3.0 * fps)


# ------------------------------------------------------------------ animals
def build_birds(sc, fps, gulls, pigeons=False):
    body_c = (0.92, 0.92, 0.90) if gulls else (0.52, 0.54, 0.60) if pigeons else (0.22, 0.19, 0.17)
    wing_c = (0.55, 0.57, 0.60) if gulls else (0.38, 0.40, 0.46) if pigeons else (0.15, 0.13, 0.12)
    mb, mw, mk = VK.mat("BirdBody", body_c, 0.9), VK.mat("BirdWing", wing_c, 0.9), VK.mat("Beak", (0.95, 0.65, 0.10), 0.6)
    for k in range(4 if gulls else 5):
        piv = bpy.data.objects.new("BirdPivot_%d" % k, None)
        sc.collection.objects.link(piv)
        piv.location = (rnd.uniform(-1.5, 1.5), rnd.uniform(1.5, 4.0), rnd.uniform(2.6, 4.2))
        r, period, ccw = rnd.uniform(2.5, 4.5), rnd.uniform(9, 14), k % 2 == 0
        body = VK.mesh_obj("Bird_%d" % k, [((0, 0, 0), (0.05, 0.14, 0.045), 0), ((0, 0.08, 0.015), (0.045, 0.045, 0.04), 0),
                                           ((0, 0.11, 0.01), (0.015, 0.03, 0.012), 2), ((0, -0.08, 0.0), (0.04, 0.04, 0.012), 1)],
                           [mb, mw, mk], parent=piv, coll=sc.collection)
        body.location = (r, 0, 0)
        body.rotation_euler = (0, 0, 0 if ccw else math.pi)            # face along the circle
        for s in (-1, 1):
            wing = VK.mesh_obj("Wing_%d_%d" % (k, s), [((0.065 * s, 0, 0), (0.11, 0.07, 0.01), 0)], [mw], parent=body, coll=sc.collection)
            wing.location = (0.022 * s, 0, 0.012)
            _wave(wing, "rotation_euler", 1, 0.6 * s, fps / 3.2, phase=k)
        _loop(piv, "rotation_euler", 2, 0.0, 2 * math.pi * (1 if ccw else -1), round(period * fps))
        _wave(piv, "location", 2, 0.12, 4.0 * fps, phase=k * 1.3)


def build_cat(sc, fps, spot):
    fur, white = VK.mat("CatFur", (0.86, 0.50, 0.20), 0.95), VK.mat("CatWhite", (0.95, 0.93, 0.90), 0.95)
    eye, nose = VK.mat("CatEye", (0.30, 0.75, 0.25), 0.3), VK.mat("CatNose", (0.95, 0.55, 0.60), 0.6)
    root = bpy.data.objects.new("Cat", None)
    sc.collection.objects.link(root)
    root.location = (spot[0], spot[1], 0)
    root.rotation_euler = (0, 0, math.atan2(-spot[0], spot[1]) * 0.5)  # turned partly toward the middle of the island
    VK.mesh_obj("CatBody", [((0, 0, 0.06), (0.14, 0.18, 0.12), 0), ((0, -0.05, 0.16), (0.12, 0.10, 0.10), 0),
                            ((0, -0.101, 0.14), (0.07, 0.004, 0.08), 1), ((-0.04, -0.09, 0.015), (0.04, 0.05, 0.03), 1),
                            ((0.04, -0.09, 0.015), (0.04, 0.05, 0.03), 1)], [fur, white], parent=root, coll=sc.collection)
    head = VK.mesh_obj("CatHead", [((0, 0, 0), (0.12, 0.10, 0.10), 0), ((-0.035, 0, 0.065), (0.03, 0.02, 0.035), 0),
                                   ((0.035, 0, 0.065), (0.03, 0.02, 0.035), 0), ((-0.03, -0.051, 0.012), (0.022, 0.004, 0.02), 2),
                                   ((0.03, -0.051, 0.012), (0.022, 0.004, 0.02), 2), ((0, -0.052, -0.012), (0.016, 0.004, 0.012), 3),
                                   ((0, -0.051, -0.03), (0.05, 0.004, 0.03), 1)], [fur, white, eye, nose], parent=root, coll=sc.collection)
    head.location = (0, -0.06, 0.26)
    _wave(head, "rotation_euler", 2, 0.35, 7.0 * fps)                  # looks around slowly
    tail = VK.mesh_obj("CatTail", [((0, 0.06, 0.0), (0.03, 0.12, 0.03), 0), ((0, 0.11, 0.06), (0.03, 0.03, 0.1), 0)],
                       [fur], parent=root, coll=sc.collection)
    tail.location = (0, 0.08, 0.03)
    _wave(tail, "rotation_euler", 2, 0.5, 2.2 * fps)                   # tail swish


def cat_spot(P):
    """The spot on the island farthest from every block and from grandpa's path."""
    pts = [Vector(p[:2]) for p in P["path"][::10]]
    for b in P["blocks"]:
        st = P["structures"][b["struct"]]
        pts.append(Vector((st["at"][0] + b["pos"][0], st["at"][1] + b["pos"][1])))
    best, cand = None, []
    for k in range(24):
        a = 2 * math.pi * k / 24
        for r in (1.6, 1.9, 2.2):
            c = Vector((r * math.cos(a), r * math.sin(a)))
            if min((c - Vector(s)).length for s in PROP_SPOTS) < 0.4:
                continue
            score = min((c - p).length for p in pts) + (0.3 if c.y > 0 else 0)   # prefer the back half (in view)
            cand.append((score, tuple(c)))
    return max(cand)[1]


def build_world(sc, P):
    """Ground, props, sky, light, weather and animals for P['world']. Returns the view exposure to use."""
    W = P["world"]
    fps = P["fps"]
    build_ground(sc, W["ground"])
    exposure = build_sky(sc, W["time"], W["weather"])
    if W["weather"] in ("clouds", "rain", "snow"):
        build_clouds(sc, P["nf"], dark=W["weather"] == "rain")
    if W["weather"] in ("rain", "snow"):
        build_precip(sc, W["weather"], fps)
    for a in W["animals"]:
        if a in ("birds", "seagulls", "pigeons"):
            build_birds(sc, fps, gulls=a == "seagulls", pigeons=a == "pigeons")
        elif a == "cat":
            build_cat(sc, fps, cat_spot(P))
    return exposure

In [ ]:
%%writefile bl_voice.py
"""Block Lab: grandpa's lines -> build/audio/voice.wav + build/dialogue.json (timings, mouth curve), then the plan.

    python bl_voice.py --story stories/giants.json
(KOKORO_MODEL / KOKORO_VOICES env vars point at the Kokoro model files.)
"""
import json
import math
import os
import shutil

import numpy as np
import scipy.io.wavfile as wav
from scipy.signal import butter, resample_poly, sosfilt

import bl_common as C

SR = 48000
MODEL = os.environ.get("KOKORO_MODEL", "/home/user/tts/kokoro-q8-timed.onnx")
VOICES = os.environ.get("KOKORO_VOICES", "/home/user/tts/voices.bin")


def main():
    path = C.story_path()
    story = C.load_story(path)
    os.makedirs(os.path.join(C.BUILD, "audio"), exist_ok=True)
    if os.path.abspath(path) != os.path.join(C.BUILD, "story.json"):
        shutil.copy(path, os.path.join(C.BUILD, "story.json"))
    from kokoro_onnx import Kokoro
    k = Kokoro(MODEL, VOICES)
    clips = []
    for ln in story["lines"]:
        a, sr = k.create(ln["text"], voice=ln.get("voice", story["voice"]), speed=ln.get("speed", story["speed"]))[:2]
        a = resample_poly(np.asarray(a, dtype=np.float64), SR // 1000, sr // 1000)
        env = np.convolve(np.abs(a), np.ones(240) / 240, mode="same")
        idx = np.where(env > 0.01 * env.max())[0]
        clips.append(a[max(0, idx[0] - 480): idx[-1] + 960])
    meta, t = [], story["start"]
    for ln, a in zip(story["lines"], clips):
        meta.append(dict(id=ln["id"], text=ln["text"], start=round(t, 3), end=round(t + len(a) / SR, 3)))
        t += len(a) / SR + ln.get("pause", 0.45)
    film = math.ceil((meta[-1]["end"] + story["tail"]) * C.FPS) / C.FPS
    nf = int(round(film * C.FPS))
    N = int(film * SR)
    track = np.zeros(N)
    for m, a in zip(meta, clips):
        j = int(round(m["start"] * SR))
        track[j:j + len(a)] += a[:N - j]
    track = sosfilt(butter(2, 90, "highpass", fs=SR, output="sos"), track)
    track = sosfilt(butter(2, 9000, "lowpass", fs=SR, output="sos"), track)
    track *= 0.7 / np.abs(track).max()
    spf = SR // C.FPS
    rms = np.array([np.sqrt(np.mean(track[f * spf:(f + 1) * spf] ** 2)) for f in range(nf)])
    mouth = np.clip((rms / (rms.max() + 1e-9) - 0.06) / 0.55, 0, 1) ** 0.8
    mouth = np.convolve(mouth, [0.25, 0.5, 0.25], mode="same")
    wav.write(os.path.join(C.BUILD, "audio", "voice.wav"), SR, (track * 32767).astype(np.int16))
    json.dump(dict(voice=story["voice"], film=film, nf=nf, lines=meta, mouth=[round(float(m), 3) for m in mouth]),
              open(os.path.join(C.BUILD, "dialogue.json"), "w"), indent=1)
    for m in meta:
        print("%6.2f-%6.2f  %s" % (m["start"], m["end"], m["text"]))
    plan = C.make_plan(story)
    C.save_plan(plan)
    print("film %.2f s (%d frames); plan written" % (film, nf))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bl_scene.py
"""Block Lab: build the Blender scene for the active story (build/plan.json) and key every frame.

    blender -b -P bl_scene.py [-- --preview 60 300 ...]     |   python3 bl_scene.py  (bpy module)
Writes build/blocklab.blend and build/anchors.json.
"""
import json
import math
import os
import random
import sys

import bpy
import bmesh
from bpy_extras.object_utils import world_to_camera_view
from mathutils import Euler, Matrix, Vector

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bl_common as C
import bl_world as BW
import voxelkit as VK
from voxelkit import aim, cl, rx, rz, smooth, smoother, two_bone

P = C.load_plan()
FPS, NF, B = P["fps"], P["nf"], P["block"]
W_PX, H_PX = P["w"], P["h"]
HIDE = Vector((0, 0, -6))
JAW_DROP = 0.04
rnd = random.Random(11)
PALETTE = {                                    # (dark, light) texel colours
    "sandstone": ((0.66, 0.50, 0.29), (0.90, 0.76, 0.50)),
    "stone": ((0.36, 0.37, 0.39), (0.62, 0.63, 0.65)),
    "wood": ((0.36, 0.22, 0.11), (0.58, 0.40, 0.22)),
    "brick": ((0.46, 0.14, 0.09), (0.72, 0.30, 0.20)),
    "gold": ((0.70, 0.50, 0.08), (1.00, 0.84, 0.30)),
    "grass": ((0.20, 0.42, 0.14), (0.40, 0.66, 0.26)),
    "blue": ((0.12, 0.28, 0.62), (0.30, 0.52, 0.90)),
    "iron": ((0.27, 0.20, 0.14), (0.47, 0.37, 0.27)),          # Eiffel Tower brown
    "rubber": ((0.05, 0.05, 0.06), (0.15, 0.15, 0.17)),        # tyres
    "red": ((0.55, 0.04, 0.05), (0.86, 0.13, 0.12)),           # car paint
    "steel": ((0.52, 0.55, 0.60), (0.82, 0.85, 0.90)),         # keys, springs, tools
}


def acts(kind):
    return [a for a in P["actions"] if a["type"] == kind]


# ============================================================ scene pieces
def block_material(name, kind, crack=False):
    d, l = PALETTE.get(kind, PALETTE["sandstone"])
    m = VK.pixel_material(name, d, l, B / 16, per_object=True)
    if not crack:
        return m, None
    N, Lk = m.node_tree.nodes, m.node_tree.links
    p = N["Principled BSDF"]
    tc = N.new("ShaderNodeTexCoord")
    vor = N.new("ShaderNodeTexVoronoi"); vor.feature = "DISTANCE_TO_EDGE"; vor.inputs["Scale"].default_value = 28
    Lk.new(tc.outputs["Object"], vor.inputs["Vector"])
    edge = N.new("ShaderNodeMath"); edge.operation = "LESS_THAN"; edge.inputs[1].default_value = 0.035
    Lk.new(vor.outputs["Distance"], edge.inputs[0])
    amount = N.new("ShaderNodeValue"); amount.name = "Crack"
    mul = N.new("ShaderNodeMath"); mul.operation = "MULTIPLY"
    Lk.new(edge.outputs[0], mul.inputs[0]); Lk.new(amount.outputs[0], mul.inputs[1])
    s = N.new("ShaderNodeMath"); s.operation = "MULTIPLY"; s.inputs[1].default_value = 12.0
    Lk.new(mul.outputs[0], s.inputs[0])
    p.inputs["Emission Color"].default_value = (1.0, 0.25, 0.06, 1)
    Lk.new(s.outputs[0], p.inputs["Emission Strength"])
    return m, amount


def block_mesh(name, m, shape=(1, 1, 1)):
    """One block. shape scales it inside its cell (e.g. [1, 1, 0.35] = a plank, [0.15, 0.15, 1] = a rope), top kept flush."""
    bm = bmesh.new()
    bmesh.ops.create_cube(bm, size=B * 0.996)
    sx, sy, sz = shape
    bmesh.ops.scale(bm, vec=(sx, sy, sz), verts=bm.verts)
    bmesh.ops.translate(bm, vec=(0, 0, (1 - sz) / 2 * B * 0.996), verts=bm.verts)
    bmesh.ops.bevel(bm, geom=bm.edges[:], offset=B * 0.03 * min(shape), segments=2, affect="EDGES", profile=0.5)
    me = bpy.data.meshes.new(name)
    bm.to_mesh(me); bm.free()
    me.materials.append(m)
    return me


def components(cells):
    """Connected groups of cells (by x/y adjacency within the part) -> list of cell lists."""
    left, out = set(map(tuple, cells)), []
    while left:
        stack, comp = [left.pop()], []
        while stack:
            c = stack.pop()
            comp.append(c)
            for n in ((c[0] + 1, c[1], c[2]), (c[0] - 1, c[1], c[2]), (c[0], c[1] + 1, c[2]), (c[0], c[1] - 1, c[2]),
                      (c[0], c[1], c[2] + 1), (c[0], c[1], c[2] - 1)):
                if n in left:
                    left.remove(n); stack.append(n)
        out.append(comp)
    return out


def key_visible(o, t0, t1):
    f0, f1 = int(round(t0 * FPS)), int(round(t1 * FPS))
    for f, v in ((0, 0), (f0 - 1, 0), (f0, 1), (f1, 1), (f1 + 1, 0)):
        if 0 <= f < NF:
            o.scale = (v, v, v)
            o.keyframe_insert("scale", frame=f)
    VK.set_interp(o, "CONSTANT")


def build_scene():
    bpy.ops.wm.read_factory_settings(use_empty=True)
    sc = bpy.context.scene
    sc.render.fps = FPS
    sc.frame_start, sc.frame_end = 0, NF - 1
    exposure = BW.build_world(sc, P)                    # ground, props, sky, sun, weather, animals
    roots, objs, crack_vals = {}, [], {}
    for name, st in P["structures"].items():
        r = bpy.data.objects.new("S_" + name, None)
        sc.collection.objects.link(r)
        r.location = (st["at"][0], st["at"][1], 0)
        roots[name] = r
        m, _ = block_material("M_" + name, st["material"])
        me = block_mesh("Block_" + name, m, st["block_shape"])
        crack_cells, me_c = set(), None
        for a in acts("crack"):
            if a["target"] == name:
                crack_cells |= set(map(tuple, st["parts"][a["part"]]))
                mc, val = block_material("MC_" + name, st["material"], crack=True)
                me_c = block_mesh("CrackBlock_" + name, mc, st["block_shape"])
                crack_vals[name] = (val, a)
        for b in [b for b in P["blocks"] if b["struct"] == name]:
            o = bpy.data.objects.new("%s_%d_%d_%d" % ((name,) + tuple(b["cell"])), me_c if tuple(b["cell"]) in crack_cells else me)
            sc.collection.objects.link(o)
            o.parent = r
            o.rotation_mode = "XYZ"
            o["settle_frame"], o["struct"] = b["settle"], name
            pos = Vector(b["pos"])
            if b["prebuilt"]:
                o.location = pos
            else:
                o.location = HIDE; o.keyframe_insert("location", frame=0)
                if b["appear"] > 1:
                    o.keyframe_insert("location", frame=b["appear"] - 1)
                o.location = pos + Vector((0, 0, 0.35 * B)); o.keyframe_insert("location", frame=b["appear"])
                o.location = pos + Vector((0, 0, 0.08 * B)); o.keyframe_insert("location", frame=b["appear"] + 1)
                o.location = pos; o.keyframe_insert("location", frame=b["settle"])
            if b.get("vanish"):
                v = b["vanish"]
                o.location = pos; o.keyframe_insert("location", frame=v - 1)
                o.location = pos + Vector((0, 0, 0.3 * B)); o.keyframe_insert("location", frame=v)
                o.location = HIDE; o.keyframe_insert("location", frame=v + 1)
            objs.append((o, b))
        if st["eyes"]:                                  # two eyes on the front of the top row
            eye_m = VK.mat("Eye_" + name, (0.03, 0.03, 0.04), 0.3)
            top = max(c[2] for c in st["cells"])
            front = min(c[1] for c in st["cells"] if c[2] == top)
            row = sorted([c for c in st["cells"] if c[2] == top and c[1] == front], key=lambda c: c[0])
            hb = [(o, b) for o, b in objs if b["struct"] == name and tuple(b["cell"]) in set(map(tuple, row))]
            hb.sort(key=lambda ob: ob[1]["cell"][0])
            if len(hb) == 1:
                for ex in (-0.22 * B, 0.22 * B):
                    VK.mesh_obj("eye", [((ex, -0.51 * B, 0.08 * B), (0.16 * B, 0.04 * B, 0.2 * B), 0)], [eye_m], parent=hb[0][0])
            else:
                mid = len(hb) // 2
                for (o, b), ex in ((hb[mid - 1], 0.2 * B), (hb[mid], -0.2 * B)):
                    VK.mesh_obj("eye", [((ex, -0.51 * B, -0.3 * B), (0.3 * B, 0.04 * B, 0.35 * B), 0)], [eye_m], parent=o)
    # highlight rings (cross-sections)
    glow = VK.emission_material("Highlight", (0.25, 0.85, 1.0), 6.0)
    for a in acts("highlight"):
        st = P["structures"][a["target"]]
        for comp in components(st["parts"][a["part"]]):
            xs = [c[0] for c in comp]; ys = [c[1] for c in comp]; zs = [c[2] for c in comp]
            p0 = Vector(C.local_pos((min(xs), min(ys), min(zs)), st["cells"], st["centered"]))
            p1 = Vector(C.local_pos((max(xs), max(ys), max(zs)), st["cells"], st["centered"]))
            cx, cy, cz = (p0.x + p1.x) / 2, (p0.y + p1.y) / 2, (p0.z + p1.z) / 2
            w, d = (max(xs) - min(xs) + 1) * B * 1.12, (max(ys) - min(ys) + 1) * B * 1.12
            t, h = max(w, d) * 0.08, 0.012 * (1 + 0.3 * (len(comp) > 2))
            boxes = [((0, d / 2, 0), (w + t, t, h), 0), ((0, -d / 2, 0), (w + t, t, h), 0),
                     ((w / 2, 0, 0), (t, d + t, h), 0), ((-w / 2, 0, 0), (t, d + t, h), 0)]
            o = VK.mesh_obj("ring_" + a["target"], boxes, [glow], parent=roots[a["target"]])
            o.location = (cx, cy, cz)
            key_visible(o, a["t0"], a["t1"])
    for name, (val, a) in crack_vals.items():
        val.outputs[0].default_value = 0.0
        val.outputs[0].keyframe_insert("default_value", frame=0)
        val.outputs[0].keyframe_insert("default_value", frame=int(a["t0"] * FPS))
        val.outputs[0].default_value = 1.0
        val.outputs[0].keyframe_insert("default_value", frame=int(a["t1"] * FPS))
        if a.get("c0") is not None:                             # "cool_from" / "cool_until": the glow fades away again
            val.outputs[0].keyframe_insert("default_value", frame=int(a["c0"] * FPS))
            val.outputs[0].default_value = 0.0
            val.outputs[0].keyframe_insert("default_value", frame=int(a["c1"] * FPS))

    arm = VK.build_grandpa(sc, None, P["outfit"])
    dust_m = VK.pixel_material("Dust", (0.80, 0.66, 0.46), (0.93, 0.82, 0.62), 0.05)
    cube = bpy.data.meshes.new("Puff")
    bm = bmesh.new(); bmesh.ops.create_cube(bm, size=1.0); bm.to_mesh(cube); bm.free()
    cube.materials.append(dust_m)
    puffs = []
    for k in range(46 * len(acts("collapse"))):
        o = bpy.data.objects.new("dust_%02d" % k, cube)
        sc.collection.objects.link(o)
        puffs.append(o)

    cam_d = bpy.data.cameras.new("Camera")
    cam_d.lens, cam_d.sensor_fit, cam_d.sensor_width, cam_d.clip_start = 35, "HORIZONTAL", 36, 0.05
    cam = bpy.data.objects.new("Camera", cam_d)
    cam.rotation_mode = "QUATERNION"
    sc.collection.objects.link(cam)
    sc.camera = cam
    r = sc.render
    r.engine = "CYCLES"
    r.resolution_x, r.resolution_y, r.resolution_percentage = W_PX, H_PX, 100
    r.image_settings.file_format = "PNG"
    r.use_persistent_data = True
    cy = sc.cycles
    cy.device, cy.samples, cy.adaptive_threshold = "CPU", 16, 0.05
    cy.use_denoising, cy.denoiser = True, "OPENIMAGEDENOISE"
    cy.denoising_quality, cy.denoising_prefilter = "FAST", "FAST"
    cy.max_bounces, cy.diffuse_bounces, cy.glossy_bounces = 2, 1, 0
    cy.transmission_bounces = cy.transparent_max_bounces = cy.volume_bounces = 0
    sc.view_settings.view_transform = "Standard"
    sc.view_settings.look = "Medium High Contrast"
    sc.view_settings.exposure = exposure
    animate(sc, arm, roots, objs, puffs, cam)
    return sc


# ============================================================ animation
def cam_at(t):
    keys = P["camera"] or [dict(t=0.0, target=[0, 0.1, 0.4], az=-60, el=20, dist=3.0)]
    if t <= keys[0]["t"]:
        k0 = k1 = keys[0]; u = 0.0
    else:
        k0 = k1 = keys[-1]; u = 0.0
        for a, b in zip(keys, keys[1:]):
            if a["t"] <= t <= b["t"]:
                k0, k1, u = a, b, smoother((t - a["t"]) / max(1e-6, b["t"] - a["t"]))
                break
    lerp = lambda p, q: p + (q - p) * u
    tgt = Vector(k0["target"]).lerp(Vector(k1["target"]), u)
    az, el = math.radians(lerp(k0["az"], k1["az"])), math.radians(lerp(k0["el"], k1["el"]))
    d = k0["dist"] * (k1["dist"] / k0["dist"]) ** u
    eye = tgt + d * Vector((math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)))
    for a in acts("collapse"):
        tc = a["t0"]
        if tc <= t < tc + 0.9:
            k = math.exp(-(t - tc) * 4.5)
            eye += Vector((math.sin(t * 57), math.sin(t * 43), math.sin(t * 51))) * 0.05 * k
    return eye, tgt


MOTIONS = ("slide", "spin", "swing", "fly")
AXES = {"x": Vector((1, 0, 0)), "y": Vector((0, 1, 0)), "z": Vector((0, 0, 1))}


def motion_pivot(a):
    """Pivot in the structure's own (unscaled) frame: centre of block 'pivot' [x,y,z], else the structure's centroid."""
    st = P["structures"][a["target"]]
    if a.get("pivot") is not None:
        return Vector(C.local_pos(a["pivot"], st["cells"], st["centered"]))
    ps = [Vector(C.local_pos(c, st["cells"], st["centered"])) for c in st["cells"]]
    return sum(ps, Vector()) / len(ps)


def motion_offset(name, t, local=False):
    """World offset of a structure; local=True: the slides marked \"local\", which turn with its spins and swings."""
    off = Vector()
    for a in acts("slide"):
        if a["target"] == name and bool(a.get("local")) == local:
            off += Vector(a["by"]) * smoother((t - a["t0"]) / max(1e-6, a["t1"] - a["t0"]))
    if local:
        return off
    for a in acts("fly"):
        if a["target"] == name:
            off += Vector(C.fly_offset(a, t))
    return off


def motion_rotation(name, t):
    R = Matrix.Identity(4)
    for a in acts("spin") + acts("swing"):
        if a["target"] == name:
            p = motion_pivot(a)
            R = R @ Matrix.Translation(p) @ Matrix.Rotation(math.radians(C.motion_angle(a, t)), 4, AXES[a.get("axis", "z")]) @ Matrix.Translation(-p)
    return R


def collapse_plan(objs):
    plan = {}
    for a in acts("collapse"):
        tc = a["t0"]
        for o, b in objs:
            if b["struct"] != a["target"]:
                continue
            p0 = Vector(b["pos"])
            k = b["cell"][2]
            delay = 0.03 * k + rnd.uniform(0, 0.06)
            out = Vector((rnd.gauss(0, 0.09) + p0.x * 0.4, rnd.gauss(0, 0.09), 0))
            rest_z = 0.05 + max(0.0, 0.22 - 0.5 * (abs(p0.x + out.x) + abs(p0.y + out.y))) * rnd.uniform(0.4, 1.0)
            rest = Vector((p0.x + out.x, p0.y + out.y, min(rest_z, p0.z)))      # rubble never rises
            rot = Euler((rnd.uniform(-1.2, 1.2), rnd.uniform(-1.2, 1.2), rnd.uniform(-0.6, 0.6)))
            plan[o.name] = (tc + delay, p0, rest, rot, a["target"])
    for a in acts("melt"):                                     # straight down into a flat slab, bottom blocks first
        st, tgt = P["structures"][a["target"]], P["melt"][a["target"]]
        for o, b in objs:
            if b["struct"] == a["target"]:
                rest = Vector(C.local_pos(tgt[str(b["id"])], st["cells"], st["centered"]))
                plan[o.name] = (a["t0"] + 0.025 * b["cell"][2] + rnd.uniform(0, 0.05), Vector(b["pos"]), rest, Euler((0, 0, 0)), a["target"])
    return plan


def animate(sc, arm, roots, objs, puffs, cam):
    path, phase, amp = P["path"], P["phase"], P["amp"]
    blocks = P["blocks"]
    S = P["structures"]
    pb = arm.pose.bones
    rest = {b_.name: b_.matrix_local.copy() for b_ in arm.data.bones}
    for p in pb:
        p.rotation_mode = "QUATERNION"
    arm.rotation_mode = "XYZ"
    grows = acts("grow")
    moving = {a["target"] for a in P["actions"] if a["type"] in MOTIONS}
    for r in roots.values():
        r.rotation_mode = "QUATERNION"
    plan = collapse_plan(objs)
    scale_at = {}
    # focus windows: what grandpa faces / looks at
    focus_acts = [a for a in P["actions"] if a["type"] in ("face_camera", "build", "point", "grow", "crack", "collapse", "melt") + MOTIONS]
    for i, a in enumerate(focus_acts):
        a["_t1"] = a["t1"] if a["type"] in ("face_camera", "build", "point") + MOTIONS else \
            (focus_acts[i + 1]["t0"] if i + 1 < len(focus_acts) else P["film"])
    yaw_prev = 0.0
    BREATH = P["film"] / max(1, round(P["film"] / 3.2)) if P.get("loop") else 3.2   # a loop breathes a whole number of times
    for f in range(NF):
        t = f / FPS
        # ---- structures: grow (scale about the base, optionally moving), slide / spin / swing, collapse
        for name, r in roots.items():
            base = Vector((S[name]["at"][0], S[name]["at"][1], 0))
            loc, s = base, 1.0
            for g in grows:                                     # several grows on one structure multiply in order
                if g["target"] != name:
                    continue
                u = smoother((t - g["t0"]) / max(1e-6, g["t1"] - g["t0"]))
                if g.get("to"):
                    loc = loc.lerp(Vector((g["to"][0], g["to"][1], 0)), u)
                s *= 1 + (g["factor"] - 1) * u
            M = (Matrix.Translation(loc + motion_offset(name, t)) @ Matrix.Scale(s, 4) @ motion_rotation(name, t)
                 @ Matrix.Translation(motion_offset(name, t, local=True)))
            r.matrix_basis = M
            scale_at[name] = s
            if grows or name in moving:
                r.keyframe_insert("location", frame=f); r.keyframe_insert("scale", frame=f)
                r.keyframe_insert("rotation_quaternion", frame=f)
        for o, b in objs:
            pl = plan.get(o.name)
            if not pl or t < pl[0] - 2 / FPS:
                continue
            ts, p0, pr, rot, tgt_name = pl
            g_local = 9.81 / max(1.0, scale_at[tgt_name]) * 0.8
            tau = max(0.0, t - ts)
            h = p0.z - pr.z
            tf = math.sqrt(2 * max(1e-6, h) / g_local)
            v = min(1.0, tau / tf) if h > 1e-6 else min(1.0, tau / 0.4)
            z = max(pr.z, p0.z - 0.5 * g_local * tau * tau)
            bounce = 0.02 * math.exp(-(tau - tf) * 9) * abs(math.sin((tau - tf) * 18)) if tau > tf else 0.0
            o.location = Vector((p0.x + (pr.x - p0.x) * smooth(v), p0.y + (pr.y - p0.y) * smooth(v), z + bounce))
            o.rotation_euler = Euler((rot.x * smooth(v), rot.y * smooth(v), rot.z * smooth(v)))
            o.keyframe_insert("location", frame=f)
            o.keyframe_insert("rotation_euler", frame=f)

        # ---- grandpa: what is he doing?
        x, y, zg = path[f]                                     # zg: standing height (ground, scaffold or ladder)
        vz = (path[min(NF - 1, f + 1)][2] - path[max(0, f - 1)][2]) * FPS / 2
        c_w = smooth((abs(vz) - 0.05) / 0.15)                   # climbing weight: moving up or down a ladder
        phc = 2 * math.pi * zg / 0.24                           # one hand-over-hand cycle every two rungs
        r_sw, aimp = 0.0, None
        for sw in P["swings"]:
            if sw["s"] - 0.05 <= t <= sw["e"] + 0.05:
                uu = (t - sw["s"]) / (sw["e"] - sw["s"])
                r_sw = smooth(uu / 0.22) * (1 - smooth((uu - 0.80) / 0.20))
                pa = Vector(roots[sw["target"]].location) + Vector(blocks[sw["ids"][0]]["pos"]) * scale_at[sw["target"]]
                pbb = Vector(roots[sw["target"]].location) + Vector(blocks[sw["ids"][-1]]["pos"]) * scale_at[sw["target"]]
                aimp = pa.lerp(pbb, cl((uu - 0.22) / 0.58))
        for a in acts("point"):
            w_pt = smooth((t - a["t0"]) / 0.4) * (1 - smooth((t - (a["t1"] - 0.4)) / 0.4))
            if w_pt > 0:
                if a.get("pos"):
                    aimp = Vector(a["pos"])
                else:
                    off = Vector(a.get("offset", [0, 0, 0.2]))
                    aimp = Vector(roots[a["target"]].location) + off * scale_at[a["target"]]
                aimp = aimp + Vector((0, 0, 0.02 * math.sin(t * 5)))
                r_sw = max(r_sw, w_pt) if aimp is not None else r_sw
        w_sh = max([smooth((t - a["t0"]) / 0.3) * (1 - smooth((t - (a["t1"] - 0.35)) / 0.35)) for a in acts("shrug")] + [0.0])
        startle = max([smooth((t - a["t0"]) / 0.15) * (1 - smooth((t - a["t0"] - 0.9) / 0.6)) for a in acts("collapse")] + [0.0])
        active = [a for a in focus_acts if a["t0"] <= t <= a["_t1"]]
        fa = active[-1] if active else None
        eye_now = Vector(cam_at(t)[0])
        if fa is None:
            focus = Vector((x, y - 1, 0))
        elif fa["type"] == "face_camera":
            focus = eye_now
        elif fa["type"] in ("build",):
            focus = Vector(roots[fa["target"]].location)
        elif fa["type"] == "point":
            focus = Vector(fa["pos"]) if fa.get("pos") else Vector(roots[fa["target"]].location)
        else:
            focus = Vector(roots[fa["target"]].location)
        if c_w > 0.3:
            mv = [a for a in acts("move") if a.get("face") and a["t0"] - 0.5 <= t <= a["t1"] + 0.5]
            if mv:
                focus = Vector((mv[-1]["face"][0], mv[-1]["face"][1], zg))
        fx, fy = focus.x - x, focus.y - y
        yaw_t = math.atan2(fx, -fy) if math.hypot(fx, fy) > 0.05 else yaw_prev
        while yaw_t - yaw_prev > math.pi:
            yaw_t -= 2 * math.pi
        while yaw_t - yaw_prev < -math.pi:
            yaw_t += 2 * math.pi
        yaw = yaw_prev + (yaw_t - yaw_prev) * (0.15 if f else 1.0)
        yaw_prev = yaw
        Rw = rz(yaw)
        Wroot = Matrix.Translation((x, y, zg)) @ Rw.to_4x4()
        Winv = Wroot.inverted()
        to_char = lambda p: Winv @ p
        arm.location, arm.rotation_euler = (x, y, zg), (0, 0, yaw)
        arm.keyframe_insert("location", frame=f)
        arm.keyframe_insert("rotation_euler", frame=f)
        a_walk = amp[f]
        cyc = 2 * math.pi * phase[f]
        lean = (0.18 + 0.10 * r_sw - 0.16 * startle - 0.06 * w_sh) * (1 - 0.8 * c_w)
        H = Vector((0, 0.03 * startle, 0.46 - 0.010 * a_walk * (1 - math.cos(2 * cyc)) + 0.004 * math.sin(2 * math.pi * t / BREATH)))
        twist = 0.0
        if aimp is not None:
            ac = to_char(aimp)
            twist = 0.3 * math.atan2(ac.x, -ac.y) * r_sw
        Qs = rz(twist) @ rx(lean)
        S_R, S_L = H + Qs @ Vector((-0.19, 0, 0.31)), H + Qs @ Vector((0.19, 0, 0.31))
        neck = H + Qs @ Vector((0, 0, 0.36))
        if aimp is not None and r_sw > 0.05:
            look = aimp
        elif fa is not None and fa["type"] in ("grow", "crack", "collapse", "melt"):
            sz = S[fa["target"]]["size"][2] * scale_at[fa["target"]]
            look = Vector(roots[fa["target"]].location) + Vector((0, 0, sz * (0.55 if fa["type"] != "collapse" else 0.2)))
        elif fa is not None and fa["type"] in MOTIONS:
            look = Vector(roots[fa["target"]].location) + motion_pivot(fa) * scale_at[fa["target"]]
        elif fa is not None and fa["type"] == "face_camera":
            look = eye_now
        else:
            look = focus + Vector((0, 0, 0.3))
        lc = to_char(look) - (neck + Qs @ Vector((0, 0, 0.12)))
        hy = max(-1.1, min(1.1, math.atan2(lc.x, -lc.y)))
        hp = max(-0.6, min(0.45, 0.6 * math.atan2(-lc.z, math.hypot(lc.x, lc.y)))) + 0.12 * w_sh
        Qh = rz(hy) @ rx(hp)
        sw_off = 0.06 * a_walk * math.sin(cyc)
        TR = S_R + Vector((0.01, -0.05 - sw_off, -0.405))
        TL_ = S_L + Vector((-0.01, -0.05 + sw_off, -0.405))
        poleR, poleL = Vector((-0.4, 1.0, 0.0)), Vector((0.4, 1.0, 0.0))
        if aimp is not None and r_sw > 0:
            ac = to_char(aimp)
            TR = TR.lerp(S_R + (ac - S_R).normalized() * (VK.L_UP + VK.L_FORE - 0.01), r_sw)
        shR, shL = H + Qs @ Vector((-0.30, -0.14, 0.30)), H + Qs @ Vector((0.30, -0.14, 0.30))
        TR, TL_ = TR.lerp(shR, w_sh), TL_.lerp(shL, w_sh)
        poleR, poleL = poleR.lerp(Vector((-1, 0.2, -0.6)), w_sh), poleL.lerp(Vector((1, 0.2, -0.6)), w_sh)
        upR, upL = H + Qs @ Vector((-0.22, -0.16, 0.48)), H + Qs @ Vector((0.22, -0.16, 0.48))
        TR, TL_ = TR.lerp(upR, startle * 0.7), TL_.lerp(upL, startle * 0.7)
        if "hammer" in P["outfit"]["extras"] and r_sw > 0:       # little hammer taps while placing
            TR = TR + Vector((0, 0, 0.03 * abs(math.sin(2 * math.pi * 3.5 * t)))) * r_sw
        if c_w > 0:                                             # hand over hand on the rungs in front of him
            rungR = S_R + Vector((0.02, -0.17, 0.12 + 0.07 * math.sin(phc)))
            rungL = S_L + Vector((-0.02, -0.17, 0.12 - 0.07 * math.sin(phc)))
            TR, TL_ = TR.lerp(rungR, c_w), TL_.lerp(rungL, c_w)
        E_R, W_R, _ = two_bone(S_R, TR, VK.L_UP, VK.L_FORE, poleR)
        E_L, W_L, _ = two_bone(S_L, TL_, VK.L_UP, VK.L_FORE, poleL)
        fn = min(NF - 1, f + 1)
        vel = Vector((path[fn][0] - path[max(0, fn - 2)][0], path[fn][1] - path[max(0, fn - 2)][1], 0))
        vc = Rw.inverted() @ vel
        vdir = vc.normalized() if vc.length > 1e-6 else Vector((0, -1, 0))
        feet = {}
        for s_, xo, off in (("R", -0.07, 0.0), ("L", 0.07, 0.5)):
            ph = cyc + 2 * math.pi * off
            p = Vector((xo, 0.0, 0.0)) + vdir * (C.STRIDE / 2 * -math.cos(ph) * a_walk)
            feet[s_] = p + Vector((0, 0, 0.05 * a_walk * max(0.0, math.sin(ph))))
            if c_w > 0:                                         # alternate feet stepping up the rungs
                step = 0.07 * max(0.0, math.sin(phc + (0 if s_ == "R" else math.pi)))
                feet[s_] = feet[s_].lerp(Vector((xo, -0.06, step)), c_w)
        hipR, hipL = H + Vector((-0.07, 0, 0)), H + Vector((0.07, 0, 0))
        K_R, F_R, _ = two_bone(hipR, feet["R"], VK.L_THIGH, VK.L_SHIN, Vector((0, -1, 0.2)))
        K_L, F_L, _ = two_bone(hipL, feet["L"], VK.L_THIGH, VK.L_SHIN, Vector((0, -1, 0.2)))
        Qid, dn = Matrix.Identity(3), Vector((0, 0, -1))
        Qu_R = aim(Qs, dn, E_R - S_R); Qf_R = aim(Qu_R, dn, W_R - E_R)
        Qu_L = aim(Qs, dn, E_L - S_L); Qf_L = aim(Qu_L, dn, W_L - E_L)
        Qt_R = aim(Qid, dn, K_R - hipR); Qn_R = aim(Qt_R, dn, F_R - K_R)
        Qt_L = aim(Qid, dn, K_L - hipL); Qn_L = aim(Qt_L, dn, F_L - K_L)
        mouth = P["mouth"][min(f, len(P["mouth"]) - 1)]
        want = {"hips": (H, Qid), "spine": (H, Qs), "head": (neck, Qh),
                "jaw": (neck + Qh @ Vector((0, -0.1, 0.04 - JAW_DROP * mouth)), Qh),
                "upper_arm.R": (S_R, Qu_R), "forearm.R": (E_R, Qf_R), "upper_arm.L": (S_L, Qu_L), "forearm.L": (E_L, Qf_L),
                "thigh.R": (hipR, Qt_R), "shin.R": (K_R, Qn_R), "thigh.L": (hipL, Qt_L), "shin.L": (K_L, Qn_L)}
        Mw = {}
        for name in VK.BONE_ORDER:
            p_, Q = want[name]
            M = Matrix.Translation(p_) @ (Q @ rest[name].to_3x3()).to_4x4()
            Mw[name] = M
            par = arm.data.bones[name].parent
            basis = ((Mw[par.name] @ rest[par.name].inverted() @ rest[name]).inverted() @ M) if par else rest[name].inverted() @ M
            loc, rot, _ = basis.decompose()
            pb[name].location, pb[name].rotation_quaternion = loc, rot
            pb[name].keyframe_insert("location", frame=f)
            pb[name].keyframe_insert("rotation_quaternion", frame=f)
        eye, tgt = cam_at(t)
        cam.location, cam.rotation_quaternion = eye, (tgt - eye).to_track_quat("-Z", "Y")
        cam.keyframe_insert("location", frame=f)
        cam.keyframe_insert("rotation_quaternion", frame=f)

    k = 0
    for a in acts("collapse"):
        st, tc = S[a["target"]], a["t0"]
        sc_ = scale_at[a["target"]]
        base = Vector(roots[a["target"]].location)
        for _ in range(46):
            o = puffs[k]; k += 1
            ts = tc + 0.15 + rnd.uniform(0, 0.9)
            f0 = int(ts * FPS)
            src = base + Vector((rnd.uniform(-0.6, 0.6) * st["size"][0] * sc_, rnd.uniform(-0.6, 0.6) * max(0.2, st["size"][1]) * sc_, 0.1))
            size = rnd.uniform(0.18, 0.45) * min(1.0, sc_ / 5)
            o.rotation_euler = (rnd.uniform(0, 1), rnd.uniform(0, 1), rnd.uniform(0, 3))
            for fk, loc, s_ in ((0, src, 0), (f0 - 1, src, 0), (f0, src, size * 0.3),
                                (f0 + 12, src + Vector((rnd.uniform(-0.3, 0.3), rnd.uniform(-0.3, 0.3), 0.35)), size),
                                (f0 + 40, src + Vector((rnd.uniform(-0.6, 0.6), rnd.uniform(-0.6, 0.6), 0.7)), 0)):
                if 0 <= fk < NF:
                    o.location, o.scale = loc, (s_, s_, s_)
                    o.keyframe_insert("location", frame=fk); o.keyframe_insert("scale", frame=fk)
    for o, b in objs:
        for fc in VK.fcurves_of(o):
            for kp in fc.keyframe_points:
                kp.interpolation = "CONSTANT" if ((not b["prebuilt"] and int(round(kp.co.x)) in (0, b["appear"] - 1))
                                                  or int(round(kp.co.x)) == b.get("vanish", -9)) else "LINEAR"
    for o in [arm, cam] + list(roots.values()) + puffs:
        VK.set_interp(o, "LINEAR")

    # ---- screen anchors for the overlay
    pts = {"grandpa": lambda: arm.matrix_world @ arm.pose.bones["head"].tail}
    for name, st in S.items():
        hz = st["size"][2]
        if st["centered"]:
            fc = Vector((0, 0, 0))
        else:                                                     # footprint centre of a corner-anchored structure
            xs, ys = [c[0] for c in st["cells"]], [c[1] for c in st["cells"]]
            fc = Vector(((min(xs) + max(xs)) / 2 * B, (min(ys) + max(ys)) / 2 * B, 0))
        pts[name + ":top"] = (lambda r=roots[name], hz=hz, fc=fc: r.matrix_world @ (fc + Vector((0, 0, hz + 0.05))))
        pts[name + ":center"] = (lambda r=roots[name], hz=hz, fc=fc: r.matrix_world @ (fc + Vector((0, 0, hz * 0.5))))
        for pn, cells in st["parts"].items():
            if not cells:
                continue
            comp = sorted(components(cells), key=lambda c: -sum(cc[0] for cc in c) / len(c))[0]
            ctr = sum((Vector(C.local_pos(c, st["cells"], st["centered"])) for c in comp), Vector()) / len(comp)
            front = ctr + Vector((0, -st["size"][1] / 2 - 0.01, 0))
            pts["%s:part:%s" % (name, pn)] = (lambda r=roots[name], p=front: r.matrix_world @ p)
    anchors = {}
    for f in range(NF):
        sc.frame_set(f)
        a = {}
        for k_, fn_ in pts.items():
            v = world_to_camera_view(sc, cam, fn_())
            a[k_] = [round(v.x * W_PX, 1), round((1 - v.y) * H_PX, 1), 1 if v.z > 0 else 0]
        anchors[f] = a
    json.dump(anchors, open(os.path.join(C.BUILD, "anchors.json"), "w"))


def preview(frames, scale=25, samples=8):
    sc = bpy.context.scene
    sc.render.resolution_percentage, sc.cycles.samples = scale, samples
    os.makedirs(os.path.join(C.BUILD, "previews"), exist_ok=True)
    for f in frames:
        sc.frame_set(f)
        sc.render.filepath = os.path.join(C.BUILD, "previews", "%04d.png" % f)
        bpy.ops.render.render(write_still=True)
    sc.render.resolution_percentage, sc.cycles.samples = 100, 16


if __name__ == "__main__":
    build_scene()
    bpy.ops.wm.save_as_mainfile(filepath=os.path.join(C.BUILD, "blocklab.blend"))
    if "--preview" in sys.argv:
        preview([int(a) for a in sys.argv[sys.argv.index("--preview") + 1:]])

In [ ]:
%%writefile bl_overlay.py
"""Block Lab: draw the story's cards + captions over the rendered frames and encode the MP4 (vertical 1080x1920 inside the
Shorts safe zone, or "format": "wide" 1920x1080). Card styles: title, bubble ({count:STRUCT} = live block count), tag, panel, impact, end.

    python bl_overlay.py [--frames DIR] [--out NAME.mp4] [--still 60 300 ...]
"""
import json
import math
import os
import subprocess
import sys
from multiprocessing import Pool

import numpy as np
from PIL import Image, ImageDraw, ImageFont

import bl_common as C

P = C.load_plan()
FPS, NF, W, H = P["fps"], P["nf"], P["w"], P["h"]
ANCH = json.load(open(os.path.join(C.BUILD, "anchors.json")))
smooth = C.smooth
COL = {"white": (255, 255, 255), "amber": (255, 196, 64), "cyan": (110, 225, 255), "red": (255, 80, 64),
       "green": (120, 230, 120), "pink": (255, 130, 190)}
INK = (24, 20, 16)
WIDE = W > H                                   # "format": "wide" stories are 1920x1080
CX = W // 2
LAY = dict(tag=(60, 50), cap_y=H - 115, cap_size=56, cap_chars=30, clamp=(120, W - 120, 150, H - 190),
           end=(H * 0.36, H * 0.48, H * 0.57, H * 0.64), panel_w=520) if WIDE else \
      dict(tag=(90, 250), cap_y=1330, cap_size=64, cap_chars=16, clamp=(120, 960, 280, 1400),
           end=(760, 890, 1010, 1090), panel_w=450)
SETTLES = {}
for b in P["blocks"]:
    SETTLES.setdefault(b["struct"], []).append(b["settle"])


def font_dir():
    try:
        import matplotlib
        return os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")
    except ImportError:
        return "/usr/share/fonts/truetype/dejavu"


FD, _fc = font_dir(), {}


def F(kind, size):
    size = max(8, int(size))
    if (kind, size) not in _fc:
        _fc[(kind, size)] = ImageFont.truetype(os.path.join(FD, {"bold": "DejaVuSans-Bold.ttf", "reg": "DejaVuSans.ttf"}[kind]), size)
    return _fc[(kind, size)]


def env(t, a, b, fi=0.25, fo=0.25):
    return smooth((t - a) / fi) * (1 - smooth((t - (b - fo)) / fo))


def pop(t, a):
    u = max(0.0, min(1.0, (t - a) / 0.22))
    return 0.6 + 0.4 * u + 0.18 * math.sin(math.pi * u) if u < 1 else 1.0


def outlined(d, xy, s, font, fill, a=1.0, anchor="mm", stroke=6):
    if a > 0.01:
        d.text(xy, s, font=font, fill=fill + (int(255 * a),), anchor=anchor, stroke_width=stroke, stroke_fill=INK + (int(235 * a),))


def anchor(f, key, dy=0):
    a = ANCH[str(min(f, NF - 1))].get(key)
    if not a or not a[2]:
        return None
    x0, x1, y0, y1 = LAY["clamp"]
    return min(x1, max(x0, a[0])), min(y1, max(y0, a[1] + dy))


def fill_text(s, f):
    """{count:NAME} = blocks of NAME placed so far; {count:NAME*} sums every structure whose name starts with NAME."""
    out = s
    for key in set(__import__("re").findall(r"\{count:([^}]+)\}", s)):
        names = [n for n in SETTLES if (n.startswith(key[:-1]) if key.endswith("*") else n == key)]
        out = out.replace("{count:%s}" % key, str(sum(1 for n in names for x in SETTLES[n] if x <= f)))
    return out


def card(d, ov, c, t, f):
    style = c.get("style", "title")
    col = COL.get(c.get("color", "white"), COL["white"])
    if style == "end":
        k = smooth((t - c["t0"]) / 0.4)
        if k <= 0:
            return
        ov.alpha_composite(Image.new("RGBA", (W, H), (12, 14, 18, int(200 * k))))
        d = ImageDraw.Draw(ov)
        fit = lambda s_: F("bold", min(110, 110 * 940 / max(1.0, F("bold", 110).getlength(s_))))   # shrink long titles to the safe width
        e1, e2, e3, e4 = LAY["end"]
        outlined(d, (CX, e1), c.get("title1", ""), fit(c.get("title1", "")), COL["white"], k)
        outlined(d, (CX, e2), c.get("title2", ""), fit(c.get("title2", "")), COL["amber"], k)
        d.text((CX, e3), c.get("sub", ""), font=F("reg", 30), fill=(210, 214, 220, int(255 * k)), anchor="mm")
        d.text((CX, e4), c.get("cta", ""), font=F("bold", 40), fill=COL["cyan"] + (int(255 * k),), anchor="mm")
        return
    a = env(t, c["t0"], c["t1"], fi=0.2 if style != "impact" else 0.06, fo=0.25 if style != "impact" else 0.35)
    if a <= 0.01:
        return
    if style == "title":
        outlined(d, (CX, c.get("y", 470)), fill_text(c["text"], f), F("bold", c.get("size", 96) * pop(t, c["t0"])), col, a)
    elif style == "impact":
        jx = 8 * math.sin(t * 90) * math.exp(-(t - c["t0"]) * 5)
        outlined(d, (CX + jx, c.get("y", 640)), c["text"], F("bold", c.get("size", 150) * pop(t, c["t0"])), col, a, stroke=8)
    elif style in ("bubble", "tag"):
        p = anchor(f, c["anchor"], c.get("dy", -40 if style == "bubble" else 0))
        if not p:
            return
        x, y = p
        size = c.get("size", 40 if style == "bubble" else 34) * (pop(t, c["t0"]) if style == "bubble" else 1)
        s = fill_text(c["text"], f)
        fnt = F("bold", size)
        tw = fnt.getlength(s)
        if style == "bubble":
            box = (x - tw / 2 - 24, y - size * 0.9 - 26, x + tw / 2 + 24, y - 26)
        else:
            box = (x - tw / 2 - 20, y - size * 0.6, x + tw / 2 + 20, y + size * 0.6)
        d.rounded_rectangle(box, radius=18, fill=(255, 250, 238, int(240 * a)), outline=col + (int(255 * a),), width=5)
        if style == "bubble":
            d.polygon([(x - 14, y - 28), (x + 14, y - 28), (x, y - 6)], fill=col + (int(255 * a),))
        d.text((x, (box[1] + box[3]) / 2), s, font=fnt, fill=INK + (int(255 * a),), anchor="mm")
    elif style == "panel":
        vis = [r for r in c["rows"] if t >= r["t"]]
        if not vis:
            return
        y0 = c.get("y", 300)
        heights = [int(r.get("size", 46) * 1.6) for r in vis]
        hgt = 30 + sum(heights)
        border = COL.get(c.get("border", "cyan"), COL["cyan"])
        d.rounded_rectangle((CX - LAY["panel_w"], y0, CX + LAY["panel_w"], y0 + hgt), radius=26, fill=(16, 22, 30, int(200 * a)), outline=border + (int(200 * a),), width=4)
        yy = y0 + 15
        for r, hh in zip(vis, heights):
            aa = smooth((t - r["t"]) / 0.25) * a
            d.text((CX, yy + hh / 2), fill_text(r["text"], f), font=F("bold", r.get("size", 46)),
                   fill=COL.get(r.get("color", "white"), COL["white"]) + (int(255 * aa),), anchor="mm")
            yy += hh


def captions(d, t):
    for ln in P["lines"]:
        if ln["start"] - 0.05 <= t <= ln["end"] + 0.2:
            chunks, cur = [], []
            for w_ in ln["text"].split():
                cur.append(w_)
                if len(" ".join(cur)) > LAY["cap_chars"] or w_.endswith((".", ",", "?", "!")):
                    chunks.append(" ".join(cur)); cur = []
            if cur:
                chunks.append(" ".join(cur))
            n = sum(len(c) for c in chunks)
            t0 = ln["start"]
            for c in chunks:
                dur = (ln["end"] - ln["start"]) * len(c) / n
                if t0 - 0.05 <= t < t0 + dur + 0.12:
                    outlined(d, (CX, LAY["cap_y"]), c.replace("...", "…"), F("bold", LAY["cap_size"]), COL["white"], 1.0, stroke=7)
                    return
                t0 += dur


def compose(args):
    f, src = args
    t = f / FPS
    base = Image.open(os.path.join(src, "%04d.png" % f)).convert("RGB")
    if base.size != (W, H):
        base = base.resize((W, H), Image.BICUBIC)
    img = base.convert("RGBA")
    ov = Image.new("RGBA", (W, H))
    d = ImageDraw.Draw(ov)
    tag = P["channel_tag"]
    if tag:
        tw = F("bold", 28).getlength(tag)
        tx, ty = LAY["tag"]
        d.rounded_rectangle((tx, ty, tx + tw + 50, ty + 62), radius=30, fill=(255, 196, 64, 235))
        d.text((tx + (tw + 50) / 2, ty + 31), tag, font=F("bold", 28), fill=INK + (255,), anchor="mm")
    end_t = min([c["t0"] for c in P["cards"] if c.get("style") == "end"] + [P["film"] + 1])
    for c in P["cards"]:
        card(d, ov, c, t, f)
        d = ImageDraw.Draw(ov)
    if t < end_t:
        captions(d, t)
    img.alpha_composite(ov)
    return img.convert("RGB").tobytes()


def main():
    args = sys.argv[1:]
    src = os.path.abspath(args[args.index("--frames") + 1]) if "--frames" in args else os.path.join(C.BUILD, "frames")
    out = args[args.index("--out") + 1] if "--out" in args else os.path.join(C.HERE, P["slug"] + ".mp4")
    if "--still" in args:
        os.makedirs(os.path.join(C.BUILD, "stills"), exist_ok=True)
        for f in [int(a) for a in args[args.index("--still") + 1:] if a.isdigit()]:
            Image.frombytes("RGB", (W, H), compose((f, src))).save(os.path.join(C.BUILD, "stills", "%04d.png" % f))
        return
    ff = subprocess.Popen(["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", "%dx%d" % (W, H),
                           "-r", str(FPS), "-i", "-", "-i", os.path.join(C.BUILD, "audio", "mix.wav"), "-map", "0:v", "-map", "1:a",
                           "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p", "-profile:v", "high",
                           "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-shortest", "-movflags", "+faststart", out],
                          stdin=subprocess.PIPE)
    with Pool(max(1, os.cpu_count() or 2)) as pool:
        for buf in pool.imap(compose, [(f, src) for f in range(NF)], chunksize=4):
            ff.stdin.write(buf)
    ff.stdin.close(); ff.wait()
    print("wrote", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bl_audio.py
"""Block Lab: procedural sound for the active story (from build/plan.json) + grandpa's voice -> build/audio/mix.wav.

Every sound comes from a story event: block settles, footsteps, card pops, highlights, grow, crack, collapse.
"""
import json
import os
import subprocess

import numpy as np
import scipy.io.wavfile as wav
from scipy.signal import butter, sosfilt

import bl_common as C

P = C.load_plan()
OUT = os.path.join(C.BUILD, "audio")
SR, FPS = 48000, P["fps"]
N0 = int(P["film"] * SR)
NL = int(0.4 * SR) if P.get("loop") else 0          # looping Short: render a little past the end, then fold it onto the start
N = N0 + NL
t_all = np.arange(N) / SR
rng = np.random.default_rng(9)


def bp(x, lo, hi, o=2):
    return sosfilt(butter(o, [lo, hi], "bandpass", fs=SR, output="sos"), x)


def lp(x, f, o=2):
    return sosfilt(butter(o, f, "lowpass", fs=SR, output="sos"), x)


def hp(x, f, o=2):
    return sosfilt(butter(o, f, "highpass", fs=SR, output="sos"), x)


def tt(d):
    return np.arange(int(d * SR)) / SR


def place(buf, t, x, g=1.0):
    j = int(round(t * SR))
    if 0 <= j < len(buf):
        e = min(len(buf), j + len(x))
        buf[j:e] += x[:e - j] * g


def click(p=1.0):
    t = tt(0.07)
    x = sum(a * np.sin(2 * np.pi * f * p * t + rng.uniform(0, 6.3)) * np.exp(-t * dmp) for f, a, dmp in ((1850, 0.9, 70), (2930, 0.6, 95), (4320, 0.4, 130)))
    return (x * 0.5 + bp(rng.standard_normal(len(t)), 2500, 9000) * np.exp(-t * 900) * 0.8) * (1 - np.exp(-t * 4000))


def thunk(p=1.0, d=0.16):
    t = tt(d)
    body = np.sin(2 * np.pi * np.cumsum(150 * p * (1 + 0.25 * np.exp(-t * 60))) / SR) * np.exp(-t * 30)
    return (body + lp(rng.standard_normal(len(t)), 600) * np.exp(-t * 70) * 0.8) * (1 - np.exp(-t * 2500))


def modes(t, p, parts):
    """Sum of decaying partials [(Hz, amplitude, decay per second), ...] with random phases."""
    return sum(a * np.sin(2 * np.pi * f * p * t + rng.uniform(0, 6.3)) * np.exp(-t * d) for f, a, d in parts)


BLOCK_SOUND = {                                                  # material -> (length s, loudness) of its placement sound
    "wood": (0.20, 0.95), "stone": (0.16, 1.0), "sandstone": (0.15, 0.85), "brick": (0.11, 0.9),
    "gold": (0.45, 0.6), "grass": (0.13, 0.9), "blue": (0.13, 0.9), "iron": (0.35, 0.75), "rubber": (0.13, 0.9), "red": (0.25, 0.8),
    "steel": (0.30, 0.6)}


def block_sound(material, p=1.0):
    """One block settling, voiced by what it is made of: wood knocks, stone clacks, sandstone taps softly,
    brick taps dry, gold tinks, grass / blue pop."""
    dur, loud = BLOCK_SOUND.get(material, BLOCK_SOUND["stone"])
    t = tt(dur)
    n = rng.standard_normal(len(t))
    if material == "wood":        # hollow knock: low body resonances, little top end
        x = modes(t, p, ((380, 1.0, 38), (610, 0.6, 45), (980, 0.35, 60))) + bp(n, 300, 1800) * np.exp(-t * 400) * 0.6 \
            + np.sin(2 * np.pi * 120 * p * t) * np.exp(-t * 35) * 0.5
    elif material == "sandstone":  # soft gritty tap: grainy noise, a little tone
        grain = np.repeat(rng.uniform(0.3, 1.0, len(t) // 96 + 1), 96)[:len(t)]
        x = bp(n, 450, 2200) * grain * np.exp(-t * 120) * 0.8 + modes(t, p, ((700, 0.25, 90),)) \
            + np.sin(2 * np.pi * 110 * p * t) * np.exp(-t * 40) * 0.8
    elif material == "brick":     # dry sharp tap: short bright partials, almost no ring
        x = modes(t, p, ((1600, 0.8, 140), (2450, 0.6, 170), (3500, 0.35, 200))) + bp(n, 2000, 8000) * np.exp(-t * 1100) * 0.8 \
            + np.sin(2 * np.pi * 180 * p * t) * np.exp(-t * 60) * 0.3
    elif material == "gold":      # bright tink: high, slightly inharmonic, rings a little
        x = modes(t, p, ((2960, 1.0, 14), (4410, 0.55, 18), (6120, 0.3, 24), (7830, 0.15, 30))) \
            + bp(n, 4000, 12000) * np.exp(-t * 1500) * 0.4 + np.sin(2 * np.pi * 220 * p * t) * np.exp(-t * 50) * 0.2
    elif material == "iron":      # metallic clank: inharmonic ring over a dull hit, like a girder being set
        x = modes(t, p, ((520, 1.0, 22), (1410, 0.6, 28), (2380, 0.45, 35), (3720, 0.25, 45))) \
            + bp(n, 1200, 6000) * np.exp(-t * 600) * 0.6 + np.sin(2 * np.pi * 140 * p * t) * np.exp(-t * 40) * 0.5
    elif material == "steel":     # small bright steel: a light high ting, like keys on a ring
        x = modes(t, p, ((2350, 1.0, 20), (3900, 0.6, 26), (5600, 0.35, 34))) + bp(n, 3000, 10000) * np.exp(-t * 1400) * 0.5
    elif material == "red":       # painted steel panel: a bright, short clank
        x = modes(t, p, ((780, 1.0, 30), (1960, 0.5, 42), (3100, 0.25, 55))) + bp(n, 1500, 7000) * np.exp(-t * 700) * 0.5
    elif material in ("grass", "blue", "rubber"):  # muffled pop
        x = lp(n, 600) * np.exp(-t * 60) * 0.9 + np.sin(2 * np.pi * 95 * p * t) * np.exp(-t * 45) * 0.7
    else:                         # stone (and anything new): solid clack with a heavy thunk under it
        x = modes(t, p, ((1150, 0.7, 80), (1720, 0.5, 95), (2600, 0.3, 120))) + bp(n, 1500, 7000) * np.exp(-t * 700) * 0.7 \
            + thunk(p, dur) * 0.6
    x = x * (1 - np.exp(-t * 4000))                             # same instant attack for every material (keeps click sync)
    return x / max(1e-9, np.abs(x).max()) * loud


def step():
    t = tt(0.16)
    crunch = np.zeros(len(t))
    for _ in range(12):
        j = int(rng.uniform(0, 0.07) * SR)
        g = bp(rng.standard_normal(int(0.006 * SR)), 1200, 5500) * np.hanning(int(0.006 * SR))
        crunch[j:j + len(g)] += g * rng.uniform(0.2, 0.7)
    return np.sin(2 * np.pi * 85 * t) * np.exp(-t * 45) * 0.6 + crunch


def soft_hit(f0=70, d=0.6):
    t = tt(d)
    return np.sin(2 * np.pi * np.cumsum(f0 * (1 + 0.6 * np.exp(-t * 20))) / SR) * np.exp(-t * 7)


def chime(f=880, d=1.2):
    t = tt(d)
    return sum(a * np.sin(2 * np.pi * f * k * t) for k, a in ((1, 1.0), (2.01, 0.35), (3.0, 0.15))) * np.exp(-t * 3.5) * 0.4


def blip(f=1500, d=0.08):
    t = tt(d)
    return np.sin(2 * np.pi * f * t) * np.exp(-t * 45)


def snap():
    t = tt(0.05)
    return bp(rng.standard_normal(len(t)), 1500, 9000) * np.exp(-t * 160) + np.sin(2 * np.pi * 320 * t) * np.exp(-t * 90) * 0.5


def creak(d=0.5):
    t = tt(d)
    f = 70 + 40 * np.sin(2 * np.pi * 3 * t)
    return np.sign(np.sin(2 * np.pi * np.cumsum(f) / SR)) * bp(rng.standard_normal(len(t)), 200, 1500) * np.hanning(len(t)) * 0.4


def boom(d=2.5):
    t = tt(d)
    return (np.sin(2 * np.pi * np.cumsum(42 * (0.6 + 0.8 * np.exp(-t * 4))) / SR) * np.exp(-t * 2) * 1.6 +
            lp(rng.standard_normal(len(t)), 260) * np.exp(-t * 3) * 2.4) * (1 - np.exp(-t * 500))


stems = {k: np.zeros(N + 4 * SR) for k in ("build", "fx", "ambience")}
counts = {n: sum(1 for b in P["blocks"] if b["struct"] == n) for n in P["structures"]}
for b in P["blocks"]:
    if b["prebuilt"]:
        continue
    ts = b["settle"] / FPS
    many = counts[b["struct"]] > 24
    p = rng.uniform(0.92, 1.08) * (0.9 if many else 1.0)
    g = rng.uniform(0.3, 0.42) if many else 0.75
    place(stems["build"], ts, block_sound(P["structures"][b["struct"]]["material"], p), g * 1.1)
for s in P["steps"]:
    place(stems["build"], s["frame"] / FPS, step(), 0.3)
if "hammer" in P.get("outfit", {}).get("extras", []):          # a light hammer tap just after each block lands
    for b in P["blocks"]:
        if not b["prebuilt"]:
            x = tt(0.09)
            place(stems["build"], b["settle"] / FPS + 0.09, modes(x, rng.uniform(0.95, 1.05), ((3100, 1.0, 60), (4700, 0.5, 80))) * np.exp(-x * 40), 0.12)
for b in P["blocks"]:                                           # blocks being taken down: a soft lift-off
    if b.get("vanish"):
        place(stems["build"], b["vanish"] / FPS, block_sound(P["structures"][b["struct"]]["material"], rng.uniform(1.05, 1.2)),
              0.35 if counts[b["struct"]] <= 24 else 0.12)
zs = [p[2] for p in P["path"]]                                  # boots on ladder rungs while climbing
for f in range(1, len(zs)):
    if abs(zs[f] - zs[f - 1]) * FPS > 0.05 and int(zs[f] / 0.1) != int(zs[f - 1] / 0.1):
        place(stems["build"], f / FPS, block_sound("wood", rng.uniform(0.8, 0.9)), 0.3)
for c in P["cards"]:
    st = c.get("style", "title")
    if st == "title":
        place(stems["fx"], c["t0"], soft_hit(70, 0.7), 0.5)
    elif st in ("bubble", "tag"):
        place(stems["fx"], c["t0"], blip(1500), 0.3)
    elif st == "panel":
        for r in c["rows"]:
            place(stems["fx"], r["t"], blip(1400), 0.35)
    elif st == "end":
        place(stems["fx"], c["t0"], soft_hit(50, 1.2), 0.6)
for a in P["actions"]:
    if a["type"] == "highlight":
        place(stems["fx"], a["t0"], chime(988), 0.4)
    elif a["type"] == "grow" and a.get("sound") == "none":
        pass
    elif a["type"] == "grow":
        x = tt(a["t1"] - a["t0"] + 0.4)
        rise = np.sin(2 * np.pi * np.cumsum(120 + 500 * (x / x[-1]) ** 1.5) / SR) * 0.25 + bp(rng.standard_normal(len(x)), 500, 6000) * 0.35
        shimmer = sum(np.sin(2 * np.pi * f * x) for f in (1320, 1760, 2217)) * 0.06 * (x / x[-1])
        place(stems["fx"], a["t0"], (rise + shimmer) * np.hanning(len(x)) ** 0.7, 0.7)
        place(stems["fx"], a["t0"], lp(rng.standard_normal(len(x)), 120) * (x / x[-1]) * 1.2, 0.6)
    elif a["type"] == "crack" and a.get("sound") == "none":
        pass
    elif a["type"] == "crack" and a.get("sound") == "spark":     # spark plug: a sharp electric snap
        for k in range(3):
            place(stems["fx"], a["t0"] + 0.025 * k, snap(), 0.5)
    elif a["type"] == "crack" and a.get("sound") == "bang":      # the fuel burning: a punchy little explosion
        place(stems["fx"], a["t0"], boom(0.9), 0.55)
        place(stems["fx"], a["t0"], snap(), 0.6)
    elif a["type"] == "crack" and a.get("sound") == "sizzle":    # hot metal: a hiss as it heats, steam as it cools
        x = tt(max(0.4, a["t1"] - a["t0"]) + 0.6)
        place(stems["fx"], a["t0"], bp(rng.standard_normal(len(x)), 2500, 9000) * np.exp(-x * 2.5) * (1 - np.exp(-x * 20)), 0.35)
        if a.get("c0") is not None:
            x = tt(a["c1"] - a["c0"] + 0.5)
            hiss = bp(rng.standard_normal(len(x)), 1800, 7000) * (1 - x / x[-1]) * (0.7 + 0.3 * np.sin(2 * np.pi * 9 * x))
            place(stems["fx"], a["c0"], hiss, 0.4)
    elif a["type"] == "crack":
        for k in range(22):
            place(stems["fx"], a["t0"] + (a["t1"] - a["t0"]) * (k / 22) ** 0.6, snap(), rng.uniform(0.35, 0.7))
        place(stems["fx"], a["t0"], creak(max(0.2, a["t1"] - a["t0"])), 0.6)
    elif a["type"] == "collapse":
        place(stems["fx"], a["t0"], boom(2.8), 1.2)
        for k in range(70):
            place(stems["fx"], a["t0"] + 0.15 + rng.gamma(2.0, 0.22), thunk(rng.uniform(0.5, 0.8), 0.3), rng.uniform(0.25, 0.6))
        x = tt(2.0)
        place(stems["fx"], a["t0"] + 0.2, bp(rng.standard_normal(len(x)), 300, 3000) * np.exp(-x * 1.5) * 0.4)
    elif a["type"] == "melt":                                   # hot hiss, then the blocks clink down into the slab
        mat_ = P["structures"][a["target"]]["material"]
        x = tt(2.2)
        place(stems["fx"], a["t0"] - 0.3, bp(rng.standard_normal(len(x)), 2500, 9000) * np.exp(-x * 1.4) * (1 - np.exp(-x * 8)), 0.5)
        place(stems["fx"], a["t0"], lp(rng.standard_normal(len(x)), 180) * np.exp(-x * 1.8), 0.9)
        for k in range(60):
            place(stems["fx"], a["t0"] + rng.gamma(2.0, 0.25), block_sound(mat_, rng.uniform(0.85, 1.1)), rng.uniform(0.15, 0.4))
    elif a["type"] == "fly":                                    # a whoosh through the air, a thud if it lands
        x = tt(max(0.2, a["t1"] - a["t0"]))
        whoosh = bp(rng.standard_normal(len(x)), 500, 3000) * (0.3 + 0.7 * np.sin(np.pi * x / x[-1]))
        place(stems["fx"], a["t0"], whoosh, 0.25)
        if a.get("land"):
            place(stems["fx"], a["t1"], thunk(0.6, 0.35), 0.9)
            place(stems["fx"], a["t1"], boom(0.7), 0.35)
    elif a["type"] == "slide" and a.get("sound") == "none":
        pass
    elif a["type"] == "slide" and a.get("sound") == "click":     # a lock pin: a small bright click as it seats
        place(stems["fx"], a["t1"] - 0.01, click(rng.uniform(0.9, 1.1)), 0.4)
        place(stems["fx"], a["t1"] - 0.01, thunk(1.6, 0.04), 0.12)
    elif a["type"] == "slide" and a.get("sound") == "clunk":     # a bolt or a jammed part: a solid metal clunk
        place(stems["fx"], a["t1"] - 0.02, thunk(0.9, 0.1), 0.6)
        place(stems["fx"], a["t1"] - 0.02, click(0.45), 0.25)
    elif a["type"] == "slide" and a.get("sound") == "chug":      # engine part: a mechanical knock at the end of its stroke
        place(stems["fx"], a["t1"] - 0.02, thunk(0.75, 0.12), 0.45)
        place(stems["fx"], a["t1"] - 0.02, click(0.5), 0.12)
    elif a["type"] == "slide" and P["structures"][a["target"]]["material"] == "wood":   # crate on a pulley: squeak + rope
        x = tt(a["t1"] - a["t0"] + 0.1)
        sp = np.array([C.smoother((k / SR) / max(1e-6, a["t1"] - a["t0"])) for k in range(len(x))])
        speed = np.abs(np.gradient(sp)); speed /= max(1e-9, speed.max())
        squeak = np.sin(2 * np.pi * np.cumsum(1050 + 180 * np.sin(2 * np.pi * 7 * x)) / SR) * (0.5 + 0.5 * np.sin(2 * np.pi * 3.1 * x) ** 2)
        rope = bp(rng.standard_normal(len(x)), 300, 2000) * 0.5
        place(stems["fx"], a["t0"], (squeak * 0.25 + rope) * speed, 0.45)
        place(stems["fx"], a["t1"], block_sound("wood", 0.7), 0.6)
    elif a["type"] == "slide":                                  # stone scrape that follows the slide's speed
        x = tt(a["t1"] - a["t0"] + 0.1)
        sp = np.array([C.smoother((k / SR) / max(1e-6, a["t1"] - a["t0"])) for k in range(len(x))])
        speed = np.abs(np.gradient(sp)); speed /= max(1e-9, speed.max())
        grind = bp(rng.standard_normal(len(x)), 150, 1400) * (0.6 + 0.4 * np.sign(np.sin(2 * np.pi * 23 * x)))
        place(stems["fx"], a["t0"], grind * speed, 0.5)
    elif a["type"] == "spin" and a.get("ticks_per_turn", 12):  # a ratchet tick every 1/ticks_per_turn of a turn (0 = silent)
        step_deg = 360.0 / a.get("ticks_per_turn", 12)
        last = 0
        for k in range(int((a["t1"] - a["t0"]) * 1000) + 1):
            tk = a["t0"] + k / 1000
            n = int(abs(C.motion_angle(a, tk)) // step_deg)
            if n > last:
                place(stems["fx"], tk, click(rng.uniform(0.55, 0.65)), 0.35)
                place(stems["fx"], tk, thunk(1.4, 0.06), 0.15)
                last = n
    elif a["type"] == "swing":                                  # a whoosh each time it sweeps through the bottom
        half = a.get("period", 1.6) / 2
        tk = a["t0"] + half
        while tk < a["t1"] - 0.2:
            x = tt(0.5)
            w = abs(C.motion_angle(a, tk - half / 2)) / max(1e-6, abs(a["degrees"]))
            place(stems["fx"], tk - 0.25, bp(rng.standard_normal(len(x)), 400, 2500) * np.hanning(len(x)) * w, 0.4)
            tk += half
W = P.get("world", {})


def ambience():
    """Background bed that matches the world: wind, rain, waves, crickets at night, birds / gulls calling."""
    wind = lp(rng.standard_normal(N), 700) * (0.6 + 0.4 * np.sin(2 * np.pi * t_all / 7.3))
    amb = wind * (0.35 if W.get("time") == "night" or W.get("weather") == "snow" else 0.6)
    if W.get("weather") == "rain":
        amb = amb * 0.5 + bp(rng.standard_normal(N), 900, 9000) * 0.5
        drops = np.zeros(N)
        for tk in np.sort(rng.uniform(0, N / SR, int(N / SR * 40))):
            j = int(tk * SR); x = bp(rng.standard_normal(240), 2000, 8000) * np.exp(-np.arange(240) / 30)
            drops[j:j + len(x)] += x[:N - j] * rng.uniform(0.2, 1.0)
        amb = amb + drops * 0.6
    if W.get("ground") == "paris":                                  # distant city murmur
        amb = amb * 0.7 + lp(bp(rng.standard_normal(N), 150, 900), 600) * (0.8 + 0.2 * np.sin(2 * np.pi * t_all / 11)) * 0.5
    if W.get("ground") == "beach":                                  # slow waves
        swell = 0.5 + 0.5 * np.sin(2 * np.pi * t_all / 6.5) ** 2
        amb = amb + lp(rng.standard_normal(N), 1500) * swell * 0.8
    if W.get("time") == "night" and W.get("weather") not in ("rain", "snow"):   # crickets: trains of short 4.6 kHz pulses
        tk = rng.uniform(0.2, 0.9)
        while tk < N / SR - 0.3:
            for q in range(3):
                x = tt(0.018); j = int((tk + q * 0.045) * SR)
                c = np.sin(2 * np.pi * 4600 * x) * np.hanning(len(x))
                amb[j:j + len(c)] += c[:N - j] * 0.25
            tk += rng.uniform(0.7, 1.4)
    for kind in W.get("animals", []):
        tk = rng.uniform(1.0, 3.0)
        while tk < N / SR - 1.0:
            if kind == "birds":                                     # two or three quick rising tweets
                for q in range(rng.integers(2, 4)):
                    x = tt(0.07); f = 3200 + 1600 * x / x[-1]
                    c = np.sin(2 * np.pi * np.cumsum(f) / SR) * np.hanning(len(x))
                    j = int((tk + q * 0.11) * SR); amb[j:j + len(c)] += c[:N - j] * 0.12
                tk += rng.uniform(3.0, 6.0)
            elif kind == "pigeons":                                 # soft "hoo-hoo-rrroo" coo
                for q, (f0, d) in enumerate(((420, 0.16), (440, 0.16), (380, 0.45))):
                    x = tt(d); f = f0 * (1 + 0.04 * np.sin(2 * np.pi * 28 * x))
                    c = np.sin(2 * np.pi * np.cumsum(f) / SR) * np.hanning(len(x))
                    j = int((tk + q * 0.22) * SR); amb[j:j + len(c)] += c[:N - j] * 0.10
                tk += rng.uniform(4.0, 8.0)
            elif kind == "seagulls":                                # falling "kee-ow"
                x = tt(0.45); f = 1300 - 600 * (x / x[-1]) ** 0.7
                ph = 2 * np.pi * np.cumsum(f) / SR
                c = (np.sin(ph) + 0.4 * np.sin(2 * ph) + 0.2 * np.sin(3 * ph)) * np.hanning(len(x))
                j = int(tk * SR); amb[j:j + len(c)] += c[:N - j] * 0.08
                tk += rng.uniform(5.0, 9.0)
            else:
                break
    return amb


AMB_W = {"rain": 0.22, "snow": 0.08}.get(W.get("weather"), 0.12)
stems["ambience"][:N] += ambience()

for k in stems:
    stems[k] = stems[k][:N]
    if NL:                                          # what plays past the cut becomes the opening, crossfading into it
        w = np.linspace(0.0, 1.0, NL)
        stems[k][:NL] = stems[k][:NL] * w + stems[k][N0:N0 + NL] * (1 - w)
    stems[k] = stems[k][:N0]
N = N0
t_all = t_all[:N0]
norm = lambda s: s / max(1e-9, np.abs(s).max())
bed = hp(0.8 * norm(stems["build"]) + 0.9 * norm(stems["fx"]) + AMB_W * norm(stems["ambience"]), 30)
_, v = wav.read(os.path.join(OUT, "voice.wav"))
v = v.astype(np.float64)[:N] / 32768
v = np.pad(v, (0, N - len(v)))
active = lp((np.abs(v) > 0.02).astype(float), 3)
duck = 1 - 0.5 * np.clip(active / max(1e-9, active.max()), 0, 1)
for ln in P["lines"]:
    a, z = int(ln["start"] * SR), int(ln["end"] * SR)
    need = 10 ** (9 / 20) * np.sqrt(np.mean((bed[a:z] * duck[a:z]) ** 2) + 1e-12) / np.sqrt(np.mean(v[a:z] ** 2))
    v[a:z] *= max(need, 1.0)
mix = bed * duck + v
stems["voice"] = v
mix = mix / np.abs(mix).max()
mix = mix * 0.5
pre = os.path.join(OUT, "mix_pre.wav")
wav.write(pre, SR, (np.stack([mix, mix], 1) * 32767).astype(np.int16))
def integrated_lufs(path):
    p = subprocess.run(["ffmpeg", "-hide_banner", "-i", path, "-af", "loudnorm=I=-14:TP=-1.5:LRA=11:print_format=json", "-f", "null", "-"],
                       capture_output=True, text=True)
    return float(json.loads(p.stderr[p.stderr.rindex("{"):p.stderr.rindex("}") + 1])["input_i"])


# -14 LUFS: add the missing gain, then a brick-wall peak limiter (which gives a little back), and repeat until within 0.3 dB
gain, out_wav = 0.0, os.path.join(OUT, "mix.wav")
lufs = integrated_lufs(pre)
for _ in range(4):
    gain += -14.0 - lufs
    af = "volume=%.2fdB,alimiter=limit=0.75:attack=2:release=60:level=false" % gain
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", pre, "-af", af, "-ar", "48000", out_wav], check=True)
    lufs = integrated_lufs(out_wav)
    if abs(lufs + 14.0) < 0.3:
        break
os.remove(pre)
for k, s in stems.items():
    wav.write(os.path.join(OUT, "stem_%s.wav" % k), SR, (np.clip(norm(s) * 0.7, -1, 1) * 32767).astype(np.int16))
print("audio ok: %.2f s, %d block sounds, %d footsteps" % (N / SR, sum(1 for b in P["blocks"] if not b["prebuilt"]), len(P["steps"])))

In [ ]:
%%writefile render.py
"""Render frames from blocklab.blend (resumable: existing frames are skipped).

    python3 render.py                 # full 1080x1920 -> frames/0000.png ...  (bpy module)
    blender -b -P render.py           # same, with a downloaded Blender build
    python3 render.py --draft         # 270x480 quick motion check -> draft/
    python3 render.py --draft --scale 50 --samples 8   # sharper preview, still in draft/
Env: GPU=1 uses OptiX/CUDA if available (falls back to CPU); FRAMES_DIR=/path overrides the output folder.
"""
import os
import sys
import time

import bpy

HERE = os.path.dirname(os.path.abspath(__file__))
draft = "--draft" in sys.argv
bpy.ops.wm.open_mainfile(filepath=os.path.join(HERE, "build", "blocklab.blend"))
sc = bpy.context.scene
out = os.environ.get("FRAMES_DIR", os.path.join(HERE, "build", "draft" if draft else "frames"))
os.makedirs(out, exist_ok=True)

if os.environ.get("GPU") == "1":
    prefs = bpy.context.preferences.addons["cycles"].preferences
    for kind in ("OPTIX", "CUDA"):
        try:
            prefs.compute_device_type = kind
        except TypeError:
            continue
        prefs.get_devices()
        devs = [d for d in prefs.devices if d.type == kind]
        if devs:
            for d in prefs.devices:
                d.use = d.type == kind
            sc.cycles.device = "GPU"
            sc.cycles.denoising_use_gpu = True
            print("Rendering on GPU (%s): %s" % (kind, ", ".join(d.name for d in devs)), flush=True)
            break
    else:
        print("No CUDA/OptiX GPU found -- rendering on CPU", flush=True)

if draft:
    sc.render.resolution_percentage = 25
    sc.cycles.samples = 4
    sc.cycles.use_denoising = False
if "--scale" in sys.argv:                          # e.g. --scale 50 --samples 8: a sharper preview than --draft
    sc.render.resolution_percentage = int(sys.argv[sys.argv.index("--scale") + 1])
if "--samples" in sys.argv:
    sc.cycles.samples = int(sys.argv[sys.argv.index("--samples") + 1])
    sc.cycles.use_denoising = True                 # a chosen sample count means "clean it up"


def black_fraction(path):
    """Share of pixels that are exactly black: a sign of a glitched render (good frames have ~none)."""
    import numpy as np
    img = bpy.data.images.load(path, check_existing=False)
    px = np.empty(len(img.pixels), dtype=np.float32)
    img.pixels.foreach_get(px)
    bpy.data.images.remove(img)
    return float((px.reshape(-1, 4)[:, :3].sum(axis=1) == 0).mean())


t0 = time.time()
done = 0
for f in range(sc.frame_start, sc.frame_end + 1):
    path = os.path.join(out, "%04d.png" % f)
    if os.path.exists(path):
        continue
    sc.frame_set(f)
    sc.render.filepath = path
    bpy.ops.render.render(write_still=True)
    for attempt in (1, 2):                         # re-render a glitched frame from scratch (caches dropped)
        bad = black_fraction(path)
        if bad < 0.001:
            break
        print("RETRY frame %d (%.2f%% pure black), attempt %d" % (f, 100 * bad, attempt), flush=True)
        keep = sc.render.use_persistent_data
        sc.render.use_persistent_data = False
        bpy.ops.render.render(write_still=True)
        sc.render.use_persistent_data = keep
    done += 1
    if done % 10 == 0:
        el = time.time() - t0
        print("RENDERED %d (frame %d) %.1fs/frame" % (done, f, el / done), flush=True)
print("RENDER DONE", done, "frames in %.0fs" % (time.time() - t0), flush=True)

In [ ]:
%%writefile bl_verify.py
"""Block Lab: automatic checks for the active story (build/) and its MP4. Writes build/verification.txt.

    blender -b -P bl_verify.py       |   python3 bl_verify.py (bpy module)     (needs scipy)
"""
import json
import math
import os
import subprocess
import sys

import bpy
import numpy as np
import scipy.io.wavfile as wav
from mathutils import Vector

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bl_common as C

P = C.load_plan()
FPS, NF = P["fps"], P["nf"]
out, ok_all = [], True


def check(name, ok, detail=""):
    global ok_all
    ok_all &= bool(ok)
    out.append("[%s] %s%s" % ("PASS" if ok else "FAIL", name, (" -- " + detail) if detail else ""))
    print(out[-1], flush=True)


acts = lambda k: [a for a in P["actions"] if a["type"] == k]
for name, st in P["structures"].items():
    n = sum(1 for b in P["blocks"] if b["struct"] == name)
    if st["expect"] is not None:
        check("%s has %d blocks" % (name, st["expect"]), n == st["expect"], "%d built" % n)
    cells = [tuple(b["cell"]) for b in P["blocks"] if b["struct"] == name]
    check("%s: every block in its own grid cell" % name, len(set(cells)) == len(cells))

bpy.ops.wm.open_mainfile(filepath=os.path.join(C.BUILD, "blocklab.blend"))
sc = bpy.context.scene
objs = [o for o in sc.objects if "settle_frame" in o]
vanish = {"%s_%d_%d_%d" % ((b["struct"],) + tuple(b["cell"])): b["vanish"] for b in P["blocks"] if b.get("vanish")}
collapse_at = {a["target"]: int(a["t0"] * FPS) for a in acts("collapse") + acts("melt")}
loc = {o.name: [] for o in objs}
wz = {o.name: [] for o in objs}
for f in range(NF):
    sc.frame_set(f)
    for o in objs:
        loc[o.name].append(o.location.copy())
        wz[o.name].append(o.matrix_world.translation.z)
moved = 0
for o in objs:
    s = o["settle_frame"]
    end = min(collapse_at.get(o["struct"], NF), vanish.get(o.name, NF)) - 2
    seg = loc[o.name][s:max(s + 1, end)]
    moved += max((p - seg[0]).length for p in seg) > 1e-6
check("blocks hold still once placed%s" % (" (until the collapse)" if collapse_at else ""), moved == 0, "%d moved" % moved)
for g in acts("grow"):
    r = bpy.data.objects["S_" + g["target"]]
    sc.frame_set(min(NF - 1, int(g["t1"] * FPS) + 1))
    want = 1.0                                                     # grows on one structure multiply in order
    for g2 in acts("grow"):
        if g2["target"] == g["target"] and g2["t1"] <= g["t1"]:
            want *= g2["factor"]
    nxt = [g2["t0"] for g2 in acts("grow") if g2["target"] == g["target"] and g2["t0"] > g["t1"]]
    if not nxt or nxt[0] * FPS > int(g["t1"] * FPS) + 1:
        check("%s grows x%g" % (g["target"], g["factor"]), abs(r.scale.x - want) < 1e-3, "scale %.3f (expected %.3f)" % (r.scale.x, want))
def root_at(name, t):
    sc.frame_set(max(0, min(NF - 1, int(round(t * FPS)))))
    return bpy.data.objects["S_" + name].matrix_world.copy()


def scripted(name, t):
    """Where the story says structure `name` should be at time t: grows, slides, spins and swings combined."""
    from mathutils import Matrix
    st = P["structures"][name]
    loc, s = Vector((st["at"][0], st["at"][1], 0)), 1.0
    for g in acts("grow"):
        if g["target"] == name:
            u = C.smoother((t - g["t0"]) / max(1e-6, g["t1"] - g["t0"]))
            if g.get("to"):
                loc = loc.lerp(Vector((g["to"][0], g["to"][1], 0)), u)
            s *= 1 + (g["factor"] - 1) * u
    loc_l = Vector()                                               # "local" slides turn with the structure
    for a in acts("slide"):
        if a["target"] == name:
            d = Vector(a["by"]) * C.smoother((t - a["t0"]) / max(1e-6, a["t1"] - a["t0"]))
            if a.get("local"):
                loc_l = loc_l + d
            else:
                loc = loc + d
    for a in acts("fly"):
        if a["target"] == name:
            loc = loc + Vector(C.fly_offset(a, t))
    R = Matrix.Identity(4)
    for a in acts("spin") + acts("swing"):
        if a["target"] == name:
            if a.get("pivot") is not None:
                p = Vector(C.local_pos(a["pivot"], st["cells"], st["centered"]))
            else:
                p = sum((Vector(C.local_pos(c, st["cells"], st["centered"])) for c in st["cells"]), Vector()) / len(st["cells"])
            axis = {"x": "X", "y": "Y", "z": "Z"}[a.get("axis", "z")]
            R = R @ Matrix.Translation(p) @ Matrix.Rotation(math.radians(C.motion_angle(a, t)), 4, axis) @ Matrix.Translation(-p)
    return Matrix.Translation(loc) @ Matrix.Scale(s, 4) @ R @ Matrix.Translation(loc_l)


slide_err = {}
for a in acts("slide"):                                           # each slide follows the script at its start, middle and end
    worst = 0.0
    for tt_ in (a["t0"], (a["t0"] + a["t1"]) / 2, a["t1"]):
        f = max(0, min(NF - 1, int(round(tt_ * FPS))))
        sc.frame_set(f)
        got = bpy.data.objects["S_" + a["target"]].matrix_world
        want = scripted(a["target"], f / FPS)
        worst = max(worst, (got.translation - want.translation).length, max(abs(got[i][j] - want[i][j]) for i in range(3) for j in range(3)))
    slide_err.setdefault(a["target"], []).append(worst)
for name, errs in slide_err.items():
    check("%s slides as scripted (%d moves)" % (name, len(errs)), max(errs) < 2e-3, "worst difference %.4f" % max(errs))
move_err = {}
for a in acts("fly"):                                              # a thrown object follows its arc
    worst = 0.0
    for k in range(5):
        f = max(0, min(NF - 1, int(round((a["t0"] + (a["t1"] - a["t0"]) * k / 4) * FPS))))
        sc.frame_set(f)
        worst = max(worst, (bpy.data.objects["S_" + a["target"]].matrix_world.translation - scripted(a["target"], f / FPS).translation).length)
    check("%s flies on its arc" % a["target"], worst < 2e-3, "worst difference %.4f" % worst)
for a in acts("spin") + acts("swing"):                            # every turn follows the script at its start, middle and end
    worst = 0.0
    for tt_ in (a["t0"], (a["t0"] + a["t1"]) / 2, a["t1"]):
        f = max(0, min(NF - 1, int(round(tt_ * FPS))))
        sc.frame_set(f)
        got = bpy.data.objects["S_" + a["target"]].matrix_world
        want = scripted(a["target"], f / FPS)
        worst = max(worst, (got.translation - want.translation).length, max(abs(got[i][j] - want[i][j]) for i in range(3) for j in range(3)))
    move_err.setdefault((a["target"], a["type"]), []).append((worst, a["degrees"]))
for (name, kind), errs in move_err.items():
    total = sum(d for _, d in errs) if kind == "spin" else max(abs(d) for _, d in errs)
    check("%s %s as scripted (%d %s, %g deg %s)" % (name, "spins" if kind == "spin" else "swings", len(errs), "turns" if kind == "spin" else "swings",
          total, "in all" if kind == "spin" else "peak"), max(e for e, _ in errs) < 2e-3, "worst difference %.4f" % max(e for e, _ in errs))
Wd = P.get("world", {})
if Wd:
    def moves(obj_name, attr):
        o = bpy.data.objects.get(obj_name)
        if o is None:
            return False
        vals = []
        for f in (0, min(NF - 1, 15)):
            sc.frame_set(f)
            vals.append(tuple(getattr(o, attr)))
        return vals[0] != vals[1]
    need = ["Island"] + (["Stars"] if Wd["time"] == "night" and Wd["weather"] not in ("rain", "snow") else [])
    missing = [n for n in need if n not in bpy.data.objects]
    check("world built: %s ground, %s, %s" % (Wd["ground"], Wd["time"], Wd["weather"]), not missing, "missing %s" % missing if missing else "")
    if Wd["weather"] in ("rain", "snow"):
        check("%s keeps falling" % Wd["weather"], moves(Wd["weather"].capitalize(), "location"))
    for a in Wd["animals"]:
        if a in ("birds", "seagulls", "pigeons"):
            check("%s fly" % a, moves("BirdPivot_0", "rotation_euler"))
        elif a == "cat":
            cat = bpy.data.objects.get("Cat")
            gap = min((Vector((cat.location.x, cat.location.y)) - Vector((P["structures"][b["struct"]]["at"][0] + b["pos"][0],
                       P["structures"][b["struct"]]["at"][1] + b["pos"][1]))).length for b in P["blocks"]) if cat else 0
            check("cat sits clear of the build", cat is not None and gap > 0.3, "%.2f m from the nearest block" % gap)
for a in acts("melt"):                                            # every block lands exactly on its own slab cell
    st, tgt = P["structures"][a["target"]], P["melt"][a["target"]]
    sc.frame_set(NF - 1)
    off = 0.0
    ids = {tuple(b["cell"]): b["id"] for b in P["blocks"] if b["struct"] == a["target"]}
    for o in objs:
        if o["struct"] == a["target"]:
            cell = tuple(int(v) for v in o.name.split("_")[-3:])
            want = Vector(C.local_pos(tgt[str(ids[cell])], st["cells"], st["centered"]))
            off = max(off, (o.location - want).length)
    layers = 1 + max(c[2] for c in tgt.values())
    cells_used = {tuple(c) for c in tgt.values()}
    check("%s melts into a flat slab" % a["target"], off < 2e-3 and len(cells_used) == len(tgt),
          "%d blocks in %d layer(s), worst landing error %.4f m" % (len(tgt), layers, off))
for a in acts("remove"):                                          # taken apart: every block gone by the end
    sc.frame_set(NF - 1)
    left = sum(1 for o in objs if o["struct"] == a["target"] and o.location.z > -1)
    check("%s is taken down" % a["target"], left == 0, "%d blocks left" % left)
if any(abs(p[2]) > 0.01 for p in P["path"]):                      # grandpa never hangs in the air
    float_frames = 0
    for f in range(0, NF, 5):
        x, y, z = P["path"][f]
        vz = abs(P["path"][min(NF - 1, f + 1)][2] - P["path"][max(0, f - 1)][2]) * FPS / 2
        if z < 0.02 or vz > 0.05:
            continue                                               # on the ground, or on the ladder
        sc.frame_set(f)
        under = [o for o in objs if o.location.z > -1 and abs(o.matrix_world.translation.x - x) < 0.16
                 and abs(o.matrix_world.translation.y - y) < 0.16 and abs(o.matrix_world.translation.z + C.B / 2 - z) < 0.06]
        float_frames += not under
    check("grandpa always stands on something when up high", float_frames == 0, "%d sampled frames floating" % float_frames)
for name, fc in collapse_at.items():
    low = min(z for o in objs if o["struct"] == name for z in wz[o.name][fc:])
    check("%s's rubble stays on the ground" % name, low > -0.01, "lowest block centre %.3f m" % low)

arm = bpy.data.objects["Grandpa"]
from bpy_extras.object_utils import world_to_camera_view
out_frames, worst = [], 1.0
for f in range(NF):                                                # grandpa never leaves the picture: hat to boots, every frame
    sc.frame_set(f)
    head = arm.matrix_world @ arm.pose.bones["head"].tail
    for p in (head + Vector((0, 0, 0.1)), arm.matrix_world.translation, (head + arm.matrix_world.translation) / 2):
        v = world_to_camera_view(sc, sc.camera, p)
        m = min(v.x, 1 - v.x, v.y, 1 - v.y) if v.z > 0 else -1.0
        worst = min(worst, m)
        if m < 0.03:
            out_frames.append(f)
            break
check("grandpa stays in the picture every frame (hat to boots)", not out_frames,
      "%d frames out%s; closest to the edge %.1f%% of the frame" % (len(out_frames), (" from frame %d" % out_frames[0]) if out_frames else "", worst * 100))
jr = (arm.data.bones["head"].matrix_local.inverted() @ arm.data.bones["jaw"].matrix_local).translation
jaw = []
for f in range(NF):
    sc.frame_set(f)
    pb = arm.pose.bones
    jaw.append(((pb["head"].matrix.inverted() @ pb["jaw"].matrix).translation - jr).length)
spoken = set()
for ln in P["lines"]:
    fs = range(int(ln["start"] * FPS), int(ln["end"] * FPS) + 1)
    spoken |= set(range(fs.start - 2, fs.stop + 3))
    check("mouth moves on: \"%s\"" % ln["text"][:44], max(jaw[f] for f in fs) > 0.02)
check("mouth closed between lines", max(jaw[f] for f in range(NF) if f not in spoken) < 1e-4)

sr, x = wav.read(os.path.join(C.BUILD, "audio", "stem_build.wav"))
x = x.astype(float)
e = np.convolve(x ** 2, np.ones(48) / 48, mode="same")
d = np.diff(e, prepend=0)
offs = []
for b in P["blocks"]:
    if b["prebuilt"]:
        continue
    ts = b["settle"] / FPS
    j0, j1 = int((ts - 0.015) * sr), int((ts + 0.015) * sr)
    offs.append(abs((j0 + np.argmax(d[j0:j1])) / sr - ts) * 1000)
if offs:
    check("every block click lands on its settle frame", max(offs) < 16.7, "median %.1f ms, max %.1f ms" % (np.median(offs), max(offs)))

mp4 = os.path.join(C.HERE, P["slug"] + ".mp4")
if P.get("loop") and os.path.exists(mp4):                         # a looping Short: the last frame flows into the first
    def grab(n):
        raw = subprocess.run(["ffmpeg", "-v", "error", "-i", mp4, "-vf", "select=eq(n\\,%d),scale=108:192" % n, "-frames:v", "1",
                              "-f", "rawvideo", "-pix_fmt", "gray", "-"], capture_output=True).stdout
        return np.frombuffer(raw, np.uint8).astype(float)
    first, last = grab(0), grab(NF - 1)
    d = float(np.abs(first - last).mean()) if len(first) == len(last) > 0 else 999.0
    check("loops seamlessly (last frame matches the first)", d < 4.0, "mean difference %.2f of 255" % d)
    _, mx = wav.read(os.path.join(C.BUILD, "audio", "mix.wav"))
    mx = mx.astype(float)[:, 0] / 32768
    jump, step = abs(mx[0] - mx[-1]), float(np.abs(np.diff(mx)).mean())
    check("sound flows across the loop (no click)", jump < 4 * step + 1e-4, "jump %.4f vs typical step %.4f" % (jump, step))
if os.path.exists(mp4):
    pr = json.loads(subprocess.run(["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", mp4],
                                   capture_output=True, text=True).stdout)
    v = [s for s in pr["streams"] if s["codec_type"] == "video"][0]
    a = [s for s in pr["streams"] if s["codec_type"] == "audio"]
    check("MP4 %dx%d H.264 yuv420p 30 fps" % (P["w"], P["h"]), (v["width"], v["height"], v["codec_name"], v["pix_fmt"], v["r_frame_rate"]) ==
          (P["w"], P["h"], "h264", "yuv420p", "30/1"))
    check("MP4 AAC audio, full length", bool(a) and abs(float(pr["format"]["duration"]) - P["film"]) < 0.1 and int(v.get("nb_frames", 0)) == NF,
          "%.2f s, %s frames" % (float(pr["format"]["duration"]), v.get("nb_frames")))
    lo = subprocess.run(["ffmpeg", "-hide_banner", "-i", mp4, "-af", "ebur128=peak=true", "-f", "null", "-"], capture_output=True, text=True).stderr
    I = float(lo.split("I:")[-1].split("LUFS")[0]); pk = float(lo.split("Peak:")[-1].split("dBFS")[0])
    check("loudness about -14 LUFS, no clipping", abs(I + 14) < 1 and pk < -1.0, "%.1f LUFS, peak %.1f dBFS" % (I, pk))
else:
    check("final MP4 exists", False, mp4)
out.append("\nOVERALL: %s" % ("ALL CHECKS PASSED" if ok_all else "SOME CHECKS FAILED"))
open(os.path.join(C.BUILD, "verification.txt"), "w").write("\n".join(out) + "\n")
print(out[-1])

In [ ]:
%%writefile stories/giants.json
{
  "title": "Why giants can't exist",
  "slug": "giants",
  "voice": "am_santa",
  "speed": 0.95,
  "start": 0.3,
  "tail": 1.6,
  "channel_tag": "GRANDPA'S BLOCK LAB",

  "lines": [
    {"id": "hook",   "text": "Make something twice as big... and it gets weaker. Watch.", "pause": 0.5},
    {"id": "small",  "text": "This little fella? Nine blocks.", "pause": 0.45},
    {"id": "big",    "text": "Now twice as tall, twice as wide, twice as deep. Seventy-two blocks. Eight times the weight!", "pause": 0.5},
    {"id": "legs",   "text": "But his legs only got four times thicker. So every leg carries twice the load.", "pause": 0.6},
    {"id": "giant",  "text": "Make him ten times bigger... and... crack.", "pause": 1.8},
    {"id": "payoff", "text": "That's why elephants need thick legs. And why real giants can't exist.", "pause": 0.0}
  ],

  "grandpa": {"start": [-0.28, 0.36]},

  "structures": {
    "small": {"cells": [[0,0,0],[2,0,0],[0,0,1],[1,0,1],[2,0,1],[0,0,2],[1,0,2],[2,0,2],[1,0,3]],
              "at": [-0.40, 0.0], "material": "sandstone", "eyes": true, "stand": "behind",
              "parts": {"legs": "z < 1"}, "expect_blocks": 9},
    "big":   {"copy_of": "small", "scale": 2, "at": [0.38, 0.0], "material": "sandstone", "eyes": true, "stand": "right",
              "parts": {"legs": "z < 2"}, "expect_blocks": 72}
  },

  "actions": [
    {"type": "face_camera", "when": 0.0, "until": ["hook", 0.15]},
    {"type": "point", "pos": [-0.40, 0.0, 0.05], "when": ["hook", 0.15], "until": ["hook", "end", -0.3]},
    {"type": "build", "target": "small", "order": "z", "when": ["small", "start", -0.25], "until": ["small", "start", 1.95]},
    {"type": "build", "target": "big", "order": "z", "when": ["big", "start", 0.2], "until": ["big", "end", -1.6]},
    {"type": "highlight", "target": "small", "part": "legs", "when": ["legs", "start", 0.4], "until": ["legs", "end", 0.4]},
    {"type": "highlight", "target": "big", "part": "legs", "when": ["legs", "start", 0.4], "until": ["legs", "end", 0.4]},
    {"type": "point", "target": "big", "offset": [0.2, -0.06, 0.14], "when": ["legs", "start", 0.3], "until": ["legs", "end", -0.2]},
    {"type": "move", "to": [0.83, -0.45], "when": ["giant", "start", 0.1], "until": ["giant", "start", 0.7]},
    {"type": "move", "to": [-0.85, -0.45], "when": ["giant", "start", 0.7], "until": ["giant", "start", 1.9]},
    {"type": "grow", "target": "big", "factor": 5, "to": [0.25, 1.45], "when": ["giant", "start", 0.3], "until": ["giant", "start", 1.9]},
    {"type": "crack", "target": "big", "part": "legs", "when": ["giant", "start", 2.1], "until": ["giant", "end"]},
    {"type": "collapse", "target": "big", "when": ["giant", "end", 0.05]},
    {"type": "shrug", "when": ["payoff", "start", 0.2], "until": ["payoff", "start", 1.3]},
    {"type": "face_camera", "when": ["payoff", "start"], "until": "end"}
  ],

  "camera": [
    {"at": 0.0,                        "target": [-0.38, 0.12, 0.42], "az": -62, "el": 15, "dist": 1.9},
    {"at": ["hook", "end"],            "target": [-0.36, 0.12, 0.40], "az": -62, "el": 16, "dist": 2.0},
    {"at": ["small", "end", 0.3],      "target": [-0.02, 0.15, 0.42], "az": -60, "el": 21, "dist": 2.9},
    {"at": ["big", "end"],             "target": [0.0, 0.15, 0.42],   "az": -58, "el": 21, "dist": 2.75},
    {"at": ["legs", "start", 0.6],     "target": [0.15, 0.08, 0.30],  "az": -56, "el": 12, "dist": 2.5},
    {"at": ["legs", "end"],            "target": [0.17, 0.08, 0.30],  "az": -56, "el": 12, "dist": 2.45},
    {"at": ["giant", "start", 0.5],    "target": [0.0, 0.05, 0.30],   "az": -60, "el": 12, "dist": 2.6},
    {"at": ["giant", "start", 1.9],    "target": [-0.1, 0.85, 1.75],  "az": -62, "el": 9,  "dist": 7.6},
    {"at": ["giant", "end", 1.25],     "target": [-0.1, 0.85, 1.35],  "az": -62, "el": 10, "dist": 7.4},
    {"at": ["payoff", "start", 0.5],   "target": [-0.6, 0.1, 0.75],   "az": -100, "el": 12, "dist": 3.6},
    {"at": "end",                      "target": [-0.66, 0.0, 0.78],  "az": -100, "el": 12, "dist": 3.2}
  ],

  "cards": [
    {"style": "title", "text": "TWICE AS BIG", "color": "white", "y": 470, "when": ["hook", "start", -0.05], "until": ["hook", "end", 0.2]},
    {"style": "title", "text": "= WEAKER?", "color": "amber", "y": 590, "size": 110, "when": 1.3, "until": ["hook", "end", 0.2]},
    {"style": "bubble", "text": "9 BLOCKS", "anchor": "small:top", "color": "amber", "when": ["small", "start", 1.6], "until": ["big", "start", 0.8]},
    {"style": "bubble", "text": "{count:big} BLOCKS", "anchor": "big:top", "color": "cyan", "when": ["big", "start", 0.2], "until": ["legs", "start", 0.3]},
    {"style": "title", "text": "2× SIZE", "color": "cyan", "y": 420, "size": 92, "when": ["big", "start"], "until": ["big", 0.62]},
    {"style": "title", "text": "72 ÷ 9 = ×8 WEIGHT", "color": "amber", "y": 420, "size": 76, "when": ["big", 0.78], "until": ["legs", "start", 0.2]},
    {"style": "tag", "text": "AREA 1", "anchor": "small:part:legs", "color": "cyan", "dy": 70, "when": ["legs", "start", 0.3], "until": ["legs", "end", 0.4]},
    {"style": "tag", "text": "AREA 4", "anchor": "big:part:legs", "color": "cyan", "dy": 70, "when": ["legs", "start", 0.3], "until": ["legs", "end", 0.4]},
    {"style": "panel", "y": 300, "border": "cyan", "when": ["legs", "start", 0.3], "until": ["legs", "end", 0.4], "rows": [
      {"text": "LEG AREA   ×4", "color": "cyan", "at": ["legs", 0.05]},
      {"text": "WEIGHT   ×8", "color": "amber", "at": ["legs", 0.45]},
      {"text": "LOAD PER LEG   8 ÷ 4 = ×2", "color": "red", "at": ["legs", 0.62]}]},
    {"style": "title", "text": "10× BIGGER", "color": "white", "y": 430, "size": 104, "when": ["giant", "start", 0.3], "until": ["giant", "start", 2.2]},
    {"style": "panel", "y": 300, "border": "red", "when": ["giant", "start", 2.1], "until": ["giant", "end", 0.55], "rows": [
      {"text": "WEIGHT   ×1000", "color": "amber", "at": ["giant", "start", 2.1]},
      {"text": "LEG AREA   ×100", "color": "cyan", "at": ["giant", "start", 2.45]},
      {"text": "LOAD PER LEG   ×10", "color": "red", "at": ["giant", "start", 2.8]}]},
    {"style": "impact", "text": "CRACK!", "color": "red", "y": 640, "when": ["giant", "end"], "until": ["giant", "end", 1.05]},
    {"style": "panel", "y": 300, "border": "amber", "when": ["payoff", "start", 0.2], "until": ["payoff", "end", 0.4], "rows": [
      {"text": "THE SQUARE–CUBE LAW", "color": "amber", "at": ["payoff", "start", 0.2], "size": 52},
      {"text": "size ×2  →  strength ×4", "color": "cyan", "at": ["payoff", "start", 0.2], "size": 42},
      {"text": "weight ×8", "color": "red", "at": ["payoff", "start", 0.2], "size": 42}]},
    {"style": "end", "title1": "WHY GIANTS", "title2": "CAN'T EXIST", "sub": "J.B.S. Haldane, \"On Being the Right Size\" (1926)",
     "cta": "Follow for more Block Lab", "when": ["payoff", "end", 0.15]}
  ]
}

In [ ]:
%%writefile stories/odd_squares.json
{
  "title": "Odd numbers make squares",
  "slug": "odd_squares",
  "voice": "am_santa",
  "speed": 0.95,
  "channel_tag": "GRANDPA'S BLOCK LAB",

  "lines": [
    {"id": "hook",    "text": "Want a magic trick? Add up odd numbers... and you always get a perfect square.", "pause": 0.5},
    {"id": "one",     "text": "One block. One by one.", "pause": 0.4},
    {"id": "three",   "text": "Add three, around the corner. Two by two. That's four.", "pause": 0.4},
    {"id": "five",    "text": "Add five. Three by three. Nine.", "pause": 0.4},
    {"id": "more",    "text": "Seven makes sixteen. Nine makes twenty-five. A perfect square, every single time!", "pause": 0.6},
    {"id": "payoff",  "text": "So one plus three, all the way to nineteen? That's ten odd numbers. Ten by ten. One hundred!", "pause": 0.0}
  ],

  "grandpa": {"start": [-0.45, 0.55]},

  "structures": {
    "odd1":    {"gnomon": 0, "at": [-0.75, -0.25], "centered": false, "material": "gold",  "stand": "left", "expect_blocks": 1},
    "odd3":    {"gnomon": 1, "at": [-0.75, -0.25], "centered": false, "material": "blue",  "stand": "left", "expect_blocks": 3},
    "odd5":    {"gnomon": 2, "at": [-0.75, -0.25], "centered": false, "material": "grass", "stand": "left", "expect_blocks": 5},
    "odd7":    {"gnomon": 3, "at": [-0.75, -0.25], "centered": false, "material": "brick", "stand": "left", "expect_blocks": 7},
    "odd9":    {"gnomon": 4, "at": [-0.75, -0.25], "centered": false, "material": "wood",  "stand": "left", "expect_blocks": 9},
    "hundred": {"box": [10, 10, 1], "at": [0.05, -0.55], "centered": false, "material": "stone", "stand": "behind", "expect_blocks": 100}
  },

  "actions": [
    {"type": "face_camera", "when": 0.0, "until": ["hook", 0.5]},
    {"type": "build", "target": "odd1", "when": ["one", "start", -0.1], "until": ["one", 0.35]},
    {"type": "build", "target": "odd3", "order": "x", "when": ["three", 0.05], "until": ["three", 0.45]},
    {"type": "build", "target": "odd5", "order": "x", "when": ["five", 0.0], "until": ["five", 0.45]},
    {"type": "build", "target": "odd7", "order": "x", "when": ["more", 0.0], "until": ["more", 0.24]},
    {"type": "build", "target": "odd9", "order": "x", "when": ["more", 0.27], "until": ["more", 0.52]},
    {"type": "build", "target": "hundred", "order": "y", "when": ["payoff", 0.45], "until": ["payoff", "end", -0.2]},
    {"type": "face_camera", "when": ["payoff", "end", 0.1], "until": "end"}
  ],

  "camera": [
    {"at": 0.0,                     "target": [-0.45, 0.1, 0.35], "az": -62, "el": 18, "dist": 1.9},
    {"at": ["hook", "end"],         "target": [-0.62, 0.02, 0.16], "az": -62, "el": 40, "dist": 2.25},
    {"at": ["more", "end"],         "target": [-0.62, 0.02, 0.16], "az": -60, "el": 42, "dist": 2.3},
    {"at": ["payoff", 0.45],        "target": [0.05, 0.0, 0.1],   "az": -60, "el": 52, "dist": 3.3},
    {"at": "end",                   "target": [0.05, 0.0, 0.1],   "az": -64, "el": 52, "dist": 3.1}
  ],

  "cards": [
    {"style": "title", "text": "ODD NUMBERS", "color": "white", "y": 470, "when": ["hook", "start"], "until": ["hook", "end", 0.2]},
    {"style": "title", "text": "= SQUARES?", "color": "amber", "y": 590, "size": 110, "when": ["hook", 0.45], "until": ["hook", "end", 0.2]},
    {"style": "title", "text": "1 = 1×1", "color": "amber", "y": 420, "size": 90, "when": ["one", 0.4], "until": ["three", "start", -0.1]},
    {"style": "title", "text": "1+3 = 4 = 2×2", "color": "cyan", "y": 420, "size": 84, "when": ["three", 0.5], "until": ["five", "start", -0.1]},
    {"style": "title", "text": "1+3+5 = 9 = 3×3", "color": "green", "y": 420, "size": 80, "when": ["five", 0.5], "until": ["more", "start", -0.1]},
    {"style": "title", "text": "… + 7 = 16 = 4×4", "color": "red", "y": 420, "size": 78, "when": ["more", 0.26], "until": ["more", 0.53]},
    {"style": "title", "text": "… + 9 = 25 = 5×5", "color": "amber", "y": 420, "size": 78, "when": ["more", 0.54], "until": ["more", "end", 0.4]},
    {"style": "bubble", "text": "{count:hundred} BLOCKS", "anchor": "hundred:top", "color": "cyan", "when": ["payoff", 0.45], "until": ["payoff", "end", 0.5]},
    {"style": "panel", "y": 300, "border": "amber", "when": ["payoff", "start"], "until": ["payoff", "end", 0.5], "rows": [
      {"text": "1+3+5+…+19", "color": "white", "at": ["payoff", 0.05], "size": 54},
      {"text": "10 odd numbers", "color": "cyan", "at": ["payoff", 0.48], "size": 44},
      {"text": "= 10×10 = 100", "color": "amber", "at": ["payoff", 0.8], "size": 54}]},
    {"style": "end", "title1": "ODD NUMBERS", "title2": "MAKE SQUARES", "sub": "The first n odd numbers always add up to n × n",
     "cta": "Follow for more Block Lab", "when": ["payoff", "end", 0.2]}
  ]
}

In [ ]:
%%writefile stories/stone_arch.json
{
 "title": "Why stone arches stand",
 "slug": "stone_arch",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "lines": [
  {
   "id": "hook",
   "text": "Roman bridges have stood for two thousand years... with no glue at all. How?",
   "pause": 0.5
  },
  {
   "id": "flat",
   "text": "Lay stone flat across a gap, and the middle sags. Its bottom gets stretched...",
   "pause": 0.15
  },
  {
   "id": "crack",
   "text": "and stone hates being stretched!",
   "pause": 1.0
  },
  {
   "id": "arch",
   "text": "Now curve the stones into an arch. Each one leans on the next.",
   "pause": 0.3
  },
  {
   "id": "key",
   "text": "The keystone at the top locks them all together.",
   "pause": 0.3
  },
  {
   "id": "squeeze",
   "text": "Now every stone gets squeezed instead. And stone loves being squeezed.",
   "pause": 0.4
  },
  {
   "id": "payoff",
   "text": "Pile on more weight? It only squeezes tighter. That's why Roman arches are still standing!",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   0.0,
   0.2
  ],
  "outfit": {
   "hat": "hardhat",
   "extras": [
    "vest"
   ]
  }
 },
 "structures": {
  "flat": {
   "cells": [
    [
     0,
     0,
     0
    ],
    [
     1,
     0,
     0
    ],
    [
     9,
     0,
     0
    ],
    [
     10,
     0,
     0
    ],
    [
     0,
     1,
     0
    ],
    [
     1,
     1,
     0
    ],
    [
     9,
     1,
     0
    ],
    [
     10,
     1,
     0
    ],
    [
     0,
     0,
     1
    ],
    [
     1,
     0,
     1
    ],
    [
     9,
     0,
     1
    ],
    [
     10,
     0,
     1
    ],
    [
     0,
     1,
     1
    ],
    [
     1,
     1,
     1
    ],
    [
     9,
     1,
     1
    ],
    [
     10,
     1,
     1
    ],
    [
     0,
     0,
     2
    ],
    [
     1,
     0,
     2
    ],
    [
     9,
     0,
     2
    ],
    [
     10,
     0,
     2
    ],
    [
     0,
     1,
     2
    ],
    [
     1,
     1,
     2
    ],
    [
     9,
     1,
     2
    ],
    [
     10,
     1,
     2
    ],
    [
     0,
     0,
     3
    ],
    [
     1,
     0,
     3
    ],
    [
     9,
     0,
     3
    ],
    [
     10,
     0,
     3
    ],
    [
     0,
     1,
     3
    ],
    [
     1,
     1,
     3
    ],
    [
     9,
     1,
     3
    ],
    [
     10,
     1,
     3
    ],
    [
     0,
     0,
     4
    ],
    [
     1,
     0,
     4
    ],
    [
     9,
     0,
     4
    ],
    [
     10,
     0,
     4
    ],
    [
     0,
     1,
     4
    ],
    [
     1,
     1,
     4
    ],
    [
     9,
     1,
     4
    ],
    [
     10,
     1,
     4
    ],
    [
     0,
     0,
     5
    ],
    [
     1,
     0,
     5
    ],
    [
     9,
     0,
     5
    ],
    [
     10,
     0,
     5
    ],
    [
     0,
     1,
     5
    ],
    [
     1,
     1,
     5
    ],
    [
     9,
     1,
     5
    ],
    [
     10,
     1,
     5
    ],
    [
     0,
     0,
     6
    ],
    [
     1,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     6,
     0,
     6
    ],
    [
     7,
     0,
     6
    ],
    [
     8,
     0,
     6
    ],
    [
     9,
     0,
     6
    ],
    [
     10,
     0,
     6
    ],
    [
     0,
     1,
     6
    ],
    [
     1,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     6,
     1,
     6
    ],
    [
     7,
     1,
     6
    ],
    [
     8,
     1,
     6
    ],
    [
     9,
     1,
     6
    ],
    [
     10,
     1,
     6
    ]
   ],
   "at": [
    -0.88,
    0.0
   ],
   "material": "sandstone",
   "stand": "right",
   "parts": {
    "mid": "z == 6 and 3 <= x <= 7"
   },
   "expect_blocks": 70
  },
  "arch": {
   "cells": [
    [
     3,
     0,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     6,
     0,
     6
    ],
    [
     6,
     1,
     6
    ],
    [
     7,
     0,
     6
    ],
    [
     7,
     1,
     6
    ],
    [
     2,
     0,
     5
    ],
    [
     2,
     1,
     5
    ],
    [
     3,
     0,
     5
    ],
    [
     3,
     1,
     5
    ],
    [
     7,
     0,
     5
    ],
    [
     7,
     1,
     5
    ],
    [
     8,
     0,
     5
    ],
    [
     8,
     1,
     5
    ],
    [
     1,
     0,
     4
    ],
    [
     1,
     1,
     4
    ],
    [
     2,
     0,
     4
    ],
    [
     2,
     1,
     4
    ],
    [
     8,
     0,
     4
    ],
    [
     8,
     1,
     4
    ],
    [
     9,
     0,
     4
    ],
    [
     9,
     1,
     4
    ],
    [
     0,
     0,
     3
    ],
    [
     0,
     1,
     3
    ],
    [
     1,
     0,
     3
    ],
    [
     1,
     1,
     3
    ],
    [
     9,
     0,
     3
    ],
    [
     9,
     1,
     3
    ],
    [
     10,
     0,
     3
    ],
    [
     10,
     1,
     3
    ],
    [
     0,
     0,
     2
    ],
    [
     0,
     1,
     2
    ],
    [
     1,
     0,
     2
    ],
    [
     1,
     1,
     2
    ],
    [
     9,
     0,
     2
    ],
    [
     9,
     1,
     2
    ],
    [
     10,
     0,
     2
    ],
    [
     10,
     1,
     2
    ],
    [
     0,
     0,
     1
    ],
    [
     0,
     1,
     1
    ],
    [
     1,
     0,
     1
    ],
    [
     1,
     1,
     1
    ],
    [
     9,
     0,
     1
    ],
    [
     9,
     1,
     1
    ],
    [
     10,
     0,
     1
    ],
    [
     10,
     1,
     1
    ],
    [
     0,
     0,
     0
    ],
    [
     0,
     1,
     0
    ],
    [
     1,
     0,
     0
    ],
    [
     1,
     1,
     0
    ],
    [
     9,
     0,
     0
    ],
    [
     9,
     1,
     0
    ],
    [
     10,
     0,
     0
    ],
    [
     10,
     1,
     0
    ]
   ],
   "at": [
    0.32,
    -0.05
   ],
   "centered": false,
   "material": "stone",
   "stand": "left",
   "expect_blocks": 56
  },
  "keystone": {
   "cells": [
    [
     5,
     0,
     6
    ],
    [
     5,
     1,
     6
    ]
   ],
   "at": [
    0.32,
    -0.05
   ],
   "centered": false,
   "material": "gold",
   "stand": "right",
   "parts": {
    "all": "z >= 0"
   },
   "expect_blocks": 2
  },
  "load": {
   "cells": [
    [
     3,
     0,
     7
    ],
    [
     4,
     0,
     7
    ],
    [
     5,
     0,
     7
    ],
    [
     6,
     0,
     7
    ],
    [
     7,
     0,
     7
    ],
    [
     3,
     1,
     7
    ],
    [
     4,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     6,
     1,
     7
    ],
    [
     7,
     1,
     7
    ],
    [
     3,
     0,
     8
    ],
    [
     4,
     0,
     8
    ],
    [
     5,
     0,
     8
    ],
    [
     6,
     0,
     8
    ],
    [
     7,
     0,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     5,
     1,
     8
    ],
    [
     6,
     1,
     8
    ],
    [
     7,
     1,
     8
    ]
   ],
   "at": [
    0.32,
    -0.05
   ],
   "centered": false,
   "material": "brick",
   "stand": "right",
   "expect_blocks": 20
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": [
    "hook",
    "end"
   ]
  },
  {
   "type": "build",
   "target": "flat",
   "order": "z",
   "when": [
    "hook",
    0.15
   ],
   "until": [
    "flat",
    0.25
   ]
  },
  {
   "type": "point",
   "target": "flat",
   "offset": [
    0,
    0,
    0.62
   ],
   "when": [
    "flat",
    0.3
   ],
   "until": [
    "crack",
    "end"
   ]
  },
  {
   "type": "crack",
   "target": "flat",
   "part": "mid",
   "when": [
    "flat",
    0.55
   ],
   "until": [
    "crack",
    0.7
   ]
  },
  {
   "type": "collapse",
   "target": "flat",
   "when": [
    "crack",
    "end",
    0.05
   ]
  },
  {
   "type": "build",
   "target": "arch",
   "order": "z",
   "when": [
    "arch",
    0.1
   ],
   "until": [
    "arch",
    "end",
    0.1
   ]
  },
  {
   "type": "build",
   "target": "keystone",
   "stand_at": [
    0.0,
    0.2
   ],
   "when": [
    "key",
    0.1
   ],
   "until": [
    "key",
    0.4
   ]
  },
  {
   "type": "highlight",
   "target": "keystone",
   "part": "all",
   "when": [
    "key",
    0.4
   ],
   "until": [
    "squeeze",
    "start"
   ]
  },
  {
   "type": "point",
   "target": "arch",
   "offset": [
    0.55,
    0.05,
    0.45
   ],
   "when": [
    "squeeze",
    "start"
   ],
   "until": [
    "squeeze",
    "end"
   ]
  },
  {
   "type": "build",
   "target": "load",
   "order": "z",
   "stand_at": [
    0.0,
    0.2
   ],
   "when": [
    "payoff",
    0.05
   ],
   "until": [
    "payoff",
    0.45
   ]
  },
  {
   "type": "face_camera",
   "when": [
    "payoff",
    0.55
   ],
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    -0.48,
    0.05,
    0.42
   ],
   "az": -80,
   "el": 10,
   "dist": 2.0
  },
  {
   "at": [
    "hook",
    "end"
   ],
   "target": [
    -0.6,
    0.0,
    0.38
   ],
   "az": -82,
   "el": 14,
   "dist": 2.3
  },
  {
   "at": [
    "crack",
    "end",
    0.6
   ],
   "target": [
    -0.62,
    0.0,
    0.3
   ],
   "az": -86,
   "el": 16,
   "dist": 2.4
  },
  {
   "at": [
    "arch",
    0.15
   ],
   "target": [
    0.62,
    0.0,
    0.36
   ],
   "az": -94,
   "el": 14,
   "dist": 1.85
  },
  {
   "at": [
    "squeeze",
    "end"
   ],
   "target": [
    0.66,
    0.0,
    0.38
   ],
   "az": -98,
   "el": 14,
   "dist": 1.75
  },
  {
   "at": "end",
   "target": [
    0.64,
    0.0,
    0.46
   ],
   "az": -104,
   "el": 16,
   "dist": 2.05
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "2,000 YEARS",
   "color": "white",
   "y": 470,
   "when": [
    "hook",
    "start"
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "NO GLUE?",
   "color": "amber",
   "y": 590,
   "size": 110,
   "when": [
    "hook",
    0.55
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "tag",
   "text": "STRETCHED",
   "anchor": "flat:part:mid",
   "color": "red",
   "dy": 60,
   "when": [
    "flat",
    0.7
   ],
   "until": [
    "crack",
    "end"
   ]
  },
  {
   "style": "impact",
   "text": "CRACK!",
   "color": "red",
   "y": 560,
   "when": [
    "crack",
    "end",
    0.05
   ],
   "until": [
    "crack",
    "end",
    1.1
   ]
  },
  {
   "style": "title",
   "text": "ARCH",
   "color": "cyan",
   "y": 470,
   "when": [
    "arch",
    0.2
   ],
   "until": [
    "key",
    "start"
   ]
  },
  {
   "style": "bubble",
   "text": "KEYSTONE",
   "anchor": "keystone:top",
   "color": "amber",
   "when": [
    "key",
    0.4
   ],
   "until": [
    "squeeze",
    "start"
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "cyan",
   "when": [
    "squeeze",
    "start"
   ],
   "until": [
    "payoff",
    "start",
    -0.1
   ],
   "rows": [
    {
     "text": "STONE vs STRETCHING: weak",
     "color": "red",
     "at": [
      "squeeze",
      0.05
     ],
     "size": 46
    },
    {
     "text": "STONE vs SQUEEZING: strong",
     "color": "green",
     "at": [
      "squeeze",
      0.6
     ],
     "size": 46
    }
   ]
  },
  {
   "style": "bubble",
   "text": "+{count:load} BLOCKS",
   "anchor": "load:top",
   "color": "cyan",
   "when": [
    "payoff",
    0.05
   ],
   "until": [
    "payoff",
    "end",
    0.3
   ]
  },
  {
   "style": "end",
   "title1": "ARCHES",
   "title2": "LOVE A SQUEEZE",
   "sub": "Stone resists squeezing about 10x better than stretching",
   "cta": "Follow for more Block Lab",
   "when": [
    "payoff",
    "end",
    0.2
   ]
  }
 ],
 "world": {
  "ground": "grass",
  "time": "morning",
  "weather": "clouds",
  "animals": [
   "birds",
   "cat"
  ]
 }
}

In [ ]:
%%writefile stories/gears.json
{
 "title": "How gears trade speed for force",
 "slug": "gears",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "lines": [
  {
   "id": "hook",
   "text": "How can a little gear turn a great big one? Let me show you.",
   "pause": 0.4
  },
  {
   "id": "teeth",
   "text": "This small gear has six teeth. The big one has twelve.",
   "pause": 0.4
  },
  {
   "id": "turn",
   "text": "Turn the small one... and the big one turns the other way.",
   "pause": 0.3
  },
  {
   "id": "count",
   "text": "But watch. The small gear spins twice... for just one turn of the big gear.",
   "pause": 0.4
  },
  {
   "id": "payoff",
   "text": "Slower, but with twice the turning force. That's how a bike climbs a hill!",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   0.78,
   -0.3
  ]
 },
 "structures": {
  "frame": {
   "cells": [
    [
     0,
     1,
     0
    ],
    [
     0,
     1,
     1
    ],
    [
     0,
     1,
     2
    ],
    [
     0,
     1,
     3
    ],
    [
     0,
     1,
     4
    ],
    [
     0,
     1,
     5
    ],
    [
     0,
     1,
     6
    ],
    [
     0,
     1,
     7
    ],
    [
     0,
     1,
     8
    ],
    [
     5,
     1,
     0
    ],
    [
     5,
     1,
     1
    ],
    [
     5,
     1,
     2
    ],
    [
     5,
     1,
     3
    ],
    [
     5,
     1,
     4
    ],
    [
     5,
     1,
     5
    ],
    [
     5,
     1,
     6
    ],
    [
     5,
     1,
     7
    ],
    [
     5,
     1,
     8
    ],
    [
     5,
     1,
     9
    ],
    [
     5,
     1,
     10
    ],
    [
     5,
     1,
     11
    ],
    [
     5,
     1,
     12
    ],
    [
     5,
     1,
     13
    ],
    [
     5,
     1,
     14
    ],
    [
     5,
     1,
     15
    ],
    [
     5,
     1,
     16
    ],
    [
     5,
     1,
     17
    ],
    [
     5,
     0,
     17
    ],
    [
     -2,
     1,
     0
    ],
    [
     -1,
     1,
     0
    ],
    [
     1,
     1,
     0
    ],
    [
     2,
     1,
     0
    ],
    [
     3,
     1,
     0
    ],
    [
     4,
     1,
     0
    ],
    [
     6,
     1,
     0
    ],
    [
     7,
     1,
     0
    ]
   ],
   "at": [
    -0.5,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 36
  },
  "big": {
   "cells": [
    [
     -7,
     0,
     6
    ],
    [
     -7,
     0,
     7
    ],
    [
     -7,
     0,
     9
    ],
    [
     -7,
     0,
     10
    ],
    [
     -6,
     0,
     3
    ],
    [
     -6,
     0,
     6
    ],
    [
     -6,
     0,
     7
    ],
    [
     -6,
     0,
     8
    ],
    [
     -6,
     0,
     9
    ],
    [
     -6,
     0,
     10
    ],
    [
     -6,
     0,
     13
    ],
    [
     -5,
     0,
     2
    ],
    [
     -5,
     0,
     3
    ],
    [
     -5,
     0,
     4
    ],
    [
     -5,
     0,
     5
    ],
    [
     -5,
     0,
     6
    ],
    [
     -5,
     0,
     7
    ],
    [
     -5,
     0,
     8
    ],
    [
     -5,
     0,
     9
    ],
    [
     -5,
     0,
     10
    ],
    [
     -5,
     0,
     11
    ],
    [
     -5,
     0,
     12
    ],
    [
     -5,
     0,
     13
    ],
    [
     -5,
     0,
     14
    ],
    [
     -4,
     0,
     3
    ],
    [
     -4,
     0,
     4
    ],
    [
     -4,
     0,
     5
    ],
    [
     -4,
     0,
     6
    ],
    [
     -4,
     0,
     7
    ],
    [
     -4,
     0,
     8
    ],
    [
     -4,
     0,
     9
    ],
    [
     -4,
     0,
     10
    ],
    [
     -4,
     0,
     11
    ],
    [
     -4,
     0,
     12
    ],
    [
     -4,
     0,
     13
    ],
    [
     -3,
     0,
     3
    ],
    [
     -3,
     0,
     4
    ],
    [
     -3,
     0,
     5
    ],
    [
     -3,
     0,
     6
    ],
    [
     -3,
     0,
     7
    ],
    [
     -3,
     0,
     8
    ],
    [
     -3,
     0,
     9
    ],
    [
     -3,
     0,
     10
    ],
    [
     -3,
     0,
     11
    ],
    [
     -3,
     0,
     12
    ],
    [
     -3,
     0,
     13
    ],
    [
     -2,
     0,
     1
    ],
    [
     -2,
     0,
     2
    ],
    [
     -2,
     0,
     3
    ],
    [
     -2,
     0,
     4
    ],
    [
     -2,
     0,
     5
    ],
    [
     -2,
     0,
     6
    ],
    [
     -2,
     0,
     7
    ],
    [
     -2,
     0,
     8
    ],
    [
     -2,
     0,
     9
    ],
    [
     -2,
     0,
     10
    ],
    [
     -2,
     0,
     11
    ],
    [
     -2,
     0,
     12
    ],
    [
     -2,
     0,
     13
    ],
    [
     -2,
     0,
     14
    ],
    [
     -2,
     0,
     15
    ],
    [
     -1,
     0,
     1
    ],
    [
     -1,
     0,
     2
    ],
    [
     -1,
     0,
     3
    ],
    [
     -1,
     0,
     4
    ],
    [
     -1,
     0,
     5
    ],
    [
     -1,
     0,
     6
    ],
    [
     -1,
     0,
     7
    ],
    [
     -1,
     0,
     8
    ],
    [
     -1,
     0,
     9
    ],
    [
     -1,
     0,
     10
    ],
    [
     -1,
     0,
     11
    ],
    [
     -1,
     0,
     12
    ],
    [
     -1,
     0,
     13
    ],
    [
     -1,
     0,
     14
    ],
    [
     -1,
     0,
     15
    ],
    [
     0,
     0,
     2
    ],
    [
     0,
     0,
     3
    ],
    [
     0,
     0,
     4
    ],
    [
     0,
     0,
     5
    ],
    [
     0,
     0,
     6
    ],
    [
     0,
     0,
     7
    ],
    [
     0,
     0,
     8
    ],
    [
     0,
     0,
     9
    ],
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ],
    [
     0,
     0,
     13
    ],
    [
     0,
     0,
     14
    ],
    [
     1,
     0,
     1
    ],
    [
     1,
     0,
     2
    ],
    [
     1,
     0,
     3
    ],
    [
     1,
     0,
     4
    ],
    [
     1,
     0,
     5
    ],
    [
     1,
     0,
     6
    ],
    [
     1,
     0,
     7
    ],
    [
     1,
     0,
     8
    ],
    [
     1,
     0,
     9
    ],
    [
     1,
     0,
     10
    ],
    [
     1,
     0,
     11
    ],
    [
     1,
     0,
     12
    ],
    [
     1,
     0,
     13
    ],
    [
     1,
     0,
     14
    ],
    [
     1,
     0,
     15
    ],
    [
     2,
     0,
     1
    ],
    [
     2,
     0,
     2
    ],
    [
     2,
     0,
     3
    ],
    [
     2,
     0,
     4
    ],
    [
     2,
     0,
     5
    ],
    [
     2,
     0,
     6
    ],
    [
     2,
     0,
     7
    ],
    [
     2,
     0,
     8
    ],
    [
     2,
     0,
     9
    ],
    [
     2,
     0,
     10
    ],
    [
     2,
     0,
     11
    ],
    [
     2,
     0,
     12
    ],
    [
     2,
     0,
     13
    ],
    [
     2,
     0,
     14
    ],
    [
     2,
     0,
     15
    ],
    [
     3,
     0,
     3
    ],
    [
     3,
     0,
     4
    ],
    [
     3,
     0,
     5
    ],
    [
     3,
     0,
     6
    ],
    [
     3,
     0,
     7
    ],
    [
     3,
     0,
     8
    ],
    [
     3,
     0,
     9
    ],
    [
     3,
     0,
     10
    ],
    [
     3,
     0,
     11
    ],
    [
     3,
     0,
     12
    ],
    [
     3,
     0,
     13
    ],
    [
     4,
     0,
     3
    ],
    [
     4,
     0,
     4
    ],
    [
     4,
     0,
     5
    ],
    [
     4,
     0,
     6
    ],
    [
     4,
     0,
     7
    ],
    [
     4,
     0,
     8
    ],
    [
     4,
     0,
     9
    ],
    [
     4,
     0,
     10
    ],
    [
     4,
     0,
     11
    ],
    [
     4,
     0,
     12
    ],
    [
     4,
     0,
     13
    ],
    [
     5,
     0,
     2
    ],
    [
     5,
     0,
     3
    ],
    [
     5,
     0,
     4
    ],
    [
     5,
     0,
     5
    ],
    [
     5,
     0,
     6
    ],
    [
     5,
     0,
     7
    ],
    [
     5,
     0,
     8
    ],
    [
     5,
     0,
     9
    ],
    [
     5,
     0,
     10
    ],
    [
     5,
     0,
     11
    ],
    [
     5,
     0,
     12
    ],
    [
     5,
     0,
     13
    ],
    [
     5,
     0,
     14
    ],
    [
     6,
     0,
     3
    ],
    [
     6,
     0,
     6
    ],
    [
     6,
     0,
     7
    ],
    [
     6,
     0,
     8
    ],
    [
     6,
     0,
     9
    ],
    [
     6,
     0,
     10
    ],
    [
     6,
     0,
     13
    ],
    [
     7,
     0,
     6
    ],
    [
     7,
     0,
     7
    ],
    [
     7,
     0,
     9
    ],
    [
     7,
     0,
     10
    ]
   ],
   "at": [
    -0.5,
    0.0
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 165
  },
  "small": {
   "cells": [
    [
     1,
     -1,
     16
    ],
    [
     1,
     -1,
     17
    ],
    [
     1,
     -1,
     18
    ],
    [
     2,
     -1,
     17
    ],
    [
     3,
     -1,
     14
    ],
    [
     3,
     -1,
     16
    ],
    [
     3,
     -1,
     17
    ],
    [
     3,
     -1,
     18
    ],
    [
     3,
     -1,
     20
    ],
    [
     4,
     -1,
     14
    ],
    [
     4,
     -1,
     15
    ],
    [
     4,
     -1,
     16
    ],
    [
     4,
     -1,
     17
    ],
    [
     4,
     -1,
     18
    ],
    [
     4,
     -1,
     19
    ],
    [
     4,
     -1,
     20
    ],
    [
     5,
     -1,
     15
    ],
    [
     5,
     -1,
     16
    ],
    [
     5,
     -1,
     17
    ],
    [
     5,
     -1,
     18
    ],
    [
     5,
     -1,
     19
    ],
    [
     6,
     -1,
     14
    ],
    [
     6,
     -1,
     15
    ],
    [
     6,
     -1,
     16
    ],
    [
     6,
     -1,
     17
    ],
    [
     6,
     -1,
     18
    ],
    [
     6,
     -1,
     19
    ],
    [
     6,
     -1,
     20
    ],
    [
     7,
     -1,
     14
    ],
    [
     7,
     -1,
     15
    ],
    [
     7,
     -1,
     16
    ],
    [
     7,
     -1,
     17
    ],
    [
     7,
     -1,
     18
    ],
    [
     7,
     -1,
     19
    ],
    [
     7,
     -1,
     20
    ],
    [
     8,
     -1,
     14
    ],
    [
     8,
     -1,
     17
    ],
    [
     8,
     -1,
     20
    ],
    [
     9,
     -1,
     16
    ],
    [
     9,
     -1,
     17
    ],
    [
     9,
     -1,
     18
    ]
   ],
   "at": [
    -0.5,
    0.0
   ],
   "centered": false,
   "material": "brick",
   "expect_blocks": 41
  },
  "hubs": {
   "cells": [
    [
     0,
     -1,
     8
    ],
    [
     5,
     -2,
     17
    ]
   ],
   "at": [
    -0.5,
    0.0
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 2
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": [
    "hook",
    0.45
   ]
  },
  {
   "type": "build",
   "target": "frame",
   "order": "z",
   "stand_at": [
    0.78,
    -0.3
   ],
   "when": [
    "hook",
    0.05
   ],
   "until": [
    "hook",
    0.4
   ]
  },
  {
   "type": "build",
   "target": "big",
   "order": "z",
   "stand_at": [
    0.78,
    -0.3
   ],
   "when": [
    "hook",
    0.4
   ],
   "until": [
    "teeth",
    "start"
   ]
  },
  {
   "type": "build",
   "target": "small",
   "order": "z",
   "stand_at": [
    0.78,
    -0.3
   ],
   "when": [
    "teeth",
    "start",
    0.05
   ],
   "until": [
    "teeth",
    0.45
   ]
  },
  {
   "type": "build",
   "target": "hubs",
   "stand_at": [
    0.78,
    -0.3
   ],
   "when": [
    "teeth",
    0.5
   ],
   "until": [
    "teeth",
    0.6
   ]
  },
  {
   "type": "spin",
   "target": "small",
   "axis": "y",
   "degrees": -360,
   "pivot": [
    5,
    -1,
    17
   ],
   "ticks_per_turn": 6,
   "when": [
    "turn",
    0.1
   ],
   "until": [
    "turn",
    "end",
    0.2
   ]
  },
  {
   "type": "spin",
   "target": "big",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    0,
    0,
    8
   ],
   "ticks_per_turn": 0,
   "when": [
    "turn",
    0.1
   ],
   "until": [
    "turn",
    "end",
    0.2
   ]
  },
  {
   "type": "spin",
   "target": "small",
   "axis": "y",
   "degrees": -720,
   "pivot": [
    5,
    -1,
    17
   ],
   "ticks_per_turn": 6,
   "when": [
    "count",
    0.2
   ],
   "until": [
    "count",
    "end",
    0.3
   ]
  },
  {
   "type": "spin",
   "target": "big",
   "axis": "y",
   "degrees": 360,
   "pivot": [
    0,
    0,
    8
   ],
   "ticks_per_turn": 0,
   "when": [
    "count",
    0.2
   ],
   "until": [
    "count",
    "end",
    0.3
   ]
  },
  {
   "type": "point",
   "target": "small",
   "offset": [
    0.5,
    -0.2,
    1.75
   ],
   "when": [
    "payoff",
    0.0
   ],
   "until": [
    "payoff",
    0.45
   ]
  },
  {
   "type": "face_camera",
   "when": [
    "payoff",
    0.5
   ],
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    0.5,
    0.0,
    0.75
   ],
   "az": -96,
   "el": 8,
   "dist": 1.8
  },
  {
   "at": [
    "hook",
    0.6
   ],
   "target": [
    -0.15,
    0.0,
    0.98
   ],
   "az": -90,
   "el": 8,
   "dist": 3.2
  },
  {
   "at": [
    "count",
    "end"
   ],
   "target": [
    -0.15,
    0.0,
    0.98
   ],
   "az": -92,
   "el": 8,
   "dist": 3.1
  },
  {
   "at": "end",
   "target": [
    -0.1,
    0.0,
    0.95
   ],
   "az": -88,
   "el": 8,
   "dist": 3.2
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "LITTLE GEAR",
   "color": "white",
   "y": 470,
   "when": [
    "hook",
    "start"
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "vs BIG GEAR",
   "color": "amber",
   "y": 590,
   "size": 100,
   "when": [
    "hook",
    0.35
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "bubble",
   "text": "6 TEETH",
   "anchor": "small:top",
   "color": "red",
   "when": [
    "teeth",
    0.2
   ],
   "until": [
    "teeth",
    "end",
    0.3
   ]
  },
  {
   "style": "tag",
   "text": "12 TEETH",
   "anchor": "big:center",
   "color": "cyan",
   "when": [
    "teeth",
    0.6
   ],
   "until": [
    "teeth",
    "end",
    0.3
   ]
  },
  {
   "style": "title",
   "text": "OPPOSITE WAYS",
   "color": "cyan",
   "y": 420,
   "size": 84,
   "when": [
    "turn",
    0.55
   ],
   "until": [
    "turn",
    "end",
    0.3
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "amber",
   "when": [
    "count",
    0.15
   ],
   "until": [
    "count",
    "end",
    0.4
   ],
   "rows": [
    {
     "text": "SMALL: 2 turns",
     "color": "red",
     "at": [
      "count",
      0.4
     ],
     "size": 50
    },
    {
     "text": "BIG: 1 turn",
     "color": "cyan",
     "at": [
      "count",
      0.75
     ],
     "size": 50
    }
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "green",
   "when": [
    "payoff",
    "start"
   ],
   "until": [
    "payoff",
    "end",
    0.3
   ],
   "rows": [
    {
     "text": "1/2 the speed",
     "color": "white",
     "at": [
      "payoff",
      0.05
     ],
     "size": 50
    },
    {
     "text": "2x the turning force",
     "color": "green",
     "at": [
      "payoff",
      0.3
     ],
     "size": 50
    }
   ]
  },
  {
   "style": "end",
   "title1": "GEARS TRADE",
   "title2": "SPEED FOR FORCE",
   "sub": "12 teeth / 6 teeth = 2x the force at half the speed",
   "cta": "Follow for more Block Lab",
   "when": [
    "payoff",
    "end",
    0.2
   ]
  }
 ]
}

In [ ]:
%%writefile stories/eiffel_air.json
{
 "title": "The Eiffel Tower is mostly air",
 "slug": "eiffel_air",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "world": {
  "ground": "paris",
  "time": "day",
  "weather": "clouds",
  "animals": [
   "pigeons"
  ]
 },
 "lines": [
  {
   "id": "hook",
   "text": "The Eiffel Tower is made of over seven thousand tonnes of iron. Melt it all down... how thick would it be?",
   "pause": 0.4
  },
  {
   "id": "build",
   "text": "Eighteen thousand pieces. Two and a half million rivets. Built in just over two years!",
   "pause": 0.4
  },
  {
   "id": "air",
   "text": "But look closer. It's mostly air! Criss-crossed beams, light but strong.",
   "pause": 0.4
  },
  {
   "id": "melt",
   "text": "Now melt all that iron into one slab, as wide as its base...",
   "pause": 0.9
  },
  {
   "id": "payoff",
   "text": "Just six centimetres thick! It's the shape that makes it strong, not the weight.",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   1.25,
   -0.25
  ],
  "outfit": {
   "hat": "beret",
   "shirt": "striped",
   "extras": [
    "scarf"
   ]
  }
 },
 "structures": {
  "tower": {
   "cells": [
    [
     -8,
     -8,
     0
    ],
    [
     -7,
     -8,
     0
    ],
    [
     -6,
     -8,
     0
    ],
    [
     -5,
     -8,
     0
    ],
    [
     -4,
     -8,
     0
    ],
    [
     4,
     -8,
     0
    ],
    [
     5,
     -8,
     0
    ],
    [
     6,
     -8,
     0
    ],
    [
     7,
     -8,
     0
    ],
    [
     8,
     -8,
     0
    ],
    [
     -8,
     -7,
     0
    ],
    [
     -4,
     -7,
     0
    ],
    [
     4,
     -7,
     0
    ],
    [
     8,
     -7,
     0
    ],
    [
     -8,
     -6,
     0
    ],
    [
     -4,
     -6,
     0
    ],
    [
     4,
     -6,
     0
    ],
    [
     8,
     -6,
     0
    ],
    [
     -8,
     -5,
     0
    ],
    [
     -4,
     -5,
     0
    ],
    [
     4,
     -5,
     0
    ],
    [
     8,
     -5,
     0
    ],
    [
     -8,
     -4,
     0
    ],
    [
     -7,
     -4,
     0
    ],
    [
     -6,
     -4,
     0
    ],
    [
     -5,
     -4,
     0
    ],
    [
     -4,
     -4,
     0
    ],
    [
     4,
     -4,
     0
    ],
    [
     5,
     -4,
     0
    ],
    [
     6,
     -4,
     0
    ],
    [
     7,
     -4,
     0
    ],
    [
     8,
     -4,
     0
    ],
    [
     -8,
     4,
     0
    ],
    [
     -7,
     4,
     0
    ],
    [
     -6,
     4,
     0
    ],
    [
     -5,
     4,
     0
    ],
    [
     -4,
     4,
     0
    ],
    [
     4,
     4,
     0
    ],
    [
     5,
     4,
     0
    ],
    [
     6,
     4,
     0
    ],
    [
     7,
     4,
     0
    ],
    [
     8,
     4,
     0
    ],
    [
     -8,
     5,
     0
    ],
    [
     -4,
     5,
     0
    ],
    [
     4,
     5,
     0
    ],
    [
     8,
     5,
     0
    ],
    [
     -8,
     6,
     0
    ],
    [
     -4,
     6,
     0
    ],
    [
     4,
     6,
     0
    ],
    [
     8,
     6,
     0
    ],
    [
     -8,
     7,
     0
    ],
    [
     -4,
     7,
     0
    ],
    [
     4,
     7,
     0
    ],
    [
     8,
     7,
     0
    ],
    [
     -8,
     8,
     0
    ],
    [
     -7,
     8,
     0
    ],
    [
     -6,
     8,
     0
    ],
    [
     -5,
     8,
     0
    ],
    [
     -4,
     8,
     0
    ],
    [
     4,
     8,
     0
    ],
    [
     5,
     8,
     0
    ],
    [
     6,
     8,
     0
    ],
    [
     7,
     8,
     0
    ],
    [
     8,
     8,
     0
    ],
    [
     -7,
     -7,
     1
    ],
    [
     -4,
     -7,
     1
    ],
    [
     4,
     -7,
     1
    ],
    [
     7,
     -7,
     1
    ],
    [
     -7,
     -4,
     1
    ],
    [
     -4,
     -4,
     1
    ],
    [
     4,
     -4,
     1
    ],
    [
     7,
     -4,
     1
    ],
    [
     -7,
     4,
     1
    ],
    [
     -4,
     4,
     1
    ],
    [
     4,
     4,
     1
    ],
    [
     7,
     4,
     1
    ],
    [
     -7,
     7,
     1
    ],
    [
     -4,
     7,
     1
    ],
    [
     4,
     7,
     1
    ],
    [
     7,
     7,
     1
    ],
    [
     -6,
     -6,
     2
    ],
    [
     -5,
     -6,
     2
    ],
    [
     -4,
     -6,
     2
    ],
    [
     -3,
     -6,
     2
    ],
    [
     3,
     -6,
     2
    ],
    [
     4,
     -6,
     2
    ],
    [
     5,
     -6,
     2
    ],
    [
     6,
     -6,
     2
    ],
    [
     -6,
     -5,
     2
    ],
    [
     -3,
     -5,
     2
    ],
    [
     3,
     -5,
     2
    ],
    [
     6,
     -5,
     2
    ],
    [
     -6,
     -4,
     2
    ],
    [
     -3,
     -4,
     2
    ],
    [
     3,
     -4,
     2
    ],
    [
     6,
     -4,
     2
    ],
    [
     -6,
     -3,
     2
    ],
    [
     -5,
     -3,
     2
    ],
    [
     -4,
     -3,
     2
    ],
    [
     -3,
     -3,
     2
    ],
    [
     3,
     -3,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     6,
     -3,
     2
    ],
    [
     -6,
     3,
     2
    ],
    [
     -5,
     3,
     2
    ],
    [
     -4,
     3,
     2
    ],
    [
     -3,
     3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     6,
     3,
     2
    ],
    [
     -6,
     4,
     2
    ],
    [
     -3,
     4,
     2
    ],
    [
     3,
     4,
     2
    ],
    [
     6,
     4,
     2
    ],
    [
     -6,
     5,
     2
    ],
    [
     -3,
     5,
     2
    ],
    [
     3,
     5,
     2
    ],
    [
     6,
     5,
     2
    ],
    [
     -6,
     6,
     2
    ],
    [
     -5,
     6,
     2
    ],
    [
     -4,
     6,
     2
    ],
    [
     -3,
     6,
     2
    ],
    [
     3,
     6,
     2
    ],
    [
     4,
     6,
     2
    ],
    [
     5,
     6,
     2
    ],
    [
     6,
     6,
     2
    ],
    [
     -6,
     -6,
     3
    ],
    [
     -3,
     -6,
     3
    ],
    [
     3,
     -6,
     3
    ],
    [
     6,
     -6,
     3
    ],
    [
     -3,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     -6,
     -3,
     3
    ],
    [
     -4,
     -3,
     3
    ],
    [
     -3,
     -3,
     3
    ],
    [
     3,
     -3,
     3
    ],
    [
     4,
     -3,
     3
    ],
    [
     6,
     -3,
     3
    ],
    [
     -6,
     3,
     3
    ],
    [
     -4,
     3,
     3
    ],
    [
     -3,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     -3,
     4,
     3
    ],
    [
     3,
     4,
     3
    ],
    [
     -6,
     6,
     3
    ],
    [
     -3,
     6,
     3
    ],
    [
     3,
     6,
     3
    ],
    [
     6,
     6,
     3
    ],
    [
     -5,
     -5,
     4
    ],
    [
     -4,
     -5,
     4
    ],
    [
     -3,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     -5,
     -4,
     4
    ],
    [
     -3,
     -4,
     4
    ],
    [
     -2,
     -4,
     4
    ],
    [
     -1,
     -4,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     -5,
     -3,
     4
    ],
    [
     -4,
     -3,
     4
    ],
    [
     -3,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     -4,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     -4,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     -4,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     -4,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     -4,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     -5,
     3,
     4
    ],
    [
     -4,
     3,
     4
    ],
    [
     -3,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     -5,
     4,
     4
    ],
    [
     -3,
     4,
     4
    ],
    [
     -2,
     4,
     4
    ],
    [
     -1,
     4,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     -5,
     5,
     4
    ],
    [
     -4,
     5,
     4
    ],
    [
     -3,
     5,
     4
    ],
    [
     3,
     5,
     4
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     4
    ],
    [
     -5,
     -5,
     5
    ],
    [
     -2,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     -2,
     -4,
     5
    ],
    [
     -1,
     -4,
     5
    ],
    [
     0,
     -4,
     5
    ],
    [
     1,
     -4,
     5
    ],
    [
     2,
     -4,
     5
    ],
    [
     -5,
     -2,
     5
    ],
    [
     -4,
     -2,
     5
    ],
    [
     -2,
     -2,
     5
    ],
    [
     2,
     -2,
     5
    ],
    [
     4,
     -2,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     -4,
     -1,
     5
    ],
    [
     4,
     -1,
     5
    ],
    [
     -4,
     0,
     5
    ],
    [
     4,
     0,
     5
    ],
    [
     -4,
     1,
     5
    ],
    [
     4,
     1,
     5
    ],
    [
     -5,
     2,
     5
    ],
    [
     -4,
     2,
     5
    ],
    [
     -2,
     2,
     5
    ],
    [
     2,
     2,
     5
    ],
    [
     4,
     2,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     -2,
     4,
     5
    ],
    [
     -1,
     4,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     -5,
     5,
     5
    ],
    [
     -2,
     5,
     5
    ],
    [
     2,
     5,
     5
    ],
    [
     5,
     5,
     5
    ],
    [
     -5,
     -5,
     6
    ],
    [
     -4,
     -5,
     6
    ],
    [
     -3,
     -5,
     6
    ],
    [
     -2,
     -5,
     6
    ],
    [
     -1,
     -5,
     6
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     -5,
     -4,
     6
    ],
    [
     -4,
     -4,
     6
    ],
    [
     -3,
     -4,
     6
    ],
    [
     -2,
     -4,
     6
    ],
    [
     -1,
     -4,
     6
    ],
    [
     0,
     -4,
     6
    ],
    [
     1,
     -4,
     6
    ],
    [
     2,
     -4,
     6
    ],
    [
     3,
     -4,
     6
    ],
    [
     4,
     -4,
     6
    ],
    [
     5,
     -4,
     6
    ],
    [
     -5,
     -3,
     6
    ],
    [
     -4,
     -3,
     6
    ],
    [
     -3,
     -3,
     6
    ],
    [
     -2,
     -3,
     6
    ],
    [
     -1,
     -3,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     1,
     -3,
     6
    ],
    [
     2,
     -3,
     6
    ],
    [
     3,
     -3,
     6
    ],
    [
     4,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     -5,
     -2,
     6
    ],
    [
     -4,
     -2,
     6
    ],
    [
     -3,
     -2,
     6
    ],
    [
     -2,
     -2,
     6
    ],
    [
     -1,
     -2,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     1,
     -2,
     6
    ],
    [
     2,
     -2,
     6
    ],
    [
     3,
     -2,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     5,
     -2,
     6
    ],
    [
     -5,
     -1,
     6
    ],
    [
     -4,
     -1,
     6
    ],
    [
     -3,
     -1,
     6
    ],
    [
     -2,
     -1,
     6
    ],
    [
     2,
     -1,
     6
    ],
    [
     3,
     -1,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     5,
     -1,
     6
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     6
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     -5,
     1,
     6
    ],
    [
     -4,
     1,
     6
    ],
    [
     -3,
     1,
     6
    ],
    [
     -2,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     -5,
     2,
     6
    ],
    [
     -4,
     2,
     6
    ],
    [
     -3,
     2,
     6
    ],
    [
     -2,
     2,
     6
    ],
    [
     -1,
     2,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     1,
     2,
     6
    ],
    [
     2,
     2,
     6
    ],
    [
     3,
     2,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     5,
     2,
     6
    ],
    [
     -5,
     3,
     6
    ],
    [
     -4,
     3,
     6
    ],
    [
     -3,
     3,
     6
    ],
    [
     -2,
     3,
     6
    ],
    [
     -1,
     3,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     1,
     3,
     6
    ],
    [
     2,
     3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     4,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     -5,
     4,
     6
    ],
    [
     -4,
     4,
     6
    ],
    [
     -3,
     4,
     6
    ],
    [
     -2,
     4,
     6
    ],
    [
     -1,
     4,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     -5,
     5,
     6
    ],
    [
     -4,
     5,
     6
    ],
    [
     -3,
     5,
     6
    ],
    [
     -2,
     5,
     6
    ],
    [
     -1,
     5,
     6
    ],
    [
     0,
     5,
     6
    ],
    [
     1,
     5,
     6
    ],
    [
     2,
     5,
     6
    ],
    [
     3,
     5,
     6
    ],
    [
     4,
     5,
     6
    ],
    [
     5,
     5,
     6
    ],
    [
     -5,
     -5,
     7
    ],
    [
     -3,
     -5,
     7
    ],
    [
     -1,
     -5,
     7
    ],
    [
     1,
     -5,
     7
    ],
    [
     3,
     -5,
     7
    ],
    [
     5,
     -5,
     7
    ],
    [
     -4,
     -4,
     7
    ],
    [
     -2,
     -4,
     7
    ],
    [
     2,
     -4,
     7
    ],
    [
     4,
     -4,
     7
    ],
    [
     -5,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     -4,
     -2,
     7
    ],
    [
     -2,
     -2,
     7
    ],
    [
     2,
     -2,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     -5,
     -1,
     7
    ],
    [
     5,
     -1,
     7
    ],
    [
     -5,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     -4,
     2,
     7
    ],
    [
     -2,
     2,
     7
    ],
    [
     2,
     2,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     -5,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     -4,
     4,
     7
    ],
    [
     -2,
     4,
     7
    ],
    [
     2,
     4,
     7
    ],
    [
     4,
     4,
     7
    ],
    [
     -5,
     5,
     7
    ],
    [
     -3,
     5,
     7
    ],
    [
     -1,
     5,
     7
    ],
    [
     1,
     5,
     7
    ],
    [
     3,
     5,
     7
    ],
    [
     5,
     5,
     7
    ],
    [
     -4,
     -4,
     8
    ],
    [
     -3,
     -4,
     8
    ],
    [
     -2,
     -4,
     8
    ],
    [
     -1,
     -4,
     8
    ],
    [
     1,
     -4,
     8
    ],
    [
     2,
     -4,
     8
    ],
    [
     3,
     -4,
     8
    ],
    [
     4,
     -4,
     8
    ],
    [
     -4,
     -3,
     8
    ],
    [
     -1,
     -3,
     8
    ],
    [
     1,
     -3,
     8
    ],
    [
     4,
     -3,
     8
    ],
    [
     -4,
     -2,
     8
    ],
    [
     -1,
     -2,
     8
    ],
    [
     1,
     -2,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     -4,
     -1,
     8
    ],
    [
     -3,
     -1,
     8
    ],
    [
     -2,
     -1,
     8
    ],
    [
     -1,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     3,
     -1,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     -4,
     1,
     8
    ],
    [
     -3,
     1,
     8
    ],
    [
     -2,
     1,
     8
    ],
    [
     -1,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     -4,
     2,
     8
    ],
    [
     -1,
     2,
     8
    ],
    [
     1,
     2,
     8
    ],
    [
     4,
     2,
     8
    ],
    [
     -4,
     3,
     8
    ],
    [
     -1,
     3,
     8
    ],
    [
     1,
     3,
     8
    ],
    [
     4,
     3,
     8
    ],
    [
     -4,
     4,
     8
    ],
    [
     -3,
     4,
     8
    ],
    [
     -2,
     4,
     8
    ],
    [
     -1,
     4,
     8
    ],
    [
     1,
     4,
     8
    ],
    [
     2,
     4,
     8
    ],
    [
     3,
     4,
     8
    ],
    [
     4,
     4,
     8
    ],
    [
     -3,
     -3,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     1,
     -3,
     9
    ],
    [
     3,
     -3,
     9
    ],
    [
     -3,
     -1,
     9
    ],
    [
     -1,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     3,
     -1,
     9
    ],
    [
     -3,
     1,
     9
    ],
    [
     -1,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     3,
     1,
     9
    ],
    [
     -3,
     3,
     9
    ],
    [
     -1,
     3,
     9
    ],
    [
     1,
     3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     -3,
     -3,
     10
    ],
    [
     -2,
     -3,
     10
    ],
    [
     -1,
     -3,
     10
    ],
    [
     1,
     -3,
     10
    ],
    [
     2,
     -3,
     10
    ],
    [
     3,
     -3,
     10
    ],
    [
     -3,
     -2,
     10
    ],
    [
     -1,
     -2,
     10
    ],
    [
     1,
     -2,
     10
    ],
    [
     3,
     -2,
     10
    ],
    [
     -3,
     -1,
     10
    ],
    [
     -2,
     -1,
     10
    ],
    [
     -1,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     3,
     -1,
     10
    ],
    [
     -3,
     1,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     -1,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ],
    [
     3,
     1,
     10
    ],
    [
     -3,
     2,
     10
    ],
    [
     -1,
     2,
     10
    ],
    [
     1,
     2,
     10
    ],
    [
     3,
     2,
     10
    ],
    [
     -3,
     3,
     10
    ],
    [
     -2,
     3,
     10
    ],
    [
     -1,
     3,
     10
    ],
    [
     1,
     3,
     10
    ],
    [
     2,
     3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     -3,
     -3,
     11
    ],
    [
     -1,
     -3,
     11
    ],
    [
     1,
     -3,
     11
    ],
    [
     3,
     -3,
     11
    ],
    [
     -3,
     -1,
     11
    ],
    [
     -1,
     -1,
     11
    ],
    [
     1,
     -1,
     11
    ],
    [
     3,
     -1,
     11
    ],
    [
     -3,
     1,
     11
    ],
    [
     -1,
     1,
     11
    ],
    [
     1,
     1,
     11
    ],
    [
     3,
     1,
     11
    ],
    [
     -3,
     3,
     11
    ],
    [
     -1,
     3,
     11
    ],
    [
     1,
     3,
     11
    ],
    [
     3,
     3,
     11
    ],
    [
     -3,
     -3,
     12
    ],
    [
     -2,
     -3,
     12
    ],
    [
     -1,
     -3,
     12
    ],
    [
     0,
     -3,
     12
    ],
    [
     1,
     -3,
     12
    ],
    [
     2,
     -3,
     12
    ],
    [
     3,
     -3,
     12
    ],
    [
     -3,
     -2,
     12
    ],
    [
     -2,
     -2,
     12
    ],
    [
     -1,
     -2,
     12
    ],
    [
     0,
     -2,
     12
    ],
    [
     1,
     -2,
     12
    ],
    [
     2,
     -2,
     12
    ],
    [
     3,
     -2,
     12
    ],
    [
     -3,
     -1,
     12
    ],
    [
     -2,
     -1,
     12
    ],
    [
     -1,
     -1,
     12
    ],
    [
     0,
     -1,
     12
    ],
    [
     1,
     -1,
     12
    ],
    [
     2,
     -1,
     12
    ],
    [
     3,
     -1,
     12
    ],
    [
     -3,
     0,
     12
    ],
    [
     -2,
     0,
     12
    ],
    [
     -1,
     0,
     12
    ],
    [
     0,
     0,
     12
    ],
    [
     1,
     0,
     12
    ],
    [
     2,
     0,
     12
    ],
    [
     3,
     0,
     12
    ],
    [
     -3,
     1,
     12
    ],
    [
     -2,
     1,
     12
    ],
    [
     -1,
     1,
     12
    ],
    [
     0,
     1,
     12
    ],
    [
     1,
     1,
     12
    ],
    [
     2,
     1,
     12
    ],
    [
     3,
     1,
     12
    ],
    [
     -3,
     2,
     12
    ],
    [
     -2,
     2,
     12
    ],
    [
     -1,
     2,
     12
    ],
    [
     0,
     2,
     12
    ],
    [
     1,
     2,
     12
    ],
    [
     2,
     2,
     12
    ],
    [
     3,
     2,
     12
    ],
    [
     -3,
     3,
     12
    ],
    [
     -2,
     3,
     12
    ],
    [
     -1,
     3,
     12
    ],
    [
     0,
     3,
     12
    ],
    [
     1,
     3,
     12
    ],
    [
     2,
     3,
     12
    ],
    [
     3,
     3,
     12
    ],
    [
     -3,
     -3,
     13
    ],
    [
     -1,
     -3,
     13
    ],
    [
     1,
     -3,
     13
    ],
    [
     3,
     -3,
     13
    ],
    [
     -2,
     -2,
     13
    ],
    [
     2,
     -2,
     13
    ],
    [
     -3,
     -1,
     13
    ],
    [
     3,
     -1,
     13
    ],
    [
     -3,
     1,
     13
    ],
    [
     3,
     1,
     13
    ],
    [
     -2,
     2,
     13
    ],
    [
     2,
     2,
     13
    ],
    [
     -3,
     3,
     13
    ],
    [
     -1,
     3,
     13
    ],
    [
     1,
     3,
     13
    ],
    [
     3,
     3,
     13
    ],
    [
     -2,
     -2,
     14
    ],
    [
     -1,
     -2,
     14
    ],
    [
     0,
     -2,
     14
    ],
    [
     1,
     -2,
     14
    ],
    [
     2,
     -2,
     14
    ],
    [
     -2,
     -1,
     14
    ],
    [
     2,
     -1,
     14
    ],
    [
     -2,
     0,
     14
    ],
    [
     2,
     0,
     14
    ],
    [
     -2,
     1,
     14
    ],
    [
     2,
     1,
     14
    ],
    [
     -2,
     2,
     14
    ],
    [
     -1,
     2,
     14
    ],
    [
     0,
     2,
     14
    ],
    [
     1,
     2,
     14
    ],
    [
     2,
     2,
     14
    ],
    [
     -2,
     -2,
     15
    ],
    [
     2,
     -2,
     15
    ],
    [
     -2,
     2,
     15
    ],
    [
     2,
     2,
     15
    ],
    [
     -2,
     -2,
     16
    ],
    [
     -1,
     -2,
     16
    ],
    [
     0,
     -2,
     16
    ],
    [
     1,
     -2,
     16
    ],
    [
     2,
     -2,
     16
    ],
    [
     -2,
     -1,
     16
    ],
    [
     2,
     -1,
     16
    ],
    [
     -2,
     0,
     16
    ],
    [
     2,
     0,
     16
    ],
    [
     -2,
     1,
     16
    ],
    [
     2,
     1,
     16
    ],
    [
     -2,
     2,
     16
    ],
    [
     -1,
     2,
     16
    ],
    [
     0,
     2,
     16
    ],
    [
     1,
     2,
     16
    ],
    [
     2,
     2,
     16
    ],
    [
     -2,
     -2,
     17
    ],
    [
     2,
     -2,
     17
    ],
    [
     -2,
     2,
     17
    ],
    [
     2,
     2,
     17
    ],
    [
     -1,
     -1,
     18
    ],
    [
     0,
     -1,
     18
    ],
    [
     1,
     -1,
     18
    ],
    [
     -1,
     0,
     18
    ],
    [
     1,
     0,
     18
    ],
    [
     -1,
     1,
     18
    ],
    [
     0,
     1,
     18
    ],
    [
     1,
     1,
     18
    ],
    [
     -1,
     -1,
     19
    ],
    [
     1,
     -1,
     19
    ],
    [
     -1,
     1,
     19
    ],
    [
     1,
     1,
     19
    ],
    [
     -1,
     -1,
     20
    ],
    [
     0,
     -1,
     20
    ],
    [
     1,
     -1,
     20
    ],
    [
     -1,
     0,
     20
    ],
    [
     1,
     0,
     20
    ],
    [
     -1,
     1,
     20
    ],
    [
     0,
     1,
     20
    ],
    [
     1,
     1,
     20
    ],
    [
     -1,
     -1,
     21
    ],
    [
     1,
     -1,
     21
    ],
    [
     -1,
     1,
     21
    ],
    [
     1,
     1,
     21
    ],
    [
     -1,
     -1,
     22
    ],
    [
     0,
     -1,
     22
    ],
    [
     1,
     -1,
     22
    ],
    [
     -1,
     0,
     22
    ],
    [
     1,
     0,
     22
    ],
    [
     -1,
     1,
     22
    ],
    [
     0,
     1,
     22
    ],
    [
     1,
     1,
     22
    ],
    [
     -1,
     -1,
     23
    ],
    [
     1,
     -1,
     23
    ],
    [
     -1,
     1,
     23
    ],
    [
     1,
     1,
     23
    ],
    [
     -1,
     -1,
     24
    ],
    [
     0,
     -1,
     24
    ],
    [
     1,
     -1,
     24
    ],
    [
     -1,
     0,
     24
    ],
    [
     1,
     0,
     24
    ],
    [
     -1,
     1,
     24
    ],
    [
     0,
     1,
     24
    ],
    [
     1,
     1,
     24
    ],
    [
     -1,
     -1,
     25
    ],
    [
     1,
     -1,
     25
    ],
    [
     -1,
     1,
     25
    ],
    [
     1,
     1,
     25
    ],
    [
     -1,
     -1,
     26
    ],
    [
     0,
     -1,
     26
    ],
    [
     1,
     -1,
     26
    ],
    [
     -1,
     0,
     26
    ],
    [
     1,
     0,
     26
    ],
    [
     -1,
     1,
     26
    ],
    [
     0,
     1,
     26
    ],
    [
     1,
     1,
     26
    ],
    [
     -1,
     -1,
     27
    ],
    [
     1,
     -1,
     27
    ],
    [
     -1,
     1,
     27
    ],
    [
     1,
     1,
     27
    ],
    [
     -1,
     -1,
     28
    ],
    [
     0,
     -1,
     28
    ],
    [
     1,
     -1,
     28
    ],
    [
     -1,
     0,
     28
    ],
    [
     0,
     0,
     28
    ],
    [
     1,
     0,
     28
    ],
    [
     -1,
     1,
     28
    ],
    [
     0,
     1,
     28
    ],
    [
     1,
     1,
     28
    ],
    [
     0,
     0,
     29
    ],
    [
     0,
     0,
     30
    ],
    [
     0,
     0,
     31
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "material": "iron",
   "stand": "right",
   "parts": {
    "all": "z >= 0"
   },
   "expect_blocks": 677
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": [
    "hook",
    0.4
   ]
  },
  {
   "type": "build",
   "target": "tower",
   "order": "z",
   "stand_at": [
    1.25,
    -0.25
   ],
   "when": [
    "hook",
    0.35
   ],
   "until": [
    "build",
    "end"
   ]
  },
  {
   "type": "point",
   "target": "tower",
   "offset": [
    0,
    0,
    1.3
   ],
   "when": [
    "air",
    0.15
   ],
   "until": [
    "air",
    "end"
   ]
  },
  {
   "type": "crack",
   "target": "tower",
   "part": "all",
   "when": [
    "melt",
    0.1
   ],
   "until": [
    "melt",
    0.75
   ]
  },
  {
   "type": "melt",
   "target": "tower",
   "when": [
    "melt",
    0.8
   ]
  },
  {
   "type": "point",
   "target": "tower",
   "offset": [
    0,
    0,
    0.1
   ],
   "when": [
    "payoff",
    0.0
   ],
   "until": [
    "payoff",
    0.45
   ]
  },
  {
   "type": "face_camera",
   "when": [
    "payoff",
    0.5
   ],
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    0.9,
    0.0,
    0.85
   ],
   "az": -100,
   "el": 6,
   "dist": 1.9
  },
  {
   "at": [
    "hook",
    "end"
   ],
   "target": [
    0.3,
    0.0,
    1.45
   ],
   "az": -95,
   "el": 8,
   "dist": 3.2
  },
  {
   "at": [
    "build",
    "end"
   ],
   "target": [
    0.3,
    0.0,
    1.55
   ],
   "az": -92,
   "el": 8,
   "dist": 3.4
  },
  {
   "at": [
    "air",
    "end"
   ],
   "target": [
    0.25,
    0.0,
    1.5
   ],
   "az": -96,
   "el": 9,
   "dist": 3.2
  },
  {
   "at": [
    "melt",
    0.7
   ],
   "target": [
    0.3,
    0.0,
    1.2
   ],
   "az": -100,
   "el": 14,
   "dist": 3.1
  },
  {
   "at": [
    "payoff",
    0.3
   ],
   "target": [
    0.4,
    0.0,
    0.55
   ],
   "az": -100,
   "el": 24,
   "dist": 2.5
  },
  {
   "at": "end",
   "target": [
    0.45,
    0.0,
    0.6
   ],
   "az": -104,
   "el": 22,
   "dist": 2.6
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "7,300 TONNES",
   "color": "white",
   "y": 470,
   "when": [
    "hook",
    "start"
   ],
   "until": [
    "hook",
    0.55
   ]
  },
  {
   "style": "title",
   "text": "MELT IT DOWN?",
   "color": "amber",
   "y": 470,
   "size": 100,
   "when": [
    "hook",
    0.58
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "{count:tower} BLOCKS",
   "color": "white",
   "y": 590,
   "size": 54,
   "when": [
    "hook",
    0.62
   ],
   "until": [
    "build",
    "end",
    0.3
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "amber",
   "when": [
    "build",
    "start"
   ],
   "until": [
    "build",
    "end",
    0.3
   ],
   "rows": [
    {
     "text": "18,038 PIECES",
     "color": "white",
     "at": [
      "build",
      0.05
     ],
     "size": 50
    },
    {
     "text": "2.5 MILLION RIVETS",
     "color": "amber",
     "at": [
      "build",
      0.35
     ],
     "size": 50
    },
    {
     "text": "BUILT 1887-1889",
     "color": "cyan",
     "at": [
      "build",
      0.7
     ],
     "size": 46
    }
   ]
  },
  {
   "style": "title",
   "text": "MOSTLY AIR!",
   "color": "cyan",
   "y": 430,
   "size": 100,
   "when": [
    "air",
    0.3
   ],
   "until": [
    "air",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "MELT IT!",
   "color": "red",
   "y": 430,
   "size": 100,
   "when": [
    "melt",
    0.1
   ],
   "until": [
    "melt",
    "end",
    0.2
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "green",
   "when": [
    "payoff",
    "start"
   ],
   "until": [
    "payoff",
    "end",
    0.3
   ],
   "rows": [
    {
     "text": "7,300 t of iron",
     "color": "white",
     "at": [
      "payoff",
      0.02
     ],
     "size": 48
    },
    {
     "text": "spread over its 125 m base",
     "color": "white",
     "at": [
      "payoff",
      0.1
     ],
     "size": 42
    },
    {
     "text": "= about 6 cm thick!",
     "color": "amber",
     "at": [
      "payoff",
      0.25
     ],
     "size": 56
    }
   ]
  },
  {
   "style": "end",
   "title1": "THE EIFFEL TOWER",
   "title2": "IS MOSTLY AIR",
   "sub": "7,300 t of iron over its 125 m base = about 6 cm thick",
   "cta": "Follow for more Block Lab",
   "when": [
    "payoff",
    "end",
    0.2
   ]
  }
 ]
}

In [ ]:
%%writefile stories/eiffel_build_wide.json
{
 "title": "Building the Eiffel Tower",
 "slug": "eiffel_build_wide",
 "format": "wide",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "tail": 0.9,
 "world": {
  "ground": "paris",
  "time": "sunset",
  "weather": "clouds",
  "animals": [
   "pigeons"
  ]
 },
 "lines": [
  {
   "id": "start",
   "text": "Alright... let's build the Eiffel Tower. Block by block.",
   "pause": 4.2
  },
  {
   "id": "done",
   "text": "Voila! Not bad for an old man, eh?",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   1.45,
   -0.35
  ],
  "outfit": {
   "hat": "beret",
   "shirt": "striped",
   "extras": [
    "scarf"
   ]
  }
 },
 "structures": {
  "tower": {
   "cells": [
    [
     -8,
     -8,
     0
    ],
    [
     -7,
     -8,
     0
    ],
    [
     -6,
     -8,
     0
    ],
    [
     -5,
     -8,
     0
    ],
    [
     -4,
     -8,
     0
    ],
    [
     4,
     -8,
     0
    ],
    [
     5,
     -8,
     0
    ],
    [
     6,
     -8,
     0
    ],
    [
     7,
     -8,
     0
    ],
    [
     8,
     -8,
     0
    ],
    [
     -8,
     -7,
     0
    ],
    [
     -4,
     -7,
     0
    ],
    [
     4,
     -7,
     0
    ],
    [
     8,
     -7,
     0
    ],
    [
     -8,
     -6,
     0
    ],
    [
     -4,
     -6,
     0
    ],
    [
     4,
     -6,
     0
    ],
    [
     8,
     -6,
     0
    ],
    [
     -8,
     -5,
     0
    ],
    [
     -4,
     -5,
     0
    ],
    [
     4,
     -5,
     0
    ],
    [
     8,
     -5,
     0
    ],
    [
     -8,
     -4,
     0
    ],
    [
     -7,
     -4,
     0
    ],
    [
     -6,
     -4,
     0
    ],
    [
     -5,
     -4,
     0
    ],
    [
     -4,
     -4,
     0
    ],
    [
     4,
     -4,
     0
    ],
    [
     5,
     -4,
     0
    ],
    [
     6,
     -4,
     0
    ],
    [
     7,
     -4,
     0
    ],
    [
     8,
     -4,
     0
    ],
    [
     -8,
     4,
     0
    ],
    [
     -7,
     4,
     0
    ],
    [
     -6,
     4,
     0
    ],
    [
     -5,
     4,
     0
    ],
    [
     -4,
     4,
     0
    ],
    [
     4,
     4,
     0
    ],
    [
     5,
     4,
     0
    ],
    [
     6,
     4,
     0
    ],
    [
     7,
     4,
     0
    ],
    [
     8,
     4,
     0
    ],
    [
     -8,
     5,
     0
    ],
    [
     -4,
     5,
     0
    ],
    [
     4,
     5,
     0
    ],
    [
     8,
     5,
     0
    ],
    [
     -8,
     6,
     0
    ],
    [
     -4,
     6,
     0
    ],
    [
     4,
     6,
     0
    ],
    [
     8,
     6,
     0
    ],
    [
     -8,
     7,
     0
    ],
    [
     -4,
     7,
     0
    ],
    [
     4,
     7,
     0
    ],
    [
     8,
     7,
     0
    ],
    [
     -8,
     8,
     0
    ],
    [
     -7,
     8,
     0
    ],
    [
     -6,
     8,
     0
    ],
    [
     -5,
     8,
     0
    ],
    [
     -4,
     8,
     0
    ],
    [
     4,
     8,
     0
    ],
    [
     5,
     8,
     0
    ],
    [
     6,
     8,
     0
    ],
    [
     7,
     8,
     0
    ],
    [
     8,
     8,
     0
    ],
    [
     -7,
     -7,
     1
    ],
    [
     -4,
     -7,
     1
    ],
    [
     4,
     -7,
     1
    ],
    [
     7,
     -7,
     1
    ],
    [
     -7,
     -4,
     1
    ],
    [
     -4,
     -4,
     1
    ],
    [
     4,
     -4,
     1
    ],
    [
     7,
     -4,
     1
    ],
    [
     -7,
     4,
     1
    ],
    [
     -4,
     4,
     1
    ],
    [
     4,
     4,
     1
    ],
    [
     7,
     4,
     1
    ],
    [
     -7,
     7,
     1
    ],
    [
     -4,
     7,
     1
    ],
    [
     4,
     7,
     1
    ],
    [
     7,
     7,
     1
    ],
    [
     -6,
     -6,
     2
    ],
    [
     -5,
     -6,
     2
    ],
    [
     -4,
     -6,
     2
    ],
    [
     -3,
     -6,
     2
    ],
    [
     3,
     -6,
     2
    ],
    [
     4,
     -6,
     2
    ],
    [
     5,
     -6,
     2
    ],
    [
     6,
     -6,
     2
    ],
    [
     -6,
     -5,
     2
    ],
    [
     -3,
     -5,
     2
    ],
    [
     3,
     -5,
     2
    ],
    [
     6,
     -5,
     2
    ],
    [
     -6,
     -4,
     2
    ],
    [
     -3,
     -4,
     2
    ],
    [
     3,
     -4,
     2
    ],
    [
     6,
     -4,
     2
    ],
    [
     -6,
     -3,
     2
    ],
    [
     -5,
     -3,
     2
    ],
    [
     -4,
     -3,
     2
    ],
    [
     -3,
     -3,
     2
    ],
    [
     3,
     -3,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     6,
     -3,
     2
    ],
    [
     -6,
     3,
     2
    ],
    [
     -5,
     3,
     2
    ],
    [
     -4,
     3,
     2
    ],
    [
     -3,
     3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     6,
     3,
     2
    ],
    [
     -6,
     4,
     2
    ],
    [
     -3,
     4,
     2
    ],
    [
     3,
     4,
     2
    ],
    [
     6,
     4,
     2
    ],
    [
     -6,
     5,
     2
    ],
    [
     -3,
     5,
     2
    ],
    [
     3,
     5,
     2
    ],
    [
     6,
     5,
     2
    ],
    [
     -6,
     6,
     2
    ],
    [
     -5,
     6,
     2
    ],
    [
     -4,
     6,
     2
    ],
    [
     -3,
     6,
     2
    ],
    [
     3,
     6,
     2
    ],
    [
     4,
     6,
     2
    ],
    [
     5,
     6,
     2
    ],
    [
     6,
     6,
     2
    ],
    [
     -6,
     -6,
     3
    ],
    [
     -3,
     -6,
     3
    ],
    [
     3,
     -6,
     3
    ],
    [
     6,
     -6,
     3
    ],
    [
     -3,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     -6,
     -3,
     3
    ],
    [
     -4,
     -3,
     3
    ],
    [
     -3,
     -3,
     3
    ],
    [
     3,
     -3,
     3
    ],
    [
     4,
     -3,
     3
    ],
    [
     6,
     -3,
     3
    ],
    [
     -6,
     3,
     3
    ],
    [
     -4,
     3,
     3
    ],
    [
     -3,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     -3,
     4,
     3
    ],
    [
     3,
     4,
     3
    ],
    [
     -6,
     6,
     3
    ],
    [
     -3,
     6,
     3
    ],
    [
     3,
     6,
     3
    ],
    [
     6,
     6,
     3
    ],
    [
     -5,
     -5,
     4
    ],
    [
     -4,
     -5,
     4
    ],
    [
     -3,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     -5,
     -4,
     4
    ],
    [
     -3,
     -4,
     4
    ],
    [
     -2,
     -4,
     4
    ],
    [
     -1,
     -4,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     -5,
     -3,
     4
    ],
    [
     -4,
     -3,
     4
    ],
    [
     -3,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     -4,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     -4,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     -4,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     -4,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     -4,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     -5,
     3,
     4
    ],
    [
     -4,
     3,
     4
    ],
    [
     -3,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     -5,
     4,
     4
    ],
    [
     -3,
     4,
     4
    ],
    [
     -2,
     4,
     4
    ],
    [
     -1,
     4,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     -5,
     5,
     4
    ],
    [
     -4,
     5,
     4
    ],
    [
     -3,
     5,
     4
    ],
    [
     3,
     5,
     4
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     4
    ],
    [
     -5,
     -5,
     5
    ],
    [
     -2,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     -2,
     -4,
     5
    ],
    [
     -1,
     -4,
     5
    ],
    [
     0,
     -4,
     5
    ],
    [
     1,
     -4,
     5
    ],
    [
     2,
     -4,
     5
    ],
    [
     -5,
     -2,
     5
    ],
    [
     -4,
     -2,
     5
    ],
    [
     -2,
     -2,
     5
    ],
    [
     2,
     -2,
     5
    ],
    [
     4,
     -2,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     -4,
     -1,
     5
    ],
    [
     4,
     -1,
     5
    ],
    [
     -4,
     0,
     5
    ],
    [
     4,
     0,
     5
    ],
    [
     -4,
     1,
     5
    ],
    [
     4,
     1,
     5
    ],
    [
     -5,
     2,
     5
    ],
    [
     -4,
     2,
     5
    ],
    [
     -2,
     2,
     5
    ],
    [
     2,
     2,
     5
    ],
    [
     4,
     2,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     -2,
     4,
     5
    ],
    [
     -1,
     4,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     -5,
     5,
     5
    ],
    [
     -2,
     5,
     5
    ],
    [
     2,
     5,
     5
    ],
    [
     5,
     5,
     5
    ],
    [
     -5,
     -5,
     6
    ],
    [
     -4,
     -5,
     6
    ],
    [
     -3,
     -5,
     6
    ],
    [
     -2,
     -5,
     6
    ],
    [
     -1,
     -5,
     6
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     -5,
     -4,
     6
    ],
    [
     -4,
     -4,
     6
    ],
    [
     -3,
     -4,
     6
    ],
    [
     -2,
     -4,
     6
    ],
    [
     -1,
     -4,
     6
    ],
    [
     0,
     -4,
     6
    ],
    [
     1,
     -4,
     6
    ],
    [
     2,
     -4,
     6
    ],
    [
     3,
     -4,
     6
    ],
    [
     4,
     -4,
     6
    ],
    [
     5,
     -4,
     6
    ],
    [
     -5,
     -3,
     6
    ],
    [
     -4,
     -3,
     6
    ],
    [
     -3,
     -3,
     6
    ],
    [
     -2,
     -3,
     6
    ],
    [
     -1,
     -3,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     1,
     -3,
     6
    ],
    [
     2,
     -3,
     6
    ],
    [
     3,
     -3,
     6
    ],
    [
     4,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     -5,
     -2,
     6
    ],
    [
     -4,
     -2,
     6
    ],
    [
     -3,
     -2,
     6
    ],
    [
     -2,
     -2,
     6
    ],
    [
     -1,
     -2,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     1,
     -2,
     6
    ],
    [
     2,
     -2,
     6
    ],
    [
     3,
     -2,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     5,
     -2,
     6
    ],
    [
     -5,
     -1,
     6
    ],
    [
     -4,
     -1,
     6
    ],
    [
     -3,
     -1,
     6
    ],
    [
     -2,
     -1,
     6
    ],
    [
     2,
     -1,
     6
    ],
    [
     3,
     -1,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     5,
     -1,
     6
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     6
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     -5,
     1,
     6
    ],
    [
     -4,
     1,
     6
    ],
    [
     -3,
     1,
     6
    ],
    [
     -2,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     -5,
     2,
     6
    ],
    [
     -4,
     2,
     6
    ],
    [
     -3,
     2,
     6
    ],
    [
     -2,
     2,
     6
    ],
    [
     -1,
     2,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     1,
     2,
     6
    ],
    [
     2,
     2,
     6
    ],
    [
     3,
     2,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     5,
     2,
     6
    ],
    [
     -5,
     3,
     6
    ],
    [
     -4,
     3,
     6
    ],
    [
     -3,
     3,
     6
    ],
    [
     -2,
     3,
     6
    ],
    [
     -1,
     3,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     1,
     3,
     6
    ],
    [
     2,
     3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     4,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     -5,
     4,
     6
    ],
    [
     -4,
     4,
     6
    ],
    [
     -3,
     4,
     6
    ],
    [
     -2,
     4,
     6
    ],
    [
     -1,
     4,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     -5,
     5,
     6
    ],
    [
     -4,
     5,
     6
    ],
    [
     -3,
     5,
     6
    ],
    [
     -2,
     5,
     6
    ],
    [
     -1,
     5,
     6
    ],
    [
     0,
     5,
     6
    ],
    [
     1,
     5,
     6
    ],
    [
     2,
     5,
     6
    ],
    [
     3,
     5,
     6
    ],
    [
     4,
     5,
     6
    ],
    [
     5,
     5,
     6
    ],
    [
     -5,
     -5,
     7
    ],
    [
     -3,
     -5,
     7
    ],
    [
     -1,
     -5,
     7
    ],
    [
     1,
     -5,
     7
    ],
    [
     3,
     -5,
     7
    ],
    [
     5,
     -5,
     7
    ],
    [
     -4,
     -4,
     7
    ],
    [
     -2,
     -4,
     7
    ],
    [
     2,
     -4,
     7
    ],
    [
     4,
     -4,
     7
    ],
    [
     -5,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     -4,
     -2,
     7
    ],
    [
     -2,
     -2,
     7
    ],
    [
     2,
     -2,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     -5,
     -1,
     7
    ],
    [
     5,
     -1,
     7
    ],
    [
     -5,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     -4,
     2,
     7
    ],
    [
     -2,
     2,
     7
    ],
    [
     2,
     2,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     -5,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     -4,
     4,
     7
    ],
    [
     -2,
     4,
     7
    ],
    [
     2,
     4,
     7
    ],
    [
     4,
     4,
     7
    ],
    [
     -5,
     5,
     7
    ],
    [
     -3,
     5,
     7
    ],
    [
     -1,
     5,
     7
    ],
    [
     1,
     5,
     7
    ],
    [
     3,
     5,
     7
    ],
    [
     5,
     5,
     7
    ],
    [
     -4,
     -4,
     8
    ],
    [
     -3,
     -4,
     8
    ],
    [
     -2,
     -4,
     8
    ],
    [
     -1,
     -4,
     8
    ],
    [
     1,
     -4,
     8
    ],
    [
     2,
     -4,
     8
    ],
    [
     3,
     -4,
     8
    ],
    [
     4,
     -4,
     8
    ],
    [
     -4,
     -3,
     8
    ],
    [
     -1,
     -3,
     8
    ],
    [
     1,
     -3,
     8
    ],
    [
     4,
     -3,
     8
    ],
    [
     -4,
     -2,
     8
    ],
    [
     -1,
     -2,
     8
    ],
    [
     1,
     -2,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     -4,
     -1,
     8
    ],
    [
     -3,
     -1,
     8
    ],
    [
     -2,
     -1,
     8
    ],
    [
     -1,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     3,
     -1,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     -4,
     1,
     8
    ],
    [
     -3,
     1,
     8
    ],
    [
     -2,
     1,
     8
    ],
    [
     -1,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     -4,
     2,
     8
    ],
    [
     -1,
     2,
     8
    ],
    [
     1,
     2,
     8
    ],
    [
     4,
     2,
     8
    ],
    [
     -4,
     3,
     8
    ],
    [
     -1,
     3,
     8
    ],
    [
     1,
     3,
     8
    ],
    [
     4,
     3,
     8
    ],
    [
     -4,
     4,
     8
    ],
    [
     -3,
     4,
     8
    ],
    [
     -2,
     4,
     8
    ],
    [
     -1,
     4,
     8
    ],
    [
     1,
     4,
     8
    ],
    [
     2,
     4,
     8
    ],
    [
     3,
     4,
     8
    ],
    [
     4,
     4,
     8
    ],
    [
     -3,
     -3,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     1,
     -3,
     9
    ],
    [
     3,
     -3,
     9
    ],
    [
     -3,
     -1,
     9
    ],
    [
     -1,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     3,
     -1,
     9
    ],
    [
     -3,
     1,
     9
    ],
    [
     -1,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     3,
     1,
     9
    ],
    [
     -3,
     3,
     9
    ],
    [
     -1,
     3,
     9
    ],
    [
     1,
     3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     -3,
     -3,
     10
    ],
    [
     -2,
     -3,
     10
    ],
    [
     -1,
     -3,
     10
    ],
    [
     1,
     -3,
     10
    ],
    [
     2,
     -3,
     10
    ],
    [
     3,
     -3,
     10
    ],
    [
     -3,
     -2,
     10
    ],
    [
     -1,
     -2,
     10
    ],
    [
     1,
     -2,
     10
    ],
    [
     3,
     -2,
     10
    ],
    [
     -3,
     -1,
     10
    ],
    [
     -2,
     -1,
     10
    ],
    [
     -1,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     3,
     -1,
     10
    ],
    [
     -3,
     1,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     -1,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ],
    [
     3,
     1,
     10
    ],
    [
     -3,
     2,
     10
    ],
    [
     -1,
     2,
     10
    ],
    [
     1,
     2,
     10
    ],
    [
     3,
     2,
     10
    ],
    [
     -3,
     3,
     10
    ],
    [
     -2,
     3,
     10
    ],
    [
     -1,
     3,
     10
    ],
    [
     1,
     3,
     10
    ],
    [
     2,
     3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     -3,
     -3,
     11
    ],
    [
     -1,
     -3,
     11
    ],
    [
     1,
     -3,
     11
    ],
    [
     3,
     -3,
     11
    ],
    [
     -3,
     -1,
     11
    ],
    [
     -1,
     -1,
     11
    ],
    [
     1,
     -1,
     11
    ],
    [
     3,
     -1,
     11
    ],
    [
     -3,
     1,
     11
    ],
    [
     -1,
     1,
     11
    ],
    [
     1,
     1,
     11
    ],
    [
     3,
     1,
     11
    ],
    [
     -3,
     3,
     11
    ],
    [
     -1,
     3,
     11
    ],
    [
     1,
     3,
     11
    ],
    [
     3,
     3,
     11
    ],
    [
     -3,
     -3,
     12
    ],
    [
     -2,
     -3,
     12
    ],
    [
     -1,
     -3,
     12
    ],
    [
     0,
     -3,
     12
    ],
    [
     1,
     -3,
     12
    ],
    [
     2,
     -3,
     12
    ],
    [
     3,
     -3,
     12
    ],
    [
     -3,
     -2,
     12
    ],
    [
     -2,
     -2,
     12
    ],
    [
     -1,
     -2,
     12
    ],
    [
     0,
     -2,
     12
    ],
    [
     1,
     -2,
     12
    ],
    [
     2,
     -2,
     12
    ],
    [
     3,
     -2,
     12
    ],
    [
     -3,
     -1,
     12
    ],
    [
     -2,
     -1,
     12
    ],
    [
     -1,
     -1,
     12
    ],
    [
     0,
     -1,
     12
    ],
    [
     1,
     -1,
     12
    ],
    [
     2,
     -1,
     12
    ],
    [
     3,
     -1,
     12
    ],
    [
     -3,
     0,
     12
    ],
    [
     -2,
     0,
     12
    ],
    [
     -1,
     0,
     12
    ],
    [
     0,
     0,
     12
    ],
    [
     1,
     0,
     12
    ],
    [
     2,
     0,
     12
    ],
    [
     3,
     0,
     12
    ],
    [
     -3,
     1,
     12
    ],
    [
     -2,
     1,
     12
    ],
    [
     -1,
     1,
     12
    ],
    [
     0,
     1,
     12
    ],
    [
     1,
     1,
     12
    ],
    [
     2,
     1,
     12
    ],
    [
     3,
     1,
     12
    ],
    [
     -3,
     2,
     12
    ],
    [
     -2,
     2,
     12
    ],
    [
     -1,
     2,
     12
    ],
    [
     0,
     2,
     12
    ],
    [
     1,
     2,
     12
    ],
    [
     2,
     2,
     12
    ],
    [
     3,
     2,
     12
    ],
    [
     -3,
     3,
     12
    ],
    [
     -2,
     3,
     12
    ],
    [
     -1,
     3,
     12
    ],
    [
     0,
     3,
     12
    ],
    [
     1,
     3,
     12
    ],
    [
     2,
     3,
     12
    ],
    [
     3,
     3,
     12
    ],
    [
     -3,
     -3,
     13
    ],
    [
     -1,
     -3,
     13
    ],
    [
     1,
     -3,
     13
    ],
    [
     3,
     -3,
     13
    ],
    [
     -2,
     -2,
     13
    ],
    [
     2,
     -2,
     13
    ],
    [
     -3,
     -1,
     13
    ],
    [
     3,
     -1,
     13
    ],
    [
     -3,
     1,
     13
    ],
    [
     3,
     1,
     13
    ],
    [
     -2,
     2,
     13
    ],
    [
     2,
     2,
     13
    ],
    [
     -3,
     3,
     13
    ],
    [
     -1,
     3,
     13
    ],
    [
     1,
     3,
     13
    ],
    [
     3,
     3,
     13
    ],
    [
     -2,
     -2,
     14
    ],
    [
     -1,
     -2,
     14
    ],
    [
     0,
     -2,
     14
    ],
    [
     1,
     -2,
     14
    ],
    [
     2,
     -2,
     14
    ],
    [
     -2,
     -1,
     14
    ],
    [
     2,
     -1,
     14
    ],
    [
     -2,
     0,
     14
    ],
    [
     2,
     0,
     14
    ],
    [
     -2,
     1,
     14
    ],
    [
     2,
     1,
     14
    ],
    [
     -2,
     2,
     14
    ],
    [
     -1,
     2,
     14
    ],
    [
     0,
     2,
     14
    ],
    [
     1,
     2,
     14
    ],
    [
     2,
     2,
     14
    ],
    [
     -2,
     -2,
     15
    ],
    [
     2,
     -2,
     15
    ],
    [
     -2,
     2,
     15
    ],
    [
     2,
     2,
     15
    ],
    [
     -2,
     -2,
     16
    ],
    [
     -1,
     -2,
     16
    ],
    [
     0,
     -2,
     16
    ],
    [
     1,
     -2,
     16
    ],
    [
     2,
     -2,
     16
    ],
    [
     -2,
     -1,
     16
    ],
    [
     2,
     -1,
     16
    ],
    [
     -2,
     0,
     16
    ],
    [
     2,
     0,
     16
    ],
    [
     -2,
     1,
     16
    ],
    [
     2,
     1,
     16
    ],
    [
     -2,
     2,
     16
    ],
    [
     -1,
     2,
     16
    ],
    [
     0,
     2,
     16
    ],
    [
     1,
     2,
     16
    ],
    [
     2,
     2,
     16
    ],
    [
     -2,
     -2,
     17
    ],
    [
     2,
     -2,
     17
    ],
    [
     -2,
     2,
     17
    ],
    [
     2,
     2,
     17
    ],
    [
     -1,
     -1,
     18
    ],
    [
     0,
     -1,
     18
    ],
    [
     1,
     -1,
     18
    ],
    [
     -1,
     0,
     18
    ],
    [
     1,
     0,
     18
    ],
    [
     -1,
     1,
     18
    ],
    [
     0,
     1,
     18
    ],
    [
     1,
     1,
     18
    ],
    [
     -1,
     -1,
     19
    ],
    [
     1,
     -1,
     19
    ],
    [
     -1,
     1,
     19
    ],
    [
     1,
     1,
     19
    ],
    [
     -1,
     -1,
     20
    ],
    [
     0,
     -1,
     20
    ],
    [
     1,
     -1,
     20
    ],
    [
     -1,
     0,
     20
    ],
    [
     1,
     0,
     20
    ],
    [
     -1,
     1,
     20
    ],
    [
     0,
     1,
     20
    ],
    [
     1,
     1,
     20
    ],
    [
     -1,
     -1,
     21
    ],
    [
     1,
     -1,
     21
    ],
    [
     -1,
     1,
     21
    ],
    [
     1,
     1,
     21
    ],
    [
     -1,
     -1,
     22
    ],
    [
     0,
     -1,
     22
    ],
    [
     1,
     -1,
     22
    ],
    [
     -1,
     0,
     22
    ],
    [
     1,
     0,
     22
    ],
    [
     -1,
     1,
     22
    ],
    [
     0,
     1,
     22
    ],
    [
     1,
     1,
     22
    ],
    [
     -1,
     -1,
     23
    ],
    [
     1,
     -1,
     23
    ],
    [
     -1,
     1,
     23
    ],
    [
     1,
     1,
     23
    ],
    [
     -1,
     -1,
     24
    ],
    [
     0,
     -1,
     24
    ],
    [
     1,
     -1,
     24
    ],
    [
     -1,
     0,
     24
    ],
    [
     1,
     0,
     24
    ],
    [
     -1,
     1,
     24
    ],
    [
     0,
     1,
     24
    ],
    [
     1,
     1,
     24
    ],
    [
     -1,
     -1,
     25
    ],
    [
     1,
     -1,
     25
    ],
    [
     -1,
     1,
     25
    ],
    [
     1,
     1,
     25
    ],
    [
     -1,
     -1,
     26
    ],
    [
     0,
     -1,
     26
    ],
    [
     1,
     -1,
     26
    ],
    [
     -1,
     0,
     26
    ],
    [
     1,
     0,
     26
    ],
    [
     -1,
     1,
     26
    ],
    [
     0,
     1,
     26
    ],
    [
     1,
     1,
     26
    ],
    [
     -1,
     -1,
     27
    ],
    [
     1,
     -1,
     27
    ],
    [
     -1,
     1,
     27
    ],
    [
     1,
     1,
     27
    ],
    [
     -1,
     -1,
     28
    ],
    [
     0,
     -1,
     28
    ],
    [
     1,
     -1,
     28
    ],
    [
     -1,
     0,
     28
    ],
    [
     0,
     0,
     28
    ],
    [
     1,
     0,
     28
    ],
    [
     -1,
     1,
     28
    ],
    [
     0,
     1,
     28
    ],
    [
     1,
     1,
     28
    ],
    [
     0,
     0,
     29
    ],
    [
     0,
     0,
     30
    ],
    [
     0,
     0,
     31
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "material": "iron",
   "expect_blocks": 677
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": [
    "start",
    0.6
   ]
  },
  {
   "type": "build",
   "target": "tower",
   "order": "z",
   "stand_at": [
    1.45,
    -0.35
   ],
   "when": [
    "start",
    0.55
   ],
   "until": [
    "done",
    "start",
    -0.25
   ]
  },
  {
   "type": "face_camera",
   "when": [
    "done",
    "start"
   ],
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    0.9,
    0.0,
    0.95
   ],
   "az": -112,
   "el": 6,
   "dist": 3.4
  },
  {
   "at": [
    "start",
    "end"
   ],
   "target": [
    0.45,
    0.0,
    1.55
   ],
   "az": -104,
   "el": 7,
   "dist": 8.0
  },
  {
   "at": [
    "done",
    "start"
   ],
   "target": [
    0.4,
    0.0,
    1.6
   ],
   "az": -72,
   "el": 9,
   "dist": 8.2
  },
  {
   "at": "end",
   "target": [
    0.5,
    0.0,
    1.55
   ],
   "az": -66,
   "el": 8,
   "dist": 7.6
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "THE EIFFEL TOWER",
   "color": "white",
   "y": 78,
   "size": 60,
   "when": [
    "start",
    0.1
   ],
   "until": [
    "start",
    "end",
    0.6
   ]
  },
  {
   "style": "title",
   "text": "{count:tower} / 677 BLOCKS",
   "color": "amber",
   "y": 78,
   "size": 46,
   "when": [
    "start",
    "end",
    0.7
   ],
   "until": [
    "done",
    "start",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "VOILA!",
   "color": "amber",
   "y": 82,
   "size": 76,
   "when": [
    "done",
    0.05
   ],
   "until": "end"
  }
 ]
}

In [ ]:
%%writefile stories/eiffel_full_build.json
{
 "title": "Building the Eiffel Tower (full build preview)",
 "slug": "eiffel_full_build",
 "format": "wide",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "tail": 0.9,
 "world": {
  "ground": "paris",
  "time": "morning",
  "weather": "clouds",
  "animals": [
   "pigeons"
  ]
 },
 "lines": [
  {
   "id": "intro",
   "text": "Bonjour! Today we build the Eiffel Tower... the proper way. Scaffolding first!",
   "pause": 0.2
  },
  {
   "id": "band0",
   "text": "The legs. Four of them, leaning in... like my old knees.",
   "pause": 3.78
  },
  {
   "id": "band1",
   "text": "Up we go. First floor!",
   "pause": 11.57
  },
  {
   "id": "band2",
   "text": "Second floor... my knees are feeling this one.",
   "pause": 6.93
  },
  {
   "id": "band3",
   "text": "Now the long climb up the middle.",
   "pause": 7.66
  },
  {
   "id": "band4",
   "text": "Nearly there... the top deck.",
   "pause": 7.42
  },
  {
   "id": "band5",
   "text": "And the very last piece... the spire.",
   "pause": 5.61
  },
  {
   "id": "down",
   "text": "Now the scaffolding comes down...",
   "pause": 5.56
  },
  {
   "id": "voila",
   "text": "Voila! Not bad for an old man, eh?",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   0.95,
   -0.6,
   0.0
  ],
  "outfit": {
   "hat": "beret",
   "shirt": "striped",
   "extras": [
    "scarf",
    "hammer"
   ]
  }
 },
 "structures": {
  "tower0": {
   "cells": [
    [
     -8,
     -8,
     0
    ],
    [
     -7,
     -8,
     0
    ],
    [
     -6,
     -8,
     0
    ],
    [
     -5,
     -8,
     0
    ],
    [
     -4,
     -8,
     0
    ],
    [
     4,
     -8,
     0
    ],
    [
     5,
     -8,
     0
    ],
    [
     6,
     -8,
     0
    ],
    [
     7,
     -8,
     0
    ],
    [
     8,
     -8,
     0
    ],
    [
     -8,
     -7,
     0
    ],
    [
     -4,
     -7,
     0
    ],
    [
     4,
     -7,
     0
    ],
    [
     8,
     -7,
     0
    ],
    [
     -8,
     -6,
     0
    ],
    [
     -4,
     -6,
     0
    ],
    [
     4,
     -6,
     0
    ],
    [
     8,
     -6,
     0
    ],
    [
     -8,
     -5,
     0
    ],
    [
     -4,
     -5,
     0
    ],
    [
     4,
     -5,
     0
    ],
    [
     8,
     -5,
     0
    ],
    [
     -8,
     -4,
     0
    ],
    [
     -7,
     -4,
     0
    ],
    [
     -6,
     -4,
     0
    ],
    [
     -5,
     -4,
     0
    ],
    [
     -4,
     -4,
     0
    ],
    [
     4,
     -4,
     0
    ],
    [
     5,
     -4,
     0
    ],
    [
     6,
     -4,
     0
    ],
    [
     7,
     -4,
     0
    ],
    [
     8,
     -4,
     0
    ],
    [
     -8,
     4,
     0
    ],
    [
     -7,
     4,
     0
    ],
    [
     -6,
     4,
     0
    ],
    [
     -5,
     4,
     0
    ],
    [
     -4,
     4,
     0
    ],
    [
     4,
     4,
     0
    ],
    [
     5,
     4,
     0
    ],
    [
     6,
     4,
     0
    ],
    [
     7,
     4,
     0
    ],
    [
     8,
     4,
     0
    ],
    [
     -8,
     5,
     0
    ],
    [
     -4,
     5,
     0
    ],
    [
     4,
     5,
     0
    ],
    [
     8,
     5,
     0
    ],
    [
     -8,
     6,
     0
    ],
    [
     -4,
     6,
     0
    ],
    [
     4,
     6,
     0
    ],
    [
     8,
     6,
     0
    ],
    [
     -8,
     7,
     0
    ],
    [
     -4,
     7,
     0
    ],
    [
     4,
     7,
     0
    ],
    [
     8,
     7,
     0
    ],
    [
     -8,
     8,
     0
    ],
    [
     -7,
     8,
     0
    ],
    [
     -6,
     8,
     0
    ],
    [
     -5,
     8,
     0
    ],
    [
     -4,
     8,
     0
    ],
    [
     4,
     8,
     0
    ],
    [
     5,
     8,
     0
    ],
    [
     6,
     8,
     0
    ],
    [
     7,
     8,
     0
    ],
    [
     8,
     8,
     0
    ],
    [
     -7,
     -7,
     1
    ],
    [
     -4,
     -7,
     1
    ],
    [
     4,
     -7,
     1
    ],
    [
     7,
     -7,
     1
    ],
    [
     -7,
     -4,
     1
    ],
    [
     -4,
     -4,
     1
    ],
    [
     4,
     -4,
     1
    ],
    [
     7,
     -4,
     1
    ],
    [
     -7,
     4,
     1
    ],
    [
     -4,
     4,
     1
    ],
    [
     4,
     4,
     1
    ],
    [
     7,
     4,
     1
    ],
    [
     -7,
     7,
     1
    ],
    [
     -4,
     7,
     1
    ],
    [
     4,
     7,
     1
    ],
    [
     7,
     7,
     1
    ],
    [
     -6,
     -6,
     2
    ],
    [
     -5,
     -6,
     2
    ],
    [
     -4,
     -6,
     2
    ],
    [
     -3,
     -6,
     2
    ],
    [
     3,
     -6,
     2
    ],
    [
     4,
     -6,
     2
    ],
    [
     5,
     -6,
     2
    ],
    [
     6,
     -6,
     2
    ],
    [
     -6,
     -5,
     2
    ],
    [
     -3,
     -5,
     2
    ],
    [
     3,
     -5,
     2
    ],
    [
     6,
     -5,
     2
    ],
    [
     -6,
     -4,
     2
    ],
    [
     -3,
     -4,
     2
    ],
    [
     3,
     -4,
     2
    ],
    [
     6,
     -4,
     2
    ],
    [
     -6,
     -3,
     2
    ],
    [
     -5,
     -3,
     2
    ],
    [
     -4,
     -3,
     2
    ],
    [
     -3,
     -3,
     2
    ],
    [
     3,
     -3,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     6,
     -3,
     2
    ],
    [
     -6,
     3,
     2
    ],
    [
     -5,
     3,
     2
    ],
    [
     -4,
     3,
     2
    ],
    [
     -3,
     3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     6,
     3,
     2
    ],
    [
     -6,
     4,
     2
    ],
    [
     -3,
     4,
     2
    ],
    [
     3,
     4,
     2
    ],
    [
     6,
     4,
     2
    ],
    [
     -6,
     5,
     2
    ],
    [
     -3,
     5,
     2
    ],
    [
     3,
     5,
     2
    ],
    [
     6,
     5,
     2
    ],
    [
     -6,
     6,
     2
    ],
    [
     -5,
     6,
     2
    ],
    [
     -4,
     6,
     2
    ],
    [
     -3,
     6,
     2
    ],
    [
     3,
     6,
     2
    ],
    [
     4,
     6,
     2
    ],
    [
     5,
     6,
     2
    ],
    [
     6,
     6,
     2
    ],
    [
     -6,
     -6,
     3
    ],
    [
     -3,
     -6,
     3
    ],
    [
     3,
     -6,
     3
    ],
    [
     6,
     -6,
     3
    ],
    [
     -3,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     -6,
     -3,
     3
    ],
    [
     -4,
     -3,
     3
    ],
    [
     -3,
     -3,
     3
    ],
    [
     3,
     -3,
     3
    ],
    [
     4,
     -3,
     3
    ],
    [
     6,
     -3,
     3
    ],
    [
     -6,
     3,
     3
    ],
    [
     -4,
     3,
     3
    ],
    [
     -3,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     -3,
     4,
     3
    ],
    [
     3,
     4,
     3
    ],
    [
     -6,
     6,
     3
    ],
    [
     -3,
     6,
     3
    ],
    [
     3,
     6,
     3
    ],
    [
     6,
     6,
     3
    ],
    [
     -5,
     -5,
     4
    ],
    [
     -4,
     -5,
     4
    ],
    [
     -3,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     -5,
     -4,
     4
    ],
    [
     -3,
     -4,
     4
    ],
    [
     -2,
     -4,
     4
    ],
    [
     -1,
     -4,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     -5,
     -3,
     4
    ],
    [
     -4,
     -3,
     4
    ],
    [
     -3,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     -4,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     -4,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     -4,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     -4,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     -4,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     -5,
     3,
     4
    ],
    [
     -4,
     3,
     4
    ],
    [
     -3,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     -5,
     4,
     4
    ],
    [
     -3,
     4,
     4
    ],
    [
     -2,
     4,
     4
    ],
    [
     -1,
     4,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     -5,
     5,
     4
    ],
    [
     -4,
     5,
     4
    ],
    [
     -3,
     5,
     4
    ],
    [
     3,
     5,
     4
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     4
    ],
    [
     -5,
     -5,
     5
    ],
    [
     -2,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     -2,
     -4,
     5
    ],
    [
     -1,
     -4,
     5
    ],
    [
     0,
     -4,
     5
    ],
    [
     1,
     -4,
     5
    ],
    [
     2,
     -4,
     5
    ],
    [
     -5,
     -2,
     5
    ],
    [
     -4,
     -2,
     5
    ],
    [
     -2,
     -2,
     5
    ],
    [
     2,
     -2,
     5
    ],
    [
     4,
     -2,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     -4,
     -1,
     5
    ],
    [
     4,
     -1,
     5
    ],
    [
     -4,
     0,
     5
    ],
    [
     4,
     0,
     5
    ],
    [
     -4,
     1,
     5
    ],
    [
     4,
     1,
     5
    ],
    [
     -5,
     2,
     5
    ],
    [
     -4,
     2,
     5
    ],
    [
     -2,
     2,
     5
    ],
    [
     2,
     2,
     5
    ],
    [
     4,
     2,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     -2,
     4,
     5
    ],
    [
     -1,
     4,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     -5,
     5,
     5
    ],
    [
     -2,
     5,
     5
    ],
    [
     2,
     5,
     5
    ],
    [
     5,
     5,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 240
  },
  "tower1": {
   "cells": [
    [
     -5,
     -5,
     6
    ],
    [
     -4,
     -5,
     6
    ],
    [
     -3,
     -5,
     6
    ],
    [
     -2,
     -5,
     6
    ],
    [
     -1,
     -5,
     6
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     -5,
     -4,
     6
    ],
    [
     -4,
     -4,
     6
    ],
    [
     -3,
     -4,
     6
    ],
    [
     -2,
     -4,
     6
    ],
    [
     -1,
     -4,
     6
    ],
    [
     0,
     -4,
     6
    ],
    [
     1,
     -4,
     6
    ],
    [
     2,
     -4,
     6
    ],
    [
     3,
     -4,
     6
    ],
    [
     4,
     -4,
     6
    ],
    [
     5,
     -4,
     6
    ],
    [
     -5,
     -3,
     6
    ],
    [
     -4,
     -3,
     6
    ],
    [
     -3,
     -3,
     6
    ],
    [
     -2,
     -3,
     6
    ],
    [
     -1,
     -3,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     1,
     -3,
     6
    ],
    [
     2,
     -3,
     6
    ],
    [
     3,
     -3,
     6
    ],
    [
     4,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     -5,
     -2,
     6
    ],
    [
     -4,
     -2,
     6
    ],
    [
     -3,
     -2,
     6
    ],
    [
     -2,
     -2,
     6
    ],
    [
     -1,
     -2,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     1,
     -2,
     6
    ],
    [
     2,
     -2,
     6
    ],
    [
     3,
     -2,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     5,
     -2,
     6
    ],
    [
     -5,
     -1,
     6
    ],
    [
     -4,
     -1,
     6
    ],
    [
     -3,
     -1,
     6
    ],
    [
     -2,
     -1,
     6
    ],
    [
     2,
     -1,
     6
    ],
    [
     3,
     -1,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     5,
     -1,
     6
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     6
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     -5,
     1,
     6
    ],
    [
     -4,
     1,
     6
    ],
    [
     -3,
     1,
     6
    ],
    [
     -2,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     -5,
     2,
     6
    ],
    [
     -4,
     2,
     6
    ],
    [
     -3,
     2,
     6
    ],
    [
     -2,
     2,
     6
    ],
    [
     -1,
     2,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     1,
     2,
     6
    ],
    [
     2,
     2,
     6
    ],
    [
     3,
     2,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     5,
     2,
     6
    ],
    [
     -5,
     3,
     6
    ],
    [
     -4,
     3,
     6
    ],
    [
     -3,
     3,
     6
    ],
    [
     -2,
     3,
     6
    ],
    [
     -1,
     3,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     1,
     3,
     6
    ],
    [
     2,
     3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     4,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     -5,
     4,
     6
    ],
    [
     -4,
     4,
     6
    ],
    [
     -3,
     4,
     6
    ],
    [
     -2,
     4,
     6
    ],
    [
     -1,
     4,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     -5,
     5,
     6
    ],
    [
     -4,
     5,
     6
    ],
    [
     -3,
     5,
     6
    ],
    [
     -2,
     5,
     6
    ],
    [
     -1,
     5,
     6
    ],
    [
     0,
     5,
     6
    ],
    [
     1,
     5,
     6
    ],
    [
     2,
     5,
     6
    ],
    [
     3,
     5,
     6
    ],
    [
     4,
     5,
     6
    ],
    [
     5,
     5,
     6
    ],
    [
     -5,
     -5,
     7
    ],
    [
     -3,
     -5,
     7
    ],
    [
     -1,
     -5,
     7
    ],
    [
     1,
     -5,
     7
    ],
    [
     3,
     -5,
     7
    ],
    [
     5,
     -5,
     7
    ],
    [
     -4,
     -4,
     7
    ],
    [
     -2,
     -4,
     7
    ],
    [
     2,
     -4,
     7
    ],
    [
     4,
     -4,
     7
    ],
    [
     -5,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     -4,
     -2,
     7
    ],
    [
     -2,
     -2,
     7
    ],
    [
     2,
     -2,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     -5,
     -1,
     7
    ],
    [
     5,
     -1,
     7
    ],
    [
     -5,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     -4,
     2,
     7
    ],
    [
     -2,
     2,
     7
    ],
    [
     2,
     2,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     -5,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     -4,
     4,
     7
    ],
    [
     -2,
     4,
     7
    ],
    [
     2,
     4,
     7
    ],
    [
     4,
     4,
     7
    ],
    [
     -5,
     5,
     7
    ],
    [
     -3,
     5,
     7
    ],
    [
     -1,
     5,
     7
    ],
    [
     1,
     5,
     7
    ],
    [
     3,
     5,
     7
    ],
    [
     5,
     5,
     7
    ],
    [
     -4,
     -4,
     8
    ],
    [
     -3,
     -4,
     8
    ],
    [
     -2,
     -4,
     8
    ],
    [
     -1,
     -4,
     8
    ],
    [
     1,
     -4,
     8
    ],
    [
     2,
     -4,
     8
    ],
    [
     3,
     -4,
     8
    ],
    [
     4,
     -4,
     8
    ],
    [
     -4,
     -3,
     8
    ],
    [
     -1,
     -3,
     8
    ],
    [
     1,
     -3,
     8
    ],
    [
     4,
     -3,
     8
    ],
    [
     -4,
     -2,
     8
    ],
    [
     -1,
     -2,
     8
    ],
    [
     1,
     -2,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     -4,
     -1,
     8
    ],
    [
     -3,
     -1,
     8
    ],
    [
     -2,
     -1,
     8
    ],
    [
     -1,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     3,
     -1,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     -4,
     1,
     8
    ],
    [
     -3,
     1,
     8
    ],
    [
     -2,
     1,
     8
    ],
    [
     -1,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     -4,
     2,
     8
    ],
    [
     -1,
     2,
     8
    ],
    [
     1,
     2,
     8
    ],
    [
     4,
     2,
     8
    ],
    [
     -4,
     3,
     8
    ],
    [
     -1,
     3,
     8
    ],
    [
     1,
     3,
     8
    ],
    [
     4,
     3,
     8
    ],
    [
     -4,
     4,
     8
    ],
    [
     -3,
     4,
     8
    ],
    [
     -2,
     4,
     8
    ],
    [
     -1,
     4,
     8
    ],
    [
     1,
     4,
     8
    ],
    [
     2,
     4,
     8
    ],
    [
     3,
     4,
     8
    ],
    [
     4,
     4,
     8
    ],
    [
     -3,
     -3,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     1,
     -3,
     9
    ],
    [
     3,
     -3,
     9
    ],
    [
     -3,
     -1,
     9
    ],
    [
     -1,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     3,
     -1,
     9
    ],
    [
     -3,
     1,
     9
    ],
    [
     -1,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     3,
     1,
     9
    ],
    [
     -3,
     3,
     9
    ],
    [
     -1,
     3,
     9
    ],
    [
     1,
     3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     -3,
     -3,
     10
    ],
    [
     -2,
     -3,
     10
    ],
    [
     -1,
     -3,
     10
    ],
    [
     1,
     -3,
     10
    ],
    [
     2,
     -3,
     10
    ],
    [
     3,
     -3,
     10
    ],
    [
     -3,
     -2,
     10
    ],
    [
     -1,
     -2,
     10
    ],
    [
     1,
     -2,
     10
    ],
    [
     3,
     -2,
     10
    ],
    [
     -3,
     -1,
     10
    ],
    [
     -2,
     -1,
     10
    ],
    [
     -1,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     3,
     -1,
     10
    ],
    [
     -3,
     1,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     -1,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ],
    [
     3,
     1,
     10
    ],
    [
     -3,
     2,
     10
    ],
    [
     -1,
     2,
     10
    ],
    [
     1,
     2,
     10
    ],
    [
     3,
     2,
     10
    ],
    [
     -3,
     3,
     10
    ],
    [
     -2,
     3,
     10
    ],
    [
     -1,
     3,
     10
    ],
    [
     1,
     3,
     10
    ],
    [
     2,
     3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     -3,
     -3,
     11
    ],
    [
     -1,
     -3,
     11
    ],
    [
     1,
     -3,
     11
    ],
    [
     3,
     -3,
     11
    ],
    [
     -3,
     -1,
     11
    ],
    [
     -1,
     -1,
     11
    ],
    [
     1,
     -1,
     11
    ],
    [
     3,
     -1,
     11
    ],
    [
     -3,
     1,
     11
    ],
    [
     -1,
     1,
     11
    ],
    [
     1,
     1,
     11
    ],
    [
     3,
     1,
     11
    ],
    [
     -3,
     3,
     11
    ],
    [
     -1,
     3,
     11
    ],
    [
     1,
     3,
     11
    ],
    [
     3,
     3,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 260
  },
  "tower2": {
   "cells": [
    [
     -3,
     -3,
     12
    ],
    [
     -2,
     -3,
     12
    ],
    [
     -1,
     -3,
     12
    ],
    [
     0,
     -3,
     12
    ],
    [
     1,
     -3,
     12
    ],
    [
     2,
     -3,
     12
    ],
    [
     3,
     -3,
     12
    ],
    [
     -3,
     -2,
     12
    ],
    [
     -2,
     -2,
     12
    ],
    [
     -1,
     -2,
     12
    ],
    [
     0,
     -2,
     12
    ],
    [
     1,
     -2,
     12
    ],
    [
     2,
     -2,
     12
    ],
    [
     3,
     -2,
     12
    ],
    [
     -3,
     -1,
     12
    ],
    [
     -2,
     -1,
     12
    ],
    [
     -1,
     -1,
     12
    ],
    [
     0,
     -1,
     12
    ],
    [
     1,
     -1,
     12
    ],
    [
     2,
     -1,
     12
    ],
    [
     3,
     -1,
     12
    ],
    [
     -3,
     0,
     12
    ],
    [
     -2,
     0,
     12
    ],
    [
     -1,
     0,
     12
    ],
    [
     0,
     0,
     12
    ],
    [
     1,
     0,
     12
    ],
    [
     2,
     0,
     12
    ],
    [
     3,
     0,
     12
    ],
    [
     -3,
     1,
     12
    ],
    [
     -2,
     1,
     12
    ],
    [
     -1,
     1,
     12
    ],
    [
     0,
     1,
     12
    ],
    [
     1,
     1,
     12
    ],
    [
     2,
     1,
     12
    ],
    [
     3,
     1,
     12
    ],
    [
     -3,
     2,
     12
    ],
    [
     -2,
     2,
     12
    ],
    [
     -1,
     2,
     12
    ],
    [
     0,
     2,
     12
    ],
    [
     1,
     2,
     12
    ],
    [
     2,
     2,
     12
    ],
    [
     3,
     2,
     12
    ],
    [
     -3,
     3,
     12
    ],
    [
     -2,
     3,
     12
    ],
    [
     -1,
     3,
     12
    ],
    [
     0,
     3,
     12
    ],
    [
     1,
     3,
     12
    ],
    [
     2,
     3,
     12
    ],
    [
     3,
     3,
     12
    ],
    [
     -3,
     -3,
     13
    ],
    [
     -1,
     -3,
     13
    ],
    [
     1,
     -3,
     13
    ],
    [
     3,
     -3,
     13
    ],
    [
     -2,
     -2,
     13
    ],
    [
     2,
     -2,
     13
    ],
    [
     -3,
     -1,
     13
    ],
    [
     3,
     -1,
     13
    ],
    [
     -3,
     1,
     13
    ],
    [
     3,
     1,
     13
    ],
    [
     -2,
     2,
     13
    ],
    [
     2,
     2,
     13
    ],
    [
     -3,
     3,
     13
    ],
    [
     -1,
     3,
     13
    ],
    [
     1,
     3,
     13
    ],
    [
     3,
     3,
     13
    ],
    [
     -2,
     -2,
     14
    ],
    [
     -1,
     -2,
     14
    ],
    [
     0,
     -2,
     14
    ],
    [
     1,
     -2,
     14
    ],
    [
     2,
     -2,
     14
    ],
    [
     -2,
     -1,
     14
    ],
    [
     2,
     -1,
     14
    ],
    [
     -2,
     0,
     14
    ],
    [
     2,
     0,
     14
    ],
    [
     -2,
     1,
     14
    ],
    [
     2,
     1,
     14
    ],
    [
     -2,
     2,
     14
    ],
    [
     -1,
     2,
     14
    ],
    [
     0,
     2,
     14
    ],
    [
     1,
     2,
     14
    ],
    [
     2,
     2,
     14
    ],
    [
     -2,
     -2,
     15
    ],
    [
     2,
     -2,
     15
    ],
    [
     -2,
     2,
     15
    ],
    [
     2,
     2,
     15
    ],
    [
     -2,
     -2,
     16
    ],
    [
     -1,
     -2,
     16
    ],
    [
     0,
     -2,
     16
    ],
    [
     1,
     -2,
     16
    ],
    [
     2,
     -2,
     16
    ],
    [
     -2,
     -1,
     16
    ],
    [
     2,
     -1,
     16
    ],
    [
     -2,
     0,
     16
    ],
    [
     2,
     0,
     16
    ],
    [
     -2,
     1,
     16
    ],
    [
     2,
     1,
     16
    ],
    [
     -2,
     2,
     16
    ],
    [
     -1,
     2,
     16
    ],
    [
     0,
     2,
     16
    ],
    [
     1,
     2,
     16
    ],
    [
     2,
     2,
     16
    ],
    [
     -2,
     -2,
     17
    ],
    [
     2,
     -2,
     17
    ],
    [
     -2,
     2,
     17
    ],
    [
     2,
     2,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 105
  },
  "tower3": {
   "cells": [
    [
     -1,
     -1,
     18
    ],
    [
     0,
     -1,
     18
    ],
    [
     1,
     -1,
     18
    ],
    [
     -1,
     0,
     18
    ],
    [
     1,
     0,
     18
    ],
    [
     -1,
     1,
     18
    ],
    [
     0,
     1,
     18
    ],
    [
     1,
     1,
     18
    ],
    [
     -1,
     -1,
     19
    ],
    [
     1,
     -1,
     19
    ],
    [
     -1,
     1,
     19
    ],
    [
     1,
     1,
     19
    ],
    [
     -1,
     -1,
     20
    ],
    [
     0,
     -1,
     20
    ],
    [
     1,
     -1,
     20
    ],
    [
     -1,
     0,
     20
    ],
    [
     1,
     0,
     20
    ],
    [
     -1,
     1,
     20
    ],
    [
     0,
     1,
     20
    ],
    [
     1,
     1,
     20
    ],
    [
     -1,
     -1,
     21
    ],
    [
     1,
     -1,
     21
    ],
    [
     -1,
     1,
     21
    ],
    [
     1,
     1,
     21
    ],
    [
     -1,
     -1,
     22
    ],
    [
     0,
     -1,
     22
    ],
    [
     1,
     -1,
     22
    ],
    [
     -1,
     0,
     22
    ],
    [
     1,
     0,
     22
    ],
    [
     -1,
     1,
     22
    ],
    [
     0,
     1,
     22
    ],
    [
     1,
     1,
     22
    ],
    [
     -1,
     -1,
     23
    ],
    [
     1,
     -1,
     23
    ],
    [
     -1,
     1,
     23
    ],
    [
     1,
     1,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 36
  },
  "tower4": {
   "cells": [
    [
     -1,
     -1,
     24
    ],
    [
     0,
     -1,
     24
    ],
    [
     1,
     -1,
     24
    ],
    [
     -1,
     0,
     24
    ],
    [
     1,
     0,
     24
    ],
    [
     -1,
     1,
     24
    ],
    [
     0,
     1,
     24
    ],
    [
     1,
     1,
     24
    ],
    [
     -1,
     -1,
     25
    ],
    [
     1,
     -1,
     25
    ],
    [
     -1,
     1,
     25
    ],
    [
     1,
     1,
     25
    ],
    [
     -1,
     -1,
     26
    ],
    [
     0,
     -1,
     26
    ],
    [
     1,
     -1,
     26
    ],
    [
     -1,
     0,
     26
    ],
    [
     1,
     0,
     26
    ],
    [
     -1,
     1,
     26
    ],
    [
     0,
     1,
     26
    ],
    [
     1,
     1,
     26
    ],
    [
     -1,
     -1,
     27
    ],
    [
     1,
     -1,
     27
    ],
    [
     -1,
     1,
     27
    ],
    [
     1,
     1,
     27
    ],
    [
     -1,
     -1,
     28
    ],
    [
     0,
     -1,
     28
    ],
    [
     1,
     -1,
     28
    ],
    [
     -1,
     0,
     28
    ],
    [
     0,
     0,
     28
    ],
    [
     1,
     0,
     28
    ],
    [
     -1,
     1,
     28
    ],
    [
     0,
     1,
     28
    ],
    [
     1,
     1,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 33
  },
  "tower5": {
   "cells": [
    [
     0,
     0,
     29
    ],
    [
     0,
     0,
     30
    ],
    [
     0,
     0,
     31
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 3
  },
  "posts0": {
   "cells": [
    [
     10,
     -3,
     0
    ],
    [
     10,
     3,
     0
    ],
    [
     15,
     -3,
     0
    ],
    [
     15,
     3,
     0
    ],
    [
     10,
     -3,
     1
    ],
    [
     10,
     3,
     1
    ],
    [
     15,
     -3,
     1
    ],
    [
     15,
     3,
     1
    ],
    [
     10,
     -3,
     2
    ],
    [
     10,
     3,
     2
    ],
    [
     15,
     -3,
     2
    ],
    [
     15,
     3,
     2
    ],
    [
     10,
     -3,
     3
    ],
    [
     10,
     3,
     3
    ],
    [
     15,
     -3,
     3
    ],
    [
     15,
     3,
     3
    ],
    [
     10,
     -3,
     4
    ],
    [
     10,
     3,
     4
    ],
    [
     15,
     -3,
     4
    ],
    [
     15,
     3,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks0": {
   "cells": [
    [
     10,
     -3,
     5
    ],
    [
     10,
     -2,
     5
    ],
    [
     10,
     -1,
     5
    ],
    [
     10,
     0,
     5
    ],
    [
     10,
     1,
     5
    ],
    [
     10,
     2,
     5
    ],
    [
     10,
     3,
     5
    ],
    [
     11,
     -3,
     5
    ],
    [
     11,
     -2,
     5
    ],
    [
     11,
     -1,
     5
    ],
    [
     11,
     0,
     5
    ],
    [
     11,
     1,
     5
    ],
    [
     11,
     2,
     5
    ],
    [
     11,
     3,
     5
    ],
    [
     12,
     -3,
     5
    ],
    [
     12,
     -2,
     5
    ],
    [
     12,
     -1,
     5
    ],
    [
     12,
     0,
     5
    ],
    [
     12,
     1,
     5
    ],
    [
     12,
     2,
     5
    ],
    [
     12,
     3,
     5
    ],
    [
     13,
     -3,
     5
    ],
    [
     13,
     -2,
     5
    ],
    [
     13,
     -1,
     5
    ],
    [
     13,
     0,
     5
    ],
    [
     13,
     1,
     5
    ],
    [
     13,
     2,
     5
    ],
    [
     13,
     3,
     5
    ],
    [
     14,
     -3,
     5
    ],
    [
     14,
     -2,
     5
    ],
    [
     14,
     -1,
     5
    ],
    [
     14,
     0,
     5
    ],
    [
     14,
     1,
     5
    ],
    [
     14,
     2,
     5
    ],
    [
     14,
     3,
     5
    ],
    [
     15,
     -3,
     5
    ],
    [
     15,
     -2,
     5
    ],
    [
     15,
     -1,
     5
    ],
    [
     15,
     0,
     5
    ],
    [
     15,
     1,
     5
    ],
    [
     15,
     2,
     5
    ],
    [
     15,
     3,
     5
    ],
    [
     6,
     -1,
     5
    ],
    [
     6,
     0,
     5
    ],
    [
     6,
     1,
     5
    ],
    [
     7,
     -1,
     5
    ],
    [
     7,
     0,
     5
    ],
    [
     7,
     1,
     5
    ],
    [
     8,
     -1,
     5
    ],
    [
     8,
     0,
     5
    ],
    [
     8,
     1,
     5
    ],
    [
     9,
     -1,
     5
    ],
    [
     9,
     0,
     5
    ],
    [
     9,
     1,
     5
    ],
    [
     16,
     0,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 55,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails0": {
   "cells": [
    [
     16,
     -2,
     0
    ],
    [
     16,
     2,
     0
    ],
    [
     16,
     -2,
     1
    ],
    [
     16,
     2,
     1
    ],
    [
     16,
     -2,
     2
    ],
    [
     16,
     2,
     2
    ],
    [
     16,
     -2,
     3
    ],
    [
     16,
     2,
     3
    ],
    [
     16,
     -2,
     4
    ],
    [
     16,
     2,
     4
    ],
    [
     16,
     -2,
     5
    ],
    [
     16,
     2,
     5
    ],
    [
     11,
     -5,
     0
    ],
    [
     15,
     -5,
     0
    ],
    [
     11,
     -5,
     1
    ],
    [
     15,
     -5,
     1
    ],
    [
     11,
     -5,
     2
    ],
    [
     15,
     -5,
     2
    ],
    [
     11,
     -5,
     3
    ],
    [
     15,
     -5,
     3
    ],
    [
     11,
     -5,
     4
    ],
    [
     15,
     -5,
     4
    ],
    [
     11,
     -5,
     5
    ],
    [
     15,
     -5,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs0": {
   "cells": [
    [
     16,
     -1,
     0
    ],
    [
     16,
     0,
     0
    ],
    [
     16,
     1,
     0
    ],
    [
     16,
     -1,
     2
    ],
    [
     16,
     0,
     2
    ],
    [
     16,
     1,
     2
    ],
    [
     16,
     -1,
     4
    ],
    [
     16,
     0,
     4
    ],
    [
     16,
     1,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts1": {
   "cells": [
    [
     10,
     -3,
     6
    ],
    [
     10,
     3,
     6
    ],
    [
     15,
     -3,
     6
    ],
    [
     15,
     3,
     6
    ],
    [
     10,
     -3,
     7
    ],
    [
     10,
     3,
     7
    ],
    [
     15,
     -3,
     7
    ],
    [
     15,
     3,
     7
    ],
    [
     10,
     -3,
     8
    ],
    [
     10,
     3,
     8
    ],
    [
     15,
     -3,
     8
    ],
    [
     15,
     3,
     8
    ],
    [
     10,
     -3,
     9
    ],
    [
     10,
     3,
     9
    ],
    [
     15,
     -3,
     9
    ],
    [
     15,
     3,
     9
    ],
    [
     10,
     -3,
     10
    ],
    [
     10,
     3,
     10
    ],
    [
     15,
     -3,
     10
    ],
    [
     15,
     3,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks1": {
   "cells": [
    [
     10,
     -3,
     11
    ],
    [
     10,
     -2,
     11
    ],
    [
     10,
     -1,
     11
    ],
    [
     10,
     0,
     11
    ],
    [
     10,
     1,
     11
    ],
    [
     10,
     2,
     11
    ],
    [
     10,
     3,
     11
    ],
    [
     11,
     -3,
     11
    ],
    [
     11,
     -2,
     11
    ],
    [
     11,
     -1,
     11
    ],
    [
     11,
     0,
     11
    ],
    [
     11,
     1,
     11
    ],
    [
     11,
     2,
     11
    ],
    [
     11,
     3,
     11
    ],
    [
     12,
     -3,
     11
    ],
    [
     12,
     -2,
     11
    ],
    [
     12,
     -1,
     11
    ],
    [
     12,
     0,
     11
    ],
    [
     12,
     1,
     11
    ],
    [
     12,
     2,
     11
    ],
    [
     12,
     3,
     11
    ],
    [
     13,
     -3,
     11
    ],
    [
     13,
     -2,
     11
    ],
    [
     13,
     -1,
     11
    ],
    [
     13,
     0,
     11
    ],
    [
     13,
     1,
     11
    ],
    [
     13,
     2,
     11
    ],
    [
     13,
     3,
     11
    ],
    [
     14,
     -3,
     11
    ],
    [
     14,
     -2,
     11
    ],
    [
     14,
     -1,
     11
    ],
    [
     14,
     0,
     11
    ],
    [
     14,
     1,
     11
    ],
    [
     14,
     2,
     11
    ],
    [
     14,
     3,
     11
    ],
    [
     15,
     -3,
     11
    ],
    [
     15,
     -2,
     11
    ],
    [
     15,
     -1,
     11
    ],
    [
     15,
     0,
     11
    ],
    [
     15,
     1,
     11
    ],
    [
     15,
     2,
     11
    ],
    [
     15,
     3,
     11
    ],
    [
     4,
     -1,
     11
    ],
    [
     4,
     0,
     11
    ],
    [
     4,
     1,
     11
    ],
    [
     5,
     -1,
     11
    ],
    [
     5,
     0,
     11
    ],
    [
     5,
     1,
     11
    ],
    [
     6,
     -1,
     11
    ],
    [
     6,
     0,
     11
    ],
    [
     6,
     1,
     11
    ],
    [
     7,
     -1,
     11
    ],
    [
     7,
     0,
     11
    ],
    [
     7,
     1,
     11
    ],
    [
     8,
     -1,
     11
    ],
    [
     8,
     0,
     11
    ],
    [
     8,
     1,
     11
    ],
    [
     9,
     -1,
     11
    ],
    [
     9,
     0,
     11
    ],
    [
     9,
     1,
     11
    ],
    [
     16,
     0,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 61,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails1": {
   "cells": [
    [
     16,
     -2,
     6
    ],
    [
     16,
     2,
     6
    ],
    [
     16,
     -2,
     7
    ],
    [
     16,
     2,
     7
    ],
    [
     16,
     -2,
     8
    ],
    [
     16,
     2,
     8
    ],
    [
     16,
     -2,
     9
    ],
    [
     16,
     2,
     9
    ],
    [
     16,
     -2,
     10
    ],
    [
     16,
     2,
     10
    ],
    [
     16,
     -2,
     11
    ],
    [
     16,
     2,
     11
    ],
    [
     11,
     -5,
     6
    ],
    [
     15,
     -5,
     6
    ],
    [
     11,
     -5,
     7
    ],
    [
     15,
     -5,
     7
    ],
    [
     11,
     -5,
     8
    ],
    [
     15,
     -5,
     8
    ],
    [
     11,
     -5,
     9
    ],
    [
     15,
     -5,
     9
    ],
    [
     11,
     -5,
     10
    ],
    [
     15,
     -5,
     10
    ],
    [
     11,
     -5,
     11
    ],
    [
     15,
     -5,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs1": {
   "cells": [
    [
     16,
     -1,
     6
    ],
    [
     16,
     0,
     6
    ],
    [
     16,
     1,
     6
    ],
    [
     16,
     -1,
     8
    ],
    [
     16,
     0,
     8
    ],
    [
     16,
     1,
     8
    ],
    [
     16,
     -1,
     10
    ],
    [
     16,
     0,
     10
    ],
    [
     16,
     1,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts2": {
   "cells": [
    [
     10,
     -3,
     12
    ],
    [
     10,
     3,
     12
    ],
    [
     15,
     -3,
     12
    ],
    [
     15,
     3,
     12
    ],
    [
     10,
     -3,
     13
    ],
    [
     10,
     3,
     13
    ],
    [
     15,
     -3,
     13
    ],
    [
     15,
     3,
     13
    ],
    [
     10,
     -3,
     14
    ],
    [
     10,
     3,
     14
    ],
    [
     15,
     -3,
     14
    ],
    [
     15,
     3,
     14
    ],
    [
     10,
     -3,
     15
    ],
    [
     10,
     3,
     15
    ],
    [
     15,
     -3,
     15
    ],
    [
     15,
     3,
     15
    ],
    [
     10,
     -3,
     16
    ],
    [
     10,
     3,
     16
    ],
    [
     15,
     -3,
     16
    ],
    [
     15,
     3,
     16
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks2": {
   "cells": [
    [
     10,
     -3,
     17
    ],
    [
     10,
     -2,
     17
    ],
    [
     10,
     -1,
     17
    ],
    [
     10,
     0,
     17
    ],
    [
     10,
     1,
     17
    ],
    [
     10,
     2,
     17
    ],
    [
     10,
     3,
     17
    ],
    [
     11,
     -3,
     17
    ],
    [
     11,
     -2,
     17
    ],
    [
     11,
     -1,
     17
    ],
    [
     11,
     0,
     17
    ],
    [
     11,
     1,
     17
    ],
    [
     11,
     2,
     17
    ],
    [
     11,
     3,
     17
    ],
    [
     12,
     -3,
     17
    ],
    [
     12,
     -2,
     17
    ],
    [
     12,
     -1,
     17
    ],
    [
     12,
     0,
     17
    ],
    [
     12,
     1,
     17
    ],
    [
     12,
     2,
     17
    ],
    [
     12,
     3,
     17
    ],
    [
     13,
     -3,
     17
    ],
    [
     13,
     -2,
     17
    ],
    [
     13,
     -1,
     17
    ],
    [
     13,
     0,
     17
    ],
    [
     13,
     1,
     17
    ],
    [
     13,
     2,
     17
    ],
    [
     13,
     3,
     17
    ],
    [
     14,
     -3,
     17
    ],
    [
     14,
     -2,
     17
    ],
    [
     14,
     -1,
     17
    ],
    [
     14,
     0,
     17
    ],
    [
     14,
     1,
     17
    ],
    [
     14,
     2,
     17
    ],
    [
     14,
     3,
     17
    ],
    [
     15,
     -3,
     17
    ],
    [
     15,
     -2,
     17
    ],
    [
     15,
     -1,
     17
    ],
    [
     15,
     0,
     17
    ],
    [
     15,
     1,
     17
    ],
    [
     15,
     2,
     17
    ],
    [
     15,
     3,
     17
    ],
    [
     2,
     -1,
     17
    ],
    [
     2,
     0,
     17
    ],
    [
     2,
     1,
     17
    ],
    [
     3,
     -1,
     17
    ],
    [
     3,
     0,
     17
    ],
    [
     3,
     1,
     17
    ],
    [
     4,
     -1,
     17
    ],
    [
     4,
     0,
     17
    ],
    [
     4,
     1,
     17
    ],
    [
     5,
     -1,
     17
    ],
    [
     5,
     0,
     17
    ],
    [
     5,
     1,
     17
    ],
    [
     6,
     -1,
     17
    ],
    [
     6,
     0,
     17
    ],
    [
     6,
     1,
     17
    ],
    [
     7,
     -1,
     17
    ],
    [
     7,
     0,
     17
    ],
    [
     7,
     1,
     17
    ],
    [
     8,
     -1,
     17
    ],
    [
     8,
     0,
     17
    ],
    [
     8,
     1,
     17
    ],
    [
     9,
     -1,
     17
    ],
    [
     9,
     0,
     17
    ],
    [
     9,
     1,
     17
    ],
    [
     16,
     0,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails2": {
   "cells": [
    [
     16,
     -2,
     12
    ],
    [
     16,
     2,
     12
    ],
    [
     16,
     -2,
     13
    ],
    [
     16,
     2,
     13
    ],
    [
     16,
     -2,
     14
    ],
    [
     16,
     2,
     14
    ],
    [
     16,
     -2,
     15
    ],
    [
     16,
     2,
     15
    ],
    [
     16,
     -2,
     16
    ],
    [
     16,
     2,
     16
    ],
    [
     16,
     -2,
     17
    ],
    [
     16,
     2,
     17
    ],
    [
     11,
     -5,
     12
    ],
    [
     15,
     -5,
     12
    ],
    [
     11,
     -5,
     13
    ],
    [
     15,
     -5,
     13
    ],
    [
     11,
     -5,
     14
    ],
    [
     15,
     -5,
     14
    ],
    [
     11,
     -5,
     15
    ],
    [
     15,
     -5,
     15
    ],
    [
     11,
     -5,
     16
    ],
    [
     15,
     -5,
     16
    ],
    [
     11,
     -5,
     17
    ],
    [
     15,
     -5,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs2": {
   "cells": [
    [
     16,
     -1,
     12
    ],
    [
     16,
     0,
     12
    ],
    [
     16,
     1,
     12
    ],
    [
     16,
     -1,
     14
    ],
    [
     16,
     0,
     14
    ],
    [
     16,
     1,
     14
    ],
    [
     16,
     -1,
     16
    ],
    [
     16,
     0,
     16
    ],
    [
     16,
     1,
     16
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts3": {
   "cells": [
    [
     10,
     -3,
     18
    ],
    [
     10,
     3,
     18
    ],
    [
     15,
     -3,
     18
    ],
    [
     15,
     3,
     18
    ],
    [
     10,
     -3,
     19
    ],
    [
     10,
     3,
     19
    ],
    [
     15,
     -3,
     19
    ],
    [
     15,
     3,
     19
    ],
    [
     10,
     -3,
     20
    ],
    [
     10,
     3,
     20
    ],
    [
     15,
     -3,
     20
    ],
    [
     15,
     3,
     20
    ],
    [
     10,
     -3,
     21
    ],
    [
     10,
     3,
     21
    ],
    [
     15,
     -3,
     21
    ],
    [
     15,
     3,
     21
    ],
    [
     10,
     -3,
     22
    ],
    [
     10,
     3,
     22
    ],
    [
     15,
     -3,
     22
    ],
    [
     15,
     3,
     22
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks3": {
   "cells": [
    [
     10,
     -3,
     23
    ],
    [
     10,
     -2,
     23
    ],
    [
     10,
     -1,
     23
    ],
    [
     10,
     0,
     23
    ],
    [
     10,
     1,
     23
    ],
    [
     10,
     2,
     23
    ],
    [
     10,
     3,
     23
    ],
    [
     11,
     -3,
     23
    ],
    [
     11,
     -2,
     23
    ],
    [
     11,
     -1,
     23
    ],
    [
     11,
     0,
     23
    ],
    [
     11,
     1,
     23
    ],
    [
     11,
     2,
     23
    ],
    [
     11,
     3,
     23
    ],
    [
     12,
     -3,
     23
    ],
    [
     12,
     -2,
     23
    ],
    [
     12,
     -1,
     23
    ],
    [
     12,
     0,
     23
    ],
    [
     12,
     1,
     23
    ],
    [
     12,
     2,
     23
    ],
    [
     12,
     3,
     23
    ],
    [
     13,
     -3,
     23
    ],
    [
     13,
     -2,
     23
    ],
    [
     13,
     -1,
     23
    ],
    [
     13,
     0,
     23
    ],
    [
     13,
     1,
     23
    ],
    [
     13,
     2,
     23
    ],
    [
     13,
     3,
     23
    ],
    [
     14,
     -3,
     23
    ],
    [
     14,
     -2,
     23
    ],
    [
     14,
     -1,
     23
    ],
    [
     14,
     0,
     23
    ],
    [
     14,
     1,
     23
    ],
    [
     14,
     2,
     23
    ],
    [
     14,
     3,
     23
    ],
    [
     15,
     -3,
     23
    ],
    [
     15,
     -2,
     23
    ],
    [
     15,
     -1,
     23
    ],
    [
     15,
     0,
     23
    ],
    [
     15,
     1,
     23
    ],
    [
     15,
     2,
     23
    ],
    [
     15,
     3,
     23
    ],
    [
     2,
     -1,
     23
    ],
    [
     2,
     0,
     23
    ],
    [
     2,
     1,
     23
    ],
    [
     3,
     -1,
     23
    ],
    [
     3,
     0,
     23
    ],
    [
     3,
     1,
     23
    ],
    [
     4,
     -1,
     23
    ],
    [
     4,
     0,
     23
    ],
    [
     4,
     1,
     23
    ],
    [
     5,
     -1,
     23
    ],
    [
     5,
     0,
     23
    ],
    [
     5,
     1,
     23
    ],
    [
     6,
     -1,
     23
    ],
    [
     6,
     0,
     23
    ],
    [
     6,
     1,
     23
    ],
    [
     7,
     -1,
     23
    ],
    [
     7,
     0,
     23
    ],
    [
     7,
     1,
     23
    ],
    [
     8,
     -1,
     23
    ],
    [
     8,
     0,
     23
    ],
    [
     8,
     1,
     23
    ],
    [
     9,
     -1,
     23
    ],
    [
     9,
     0,
     23
    ],
    [
     9,
     1,
     23
    ],
    [
     16,
     0,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails3": {
   "cells": [
    [
     16,
     -2,
     18
    ],
    [
     16,
     2,
     18
    ],
    [
     16,
     -2,
     19
    ],
    [
     16,
     2,
     19
    ],
    [
     16,
     -2,
     20
    ],
    [
     16,
     2,
     20
    ],
    [
     16,
     -2,
     21
    ],
    [
     16,
     2,
     21
    ],
    [
     16,
     -2,
     22
    ],
    [
     16,
     2,
     22
    ],
    [
     16,
     -2,
     23
    ],
    [
     16,
     2,
     23
    ],
    [
     11,
     -5,
     18
    ],
    [
     15,
     -5,
     18
    ],
    [
     11,
     -5,
     19
    ],
    [
     15,
     -5,
     19
    ],
    [
     11,
     -5,
     20
    ],
    [
     15,
     -5,
     20
    ],
    [
     11,
     -5,
     21
    ],
    [
     15,
     -5,
     21
    ],
    [
     11,
     -5,
     22
    ],
    [
     15,
     -5,
     22
    ],
    [
     11,
     -5,
     23
    ],
    [
     15,
     -5,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs3": {
   "cells": [
    [
     16,
     -1,
     18
    ],
    [
     16,
     0,
     18
    ],
    [
     16,
     1,
     18
    ],
    [
     16,
     -1,
     20
    ],
    [
     16,
     0,
     20
    ],
    [
     16,
     1,
     20
    ],
    [
     16,
     -1,
     22
    ],
    [
     16,
     0,
     22
    ],
    [
     16,
     1,
     22
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts4": {
   "cells": [
    [
     10,
     -3,
     24
    ],
    [
     10,
     3,
     24
    ],
    [
     15,
     -3,
     24
    ],
    [
     15,
     3,
     24
    ],
    [
     10,
     -3,
     25
    ],
    [
     10,
     3,
     25
    ],
    [
     15,
     -3,
     25
    ],
    [
     15,
     3,
     25
    ],
    [
     10,
     -3,
     26
    ],
    [
     10,
     3,
     26
    ],
    [
     15,
     -3,
     26
    ],
    [
     15,
     3,
     26
    ],
    [
     10,
     -3,
     27
    ],
    [
     10,
     3,
     27
    ],
    [
     15,
     -3,
     27
    ],
    [
     15,
     3,
     27
    ],
    [
     10,
     -3,
     29
    ],
    [
     10,
     3,
     29
    ],
    [
     15,
     -3,
     29
    ],
    [
     15,
     3,
     29
    ],
    [
     10,
     -3,
     30
    ],
    [
     10,
     3,
     30
    ],
    [
     15,
     -3,
     30
    ],
    [
     15,
     3,
     30
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24
  },
  "planks4": {
   "cells": [
    [
     10,
     -3,
     28
    ],
    [
     10,
     -2,
     28
    ],
    [
     10,
     -1,
     28
    ],
    [
     10,
     0,
     28
    ],
    [
     10,
     1,
     28
    ],
    [
     10,
     2,
     28
    ],
    [
     10,
     3,
     28
    ],
    [
     11,
     -3,
     28
    ],
    [
     11,
     -2,
     28
    ],
    [
     11,
     -1,
     28
    ],
    [
     11,
     0,
     28
    ],
    [
     11,
     1,
     28
    ],
    [
     11,
     2,
     28
    ],
    [
     11,
     3,
     28
    ],
    [
     12,
     -3,
     28
    ],
    [
     12,
     -2,
     28
    ],
    [
     12,
     -1,
     28
    ],
    [
     12,
     0,
     28
    ],
    [
     12,
     1,
     28
    ],
    [
     12,
     2,
     28
    ],
    [
     12,
     3,
     28
    ],
    [
     13,
     -3,
     28
    ],
    [
     13,
     -2,
     28
    ],
    [
     13,
     -1,
     28
    ],
    [
     13,
     0,
     28
    ],
    [
     13,
     1,
     28
    ],
    [
     13,
     2,
     28
    ],
    [
     13,
     3,
     28
    ],
    [
     14,
     -3,
     28
    ],
    [
     14,
     -2,
     28
    ],
    [
     14,
     -1,
     28
    ],
    [
     14,
     0,
     28
    ],
    [
     14,
     1,
     28
    ],
    [
     14,
     2,
     28
    ],
    [
     14,
     3,
     28
    ],
    [
     15,
     -3,
     28
    ],
    [
     15,
     -2,
     28
    ],
    [
     15,
     -1,
     28
    ],
    [
     15,
     0,
     28
    ],
    [
     15,
     1,
     28
    ],
    [
     15,
     2,
     28
    ],
    [
     15,
     3,
     28
    ],
    [
     2,
     -1,
     28
    ],
    [
     2,
     0,
     28
    ],
    [
     2,
     1,
     28
    ],
    [
     3,
     -1,
     28
    ],
    [
     3,
     0,
     28
    ],
    [
     3,
     1,
     28
    ],
    [
     4,
     -1,
     28
    ],
    [
     4,
     0,
     28
    ],
    [
     4,
     1,
     28
    ],
    [
     5,
     -1,
     28
    ],
    [
     5,
     0,
     28
    ],
    [
     5,
     1,
     28
    ],
    [
     6,
     -1,
     28
    ],
    [
     6,
     0,
     28
    ],
    [
     6,
     1,
     28
    ],
    [
     7,
     -1,
     28
    ],
    [
     7,
     0,
     28
    ],
    [
     7,
     1,
     28
    ],
    [
     8,
     -1,
     28
    ],
    [
     8,
     0,
     28
    ],
    [
     8,
     1,
     28
    ],
    [
     9,
     -1,
     28
    ],
    [
     9,
     0,
     28
    ],
    [
     9,
     1,
     28
    ],
    [
     16,
     0,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails4": {
   "cells": [
    [
     16,
     -2,
     24
    ],
    [
     16,
     2,
     24
    ],
    [
     16,
     -2,
     25
    ],
    [
     16,
     2,
     25
    ],
    [
     16,
     -2,
     26
    ],
    [
     16,
     2,
     26
    ],
    [
     16,
     -2,
     27
    ],
    [
     16,
     2,
     27
    ],
    [
     16,
     -2,
     28
    ],
    [
     16,
     2,
     28
    ],
    [
     11,
     -5,
     24
    ],
    [
     15,
     -5,
     24
    ],
    [
     11,
     -5,
     25
    ],
    [
     15,
     -5,
     25
    ],
    [
     11,
     -5,
     26
    ],
    [
     15,
     -5,
     26
    ],
    [
     11,
     -5,
     27
    ],
    [
     15,
     -5,
     27
    ],
    [
     11,
     -5,
     28
    ],
    [
     15,
     -5,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs4": {
   "cells": [
    [
     16,
     -1,
     24
    ],
    [
     16,
     0,
     24
    ],
    [
     16,
     1,
     24
    ],
    [
     16,
     -1,
     26
    ],
    [
     16,
     0,
     26
    ],
    [
     16,
     1,
     26
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 6,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "crate": {
   "cells": [
    [
     12,
     -6,
     0
    ],
    [
     12,
     -6,
     1
    ],
    [
     12,
     -5,
     0
    ],
    [
     12,
     -5,
     1
    ],
    [
     12,
     -4,
     0
    ],
    [
     12,
     -4,
     1
    ],
    [
     13,
     -6,
     0
    ],
    [
     13,
     -6,
     1
    ],
    [
     13,
     -5,
     0
    ],
    [
     13,
     -5,
     1
    ],
    [
     13,
     -4,
     0
    ],
    [
     13,
     -4,
     1
    ],
    [
     14,
     -6,
     0
    ],
    [
     14,
     -6,
     1
    ],
    [
     14,
     -5,
     0
    ],
    [
     14,
     -5,
     1
    ],
    [
     14,
     -4,
     0
    ],
    [
     14,
     -4,
     1
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 18
  },
  "load1": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load2": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load3": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load4": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load5": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": 1.0
  },
  {
   "type": "build",
   "target": "posts0",
   "when": 1.0,
   "until": 2.4,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "planks0",
   "when": 2.4,
   "until": 3.4,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "rails0",
   "when": 3.4,
   "until": 4.0,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "rungs0",
   "when": 4.0,
   "until": 4.5,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "crate",
   "when": 4.5,
   "until": 5.0,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "tower0",
   "when": 5.0,
   "until": 11.0,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "load1",
   "when": 11.0,
   "until": 11.4,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    0.4
   ],
   "when": 11.4,
   "until": 12.6
  },
  {
   "type": "slide",
   "target": "load1",
   "by": [
    0,
    0,
    0.4
   ],
   "when": 11.4,
   "until": 12.6
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.0
   ],
   "when": 11.4,
   "until": 12.6,
   "via": [
    [
     1.0,
     -0.95,
     0.0
    ],
    [
     1.75,
     -0.95,
     0.0
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.6
   ],
   "when": 12.6,
   "until": 13.4,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.8,
    0.0,
    0.6
   ],
   "when": 13.4,
   "until": 14.4,
   "via": [
    [
     1.45,
     0.0,
     0.6
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load1",
   "when": 14.6,
   "until": 21.1,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower1",
   "when": 14.6,
   "until": 21.1,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "posts1",
   "when": 21.1,
   "until": 22.1,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "planks1",
   "when": 22.1,
   "until": 22.9,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "rails1",
   "when": 22.9,
   "until": 23.4,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "rungs1",
   "when": 23.4,
   "until": 23.8,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -0.4
   ],
   "when": 23.8,
   "until": 24.8
  },
  {
   "type": "build",
   "target": "load2",
   "when": 24.8,
   "until": 25.2,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    1.0
   ],
   "when": 25.2,
   "until": 26.4
  },
  {
   "type": "slide",
   "target": "load2",
   "by": [
    0,
    0,
    1.0
   ],
   "when": 25.2,
   "until": 26.4
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.6
   ],
   "when": 25.2,
   "until": 26.2,
   "via": [
    [
     1.45,
     0.0,
     0.6
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.2
   ],
   "when": 26.2,
   "until": 27.0,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.6,
    0.0,
    1.2
   ],
   "when": 27.0,
   "until": 28.0,
   "via": [
    [
     1.45,
     0.0,
     1.2
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load2",
   "when": 28.2,
   "until": 30.82,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower2",
   "when": 28.2,
   "until": 30.82,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "posts2",
   "when": 30.82,
   "until": 31.82,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "planks2",
   "when": 31.82,
   "until": 32.62,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "rails2",
   "when": 32.62,
   "until": 33.12,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "rungs2",
   "when": 33.12,
   "until": 33.52,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -1.0
   ],
   "when": 33.52,
   "until": 34.52
  },
  {
   "type": "build",
   "target": "load3",
   "when": 34.52,
   "until": 34.92,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    1.6
   ],
   "when": 34.92,
   "until": 36.12
  },
  {
   "type": "slide",
   "target": "load3",
   "by": [
    0,
    0,
    1.6
   ],
   "when": 34.92,
   "until": 36.12
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.2
   ],
   "when": 34.92,
   "until": 35.92,
   "via": [
    [
     1.45,
     0.0,
     1.2
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.8
   ],
   "when": 35.92,
   "until": 36.72,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    1.8
   ],
   "when": 36.72,
   "until": 37.72,
   "via": [
    [
     1.45,
     0.0,
     1.8
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load3",
   "when": 37.92,
   "until": 40.42,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower3",
   "when": 37.92,
   "until": 40.42,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "posts3",
   "when": 40.42,
   "until": 41.42,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "planks3",
   "when": 41.42,
   "until": 42.22,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "rails3",
   "when": 42.22,
   "until": 42.72,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "rungs3",
   "when": 42.72,
   "until": 43.12,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -1.6
   ],
   "when": 43.12,
   "until": 44.12
  },
  {
   "type": "build",
   "target": "load4",
   "when": 44.12,
   "until": 44.52,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    2.2
   ],
   "when": 44.52,
   "until": 45.72
  },
  {
   "type": "slide",
   "target": "load4",
   "by": [
    0,
    0,
    2.2
   ],
   "when": 44.52,
   "until": 45.72
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.8
   ],
   "when": 44.52,
   "until": 45.52,
   "via": [
    [
     1.45,
     0.0,
     1.8
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.4
   ],
   "when": 45.52,
   "until": 46.32,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    2.4
   ],
   "when": 46.32,
   "until": 47.32,
   "via": [
    [
     1.45,
     0.0,
     2.4
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load4",
   "when": 47.52,
   "until": 50.02,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower4",
   "when": 47.52,
   "until": 50.02,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "posts4",
   "when": 50.02,
   "until": 51.02,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "planks4",
   "when": 51.02,
   "until": 51.82,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "rails4",
   "when": 51.82,
   "until": 52.32,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "rungs4",
   "when": 52.32,
   "until": 52.72,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -2.2
   ],
   "when": 52.72,
   "until": 53.72
  },
  {
   "type": "build",
   "target": "load5",
   "when": 53.72,
   "until": 54.12,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    2.7
   ],
   "when": 54.12,
   "until": 55.32
  },
  {
   "type": "slide",
   "target": "load5",
   "by": [
    0,
    0,
    2.7
   ],
   "when": 54.12,
   "until": 55.32
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.4
   ],
   "when": 54.12,
   "until": 55.12,
   "via": [
    [
     1.45,
     0.0,
     2.4
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.9
   ],
   "when": 55.12,
   "until": 55.92,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    2.9
   ],
   "when": 55.92,
   "until": 56.92,
   "via": [
    [
     1.45,
     0.0,
     2.9
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load5",
   "when": 57.12,
   "until": 59.62,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower5",
   "when": 57.12,
   "until": 59.62,
   "stand_at": [
    0.4,
    0.0,
    2.9
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.9
   ],
   "when": 59.62,
   "until": 60.62,
   "via": [
    [
     1.45,
     0.0,
     2.9
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.0
   ],
   "when": 60.62,
   "until": 62.7,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    2.4,
    -1.4,
    0.0
   ],
   "when": 62.7,
   "until": 64.1
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -2.7
   ],
   "when": 64.1,
   "until": 65.1
  },
  {
   "type": "remove",
   "target": "rungs4",
   "when": 64.1,
   "until": 64.45,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails4",
   "when": 64.45,
   "until": 64.8,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks4",
   "when": 64.8,
   "until": 65.15,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts4",
   "when": 65.15,
   "until": 65.5,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs3",
   "when": 65.5,
   "until": 65.85,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails3",
   "when": 65.85,
   "until": 66.2,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks3",
   "when": 66.2,
   "until": 66.55,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts3",
   "when": 66.55,
   "until": 66.9,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs2",
   "when": 66.9,
   "until": 67.25,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails2",
   "when": 67.25,
   "until": 67.6,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks2",
   "when": 67.6,
   "until": 67.95,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts2",
   "when": 67.95,
   "until": 68.3,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs1",
   "when": 68.3,
   "until": 68.65,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails1",
   "when": 68.65,
   "until": 69.0,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks1",
   "when": 69.0,
   "until": 69.35,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts1",
   "when": 69.35,
   "until": 69.7,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs0",
   "when": 69.7,
   "until": 70.05,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails0",
   "when": 70.05,
   "until": 70.4,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks0",
   "when": 70.4,
   "until": 70.75,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts0",
   "when": 70.75,
   "until": 71.1,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "crate",
   "when": 71.1,
   "until": 71.5,
   "quiet": true
  },
  {
   "type": "face_camera",
   "when": 71.8,
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    0.9,
    -0.3,
    0.9
   ],
   "az": -115,
   "el": 6,
   "dist": 3.6
  },
  {
   "at": 5.0,
   "target": [
    0.7,
    0.0,
    0.8
   ],
   "az": -104,
   "el": 8,
   "dist": 5.2
  },
  {
   "at": 13.4,
   "target": [
    0.8,
    0.0,
    1.0
   ],
   "az": -95,
   "el": 9,
   "dist": 5.35
  },
  {
   "at": 27.0,
   "target": [
    0.8,
    0.0,
    1.6
   ],
   "az": -86,
   "el": 10,
   "dist": 5.7
  },
  {
   "at": 36.72,
   "target": [
    0.8,
    0.0,
    2.2
   ],
   "az": -77,
   "el": 11,
   "dist": 6.05
  },
  {
   "at": 46.32,
   "target": [
    0.8,
    0.0,
    2.8
   ],
   "az": -68,
   "el": 12,
   "dist": 6.4
  },
  {
   "at": 55.92,
   "target": [
    0.8,
    0.0,
    3.3
   ],
   "az": -59,
   "el": 13,
   "dist": 6.75
  },
  {
   "at": 64.1,
   "target": [
    0.7,
    0.0,
    1.5
   ],
   "az": -60,
   "el": 10,
   "dist": 7.4
  },
  {
   "at": 71.8,
   "target": [
    0.5,
    0.0,
    1.6
   ],
   "az": -80,
   "el": 8,
   "dist": 7.8
  },
  {
   "at": "end",
   "target": [
    0.5,
    0.0,
    1.6
   ],
   "az": -100,
   "el": 8,
   "dist": 7.4
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "{count:tower*} / 677 BLOCKS",
   "color": "amber",
   "y": 140,
   "size": 40,
   "when": 5.0,
   "until": 60.624999999999986
  },
  {
   "style": "title",
   "text": "THE LEGS",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 5.0,
   "until": 7.4
  },
  {
   "style": "title",
   "text": "FIRST FLOOR",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 13.4,
   "until": 15.8
  },
  {
   "style": "title",
   "text": "SECOND FLOOR",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 27.0,
   "until": 29.4
  },
  {
   "style": "title",
   "text": "UP THE MIDDLE",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 36.72,
   "until": 39.12
  },
  {
   "style": "title",
   "text": "THE TOP DECK",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 46.32,
   "until": 48.72
  },
  {
   "style": "title",
   "text": "THE SPIRE",
   "color": "white",
   "y": 78,
   "size": 56,
   "when": 55.92,
   "until": 58.32
  },
  {
   "style": "title",
   "text": "VOILA!",
   "color": "amber",
   "y": 80,
   "size": 76,
   "when": 71.8,
   "until": "end"
  }
 ]
}

In [ ]:
%%writefile stories/eiffel_rivets.json
{
 "title": "Hot rivets hold the Eiffel Tower together",
 "slug": "eiffel_rivets",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "world": {
  "ground": "paris",
  "time": "morning",
  "weather": "clouds",
  "animals": [
   "pigeons"
  ]
 },
 "lines": [
  {
   "id": "hook",
   "text": "The Eiffel Tower is held together by two and a half million rivets... and they went in red hot!",
   "pause": 0.4
  },
  {
   "id": "pop",
   "text": "Heat makes iron swell up a little. Pop it through the hole... ow! Hot, hot, hot!",
   "pause": 0.3
  },
  {
   "id": "bang",
   "text": "Bang the end flat.",
   "pause": 0.5
  },
  {
   "id": "cool",
   "text": "Now, as it cools, it shrinks... and squeezes the beams together. Tight!",
   "pause": 0.5
  },
  {
   "id": "team",
   "text": "Each rivet took a team of four men. Today... it's just me.",
   "pause": 0.5
  },
  {
   "id": "payoff",
   "text": "Two and a half million times! No wonder she's still standing.",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   1.7,
   -1.3,
   0.0
  ],
  "outfit": {
   "hat": "beret",
   "shirt": "striped",
   "extras": [
    "scarf",
    "hammer"
   ]
  }
 },
 "structures": {
  "tower0": {
   "cells": [
    [
     -8,
     -8,
     0
    ],
    [
     -7,
     -8,
     0
    ],
    [
     -6,
     -8,
     0
    ],
    [
     -5,
     -8,
     0
    ],
    [
     -4,
     -8,
     0
    ],
    [
     4,
     -8,
     0
    ],
    [
     5,
     -8,
     0
    ],
    [
     6,
     -8,
     0
    ],
    [
     7,
     -8,
     0
    ],
    [
     8,
     -8,
     0
    ],
    [
     -8,
     -7,
     0
    ],
    [
     -4,
     -7,
     0
    ],
    [
     4,
     -7,
     0
    ],
    [
     8,
     -7,
     0
    ],
    [
     -8,
     -6,
     0
    ],
    [
     -4,
     -6,
     0
    ],
    [
     4,
     -6,
     0
    ],
    [
     8,
     -6,
     0
    ],
    [
     -8,
     -5,
     0
    ],
    [
     -4,
     -5,
     0
    ],
    [
     4,
     -5,
     0
    ],
    [
     8,
     -5,
     0
    ],
    [
     -8,
     -4,
     0
    ],
    [
     -7,
     -4,
     0
    ],
    [
     -6,
     -4,
     0
    ],
    [
     -5,
     -4,
     0
    ],
    [
     -4,
     -4,
     0
    ],
    [
     4,
     -4,
     0
    ],
    [
     5,
     -4,
     0
    ],
    [
     6,
     -4,
     0
    ],
    [
     7,
     -4,
     0
    ],
    [
     8,
     -4,
     0
    ],
    [
     -8,
     4,
     0
    ],
    [
     -7,
     4,
     0
    ],
    [
     -6,
     4,
     0
    ],
    [
     -5,
     4,
     0
    ],
    [
     -4,
     4,
     0
    ],
    [
     4,
     4,
     0
    ],
    [
     5,
     4,
     0
    ],
    [
     6,
     4,
     0
    ],
    [
     7,
     4,
     0
    ],
    [
     8,
     4,
     0
    ],
    [
     -8,
     5,
     0
    ],
    [
     -4,
     5,
     0
    ],
    [
     4,
     5,
     0
    ],
    [
     8,
     5,
     0
    ],
    [
     -8,
     6,
     0
    ],
    [
     -4,
     6,
     0
    ],
    [
     4,
     6,
     0
    ],
    [
     8,
     6,
     0
    ],
    [
     -8,
     7,
     0
    ],
    [
     -4,
     7,
     0
    ],
    [
     4,
     7,
     0
    ],
    [
     8,
     7,
     0
    ],
    [
     -8,
     8,
     0
    ],
    [
     -7,
     8,
     0
    ],
    [
     -6,
     8,
     0
    ],
    [
     -5,
     8,
     0
    ],
    [
     -4,
     8,
     0
    ],
    [
     4,
     8,
     0
    ],
    [
     5,
     8,
     0
    ],
    [
     6,
     8,
     0
    ],
    [
     7,
     8,
     0
    ],
    [
     8,
     8,
     0
    ],
    [
     -7,
     -7,
     1
    ],
    [
     -4,
     -7,
     1
    ],
    [
     4,
     -7,
     1
    ],
    [
     7,
     -7,
     1
    ],
    [
     -7,
     -4,
     1
    ],
    [
     -4,
     -4,
     1
    ],
    [
     4,
     -4,
     1
    ],
    [
     7,
     -4,
     1
    ],
    [
     -7,
     4,
     1
    ],
    [
     -4,
     4,
     1
    ],
    [
     4,
     4,
     1
    ],
    [
     7,
     4,
     1
    ],
    [
     -7,
     7,
     1
    ],
    [
     -4,
     7,
     1
    ],
    [
     4,
     7,
     1
    ],
    [
     7,
     7,
     1
    ],
    [
     -6,
     -6,
     2
    ],
    [
     -5,
     -6,
     2
    ],
    [
     -4,
     -6,
     2
    ],
    [
     -3,
     -6,
     2
    ],
    [
     3,
     -6,
     2
    ],
    [
     4,
     -6,
     2
    ],
    [
     5,
     -6,
     2
    ],
    [
     6,
     -6,
     2
    ],
    [
     -6,
     -5,
     2
    ],
    [
     -3,
     -5,
     2
    ],
    [
     3,
     -5,
     2
    ],
    [
     6,
     -5,
     2
    ],
    [
     -6,
     -4,
     2
    ],
    [
     -3,
     -4,
     2
    ],
    [
     3,
     -4,
     2
    ],
    [
     6,
     -4,
     2
    ],
    [
     -6,
     -3,
     2
    ],
    [
     -5,
     -3,
     2
    ],
    [
     -4,
     -3,
     2
    ],
    [
     -3,
     -3,
     2
    ],
    [
     3,
     -3,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     6,
     -3,
     2
    ],
    [
     -6,
     3,
     2
    ],
    [
     -5,
     3,
     2
    ],
    [
     -4,
     3,
     2
    ],
    [
     -3,
     3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     6,
     3,
     2
    ],
    [
     -6,
     4,
     2
    ],
    [
     -3,
     4,
     2
    ],
    [
     3,
     4,
     2
    ],
    [
     6,
     4,
     2
    ],
    [
     -6,
     5,
     2
    ],
    [
     -3,
     5,
     2
    ],
    [
     3,
     5,
     2
    ],
    [
     6,
     5,
     2
    ],
    [
     -6,
     6,
     2
    ],
    [
     -5,
     6,
     2
    ],
    [
     -4,
     6,
     2
    ],
    [
     -3,
     6,
     2
    ],
    [
     3,
     6,
     2
    ],
    [
     4,
     6,
     2
    ],
    [
     5,
     6,
     2
    ],
    [
     6,
     6,
     2
    ],
    [
     -6,
     -6,
     3
    ],
    [
     -3,
     -6,
     3
    ],
    [
     3,
     -6,
     3
    ],
    [
     6,
     -6,
     3
    ],
    [
     -3,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     -6,
     -3,
     3
    ],
    [
     -4,
     -3,
     3
    ],
    [
     -3,
     -3,
     3
    ],
    [
     3,
     -3,
     3
    ],
    [
     4,
     -3,
     3
    ],
    [
     6,
     -3,
     3
    ],
    [
     -6,
     3,
     3
    ],
    [
     -4,
     3,
     3
    ],
    [
     -3,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     -3,
     4,
     3
    ],
    [
     3,
     4,
     3
    ],
    [
     -6,
     6,
     3
    ],
    [
     -3,
     6,
     3
    ],
    [
     3,
     6,
     3
    ],
    [
     6,
     6,
     3
    ],
    [
     -5,
     -5,
     4
    ],
    [
     -4,
     -5,
     4
    ],
    [
     -3,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     -5,
     -4,
     4
    ],
    [
     -3,
     -4,
     4
    ],
    [
     -2,
     -4,
     4
    ],
    [
     -1,
     -4,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     -5,
     -3,
     4
    ],
    [
     -4,
     -3,
     4
    ],
    [
     -3,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     -4,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     -4,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     -4,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     -4,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     -4,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     -5,
     3,
     4
    ],
    [
     -4,
     3,
     4
    ],
    [
     -3,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     -5,
     4,
     4
    ],
    [
     -3,
     4,
     4
    ],
    [
     -2,
     4,
     4
    ],
    [
     -1,
     4,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     -5,
     5,
     4
    ],
    [
     -4,
     5,
     4
    ],
    [
     -3,
     5,
     4
    ],
    [
     3,
     5,
     4
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     4
    ],
    [
     -5,
     -5,
     5
    ],
    [
     -2,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     -2,
     -4,
     5
    ],
    [
     -1,
     -4,
     5
    ],
    [
     0,
     -4,
     5
    ],
    [
     1,
     -4,
     5
    ],
    [
     2,
     -4,
     5
    ],
    [
     -5,
     -2,
     5
    ],
    [
     -4,
     -2,
     5
    ],
    [
     -2,
     -2,
     5
    ],
    [
     2,
     -2,
     5
    ],
    [
     4,
     -2,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     -4,
     -1,
     5
    ],
    [
     4,
     -1,
     5
    ],
    [
     -4,
     0,
     5
    ],
    [
     4,
     0,
     5
    ],
    [
     -4,
     1,
     5
    ],
    [
     4,
     1,
     5
    ],
    [
     -5,
     2,
     5
    ],
    [
     -4,
     2,
     5
    ],
    [
     -2,
     2,
     5
    ],
    [
     2,
     2,
     5
    ],
    [
     4,
     2,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     -2,
     4,
     5
    ],
    [
     -1,
     4,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     -5,
     5,
     5
    ],
    [
     -2,
     5,
     5
    ],
    [
     2,
     5,
     5
    ],
    [
     5,
     5,
     5
    ],
    [
     -5,
     -5,
     6
    ],
    [
     -4,
     -5,
     6
    ],
    [
     -3,
     -5,
     6
    ],
    [
     -2,
     -5,
     6
    ],
    [
     -1,
     -5,
     6
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     -5,
     -4,
     6
    ],
    [
     -4,
     -4,
     6
    ],
    [
     -3,
     -4,
     6
    ],
    [
     -2,
     -4,
     6
    ],
    [
     -1,
     -4,
     6
    ],
    [
     0,
     -4,
     6
    ],
    [
     1,
     -4,
     6
    ],
    [
     2,
     -4,
     6
    ],
    [
     3,
     -4,
     6
    ],
    [
     4,
     -4,
     6
    ],
    [
     5,
     -4,
     6
    ],
    [
     -5,
     -3,
     6
    ],
    [
     -4,
     -3,
     6
    ],
    [
     -3,
     -3,
     6
    ],
    [
     -2,
     -3,
     6
    ],
    [
     -1,
     -3,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     1,
     -3,
     6
    ],
    [
     2,
     -3,
     6
    ],
    [
     3,
     -3,
     6
    ],
    [
     4,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     -5,
     -2,
     6
    ],
    [
     -4,
     -2,
     6
    ],
    [
     -3,
     -2,
     6
    ],
    [
     -2,
     -2,
     6
    ],
    [
     -1,
     -2,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     1,
     -2,
     6
    ],
    [
     2,
     -2,
     6
    ],
    [
     3,
     -2,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     5,
     -2,
     6
    ],
    [
     -5,
     -1,
     6
    ],
    [
     -4,
     -1,
     6
    ],
    [
     -3,
     -1,
     6
    ],
    [
     -2,
     -1,
     6
    ],
    [
     2,
     -1,
     6
    ],
    [
     3,
     -1,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     5,
     -1,
     6
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     6
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     -5,
     1,
     6
    ],
    [
     -4,
     1,
     6
    ],
    [
     -3,
     1,
     6
    ],
    [
     -2,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     -5,
     2,
     6
    ],
    [
     -4,
     2,
     6
    ],
    [
     -3,
     2,
     6
    ],
    [
     -2,
     2,
     6
    ],
    [
     -1,
     2,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     1,
     2,
     6
    ],
    [
     2,
     2,
     6
    ],
    [
     3,
     2,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     5,
     2,
     6
    ],
    [
     -5,
     3,
     6
    ],
    [
     -4,
     3,
     6
    ],
    [
     -3,
     3,
     6
    ],
    [
     -2,
     3,
     6
    ],
    [
     -1,
     3,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     1,
     3,
     6
    ],
    [
     2,
     3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     4,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     -5,
     4,
     6
    ],
    [
     -4,
     4,
     6
    ],
    [
     -3,
     4,
     6
    ],
    [
     -2,
     4,
     6
    ],
    [
     -1,
     4,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     -5,
     5,
     6
    ],
    [
     -4,
     5,
     6
    ],
    [
     -3,
     5,
     6
    ],
    [
     -2,
     5,
     6
    ],
    [
     -1,
     5,
     6
    ],
    [
     0,
     5,
     6
    ],
    [
     1,
     5,
     6
    ],
    [
     2,
     5,
     6
    ],
    [
     3,
     5,
     6
    ],
    [
     4,
     5,
     6
    ],
    [
     5,
     5,
     6
    ],
    [
     -5,
     -5,
     7
    ],
    [
     -3,
     -5,
     7
    ],
    [
     -1,
     -5,
     7
    ],
    [
     1,
     -5,
     7
    ],
    [
     3,
     -5,
     7
    ],
    [
     5,
     -5,
     7
    ],
    [
     -4,
     -4,
     7
    ],
    [
     -2,
     -4,
     7
    ],
    [
     2,
     -4,
     7
    ],
    [
     4,
     -4,
     7
    ],
    [
     -5,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     -4,
     -2,
     7
    ],
    [
     -2,
     -2,
     7
    ],
    [
     2,
     -2,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     -5,
     -1,
     7
    ],
    [
     5,
     -1,
     7
    ],
    [
     -5,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     -4,
     2,
     7
    ],
    [
     -2,
     2,
     7
    ],
    [
     2,
     2,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     -5,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     -4,
     4,
     7
    ],
    [
     -2,
     4,
     7
    ],
    [
     2,
     4,
     7
    ],
    [
     4,
     4,
     7
    ],
    [
     -5,
     5,
     7
    ],
    [
     -3,
     5,
     7
    ],
    [
     -1,
     5,
     7
    ],
    [
     1,
     5,
     7
    ],
    [
     3,
     5,
     7
    ],
    [
     5,
     5,
     7
    ],
    [
     -4,
     -4,
     8
    ],
    [
     -3,
     -4,
     8
    ],
    [
     -2,
     -4,
     8
    ],
    [
     -1,
     -4,
     8
    ],
    [
     1,
     -4,
     8
    ],
    [
     2,
     -4,
     8
    ],
    [
     3,
     -4,
     8
    ],
    [
     4,
     -4,
     8
    ],
    [
     -4,
     -3,
     8
    ],
    [
     -1,
     -3,
     8
    ],
    [
     1,
     -3,
     8
    ],
    [
     4,
     -3,
     8
    ],
    [
     -4,
     -2,
     8
    ],
    [
     -1,
     -2,
     8
    ],
    [
     1,
     -2,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     -4,
     -1,
     8
    ],
    [
     -3,
     -1,
     8
    ],
    [
     -2,
     -1,
     8
    ],
    [
     -1,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     3,
     -1,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     -4,
     1,
     8
    ],
    [
     -3,
     1,
     8
    ],
    [
     -2,
     1,
     8
    ],
    [
     -1,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     -4,
     2,
     8
    ],
    [
     -1,
     2,
     8
    ],
    [
     1,
     2,
     8
    ],
    [
     4,
     2,
     8
    ],
    [
     -4,
     3,
     8
    ],
    [
     -1,
     3,
     8
    ],
    [
     1,
     3,
     8
    ],
    [
     4,
     3,
     8
    ],
    [
     -4,
     4,
     8
    ],
    [
     -3,
     4,
     8
    ],
    [
     -2,
     4,
     8
    ],
    [
     -1,
     4,
     8
    ],
    [
     1,
     4,
     8
    ],
    [
     2,
     4,
     8
    ],
    [
     3,
     4,
     8
    ],
    [
     4,
     4,
     8
    ],
    [
     -3,
     -3,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     1,
     -3,
     9
    ],
    [
     3,
     -3,
     9
    ],
    [
     -3,
     -1,
     9
    ],
    [
     -1,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     3,
     -1,
     9
    ],
    [
     -3,
     1,
     9
    ],
    [
     -1,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     3,
     1,
     9
    ],
    [
     -3,
     3,
     9
    ],
    [
     -1,
     3,
     9
    ],
    [
     1,
     3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     -3,
     -3,
     10
    ],
    [
     -2,
     -3,
     10
    ],
    [
     -1,
     -3,
     10
    ],
    [
     1,
     -3,
     10
    ],
    [
     2,
     -3,
     10
    ],
    [
     3,
     -3,
     10
    ],
    [
     -3,
     -2,
     10
    ],
    [
     -1,
     -2,
     10
    ],
    [
     1,
     -2,
     10
    ],
    [
     3,
     -2,
     10
    ],
    [
     -3,
     -1,
     10
    ],
    [
     -2,
     -1,
     10
    ],
    [
     -1,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     3,
     -1,
     10
    ],
    [
     -3,
     1,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     -1,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ],
    [
     3,
     1,
     10
    ],
    [
     -3,
     2,
     10
    ],
    [
     -1,
     2,
     10
    ],
    [
     1,
     2,
     10
    ],
    [
     3,
     2,
     10
    ],
    [
     -3,
     3,
     10
    ],
    [
     -2,
     3,
     10
    ],
    [
     -1,
     3,
     10
    ],
    [
     1,
     3,
     10
    ],
    [
     2,
     3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     -3,
     -3,
     11
    ],
    [
     -1,
     -3,
     11
    ],
    [
     1,
     -3,
     11
    ],
    [
     3,
     -3,
     11
    ],
    [
     -3,
     -1,
     11
    ],
    [
     -1,
     -1,
     11
    ],
    [
     1,
     -1,
     11
    ],
    [
     3,
     -1,
     11
    ],
    [
     -3,
     1,
     11
    ],
    [
     -1,
     1,
     11
    ],
    [
     1,
     1,
     11
    ],
    [
     3,
     1,
     11
    ],
    [
     -3,
     3,
     11
    ],
    [
     -1,
     3,
     11
    ],
    [
     1,
     3,
     11
    ],
    [
     3,
     3,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 500,
   "prebuilt": true
  },
  "posts0": {
   "cells": [
    [
     10,
     -3,
     0
    ],
    [
     10,
     3,
     0
    ],
    [
     15,
     -3,
     0
    ],
    [
     15,
     3,
     0
    ],
    [
     10,
     -3,
     1
    ],
    [
     10,
     3,
     1
    ],
    [
     15,
     -3,
     1
    ],
    [
     15,
     3,
     1
    ],
    [
     10,
     -3,
     2
    ],
    [
     10,
     3,
     2
    ],
    [
     15,
     -3,
     2
    ],
    [
     15,
     3,
     2
    ],
    [
     10,
     -3,
     3
    ],
    [
     10,
     3,
     3
    ],
    [
     15,
     -3,
     3
    ],
    [
     15,
     3,
     3
    ],
    [
     10,
     -3,
     4
    ],
    [
     10,
     3,
     4
    ],
    [
     15,
     -3,
     4
    ],
    [
     15,
     3,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20,
   "prebuilt": true
  },
  "planks0": {
   "cells": [
    [
     10,
     -3,
     5
    ],
    [
     10,
     -2,
     5
    ],
    [
     10,
     -1,
     5
    ],
    [
     10,
     0,
     5
    ],
    [
     10,
     1,
     5
    ],
    [
     10,
     2,
     5
    ],
    [
     10,
     3,
     5
    ],
    [
     11,
     -3,
     5
    ],
    [
     11,
     -2,
     5
    ],
    [
     11,
     -1,
     5
    ],
    [
     11,
     0,
     5
    ],
    [
     11,
     1,
     5
    ],
    [
     11,
     2,
     5
    ],
    [
     11,
     3,
     5
    ],
    [
     12,
     -3,
     5
    ],
    [
     12,
     -2,
     5
    ],
    [
     12,
     -1,
     5
    ],
    [
     12,
     0,
     5
    ],
    [
     12,
     1,
     5
    ],
    [
     12,
     2,
     5
    ],
    [
     12,
     3,
     5
    ],
    [
     13,
     -3,
     5
    ],
    [
     13,
     -2,
     5
    ],
    [
     13,
     -1,
     5
    ],
    [
     13,
     0,
     5
    ],
    [
     13,
     1,
     5
    ],
    [
     13,
     2,
     5
    ],
    [
     13,
     3,
     5
    ],
    [
     14,
     -3,
     5
    ],
    [
     14,
     -2,
     5
    ],
    [
     14,
     -1,
     5
    ],
    [
     14,
     0,
     5
    ],
    [
     14,
     1,
     5
    ],
    [
     14,
     2,
     5
    ],
    [
     14,
     3,
     5
    ],
    [
     15,
     -3,
     5
    ],
    [
     15,
     -2,
     5
    ],
    [
     15,
     -1,
     5
    ],
    [
     15,
     0,
     5
    ],
    [
     15,
     1,
     5
    ],
    [
     15,
     2,
     5
    ],
    [
     15,
     3,
     5
    ],
    [
     6,
     -1,
     5
    ],
    [
     6,
     0,
     5
    ],
    [
     6,
     1,
     5
    ],
    [
     7,
     -1,
     5
    ],
    [
     7,
     0,
     5
    ],
    [
     7,
     1,
     5
    ],
    [
     8,
     -1,
     5
    ],
    [
     8,
     0,
     5
    ],
    [
     8,
     1,
     5
    ],
    [
     9,
     -1,
     5
    ],
    [
     9,
     0,
     5
    ],
    [
     9,
     1,
     5
    ],
    [
     16,
     0,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 55,
   "prebuilt": true,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails0": {
   "cells": [
    [
     16,
     -2,
     0
    ],
    [
     16,
     2,
     0
    ],
    [
     16,
     -2,
     1
    ],
    [
     16,
     2,
     1
    ],
    [
     16,
     -2,
     2
    ],
    [
     16,
     2,
     2
    ],
    [
     16,
     -2,
     3
    ],
    [
     16,
     2,
     3
    ],
    [
     16,
     -2,
     4
    ],
    [
     16,
     2,
     4
    ],
    [
     16,
     -2,
     5
    ],
    [
     16,
     2,
     5
    ],
    [
     11,
     -5,
     0
    ],
    [
     15,
     -5,
     0
    ],
    [
     11,
     -5,
     1
    ],
    [
     15,
     -5,
     1
    ],
    [
     11,
     -5,
     2
    ],
    [
     15,
     -5,
     2
    ],
    [
     11,
     -5,
     3
    ],
    [
     15,
     -5,
     3
    ],
    [
     11,
     -5,
     4
    ],
    [
     15,
     -5,
     4
    ],
    [
     11,
     -5,
     5
    ],
    [
     15,
     -5,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "prebuilt": true,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs0": {
   "cells": [
    [
     16,
     -1,
     0
    ],
    [
     16,
     0,
     0
    ],
    [
     16,
     1,
     0
    ],
    [
     16,
     -1,
     2
    ],
    [
     16,
     0,
     2
    ],
    [
     16,
     1,
     2
    ],
    [
     16,
     -1,
     4
    ],
    [
     16,
     0,
     4
    ],
    [
     16,
     1,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "prebuilt": true,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts1": {
   "cells": [
    [
     10,
     -3,
     6
    ],
    [
     10,
     3,
     6
    ],
    [
     15,
     -3,
     6
    ],
    [
     15,
     3,
     6
    ],
    [
     10,
     -3,
     7
    ],
    [
     10,
     3,
     7
    ],
    [
     15,
     -3,
     7
    ],
    [
     15,
     3,
     7
    ],
    [
     10,
     -3,
     8
    ],
    [
     10,
     3,
     8
    ],
    [
     15,
     -3,
     8
    ],
    [
     15,
     3,
     8
    ],
    [
     10,
     -3,
     9
    ],
    [
     10,
     3,
     9
    ],
    [
     15,
     -3,
     9
    ],
    [
     15,
     3,
     9
    ],
    [
     10,
     -3,
     10
    ],
    [
     10,
     3,
     10
    ],
    [
     15,
     -3,
     10
    ],
    [
     15,
     3,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20,
   "prebuilt": true
  },
  "planks1": {
   "cells": [
    [
     10,
     -3,
     11
    ],
    [
     10,
     -2,
     11
    ],
    [
     10,
     -1,
     11
    ],
    [
     10,
     0,
     11
    ],
    [
     10,
     1,
     11
    ],
    [
     10,
     2,
     11
    ],
    [
     10,
     3,
     11
    ],
    [
     11,
     -3,
     11
    ],
    [
     11,
     -2,
     11
    ],
    [
     11,
     -1,
     11
    ],
    [
     11,
     0,
     11
    ],
    [
     11,
     1,
     11
    ],
    [
     11,
     2,
     11
    ],
    [
     11,
     3,
     11
    ],
    [
     12,
     -3,
     11
    ],
    [
     12,
     -2,
     11
    ],
    [
     12,
     -1,
     11
    ],
    [
     12,
     0,
     11
    ],
    [
     12,
     1,
     11
    ],
    [
     12,
     2,
     11
    ],
    [
     12,
     3,
     11
    ],
    [
     13,
     -3,
     11
    ],
    [
     13,
     -2,
     11
    ],
    [
     13,
     -1,
     11
    ],
    [
     13,
     0,
     11
    ],
    [
     13,
     1,
     11
    ],
    [
     13,
     2,
     11
    ],
    [
     13,
     3,
     11
    ],
    [
     14,
     -3,
     11
    ],
    [
     14,
     -2,
     11
    ],
    [
     14,
     -1,
     11
    ],
    [
     14,
     0,
     11
    ],
    [
     14,
     1,
     11
    ],
    [
     14,
     2,
     11
    ],
    [
     14,
     3,
     11
    ],
    [
     15,
     -3,
     11
    ],
    [
     15,
     -2,
     11
    ],
    [
     15,
     -1,
     11
    ],
    [
     15,
     0,
     11
    ],
    [
     15,
     1,
     11
    ],
    [
     15,
     2,
     11
    ],
    [
     15,
     3,
     11
    ],
    [
     4,
     -1,
     11
    ],
    [
     4,
     0,
     11
    ],
    [
     4,
     1,
     11
    ],
    [
     5,
     -1,
     11
    ],
    [
     5,
     0,
     11
    ],
    [
     5,
     1,
     11
    ],
    [
     6,
     -1,
     11
    ],
    [
     6,
     0,
     11
    ],
    [
     6,
     1,
     11
    ],
    [
     7,
     -1,
     11
    ],
    [
     7,
     0,
     11
    ],
    [
     7,
     1,
     11
    ],
    [
     8,
     -1,
     11
    ],
    [
     8,
     0,
     11
    ],
    [
     8,
     1,
     11
    ],
    [
     9,
     -1,
     11
    ],
    [
     9,
     0,
     11
    ],
    [
     9,
     1,
     11
    ],
    [
     16,
     0,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 61,
   "prebuilt": true,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails1": {
   "cells": [
    [
     16,
     -2,
     6
    ],
    [
     16,
     2,
     6
    ],
    [
     16,
     -2,
     7
    ],
    [
     16,
     2,
     7
    ],
    [
     16,
     -2,
     8
    ],
    [
     16,
     2,
     8
    ],
    [
     16,
     -2,
     9
    ],
    [
     16,
     2,
     9
    ],
    [
     16,
     -2,
     10
    ],
    [
     16,
     2,
     10
    ],
    [
     16,
     -2,
     11
    ],
    [
     16,
     2,
     11
    ],
    [
     11,
     -5,
     6
    ],
    [
     15,
     -5,
     6
    ],
    [
     11,
     -5,
     7
    ],
    [
     15,
     -5,
     7
    ],
    [
     11,
     -5,
     8
    ],
    [
     15,
     -5,
     8
    ],
    [
     11,
     -5,
     9
    ],
    [
     15,
     -5,
     9
    ],
    [
     11,
     -5,
     10
    ],
    [
     15,
     -5,
     10
    ],
    [
     11,
     -5,
     11
    ],
    [
     15,
     -5,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "prebuilt": true,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs1": {
   "cells": [
    [
     16,
     -1,
     6
    ],
    [
     16,
     0,
     6
    ],
    [
     16,
     1,
     6
    ],
    [
     16,
     -1,
     8
    ],
    [
     16,
     0,
     8
    ],
    [
     16,
     1,
     8
    ],
    [
     16,
     -1,
     10
    ],
    [
     16,
     0,
     10
    ],
    [
     16,
     1,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "prebuilt": true,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "crate": {
   "cells": [
    [
     12,
     -6,
     0
    ],
    [
     12,
     -6,
     1
    ],
    [
     12,
     -5,
     0
    ],
    [
     12,
     -5,
     1
    ],
    [
     12,
     -4,
     0
    ],
    [
     12,
     -4,
     1
    ],
    [
     13,
     -6,
     0
    ],
    [
     13,
     -6,
     1
    ],
    [
     13,
     -5,
     0
    ],
    [
     13,
     -5,
     1
    ],
    [
     13,
     -4,
     0
    ],
    [
     13,
     -4,
     1
    ],
    [
     14,
     -6,
     0
    ],
    [
     14,
     -6,
     1
    ],
    [
     14,
     -5,
     0
    ],
    [
     14,
     -5,
     1
    ],
    [
     14,
     -4,
     0
    ],
    [
     14,
     -4,
     1
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 18,
   "prebuilt": true
  },
  "bench": {
   "cells": [
    [
     8,
     -14,
     2
    ],
    [
     8,
     -13,
     2
    ],
    [
     8,
     -12,
     2
    ],
    [
     9,
     -14,
     2
    ],
    [
     9,
     -13,
     2
    ],
    [
     9,
     -12,
     2
    ],
    [
     10,
     -14,
     2
    ],
    [
     10,
     -13,
     2
    ],
    [
     10,
     -12,
     2
    ],
    [
     11,
     -14,
     2
    ],
    [
     11,
     -12,
     2
    ],
    [
     12,
     -14,
     2
    ],
    [
     12,
     -13,
     2
    ],
    [
     12,
     -12,
     2
    ],
    [
     13,
     -14,
     2
    ],
    [
     13,
     -13,
     2
    ],
    [
     13,
     -12,
     2
    ],
    [
     14,
     -14,
     2
    ],
    [
     14,
     -13,
     2
    ],
    [
     14,
     -12,
     2
    ],
    [
     8,
     -14,
     0
    ],
    [
     8,
     -14,
     1
    ],
    [
     8,
     -12,
     0
    ],
    [
     8,
     -12,
     1
    ],
    [
     14,
     -14,
     0
    ],
    [
     14,
     -14,
     1
    ],
    [
     14,
     -12,
     0
    ],
    [
     14,
     -12,
     1
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 28,
   "prebuilt": true
  },
  "forge": {
   "cells": [
    [
     5,
     -14,
     0
    ],
    [
     5,
     -14,
     1
    ],
    [
     5,
     -14,
     2
    ],
    [
     5,
     -13,
     0
    ],
    [
     5,
     -13,
     1
    ],
    [
     5,
     -13,
     2
    ],
    [
     6,
     -14,
     0
    ],
    [
     6,
     -14,
     1
    ],
    [
     6,
     -14,
     2
    ],
    [
     6,
     -13,
     0
    ],
    [
     6,
     -13,
     1
    ],
    [
     6,
     -13,
     2
    ],
    [
     5,
     -14,
     3
    ],
    [
     5,
     -13,
     3
    ],
    [
     6,
     -14,
     3
    ],
    [
     6,
     -13,
     3
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "brick",
   "expect_blocks": 16,
   "prebuilt": true,
   "parts": {
    "coals": "z == 3"
   }
  },
  "plateA": {
   "cells": [
    [
     -2,
     -1,
     3
    ],
    [
     -2,
     0,
     3
    ],
    [
     -2,
     1,
     3
    ],
    [
     -1,
     -1,
     3
    ],
    [
     -1,
     0,
     3
    ],
    [
     -1,
     1,
     3
    ],
    [
     0,
     -1,
     3
    ],
    [
     0,
     1,
     3
    ],
    [
     1,
     -1,
     3
    ],
    [
     1,
     0,
     3
    ],
    [
     1,
     1,
     3
    ],
    [
     2,
     -1,
     3
    ],
    [
     2,
     0,
     3
    ],
    [
     2,
     1,
     3
    ]
   ],
   "at": [
    1.1,
    -1.3
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 14,
   "prebuilt": true
  },
  "plateB": {
   "cells": [
    [
     -2,
     -1,
     5
    ],
    [
     -2,
     0,
     5
    ],
    [
     -2,
     1,
     5
    ],
    [
     -1,
     -1,
     5
    ],
    [
     -1,
     0,
     5
    ],
    [
     -1,
     1,
     5
    ],
    [
     0,
     -1,
     5
    ],
    [
     0,
     1,
     5
    ],
    [
     1,
     -1,
     5
    ],
    [
     1,
     0,
     5
    ],
    [
     1,
     1,
     5
    ],
    [
     2,
     -1,
     5
    ],
    [
     2,
     0,
     5
    ],
    [
     2,
     1,
     5
    ]
   ],
   "at": [
    1.1,
    -1.3
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 14,
   "prebuilt": true
  },
  "rivet": {
   "cells": [
    [
     0,
     0,
     2
    ],
    [
     0,
     0,
     3
    ],
    [
     0,
     0,
     4
    ]
   ],
   "at": [
    1.1,
    -1.3
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 3,
   "parts": {
    "all": "z >= 0"
   }
  },
  "rivettop": {
   "cells": [
    [
     0,
     0,
     5
    ]
   ],
   "at": [
    1.1,
    -1.3
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 1,
   "parts": {
    "all": "z >= 0"
   }
  },
  "head": {
   "cells": [
    [
     0,
     0,
     6
    ]
   ],
   "at": [
    1.1,
    -1.3
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 1,
   "parts": {
    "all": "z >= 0"
   },
   "block_shape": [
    1,
    1,
    0.55
   ]
  }
 },
 "actions": [
  {
   "type": "crack",
   "target": "forge",
   "part": "coals",
   "when": 0.0,
   "until": 0.3,
   "sound": "none"
  },
  {
   "type": "face_camera",
   "when": 0.0,
   "until": [
    "hook",
    "end"
   ]
  },
  {
   "type": "build",
   "target": "rivet",
   "order": "z",
   "stand_at": [
    1.7,
    -1.3,
    0.0
   ],
   "when": [
    "pop",
    0.42
   ],
   "until": [
    "pop",
    0.55
   ]
  },
  {
   "type": "build",
   "target": "rivettop",
   "stand_at": [
    1.7,
    -1.3,
    0.0
   ],
   "when": [
    "pop",
    0.55
   ],
   "until": [
    "pop",
    0.6
   ]
  },
  {
   "type": "crack",
   "target": "rivet",
   "part": "all",
   "when": [
    "pop",
    0.4
   ],
   "until": [
    "pop",
    0.5
   ],
   "sound": "sizzle",
   "cool_from": [
    "cool",
    0.15
   ],
   "cool_until": [
    "cool",
    0.6
   ]
  },
  {
   "type": "crack",
   "target": "rivettop",
   "part": "all",
   "when": [
    "pop",
    0.5
   ],
   "until": [
    "pop",
    0.6
   ],
   "sound": "none",
   "cool_from": [
    "cool",
    0.15
   ],
   "cool_until": [
    "cool",
    0.6
   ]
  },
  {
   "type": "shrug",
   "when": [
    "pop",
    0.7
   ],
   "until": [
    "pop",
    "end",
    0.2
   ]
  },
  {
   "type": "build",
   "target": "head",
   "stand_at": [
    1.7,
    -1.3,
    0.0
   ],
   "when": [
    "bang",
    0.1
   ],
   "until": [
    "bang",
    0.9
   ]
  },
  {
   "type": "crack",
   "target": "head",
   "part": "all",
   "when": [
    "bang",
    0.1
   ],
   "until": [
    "bang",
    0.3
   ],
   "sound": "none",
   "cool_from": [
    "cool",
    0.15
   ],
   "cool_until": [
    "cool",
    0.6
   ]
  },
  {
   "type": "point",
   "pos": [
    1.1,
    -1.3,
    0.55
   ],
   "when": [
    "cool",
    0.0
   ],
   "until": [
    "cool",
    0.75
   ]
  },
  {
   "type": "slide",
   "target": "plateB",
   "by": [
    0,
    0,
    -0.1
   ],
   "when": [
    "cool",
    0.62
   ],
   "until": [
    "cool",
    0.8
   ]
  },
  {
   "type": "slide",
   "target": "head",
   "by": [
    0,
    0,
    -0.1
   ],
   "when": [
    "cool",
    0.62
   ],
   "until": [
    "cool",
    0.8
   ]
  },
  {
   "type": "remove",
   "target": "rivettop",
   "quiet": true,
   "when": [
    "cool",
    0.62
   ],
   "until": [
    "cool",
    0.64
   ]
  },
  {
   "type": "face_camera",
   "when": [
    "team",
    0.0
   ],
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    1.35,
    -1.3,
    0.75
   ],
   "az": -58,
   "el": 10,
   "dist": 1.9
  },
  {
   "at": [
    "hook",
    "end"
   ],
   "target": [
    1.4,
    -1.3,
    0.6
   ],
   "az": -62,
   "el": 16,
   "dist": 1.7
  },
  {
   "at": [
    "cool",
    0.2
   ],
   "target": [
    1.32,
    -1.3,
    0.5
   ],
   "az": -66,
   "el": 20,
   "dist": 1.35
  },
  {
   "at": [
    "cool",
    "end"
   ],
   "target": [
    1.32,
    -1.3,
    0.5
   ],
   "az": -68,
   "el": 20,
   "dist": 1.4
  },
  {
   "at": [
    "team",
    0.5
   ],
   "target": [
    0.8,
    -0.6,
    1.0
   ],
   "az": -80,
   "el": 10,
   "dist": 3.7
  },
  {
   "at": "end",
   "target": [
    0.8,
    -0.6,
    1.05
   ],
   "az": -88,
   "el": 10,
   "dist": 3.9
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "2,500,000 RIVETS",
   "color": "white",
   "y": 470,
   "size": 92,
   "when": [
    "hook",
    "start"
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "...RED HOT!",
   "color": "red",
   "y": 590,
   "size": 100,
   "when": [
    "hook",
    0.7
   ],
   "until": [
    "hook",
    "end",
    0.2
   ]
  },
  {
   "style": "title",
   "text": "HOT IRON SWELLS",
   "color": "amber",
   "y": 430,
   "size": 84,
   "when": [
    "pop",
    0.05
   ],
   "until": [
    "pop",
    0.65
   ]
  },
  {
   "style": "impact",
   "text": "OW!",
   "color": "red",
   "y": 560,
   "size": 150,
   "when": [
    "pop",
    0.7
   ],
   "until": [
    "pop",
    "end",
    0.2
   ]
  },
  {
   "style": "impact",
   "text": "BANG!",
   "color": "white",
   "y": 560,
   "size": 140,
   "when": [
    "bang",
    0.1
   ],
   "until": [
    "bang",
    "end",
    0.3
   ]
  },
  {
   "style": "title",
   "text": "COOLS = SHRINKS",
   "color": "cyan",
   "y": 430,
   "size": 84,
   "when": [
    "cool",
    0.1
   ],
   "until": [
    "cool",
    0.62
   ]
  },
  {
   "style": "impact",
   "text": "SQUEEZE!",
   "color": "green",
   "y": 560,
   "size": 130,
   "when": [
    "cool",
    0.64
   ],
   "until": [
    "cool",
    "end",
    0.4
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "amber",
   "when": [
    "team",
    0.0
   ],
   "until": [
    "team",
    "end",
    0.3
   ],
   "rows": [
    {
     "text": "1 RIVET = 4 WORKERS",
     "color": "amber",
     "at": [
      "team",
      0.05
     ],
     "size": 50
    },
    {
     "text": "heat \u00b7 hold \u00b7 shape \u00b7 hammer",
     "color": "white",
     "at": [
      "team",
      0.3
     ],
     "size": 38
    },
    {
     "text": "TODAY: 1 GRANDPA",
     "color": "cyan",
     "at": [
      "team",
      0.7
     ],
     "size": 46
    }
   ]
  },
  {
   "style": "end",
   "title1": "HOT RIVETS",
   "title2": "SHRINK TIGHT",
   "sub": "Heated iron swells; as it cools it shrinks and clamps the joint",
   "cta": "Follow for more Block Lab",
   "when": [
    "payoff",
    "end",
    0.2
   ]
  }
 ]
}

In [ ]:
%%writefile stories/eiffel_lessons_short.json
{
 "title": "Building the Eiffel Tower: lessons as we build",
 "slug": "eiffel_lessons_short",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "tail": 0.9,
 "world": {
  "ground": "paris",
  "time": "morning",
  "weather": "clouds",
  "animals": [
   "pigeons"
  ]
 },
 "lines": [
  {
   "id": "intro",
   "text": "Eighteen thousand iron pieces... built in about two years. Let's see how!",
   "pause": 0.65
  },
  {
   "id": "band0",
   "text": "Every piece was drawn first, then made in a factory. A giant kit!",
   "pause": 5.19
  },
  {
   "id": "band2",
   "text": "Two and a half million rivets... put in red hot! They shrink as they cool, and clamp tight.",
   "pause": 3.46
  },
  {
   "id": "band4",
   "text": "Even the cranes climbed up the tower as it grew!",
   "pause": 1.11
  },
  {
   "id": "band5",
   "text": "Finished in eighteen eighty-nine, for the World's Fair.",
   "pause": 3.53
  },
  {
   "id": "voila",
   "text": "Voila! Not bad for an old man, eh?",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   0.95,
   -0.6,
   0.0
  ],
  "outfit": {
   "hat": "beret",
   "shirt": "striped",
   "extras": [
    "scarf",
    "hammer"
   ]
  }
 },
 "structures": {
  "tower0": {
   "cells": [
    [
     -8,
     -8,
     0
    ],
    [
     -7,
     -8,
     0
    ],
    [
     -6,
     -8,
     0
    ],
    [
     -5,
     -8,
     0
    ],
    [
     -4,
     -8,
     0
    ],
    [
     4,
     -8,
     0
    ],
    [
     5,
     -8,
     0
    ],
    [
     6,
     -8,
     0
    ],
    [
     7,
     -8,
     0
    ],
    [
     8,
     -8,
     0
    ],
    [
     -8,
     -7,
     0
    ],
    [
     -4,
     -7,
     0
    ],
    [
     4,
     -7,
     0
    ],
    [
     8,
     -7,
     0
    ],
    [
     -8,
     -6,
     0
    ],
    [
     -4,
     -6,
     0
    ],
    [
     4,
     -6,
     0
    ],
    [
     8,
     -6,
     0
    ],
    [
     -8,
     -5,
     0
    ],
    [
     -4,
     -5,
     0
    ],
    [
     4,
     -5,
     0
    ],
    [
     8,
     -5,
     0
    ],
    [
     -8,
     -4,
     0
    ],
    [
     -7,
     -4,
     0
    ],
    [
     -6,
     -4,
     0
    ],
    [
     -5,
     -4,
     0
    ],
    [
     -4,
     -4,
     0
    ],
    [
     4,
     -4,
     0
    ],
    [
     5,
     -4,
     0
    ],
    [
     6,
     -4,
     0
    ],
    [
     7,
     -4,
     0
    ],
    [
     8,
     -4,
     0
    ],
    [
     -8,
     4,
     0
    ],
    [
     -7,
     4,
     0
    ],
    [
     -6,
     4,
     0
    ],
    [
     -5,
     4,
     0
    ],
    [
     -4,
     4,
     0
    ],
    [
     4,
     4,
     0
    ],
    [
     5,
     4,
     0
    ],
    [
     6,
     4,
     0
    ],
    [
     7,
     4,
     0
    ],
    [
     8,
     4,
     0
    ],
    [
     -8,
     5,
     0
    ],
    [
     -4,
     5,
     0
    ],
    [
     4,
     5,
     0
    ],
    [
     8,
     5,
     0
    ],
    [
     -8,
     6,
     0
    ],
    [
     -4,
     6,
     0
    ],
    [
     4,
     6,
     0
    ],
    [
     8,
     6,
     0
    ],
    [
     -8,
     7,
     0
    ],
    [
     -4,
     7,
     0
    ],
    [
     4,
     7,
     0
    ],
    [
     8,
     7,
     0
    ],
    [
     -8,
     8,
     0
    ],
    [
     -7,
     8,
     0
    ],
    [
     -6,
     8,
     0
    ],
    [
     -5,
     8,
     0
    ],
    [
     -4,
     8,
     0
    ],
    [
     4,
     8,
     0
    ],
    [
     5,
     8,
     0
    ],
    [
     6,
     8,
     0
    ],
    [
     7,
     8,
     0
    ],
    [
     8,
     8,
     0
    ],
    [
     -7,
     -7,
     1
    ],
    [
     -4,
     -7,
     1
    ],
    [
     4,
     -7,
     1
    ],
    [
     7,
     -7,
     1
    ],
    [
     -7,
     -4,
     1
    ],
    [
     -4,
     -4,
     1
    ],
    [
     4,
     -4,
     1
    ],
    [
     7,
     -4,
     1
    ],
    [
     -7,
     4,
     1
    ],
    [
     -4,
     4,
     1
    ],
    [
     4,
     4,
     1
    ],
    [
     7,
     4,
     1
    ],
    [
     -7,
     7,
     1
    ],
    [
     -4,
     7,
     1
    ],
    [
     4,
     7,
     1
    ],
    [
     7,
     7,
     1
    ],
    [
     -6,
     -6,
     2
    ],
    [
     -5,
     -6,
     2
    ],
    [
     -4,
     -6,
     2
    ],
    [
     -3,
     -6,
     2
    ],
    [
     3,
     -6,
     2
    ],
    [
     4,
     -6,
     2
    ],
    [
     5,
     -6,
     2
    ],
    [
     6,
     -6,
     2
    ],
    [
     -6,
     -5,
     2
    ],
    [
     -3,
     -5,
     2
    ],
    [
     3,
     -5,
     2
    ],
    [
     6,
     -5,
     2
    ],
    [
     -6,
     -4,
     2
    ],
    [
     -3,
     -4,
     2
    ],
    [
     3,
     -4,
     2
    ],
    [
     6,
     -4,
     2
    ],
    [
     -6,
     -3,
     2
    ],
    [
     -5,
     -3,
     2
    ],
    [
     -4,
     -3,
     2
    ],
    [
     -3,
     -3,
     2
    ],
    [
     3,
     -3,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     6,
     -3,
     2
    ],
    [
     -6,
     3,
     2
    ],
    [
     -5,
     3,
     2
    ],
    [
     -4,
     3,
     2
    ],
    [
     -3,
     3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     6,
     3,
     2
    ],
    [
     -6,
     4,
     2
    ],
    [
     -3,
     4,
     2
    ],
    [
     3,
     4,
     2
    ],
    [
     6,
     4,
     2
    ],
    [
     -6,
     5,
     2
    ],
    [
     -3,
     5,
     2
    ],
    [
     3,
     5,
     2
    ],
    [
     6,
     5,
     2
    ],
    [
     -6,
     6,
     2
    ],
    [
     -5,
     6,
     2
    ],
    [
     -4,
     6,
     2
    ],
    [
     -3,
     6,
     2
    ],
    [
     3,
     6,
     2
    ],
    [
     4,
     6,
     2
    ],
    [
     5,
     6,
     2
    ],
    [
     6,
     6,
     2
    ],
    [
     -6,
     -6,
     3
    ],
    [
     -3,
     -6,
     3
    ],
    [
     3,
     -6,
     3
    ],
    [
     6,
     -6,
     3
    ],
    [
     -3,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     -6,
     -3,
     3
    ],
    [
     -4,
     -3,
     3
    ],
    [
     -3,
     -3,
     3
    ],
    [
     3,
     -3,
     3
    ],
    [
     4,
     -3,
     3
    ],
    [
     6,
     -3,
     3
    ],
    [
     -6,
     3,
     3
    ],
    [
     -4,
     3,
     3
    ],
    [
     -3,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     -3,
     4,
     3
    ],
    [
     3,
     4,
     3
    ],
    [
     -6,
     6,
     3
    ],
    [
     -3,
     6,
     3
    ],
    [
     3,
     6,
     3
    ],
    [
     6,
     6,
     3
    ],
    [
     -5,
     -5,
     4
    ],
    [
     -4,
     -5,
     4
    ],
    [
     -3,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     -5,
     -4,
     4
    ],
    [
     -3,
     -4,
     4
    ],
    [
     -2,
     -4,
     4
    ],
    [
     -1,
     -4,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     -5,
     -3,
     4
    ],
    [
     -4,
     -3,
     4
    ],
    [
     -3,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     -4,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     -4,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     -4,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     -4,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     -4,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     -5,
     3,
     4
    ],
    [
     -4,
     3,
     4
    ],
    [
     -3,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     -5,
     4,
     4
    ],
    [
     -3,
     4,
     4
    ],
    [
     -2,
     4,
     4
    ],
    [
     -1,
     4,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     -5,
     5,
     4
    ],
    [
     -4,
     5,
     4
    ],
    [
     -3,
     5,
     4
    ],
    [
     3,
     5,
     4
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     4
    ],
    [
     -5,
     -5,
     5
    ],
    [
     -2,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     -2,
     -4,
     5
    ],
    [
     -1,
     -4,
     5
    ],
    [
     0,
     -4,
     5
    ],
    [
     1,
     -4,
     5
    ],
    [
     2,
     -4,
     5
    ],
    [
     -5,
     -2,
     5
    ],
    [
     -4,
     -2,
     5
    ],
    [
     -2,
     -2,
     5
    ],
    [
     2,
     -2,
     5
    ],
    [
     4,
     -2,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     -4,
     -1,
     5
    ],
    [
     4,
     -1,
     5
    ],
    [
     -4,
     0,
     5
    ],
    [
     4,
     0,
     5
    ],
    [
     -4,
     1,
     5
    ],
    [
     4,
     1,
     5
    ],
    [
     -5,
     2,
     5
    ],
    [
     -4,
     2,
     5
    ],
    [
     -2,
     2,
     5
    ],
    [
     2,
     2,
     5
    ],
    [
     4,
     2,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     -2,
     4,
     5
    ],
    [
     -1,
     4,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     -5,
     5,
     5
    ],
    [
     -2,
     5,
     5
    ],
    [
     2,
     5,
     5
    ],
    [
     5,
     5,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 240
  },
  "tower1": {
   "cells": [
    [
     -5,
     -5,
     6
    ],
    [
     -4,
     -5,
     6
    ],
    [
     -3,
     -5,
     6
    ],
    [
     -2,
     -5,
     6
    ],
    [
     -1,
     -5,
     6
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     -5,
     -4,
     6
    ],
    [
     -4,
     -4,
     6
    ],
    [
     -3,
     -4,
     6
    ],
    [
     -2,
     -4,
     6
    ],
    [
     -1,
     -4,
     6
    ],
    [
     0,
     -4,
     6
    ],
    [
     1,
     -4,
     6
    ],
    [
     2,
     -4,
     6
    ],
    [
     3,
     -4,
     6
    ],
    [
     4,
     -4,
     6
    ],
    [
     5,
     -4,
     6
    ],
    [
     -5,
     -3,
     6
    ],
    [
     -4,
     -3,
     6
    ],
    [
     -3,
     -3,
     6
    ],
    [
     -2,
     -3,
     6
    ],
    [
     -1,
     -3,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     1,
     -3,
     6
    ],
    [
     2,
     -3,
     6
    ],
    [
     3,
     -3,
     6
    ],
    [
     4,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     -5,
     -2,
     6
    ],
    [
     -4,
     -2,
     6
    ],
    [
     -3,
     -2,
     6
    ],
    [
     -2,
     -2,
     6
    ],
    [
     -1,
     -2,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     1,
     -2,
     6
    ],
    [
     2,
     -2,
     6
    ],
    [
     3,
     -2,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     5,
     -2,
     6
    ],
    [
     -5,
     -1,
     6
    ],
    [
     -4,
     -1,
     6
    ],
    [
     -3,
     -1,
     6
    ],
    [
     -2,
     -1,
     6
    ],
    [
     2,
     -1,
     6
    ],
    [
     3,
     -1,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     5,
     -1,
     6
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     6
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     6
    ],
    [
     2,
     0,
     6
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     5,
     0,
     6
    ],
    [
     -5,
     1,
     6
    ],
    [
     -4,
     1,
     6
    ],
    [
     -3,
     1,
     6
    ],
    [
     -2,
     1,
     6
    ],
    [
     2,
     1,
     6
    ],
    [
     3,
     1,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     5,
     1,
     6
    ],
    [
     -5,
     2,
     6
    ],
    [
     -4,
     2,
     6
    ],
    [
     -3,
     2,
     6
    ],
    [
     -2,
     2,
     6
    ],
    [
     -1,
     2,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     1,
     2,
     6
    ],
    [
     2,
     2,
     6
    ],
    [
     3,
     2,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     5,
     2,
     6
    ],
    [
     -5,
     3,
     6
    ],
    [
     -4,
     3,
     6
    ],
    [
     -3,
     3,
     6
    ],
    [
     -2,
     3,
     6
    ],
    [
     -1,
     3,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     1,
     3,
     6
    ],
    [
     2,
     3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     4,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     -5,
     4,
     6
    ],
    [
     -4,
     4,
     6
    ],
    [
     -3,
     4,
     6
    ],
    [
     -2,
     4,
     6
    ],
    [
     -1,
     4,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     -5,
     5,
     6
    ],
    [
     -4,
     5,
     6
    ],
    [
     -3,
     5,
     6
    ],
    [
     -2,
     5,
     6
    ],
    [
     -1,
     5,
     6
    ],
    [
     0,
     5,
     6
    ],
    [
     1,
     5,
     6
    ],
    [
     2,
     5,
     6
    ],
    [
     3,
     5,
     6
    ],
    [
     4,
     5,
     6
    ],
    [
     5,
     5,
     6
    ],
    [
     -5,
     -5,
     7
    ],
    [
     -3,
     -5,
     7
    ],
    [
     -1,
     -5,
     7
    ],
    [
     1,
     -5,
     7
    ],
    [
     3,
     -5,
     7
    ],
    [
     5,
     -5,
     7
    ],
    [
     -4,
     -4,
     7
    ],
    [
     -2,
     -4,
     7
    ],
    [
     2,
     -4,
     7
    ],
    [
     4,
     -4,
     7
    ],
    [
     -5,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     -4,
     -2,
     7
    ],
    [
     -2,
     -2,
     7
    ],
    [
     2,
     -2,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     -5,
     -1,
     7
    ],
    [
     5,
     -1,
     7
    ],
    [
     -5,
     1,
     7
    ],
    [
     5,
     1,
     7
    ],
    [
     -4,
     2,
     7
    ],
    [
     -2,
     2,
     7
    ],
    [
     2,
     2,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     -5,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     -4,
     4,
     7
    ],
    [
     -2,
     4,
     7
    ],
    [
     2,
     4,
     7
    ],
    [
     4,
     4,
     7
    ],
    [
     -5,
     5,
     7
    ],
    [
     -3,
     5,
     7
    ],
    [
     -1,
     5,
     7
    ],
    [
     1,
     5,
     7
    ],
    [
     3,
     5,
     7
    ],
    [
     5,
     5,
     7
    ],
    [
     -4,
     -4,
     8
    ],
    [
     -3,
     -4,
     8
    ],
    [
     -2,
     -4,
     8
    ],
    [
     -1,
     -4,
     8
    ],
    [
     1,
     -4,
     8
    ],
    [
     2,
     -4,
     8
    ],
    [
     3,
     -4,
     8
    ],
    [
     4,
     -4,
     8
    ],
    [
     -4,
     -3,
     8
    ],
    [
     -1,
     -3,
     8
    ],
    [
     1,
     -3,
     8
    ],
    [
     4,
     -3,
     8
    ],
    [
     -4,
     -2,
     8
    ],
    [
     -1,
     -2,
     8
    ],
    [
     1,
     -2,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     -4,
     -1,
     8
    ],
    [
     -3,
     -1,
     8
    ],
    [
     -2,
     -1,
     8
    ],
    [
     -1,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     3,
     -1,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     -4,
     1,
     8
    ],
    [
     -3,
     1,
     8
    ],
    [
     -2,
     1,
     8
    ],
    [
     -1,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     3,
     1,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     -4,
     2,
     8
    ],
    [
     -1,
     2,
     8
    ],
    [
     1,
     2,
     8
    ],
    [
     4,
     2,
     8
    ],
    [
     -4,
     3,
     8
    ],
    [
     -1,
     3,
     8
    ],
    [
     1,
     3,
     8
    ],
    [
     4,
     3,
     8
    ],
    [
     -4,
     4,
     8
    ],
    [
     -3,
     4,
     8
    ],
    [
     -2,
     4,
     8
    ],
    [
     -1,
     4,
     8
    ],
    [
     1,
     4,
     8
    ],
    [
     2,
     4,
     8
    ],
    [
     3,
     4,
     8
    ],
    [
     4,
     4,
     8
    ],
    [
     -3,
     -3,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     1,
     -3,
     9
    ],
    [
     3,
     -3,
     9
    ],
    [
     -3,
     -1,
     9
    ],
    [
     -1,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     3,
     -1,
     9
    ],
    [
     -3,
     1,
     9
    ],
    [
     -1,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     3,
     1,
     9
    ],
    [
     -3,
     3,
     9
    ],
    [
     -1,
     3,
     9
    ],
    [
     1,
     3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     -3,
     -3,
     10
    ],
    [
     -2,
     -3,
     10
    ],
    [
     -1,
     -3,
     10
    ],
    [
     1,
     -3,
     10
    ],
    [
     2,
     -3,
     10
    ],
    [
     3,
     -3,
     10
    ],
    [
     -3,
     -2,
     10
    ],
    [
     -1,
     -2,
     10
    ],
    [
     1,
     -2,
     10
    ],
    [
     3,
     -2,
     10
    ],
    [
     -3,
     -1,
     10
    ],
    [
     -2,
     -1,
     10
    ],
    [
     -1,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     3,
     -1,
     10
    ],
    [
     -3,
     1,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     -1,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ],
    [
     3,
     1,
     10
    ],
    [
     -3,
     2,
     10
    ],
    [
     -1,
     2,
     10
    ],
    [
     1,
     2,
     10
    ],
    [
     3,
     2,
     10
    ],
    [
     -3,
     3,
     10
    ],
    [
     -2,
     3,
     10
    ],
    [
     -1,
     3,
     10
    ],
    [
     1,
     3,
     10
    ],
    [
     2,
     3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     -3,
     -3,
     11
    ],
    [
     -1,
     -3,
     11
    ],
    [
     1,
     -3,
     11
    ],
    [
     3,
     -3,
     11
    ],
    [
     -3,
     -1,
     11
    ],
    [
     -1,
     -1,
     11
    ],
    [
     1,
     -1,
     11
    ],
    [
     3,
     -1,
     11
    ],
    [
     -3,
     1,
     11
    ],
    [
     -1,
     1,
     11
    ],
    [
     1,
     1,
     11
    ],
    [
     3,
     1,
     11
    ],
    [
     -3,
     3,
     11
    ],
    [
     -1,
     3,
     11
    ],
    [
     1,
     3,
     11
    ],
    [
     3,
     3,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 260
  },
  "tower2": {
   "cells": [
    [
     -3,
     -3,
     12
    ],
    [
     -2,
     -3,
     12
    ],
    [
     -1,
     -3,
     12
    ],
    [
     0,
     -3,
     12
    ],
    [
     1,
     -3,
     12
    ],
    [
     2,
     -3,
     12
    ],
    [
     3,
     -3,
     12
    ],
    [
     -3,
     -2,
     12
    ],
    [
     -2,
     -2,
     12
    ],
    [
     -1,
     -2,
     12
    ],
    [
     0,
     -2,
     12
    ],
    [
     1,
     -2,
     12
    ],
    [
     2,
     -2,
     12
    ],
    [
     3,
     -2,
     12
    ],
    [
     -3,
     -1,
     12
    ],
    [
     -2,
     -1,
     12
    ],
    [
     -1,
     -1,
     12
    ],
    [
     0,
     -1,
     12
    ],
    [
     1,
     -1,
     12
    ],
    [
     2,
     -1,
     12
    ],
    [
     3,
     -1,
     12
    ],
    [
     -3,
     0,
     12
    ],
    [
     -2,
     0,
     12
    ],
    [
     -1,
     0,
     12
    ],
    [
     0,
     0,
     12
    ],
    [
     1,
     0,
     12
    ],
    [
     2,
     0,
     12
    ],
    [
     3,
     0,
     12
    ],
    [
     -3,
     1,
     12
    ],
    [
     -2,
     1,
     12
    ],
    [
     -1,
     1,
     12
    ],
    [
     0,
     1,
     12
    ],
    [
     1,
     1,
     12
    ],
    [
     2,
     1,
     12
    ],
    [
     3,
     1,
     12
    ],
    [
     -3,
     2,
     12
    ],
    [
     -2,
     2,
     12
    ],
    [
     -1,
     2,
     12
    ],
    [
     0,
     2,
     12
    ],
    [
     1,
     2,
     12
    ],
    [
     2,
     2,
     12
    ],
    [
     3,
     2,
     12
    ],
    [
     -3,
     3,
     12
    ],
    [
     -2,
     3,
     12
    ],
    [
     -1,
     3,
     12
    ],
    [
     0,
     3,
     12
    ],
    [
     1,
     3,
     12
    ],
    [
     2,
     3,
     12
    ],
    [
     3,
     3,
     12
    ],
    [
     -3,
     -3,
     13
    ],
    [
     -1,
     -3,
     13
    ],
    [
     1,
     -3,
     13
    ],
    [
     3,
     -3,
     13
    ],
    [
     -2,
     -2,
     13
    ],
    [
     2,
     -2,
     13
    ],
    [
     -3,
     -1,
     13
    ],
    [
     3,
     -1,
     13
    ],
    [
     -3,
     1,
     13
    ],
    [
     3,
     1,
     13
    ],
    [
     -2,
     2,
     13
    ],
    [
     2,
     2,
     13
    ],
    [
     -3,
     3,
     13
    ],
    [
     -1,
     3,
     13
    ],
    [
     1,
     3,
     13
    ],
    [
     3,
     3,
     13
    ],
    [
     -2,
     -2,
     14
    ],
    [
     -1,
     -2,
     14
    ],
    [
     0,
     -2,
     14
    ],
    [
     1,
     -2,
     14
    ],
    [
     2,
     -2,
     14
    ],
    [
     -2,
     -1,
     14
    ],
    [
     2,
     -1,
     14
    ],
    [
     -2,
     0,
     14
    ],
    [
     2,
     0,
     14
    ],
    [
     -2,
     1,
     14
    ],
    [
     2,
     1,
     14
    ],
    [
     -2,
     2,
     14
    ],
    [
     -1,
     2,
     14
    ],
    [
     0,
     2,
     14
    ],
    [
     1,
     2,
     14
    ],
    [
     2,
     2,
     14
    ],
    [
     -2,
     -2,
     15
    ],
    [
     2,
     -2,
     15
    ],
    [
     -2,
     2,
     15
    ],
    [
     2,
     2,
     15
    ],
    [
     -2,
     -2,
     16
    ],
    [
     -1,
     -2,
     16
    ],
    [
     0,
     -2,
     16
    ],
    [
     1,
     -2,
     16
    ],
    [
     2,
     -2,
     16
    ],
    [
     -2,
     -1,
     16
    ],
    [
     2,
     -1,
     16
    ],
    [
     -2,
     0,
     16
    ],
    [
     2,
     0,
     16
    ],
    [
     -2,
     1,
     16
    ],
    [
     2,
     1,
     16
    ],
    [
     -2,
     2,
     16
    ],
    [
     -1,
     2,
     16
    ],
    [
     0,
     2,
     16
    ],
    [
     1,
     2,
     16
    ],
    [
     2,
     2,
     16
    ],
    [
     -2,
     -2,
     17
    ],
    [
     2,
     -2,
     17
    ],
    [
     -2,
     2,
     17
    ],
    [
     2,
     2,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 105
  },
  "tower3": {
   "cells": [
    [
     -1,
     -1,
     18
    ],
    [
     0,
     -1,
     18
    ],
    [
     1,
     -1,
     18
    ],
    [
     -1,
     0,
     18
    ],
    [
     1,
     0,
     18
    ],
    [
     -1,
     1,
     18
    ],
    [
     0,
     1,
     18
    ],
    [
     1,
     1,
     18
    ],
    [
     -1,
     -1,
     19
    ],
    [
     1,
     -1,
     19
    ],
    [
     -1,
     1,
     19
    ],
    [
     1,
     1,
     19
    ],
    [
     -1,
     -1,
     20
    ],
    [
     0,
     -1,
     20
    ],
    [
     1,
     -1,
     20
    ],
    [
     -1,
     0,
     20
    ],
    [
     1,
     0,
     20
    ],
    [
     -1,
     1,
     20
    ],
    [
     0,
     1,
     20
    ],
    [
     1,
     1,
     20
    ],
    [
     -1,
     -1,
     21
    ],
    [
     1,
     -1,
     21
    ],
    [
     -1,
     1,
     21
    ],
    [
     1,
     1,
     21
    ],
    [
     -1,
     -1,
     22
    ],
    [
     0,
     -1,
     22
    ],
    [
     1,
     -1,
     22
    ],
    [
     -1,
     0,
     22
    ],
    [
     1,
     0,
     22
    ],
    [
     -1,
     1,
     22
    ],
    [
     0,
     1,
     22
    ],
    [
     1,
     1,
     22
    ],
    [
     -1,
     -1,
     23
    ],
    [
     1,
     -1,
     23
    ],
    [
     -1,
     1,
     23
    ],
    [
     1,
     1,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 36
  },
  "tower4": {
   "cells": [
    [
     -1,
     -1,
     24
    ],
    [
     0,
     -1,
     24
    ],
    [
     1,
     -1,
     24
    ],
    [
     -1,
     0,
     24
    ],
    [
     1,
     0,
     24
    ],
    [
     -1,
     1,
     24
    ],
    [
     0,
     1,
     24
    ],
    [
     1,
     1,
     24
    ],
    [
     -1,
     -1,
     25
    ],
    [
     1,
     -1,
     25
    ],
    [
     -1,
     1,
     25
    ],
    [
     1,
     1,
     25
    ],
    [
     -1,
     -1,
     26
    ],
    [
     0,
     -1,
     26
    ],
    [
     1,
     -1,
     26
    ],
    [
     -1,
     0,
     26
    ],
    [
     1,
     0,
     26
    ],
    [
     -1,
     1,
     26
    ],
    [
     0,
     1,
     26
    ],
    [
     1,
     1,
     26
    ],
    [
     -1,
     -1,
     27
    ],
    [
     1,
     -1,
     27
    ],
    [
     -1,
     1,
     27
    ],
    [
     1,
     1,
     27
    ],
    [
     -1,
     -1,
     28
    ],
    [
     0,
     -1,
     28
    ],
    [
     1,
     -1,
     28
    ],
    [
     -1,
     0,
     28
    ],
    [
     0,
     0,
     28
    ],
    [
     1,
     0,
     28
    ],
    [
     -1,
     1,
     28
    ],
    [
     0,
     1,
     28
    ],
    [
     1,
     1,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 33
  },
  "tower5": {
   "cells": [
    [
     0,
     0,
     29
    ],
    [
     0,
     0,
     30
    ],
    [
     0,
     0,
     31
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 3
  },
  "posts0": {
   "cells": [
    [
     10,
     -3,
     0
    ],
    [
     10,
     3,
     0
    ],
    [
     15,
     -3,
     0
    ],
    [
     15,
     3,
     0
    ],
    [
     10,
     -3,
     1
    ],
    [
     10,
     3,
     1
    ],
    [
     15,
     -3,
     1
    ],
    [
     15,
     3,
     1
    ],
    [
     10,
     -3,
     2
    ],
    [
     10,
     3,
     2
    ],
    [
     15,
     -3,
     2
    ],
    [
     15,
     3,
     2
    ],
    [
     10,
     -3,
     3
    ],
    [
     10,
     3,
     3
    ],
    [
     15,
     -3,
     3
    ],
    [
     15,
     3,
     3
    ],
    [
     10,
     -3,
     4
    ],
    [
     10,
     3,
     4
    ],
    [
     15,
     -3,
     4
    ],
    [
     15,
     3,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks0": {
   "cells": [
    [
     10,
     -3,
     5
    ],
    [
     10,
     -2,
     5
    ],
    [
     10,
     -1,
     5
    ],
    [
     10,
     0,
     5
    ],
    [
     10,
     1,
     5
    ],
    [
     10,
     2,
     5
    ],
    [
     10,
     3,
     5
    ],
    [
     11,
     -3,
     5
    ],
    [
     11,
     -2,
     5
    ],
    [
     11,
     -1,
     5
    ],
    [
     11,
     0,
     5
    ],
    [
     11,
     1,
     5
    ],
    [
     11,
     2,
     5
    ],
    [
     11,
     3,
     5
    ],
    [
     12,
     -3,
     5
    ],
    [
     12,
     -2,
     5
    ],
    [
     12,
     -1,
     5
    ],
    [
     12,
     0,
     5
    ],
    [
     12,
     1,
     5
    ],
    [
     12,
     2,
     5
    ],
    [
     12,
     3,
     5
    ],
    [
     13,
     -3,
     5
    ],
    [
     13,
     -2,
     5
    ],
    [
     13,
     -1,
     5
    ],
    [
     13,
     0,
     5
    ],
    [
     13,
     1,
     5
    ],
    [
     13,
     2,
     5
    ],
    [
     13,
     3,
     5
    ],
    [
     14,
     -3,
     5
    ],
    [
     14,
     -2,
     5
    ],
    [
     14,
     -1,
     5
    ],
    [
     14,
     0,
     5
    ],
    [
     14,
     1,
     5
    ],
    [
     14,
     2,
     5
    ],
    [
     14,
     3,
     5
    ],
    [
     15,
     -3,
     5
    ],
    [
     15,
     -2,
     5
    ],
    [
     15,
     -1,
     5
    ],
    [
     15,
     0,
     5
    ],
    [
     15,
     1,
     5
    ],
    [
     15,
     2,
     5
    ],
    [
     15,
     3,
     5
    ],
    [
     6,
     -1,
     5
    ],
    [
     6,
     0,
     5
    ],
    [
     6,
     1,
     5
    ],
    [
     7,
     -1,
     5
    ],
    [
     7,
     0,
     5
    ],
    [
     7,
     1,
     5
    ],
    [
     8,
     -1,
     5
    ],
    [
     8,
     0,
     5
    ],
    [
     8,
     1,
     5
    ],
    [
     9,
     -1,
     5
    ],
    [
     9,
     0,
     5
    ],
    [
     9,
     1,
     5
    ],
    [
     16,
     0,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 55,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails0": {
   "cells": [
    [
     16,
     -2,
     0
    ],
    [
     16,
     2,
     0
    ],
    [
     16,
     -2,
     1
    ],
    [
     16,
     2,
     1
    ],
    [
     16,
     -2,
     2
    ],
    [
     16,
     2,
     2
    ],
    [
     16,
     -2,
     3
    ],
    [
     16,
     2,
     3
    ],
    [
     16,
     -2,
     4
    ],
    [
     16,
     2,
     4
    ],
    [
     16,
     -2,
     5
    ],
    [
     16,
     2,
     5
    ],
    [
     11,
     -5,
     0
    ],
    [
     15,
     -5,
     0
    ],
    [
     11,
     -5,
     1
    ],
    [
     15,
     -5,
     1
    ],
    [
     11,
     -5,
     2
    ],
    [
     15,
     -5,
     2
    ],
    [
     11,
     -5,
     3
    ],
    [
     15,
     -5,
     3
    ],
    [
     11,
     -5,
     4
    ],
    [
     15,
     -5,
     4
    ],
    [
     11,
     -5,
     5
    ],
    [
     15,
     -5,
     5
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs0": {
   "cells": [
    [
     16,
     -1,
     0
    ],
    [
     16,
     0,
     0
    ],
    [
     16,
     1,
     0
    ],
    [
     16,
     -1,
     2
    ],
    [
     16,
     0,
     2
    ],
    [
     16,
     1,
     2
    ],
    [
     16,
     -1,
     4
    ],
    [
     16,
     0,
     4
    ],
    [
     16,
     1,
     4
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts1": {
   "cells": [
    [
     10,
     -3,
     6
    ],
    [
     10,
     3,
     6
    ],
    [
     15,
     -3,
     6
    ],
    [
     15,
     3,
     6
    ],
    [
     10,
     -3,
     7
    ],
    [
     10,
     3,
     7
    ],
    [
     15,
     -3,
     7
    ],
    [
     15,
     3,
     7
    ],
    [
     10,
     -3,
     8
    ],
    [
     10,
     3,
     8
    ],
    [
     15,
     -3,
     8
    ],
    [
     15,
     3,
     8
    ],
    [
     10,
     -3,
     9
    ],
    [
     10,
     3,
     9
    ],
    [
     15,
     -3,
     9
    ],
    [
     15,
     3,
     9
    ],
    [
     10,
     -3,
     10
    ],
    [
     10,
     3,
     10
    ],
    [
     15,
     -3,
     10
    ],
    [
     15,
     3,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks1": {
   "cells": [
    [
     10,
     -3,
     11
    ],
    [
     10,
     -2,
     11
    ],
    [
     10,
     -1,
     11
    ],
    [
     10,
     0,
     11
    ],
    [
     10,
     1,
     11
    ],
    [
     10,
     2,
     11
    ],
    [
     10,
     3,
     11
    ],
    [
     11,
     -3,
     11
    ],
    [
     11,
     -2,
     11
    ],
    [
     11,
     -1,
     11
    ],
    [
     11,
     0,
     11
    ],
    [
     11,
     1,
     11
    ],
    [
     11,
     2,
     11
    ],
    [
     11,
     3,
     11
    ],
    [
     12,
     -3,
     11
    ],
    [
     12,
     -2,
     11
    ],
    [
     12,
     -1,
     11
    ],
    [
     12,
     0,
     11
    ],
    [
     12,
     1,
     11
    ],
    [
     12,
     2,
     11
    ],
    [
     12,
     3,
     11
    ],
    [
     13,
     -3,
     11
    ],
    [
     13,
     -2,
     11
    ],
    [
     13,
     -1,
     11
    ],
    [
     13,
     0,
     11
    ],
    [
     13,
     1,
     11
    ],
    [
     13,
     2,
     11
    ],
    [
     13,
     3,
     11
    ],
    [
     14,
     -3,
     11
    ],
    [
     14,
     -2,
     11
    ],
    [
     14,
     -1,
     11
    ],
    [
     14,
     0,
     11
    ],
    [
     14,
     1,
     11
    ],
    [
     14,
     2,
     11
    ],
    [
     14,
     3,
     11
    ],
    [
     15,
     -3,
     11
    ],
    [
     15,
     -2,
     11
    ],
    [
     15,
     -1,
     11
    ],
    [
     15,
     0,
     11
    ],
    [
     15,
     1,
     11
    ],
    [
     15,
     2,
     11
    ],
    [
     15,
     3,
     11
    ],
    [
     4,
     -1,
     11
    ],
    [
     4,
     0,
     11
    ],
    [
     4,
     1,
     11
    ],
    [
     5,
     -1,
     11
    ],
    [
     5,
     0,
     11
    ],
    [
     5,
     1,
     11
    ],
    [
     6,
     -1,
     11
    ],
    [
     6,
     0,
     11
    ],
    [
     6,
     1,
     11
    ],
    [
     7,
     -1,
     11
    ],
    [
     7,
     0,
     11
    ],
    [
     7,
     1,
     11
    ],
    [
     8,
     -1,
     11
    ],
    [
     8,
     0,
     11
    ],
    [
     8,
     1,
     11
    ],
    [
     9,
     -1,
     11
    ],
    [
     9,
     0,
     11
    ],
    [
     9,
     1,
     11
    ],
    [
     16,
     0,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 61,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails1": {
   "cells": [
    [
     16,
     -2,
     6
    ],
    [
     16,
     2,
     6
    ],
    [
     16,
     -2,
     7
    ],
    [
     16,
     2,
     7
    ],
    [
     16,
     -2,
     8
    ],
    [
     16,
     2,
     8
    ],
    [
     16,
     -2,
     9
    ],
    [
     16,
     2,
     9
    ],
    [
     16,
     -2,
     10
    ],
    [
     16,
     2,
     10
    ],
    [
     16,
     -2,
     11
    ],
    [
     16,
     2,
     11
    ],
    [
     11,
     -5,
     6
    ],
    [
     15,
     -5,
     6
    ],
    [
     11,
     -5,
     7
    ],
    [
     15,
     -5,
     7
    ],
    [
     11,
     -5,
     8
    ],
    [
     15,
     -5,
     8
    ],
    [
     11,
     -5,
     9
    ],
    [
     15,
     -5,
     9
    ],
    [
     11,
     -5,
     10
    ],
    [
     15,
     -5,
     10
    ],
    [
     11,
     -5,
     11
    ],
    [
     15,
     -5,
     11
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs1": {
   "cells": [
    [
     16,
     -1,
     6
    ],
    [
     16,
     0,
     6
    ],
    [
     16,
     1,
     6
    ],
    [
     16,
     -1,
     8
    ],
    [
     16,
     0,
     8
    ],
    [
     16,
     1,
     8
    ],
    [
     16,
     -1,
     10
    ],
    [
     16,
     0,
     10
    ],
    [
     16,
     1,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts2": {
   "cells": [
    [
     10,
     -3,
     12
    ],
    [
     10,
     3,
     12
    ],
    [
     15,
     -3,
     12
    ],
    [
     15,
     3,
     12
    ],
    [
     10,
     -3,
     13
    ],
    [
     10,
     3,
     13
    ],
    [
     15,
     -3,
     13
    ],
    [
     15,
     3,
     13
    ],
    [
     10,
     -3,
     14
    ],
    [
     10,
     3,
     14
    ],
    [
     15,
     -3,
     14
    ],
    [
     15,
     3,
     14
    ],
    [
     10,
     -3,
     15
    ],
    [
     10,
     3,
     15
    ],
    [
     15,
     -3,
     15
    ],
    [
     15,
     3,
     15
    ],
    [
     10,
     -3,
     16
    ],
    [
     10,
     3,
     16
    ],
    [
     15,
     -3,
     16
    ],
    [
     15,
     3,
     16
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks2": {
   "cells": [
    [
     10,
     -3,
     17
    ],
    [
     10,
     -2,
     17
    ],
    [
     10,
     -1,
     17
    ],
    [
     10,
     0,
     17
    ],
    [
     10,
     1,
     17
    ],
    [
     10,
     2,
     17
    ],
    [
     10,
     3,
     17
    ],
    [
     11,
     -3,
     17
    ],
    [
     11,
     -2,
     17
    ],
    [
     11,
     -1,
     17
    ],
    [
     11,
     0,
     17
    ],
    [
     11,
     1,
     17
    ],
    [
     11,
     2,
     17
    ],
    [
     11,
     3,
     17
    ],
    [
     12,
     -3,
     17
    ],
    [
     12,
     -2,
     17
    ],
    [
     12,
     -1,
     17
    ],
    [
     12,
     0,
     17
    ],
    [
     12,
     1,
     17
    ],
    [
     12,
     2,
     17
    ],
    [
     12,
     3,
     17
    ],
    [
     13,
     -3,
     17
    ],
    [
     13,
     -2,
     17
    ],
    [
     13,
     -1,
     17
    ],
    [
     13,
     0,
     17
    ],
    [
     13,
     1,
     17
    ],
    [
     13,
     2,
     17
    ],
    [
     13,
     3,
     17
    ],
    [
     14,
     -3,
     17
    ],
    [
     14,
     -2,
     17
    ],
    [
     14,
     -1,
     17
    ],
    [
     14,
     0,
     17
    ],
    [
     14,
     1,
     17
    ],
    [
     14,
     2,
     17
    ],
    [
     14,
     3,
     17
    ],
    [
     15,
     -3,
     17
    ],
    [
     15,
     -2,
     17
    ],
    [
     15,
     -1,
     17
    ],
    [
     15,
     0,
     17
    ],
    [
     15,
     1,
     17
    ],
    [
     15,
     2,
     17
    ],
    [
     15,
     3,
     17
    ],
    [
     2,
     -1,
     17
    ],
    [
     2,
     0,
     17
    ],
    [
     2,
     1,
     17
    ],
    [
     3,
     -1,
     17
    ],
    [
     3,
     0,
     17
    ],
    [
     3,
     1,
     17
    ],
    [
     4,
     -1,
     17
    ],
    [
     4,
     0,
     17
    ],
    [
     4,
     1,
     17
    ],
    [
     5,
     -1,
     17
    ],
    [
     5,
     0,
     17
    ],
    [
     5,
     1,
     17
    ],
    [
     6,
     -1,
     17
    ],
    [
     6,
     0,
     17
    ],
    [
     6,
     1,
     17
    ],
    [
     7,
     -1,
     17
    ],
    [
     7,
     0,
     17
    ],
    [
     7,
     1,
     17
    ],
    [
     8,
     -1,
     17
    ],
    [
     8,
     0,
     17
    ],
    [
     8,
     1,
     17
    ],
    [
     9,
     -1,
     17
    ],
    [
     9,
     0,
     17
    ],
    [
     9,
     1,
     17
    ],
    [
     16,
     0,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails2": {
   "cells": [
    [
     16,
     -2,
     12
    ],
    [
     16,
     2,
     12
    ],
    [
     16,
     -2,
     13
    ],
    [
     16,
     2,
     13
    ],
    [
     16,
     -2,
     14
    ],
    [
     16,
     2,
     14
    ],
    [
     16,
     -2,
     15
    ],
    [
     16,
     2,
     15
    ],
    [
     16,
     -2,
     16
    ],
    [
     16,
     2,
     16
    ],
    [
     16,
     -2,
     17
    ],
    [
     16,
     2,
     17
    ],
    [
     11,
     -5,
     12
    ],
    [
     15,
     -5,
     12
    ],
    [
     11,
     -5,
     13
    ],
    [
     15,
     -5,
     13
    ],
    [
     11,
     -5,
     14
    ],
    [
     15,
     -5,
     14
    ],
    [
     11,
     -5,
     15
    ],
    [
     15,
     -5,
     15
    ],
    [
     11,
     -5,
     16
    ],
    [
     15,
     -5,
     16
    ],
    [
     11,
     -5,
     17
    ],
    [
     15,
     -5,
     17
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs2": {
   "cells": [
    [
     16,
     -1,
     12
    ],
    [
     16,
     0,
     12
    ],
    [
     16,
     1,
     12
    ],
    [
     16,
     -1,
     14
    ],
    [
     16,
     0,
     14
    ],
    [
     16,
     1,
     14
    ],
    [
     16,
     -1,
     16
    ],
    [
     16,
     0,
     16
    ],
    [
     16,
     1,
     16
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts3": {
   "cells": [
    [
     10,
     -3,
     18
    ],
    [
     10,
     3,
     18
    ],
    [
     15,
     -3,
     18
    ],
    [
     15,
     3,
     18
    ],
    [
     10,
     -3,
     19
    ],
    [
     10,
     3,
     19
    ],
    [
     15,
     -3,
     19
    ],
    [
     15,
     3,
     19
    ],
    [
     10,
     -3,
     20
    ],
    [
     10,
     3,
     20
    ],
    [
     15,
     -3,
     20
    ],
    [
     15,
     3,
     20
    ],
    [
     10,
     -3,
     21
    ],
    [
     10,
     3,
     21
    ],
    [
     15,
     -3,
     21
    ],
    [
     15,
     3,
     21
    ],
    [
     10,
     -3,
     22
    ],
    [
     10,
     3,
     22
    ],
    [
     15,
     -3,
     22
    ],
    [
     15,
     3,
     22
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20
  },
  "planks3": {
   "cells": [
    [
     10,
     -3,
     23
    ],
    [
     10,
     -2,
     23
    ],
    [
     10,
     -1,
     23
    ],
    [
     10,
     0,
     23
    ],
    [
     10,
     1,
     23
    ],
    [
     10,
     2,
     23
    ],
    [
     10,
     3,
     23
    ],
    [
     11,
     -3,
     23
    ],
    [
     11,
     -2,
     23
    ],
    [
     11,
     -1,
     23
    ],
    [
     11,
     0,
     23
    ],
    [
     11,
     1,
     23
    ],
    [
     11,
     2,
     23
    ],
    [
     11,
     3,
     23
    ],
    [
     12,
     -3,
     23
    ],
    [
     12,
     -2,
     23
    ],
    [
     12,
     -1,
     23
    ],
    [
     12,
     0,
     23
    ],
    [
     12,
     1,
     23
    ],
    [
     12,
     2,
     23
    ],
    [
     12,
     3,
     23
    ],
    [
     13,
     -3,
     23
    ],
    [
     13,
     -2,
     23
    ],
    [
     13,
     -1,
     23
    ],
    [
     13,
     0,
     23
    ],
    [
     13,
     1,
     23
    ],
    [
     13,
     2,
     23
    ],
    [
     13,
     3,
     23
    ],
    [
     14,
     -3,
     23
    ],
    [
     14,
     -2,
     23
    ],
    [
     14,
     -1,
     23
    ],
    [
     14,
     0,
     23
    ],
    [
     14,
     1,
     23
    ],
    [
     14,
     2,
     23
    ],
    [
     14,
     3,
     23
    ],
    [
     15,
     -3,
     23
    ],
    [
     15,
     -2,
     23
    ],
    [
     15,
     -1,
     23
    ],
    [
     15,
     0,
     23
    ],
    [
     15,
     1,
     23
    ],
    [
     15,
     2,
     23
    ],
    [
     15,
     3,
     23
    ],
    [
     2,
     -1,
     23
    ],
    [
     2,
     0,
     23
    ],
    [
     2,
     1,
     23
    ],
    [
     3,
     -1,
     23
    ],
    [
     3,
     0,
     23
    ],
    [
     3,
     1,
     23
    ],
    [
     4,
     -1,
     23
    ],
    [
     4,
     0,
     23
    ],
    [
     4,
     1,
     23
    ],
    [
     5,
     -1,
     23
    ],
    [
     5,
     0,
     23
    ],
    [
     5,
     1,
     23
    ],
    [
     6,
     -1,
     23
    ],
    [
     6,
     0,
     23
    ],
    [
     6,
     1,
     23
    ],
    [
     7,
     -1,
     23
    ],
    [
     7,
     0,
     23
    ],
    [
     7,
     1,
     23
    ],
    [
     8,
     -1,
     23
    ],
    [
     8,
     0,
     23
    ],
    [
     8,
     1,
     23
    ],
    [
     9,
     -1,
     23
    ],
    [
     9,
     0,
     23
    ],
    [
     9,
     1,
     23
    ],
    [
     16,
     0,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails3": {
   "cells": [
    [
     16,
     -2,
     18
    ],
    [
     16,
     2,
     18
    ],
    [
     16,
     -2,
     19
    ],
    [
     16,
     2,
     19
    ],
    [
     16,
     -2,
     20
    ],
    [
     16,
     2,
     20
    ],
    [
     16,
     -2,
     21
    ],
    [
     16,
     2,
     21
    ],
    [
     16,
     -2,
     22
    ],
    [
     16,
     2,
     22
    ],
    [
     16,
     -2,
     23
    ],
    [
     16,
     2,
     23
    ],
    [
     11,
     -5,
     18
    ],
    [
     15,
     -5,
     18
    ],
    [
     11,
     -5,
     19
    ],
    [
     15,
     -5,
     19
    ],
    [
     11,
     -5,
     20
    ],
    [
     15,
     -5,
     20
    ],
    [
     11,
     -5,
     21
    ],
    [
     15,
     -5,
     21
    ],
    [
     11,
     -5,
     22
    ],
    [
     15,
     -5,
     22
    ],
    [
     11,
     -5,
     23
    ],
    [
     15,
     -5,
     23
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs3": {
   "cells": [
    [
     16,
     -1,
     18
    ],
    [
     16,
     0,
     18
    ],
    [
     16,
     1,
     18
    ],
    [
     16,
     -1,
     20
    ],
    [
     16,
     0,
     20
    ],
    [
     16,
     1,
     20
    ],
    [
     16,
     -1,
     22
    ],
    [
     16,
     0,
     22
    ],
    [
     16,
     1,
     22
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 9,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "posts4": {
   "cells": [
    [
     10,
     -3,
     24
    ],
    [
     10,
     3,
     24
    ],
    [
     15,
     -3,
     24
    ],
    [
     15,
     3,
     24
    ],
    [
     10,
     -3,
     25
    ],
    [
     10,
     3,
     25
    ],
    [
     15,
     -3,
     25
    ],
    [
     15,
     3,
     25
    ],
    [
     10,
     -3,
     26
    ],
    [
     10,
     3,
     26
    ],
    [
     15,
     -3,
     26
    ],
    [
     15,
     3,
     26
    ],
    [
     10,
     -3,
     27
    ],
    [
     10,
     3,
     27
    ],
    [
     15,
     -3,
     27
    ],
    [
     15,
     3,
     27
    ],
    [
     10,
     -3,
     29
    ],
    [
     10,
     3,
     29
    ],
    [
     15,
     -3,
     29
    ],
    [
     15,
     3,
     29
    ],
    [
     10,
     -3,
     30
    ],
    [
     10,
     3,
     30
    ],
    [
     15,
     -3,
     30
    ],
    [
     15,
     3,
     30
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 24
  },
  "planks4": {
   "cells": [
    [
     10,
     -3,
     28
    ],
    [
     10,
     -2,
     28
    ],
    [
     10,
     -1,
     28
    ],
    [
     10,
     0,
     28
    ],
    [
     10,
     1,
     28
    ],
    [
     10,
     2,
     28
    ],
    [
     10,
     3,
     28
    ],
    [
     11,
     -3,
     28
    ],
    [
     11,
     -2,
     28
    ],
    [
     11,
     -1,
     28
    ],
    [
     11,
     0,
     28
    ],
    [
     11,
     1,
     28
    ],
    [
     11,
     2,
     28
    ],
    [
     11,
     3,
     28
    ],
    [
     12,
     -3,
     28
    ],
    [
     12,
     -2,
     28
    ],
    [
     12,
     -1,
     28
    ],
    [
     12,
     0,
     28
    ],
    [
     12,
     1,
     28
    ],
    [
     12,
     2,
     28
    ],
    [
     12,
     3,
     28
    ],
    [
     13,
     -3,
     28
    ],
    [
     13,
     -2,
     28
    ],
    [
     13,
     -1,
     28
    ],
    [
     13,
     0,
     28
    ],
    [
     13,
     1,
     28
    ],
    [
     13,
     2,
     28
    ],
    [
     13,
     3,
     28
    ],
    [
     14,
     -3,
     28
    ],
    [
     14,
     -2,
     28
    ],
    [
     14,
     -1,
     28
    ],
    [
     14,
     0,
     28
    ],
    [
     14,
     1,
     28
    ],
    [
     14,
     2,
     28
    ],
    [
     14,
     3,
     28
    ],
    [
     15,
     -3,
     28
    ],
    [
     15,
     -2,
     28
    ],
    [
     15,
     -1,
     28
    ],
    [
     15,
     0,
     28
    ],
    [
     15,
     1,
     28
    ],
    [
     15,
     2,
     28
    ],
    [
     15,
     3,
     28
    ],
    [
     2,
     -1,
     28
    ],
    [
     2,
     0,
     28
    ],
    [
     2,
     1,
     28
    ],
    [
     3,
     -1,
     28
    ],
    [
     3,
     0,
     28
    ],
    [
     3,
     1,
     28
    ],
    [
     4,
     -1,
     28
    ],
    [
     4,
     0,
     28
    ],
    [
     4,
     1,
     28
    ],
    [
     5,
     -1,
     28
    ],
    [
     5,
     0,
     28
    ],
    [
     5,
     1,
     28
    ],
    [
     6,
     -1,
     28
    ],
    [
     6,
     0,
     28
    ],
    [
     6,
     1,
     28
    ],
    [
     7,
     -1,
     28
    ],
    [
     7,
     0,
     28
    ],
    [
     7,
     1,
     28
    ],
    [
     8,
     -1,
     28
    ],
    [
     8,
     0,
     28
    ],
    [
     8,
     1,
     28
    ],
    [
     9,
     -1,
     28
    ],
    [
     9,
     0,
     28
    ],
    [
     9,
     1,
     28
    ],
    [
     16,
     0,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 67,
   "block_shape": [
    1,
    1,
    0.35
   ]
  },
  "rails4": {
   "cells": [
    [
     16,
     -2,
     24
    ],
    [
     16,
     2,
     24
    ],
    [
     16,
     -2,
     25
    ],
    [
     16,
     2,
     25
    ],
    [
     16,
     -2,
     26
    ],
    [
     16,
     2,
     26
    ],
    [
     16,
     -2,
     27
    ],
    [
     16,
     2,
     27
    ],
    [
     16,
     -2,
     28
    ],
    [
     16,
     2,
     28
    ],
    [
     11,
     -5,
     24
    ],
    [
     15,
     -5,
     24
    ],
    [
     11,
     -5,
     25
    ],
    [
     15,
     -5,
     25
    ],
    [
     11,
     -5,
     26
    ],
    [
     15,
     -5,
     26
    ],
    [
     11,
     -5,
     27
    ],
    [
     15,
     -5,
     27
    ],
    [
     11,
     -5,
     28
    ],
    [
     15,
     -5,
     28
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 20,
   "block_shape": [
    0.3,
    0.3,
    1
   ]
  },
  "rungs4": {
   "cells": [
    [
     16,
     -1,
     24
    ],
    [
     16,
     0,
     24
    ],
    [
     16,
     1,
     24
    ],
    [
     16,
     -1,
     26
    ],
    [
     16,
     0,
     26
    ],
    [
     16,
     1,
     26
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 6,
   "block_shape": [
    0.3,
    1,
    0.3
   ]
  },
  "crate": {
   "cells": [
    [
     12,
     -6,
     0
    ],
    [
     12,
     -6,
     1
    ],
    [
     12,
     -5,
     0
    ],
    [
     12,
     -5,
     1
    ],
    [
     12,
     -4,
     0
    ],
    [
     12,
     -4,
     1
    ],
    [
     13,
     -6,
     0
    ],
    [
     13,
     -6,
     1
    ],
    [
     13,
     -5,
     0
    ],
    [
     13,
     -5,
     1
    ],
    [
     13,
     -4,
     0
    ],
    [
     13,
     -4,
     1
    ],
    [
     14,
     -6,
     0
    ],
    [
     14,
     -6,
     1
    ],
    [
     14,
     -5,
     0
    ],
    [
     14,
     -5,
     1
    ],
    [
     14,
     -4,
     0
    ],
    [
     14,
     -4,
     1
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 18
  },
  "load1": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load2": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load3": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load4": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  },
  "load5": {
   "cells": [
    [
     12,
     -6,
     2
    ],
    [
     12,
     -5,
     2
    ],
    [
     12,
     -4,
     2
    ],
    [
     13,
     -6,
     2
    ],
    [
     13,
     -5,
     2
    ],
    [
     13,
     -4,
     2
    ],
    [
     14,
     -6,
     2
    ],
    [
     14,
     -5,
     2
    ],
    [
     14,
     -4,
     2
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 9
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": 0.6
  },
  {
   "type": "build",
   "target": "posts0",
   "when": 0.6,
   "until": 1.2,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "planks0",
   "when": 1.2,
   "until": 1.65,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "rails0",
   "when": 1.65,
   "until": 1.95,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "rungs0",
   "when": 1.95,
   "until": 2.2,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "crate",
   "when": 2.2,
   "until": 2.5,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "tower0",
   "when": 5.96,
   "until": 10.34,
   "stand_at": [
    0.95,
    -0.6,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "load1",
   "when": 10.34,
   "until": 10.49,
   "quiet": true
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.0
   ],
   "when": 10.34,
   "until": 10.94,
   "via": [
    [
     1.0,
     -0.95,
     0.0
    ],
    [
     1.75,
     -0.95,
     0.0
    ]
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    0.4
   ],
   "when": 10.49,
   "until": 10.94
  },
  {
   "type": "slide",
   "target": "load1",
   "by": [
    0,
    0,
    0.4
   ],
   "when": 10.49,
   "until": 10.94
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.6
   ],
   "when": 10.94,
   "until": 11.32,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    0.8,
    0.0,
    0.6
   ],
   "when": 11.32,
   "until": 11.72,
   "via": [
    [
     1.45,
     0.0,
     0.6
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load1",
   "when": 11.72,
   "until": 13.88,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower1",
   "when": 11.72,
   "until": 13.88,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "posts1",
   "when": 13.88,
   "until": 14.23,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "planks1",
   "when": 14.23,
   "until": 14.48,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "rails1",
   "when": 14.48,
   "until": 14.63,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "build",
   "target": "rungs1",
   "when": 14.63,
   "until": 14.75,
   "stand_at": [
    0.8,
    0.0,
    0.6
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -0.4
   ],
   "when": 14.75,
   "until": 15.1
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.6
   ],
   "when": 14.75,
   "until": 15.15,
   "via": [
    [
     1.45,
     0.0,
     0.6
    ]
   ]
  },
  {
   "type": "build",
   "target": "load2",
   "when": 15.1,
   "until": 15.25,
   "quiet": true
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.2
   ],
   "when": 15.15,
   "until": 15.53,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    1.0
   ],
   "when": 15.25,
   "until": 15.7
  },
  {
   "type": "slide",
   "target": "load2",
   "by": [
    0,
    0,
    1.0
   ],
   "when": 15.25,
   "until": 15.7
  },
  {
   "type": "move",
   "to": [
    0.6,
    0.0,
    1.2
   ],
   "when": 15.53,
   "until": 15.93,
   "via": [
    [
     1.45,
     0.0,
     1.2
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load2",
   "when": 15.93,
   "until": 19.96,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower2",
   "when": 15.93,
   "until": 19.96,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "posts2",
   "when": 19.96,
   "until": 20.31,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "planks2",
   "when": 20.31,
   "until": 20.56,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "rails2",
   "when": 20.56,
   "until": 20.71,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "build",
   "target": "rungs2",
   "when": 20.71,
   "until": 20.83,
   "stand_at": [
    0.6,
    0.0,
    1.2
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -1.0
   ],
   "when": 20.83,
   "until": 21.18
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.2
   ],
   "when": 20.83,
   "until": 21.23,
   "via": [
    [
     1.45,
     0.0,
     1.2
    ]
   ]
  },
  {
   "type": "build",
   "target": "load3",
   "when": 21.18,
   "until": 21.33,
   "quiet": true
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.8
   ],
   "when": 21.23,
   "until": 21.6,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    1.6
   ],
   "when": 21.33,
   "until": 21.78
  },
  {
   "type": "slide",
   "target": "load3",
   "by": [
    0,
    0,
    1.6
   ],
   "when": 21.33,
   "until": 21.78
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    1.8
   ],
   "when": 21.6,
   "until": 22.0,
   "via": [
    [
     1.45,
     0.0,
     1.8
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load3",
   "when": 22.0,
   "until": 23.1,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower3",
   "when": 22.0,
   "until": 23.1,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "posts3",
   "when": 23.1,
   "until": 23.45,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "planks3",
   "when": 23.45,
   "until": 23.7,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "rails3",
   "when": 23.7,
   "until": 23.85,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "build",
   "target": "rungs3",
   "when": 23.85,
   "until": 23.97,
   "stand_at": [
    0.4,
    0.0,
    1.8
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -1.6
   ],
   "when": 23.97,
   "until": 24.32
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    1.8
   ],
   "when": 23.97,
   "until": 24.37,
   "via": [
    [
     1.45,
     0.0,
     1.8
    ]
   ]
  },
  {
   "type": "build",
   "target": "load4",
   "when": 24.32,
   "until": 24.47,
   "quiet": true
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.4
   ],
   "when": 24.37,
   "until": 24.75,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    2.2
   ],
   "when": 24.47,
   "until": 24.92
  },
  {
   "type": "slide",
   "target": "load4",
   "by": [
    0,
    0,
    2.2
   ],
   "when": 24.47,
   "until": 24.92
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    2.4
   ],
   "when": 24.75,
   "until": 25.15,
   "via": [
    [
     1.45,
     0.0,
     2.4
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load4",
   "when": 25.15,
   "until": 27.27,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower4",
   "when": 25.15,
   "until": 27.27,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "posts4",
   "when": 27.27,
   "until": 27.62,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "planks4",
   "when": 27.62,
   "until": 27.87,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "rails4",
   "when": 27.87,
   "until": 28.02,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "build",
   "target": "rungs4",
   "when": 28.02,
   "until": 28.14,
   "stand_at": [
    0.4,
    0.0,
    2.4
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -2.2
   ],
   "when": 28.14,
   "until": 28.49
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.4
   ],
   "when": 28.14,
   "until": 28.54,
   "via": [
    [
     1.45,
     0.0,
     2.4
    ]
   ]
  },
  {
   "type": "build",
   "target": "load5",
   "when": 28.49,
   "until": 28.64,
   "quiet": true
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.9
   ],
   "when": 28.54,
   "until": 28.89,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    2.7
   ],
   "when": 28.64,
   "until": 29.09
  },
  {
   "type": "slide",
   "target": "load5",
   "by": [
    0,
    0,
    2.7
   ],
   "when": 28.64,
   "until": 29.09
  },
  {
   "type": "move",
   "to": [
    0.4,
    0.0,
    2.9
   ],
   "when": 28.89,
   "until": 29.29,
   "via": [
    [
     1.45,
     0.0,
     2.9
    ]
   ]
  },
  {
   "type": "remove",
   "target": "load5",
   "when": 29.29,
   "until": 31.59,
   "quiet": true
  },
  {
   "type": "build",
   "target": "tower5",
   "when": 29.29,
   "until": 31.59,
   "stand_at": [
    0.4,
    0.0,
    2.9
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    2.9
   ],
   "when": 31.59,
   "until": 31.99,
   "via": [
    [
     1.45,
     0.0,
     2.9
    ]
   ]
  },
  {
   "type": "move",
   "to": [
    1.75,
    0.0,
    0.0
   ],
   "when": 31.99,
   "until": 33.31,
   "face": [
    1.6,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "crate",
   "by": [
    0,
    0,
    -2.7
   ],
   "when": 32.91,
   "until": 33.51
  },
  {
   "type": "move",
   "to": [
    2.2,
    -1.2,
    0.0
   ],
   "when": 33.31,
   "until": 33.91
  },
  {
   "type": "remove",
   "target": "rungs4",
   "when": 33.91,
   "until": 34.01,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails4",
   "when": 34.01,
   "until": 34.11,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks4",
   "when": 34.11,
   "until": 34.21,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts4",
   "when": 34.21,
   "until": 34.31,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs3",
   "when": 34.31,
   "until": 34.41,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails3",
   "when": 34.41,
   "until": 34.51,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks3",
   "when": 34.51,
   "until": 34.61,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts3",
   "when": 34.61,
   "until": 34.71,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs2",
   "when": 34.71,
   "until": 34.81,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails2",
   "when": 34.81,
   "until": 34.91,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks2",
   "when": 34.91,
   "until": 35.01,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts2",
   "when": 35.01,
   "until": 35.11,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs1",
   "when": 35.11,
   "until": 35.21,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails1",
   "when": 35.21,
   "until": 35.31,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks1",
   "when": 35.31,
   "until": 35.41,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts1",
   "when": 35.41,
   "until": 35.51,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rungs0",
   "when": 35.51,
   "until": 35.61,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "rails0",
   "when": 35.61,
   "until": 35.71,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "planks0",
   "when": 35.71,
   "until": 35.81,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "posts0",
   "when": 35.81,
   "until": 35.91,
   "quiet": true
  },
  {
   "type": "remove",
   "target": "crate",
   "when": 35.91,
   "until": 36.11,
   "quiet": true
  },
  {
   "type": "face_camera",
   "when": 36.11,
   "until": "end"
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    1.0,
    -0.4,
    0.75
   ],
   "az": -110,
   "el": 8,
   "dist": 2.3
  },
  {
   "at": 5.96,
   "target": [
    0.75,
    -0.2,
    0.75
   ],
   "az": -104,
   "el": 10,
   "dist": 3.1
  },
  {
   "at": 11.72,
   "target": [
    0.8,
    -0.1,
    0.91
   ],
   "az": -96,
   "el": 9,
   "dist": 3.55
  },
  {
   "at": 15.93,
   "target": [
    0.8,
    -0.1,
    1.27
   ],
   "az": -88,
   "el": 10,
   "dist": 3.8
  },
  {
   "at": 22.0,
   "target": [
    0.8,
    -0.1,
    1.63
   ],
   "az": -80,
   "el": 11,
   "dist": 4.05
  },
  {
   "at": 25.15,
   "target": [
    0.8,
    -0.1,
    1.99
   ],
   "az": -72,
   "el": 12,
   "dist": 4.3
  },
  {
   "at": 29.29,
   "target": [
    0.8,
    -0.1,
    2.29
   ],
   "az": -64,
   "el": 13,
   "dist": 4.55
  },
  {
   "at": 33.91,
   "target": [
    0.75,
    -0.1,
    1.55
   ],
   "az": -62,
   "el": 8,
   "dist": 5.2
  },
  {
   "at": 36.11,
   "target": [
    0.6,
    -0.2,
    1.6
   ],
   "az": -78,
   "el": 8,
   "dist": 5.4
  },
  {
   "at": "end",
   "target": [
    0.6,
    -0.2,
    1.6
   ],
   "az": -92,
   "el": 8,
   "dist": 5.2
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "{count:tower*} / 677 BLOCKS",
   "color": "amber",
   "y": 590,
   "size": 44,
   "when": 5.96,
   "until": 32.19
  },
  {
   "style": "title",
   "text": "18,038 PIECES",
   "color": "white",
   "y": 470,
   "size": 88,
   "when": 0.3,
   "until": 5.96
  },
  {
   "style": "title",
   "text": "A GIANT KIT",
   "color": "cyan",
   "y": 470,
   "size": 92,
   "when": 7.16,
   "until": 11.72
  },
  {
   "style": "title",
   "text": "2.5 MILLION HOT RIVETS",
   "color": "red",
   "y": 470,
   "size": 72,
   "when": 16.23,
   "until": 22.6
  },
  {
   "style": "title",
   "text": "CRANES CLIMBED TOO",
   "color": "amber",
   "y": 470,
   "size": 80,
   "when": 25.45,
   "until": 29.29
  },
  {
   "style": "title",
   "text": "DONE: 1889",
   "color": "green",
   "y": 470,
   "size": 96,
   "when": 29.79,
   "until": 33.91
  },
  {
   "style": "title",
   "text": "VOILA!",
   "color": "amber",
   "y": 470,
   "size": 110,
   "when": 36.11,
   "until": "end"
  }
 ]
}

In [ ]:
%%writefile stories/car_engine.json
{
 "title": "How a car engine works",
 "slug": "car_engine",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "tail": 3.1,
 "world": {
  "ground": "grass",
  "time": "morning",
  "weather": "clouds",
  "animals": [
   "birds",
   "cat"
  ]
 },
 "lines": [
  {
   "id": "hook",
   "text": "Your car makes thousands of little explosions every minute... let me build one and show you!",
   "pause": 0.55
  },
  {
   "id": "build",
   "text": "Frame, wheels, seat... and the heart of it all. The engine.",
   "pause": 3.09
  },
  {
   "id": "intake",
   "text": "One. The piston drops, and sucks in air and fuel.",
   "pause": 0.25
  },
  {
   "id": "squeeze",
   "text": "Two. It shoots back up, and squeezes it tight.",
   "pause": 0.25
  },
  {
   "id": "bang",
   "text": "Three. Spark! Bang! The hot gas shoves the piston down.",
   "pause": 0.25
  },
  {
   "id": "blow",
   "text": "Four. It pushes the smoke out the pipe.",
   "pause": 0.25
  },
  {
   "id": "crank",
   "text": "Those pushes turn the crank... and the crank turns the wheels.",
   "pause": 0.25
  },
  {
   "id": "payoff",
   "text": "Thousands of times a minute! Hey... wait for me!",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   -1.75,
   -0.15,
   0.0
  ],
  "outfit": {
   "hat": "hardhat",
   "shirt": "rust",
   "extras": [
    "goggles",
    "hammer"
   ]
  }
 },
 "structures": {
  "chassis": {
   "cells": [
    [
     0,
     -4,
     3
    ],
    [
     1,
     -4,
     3
    ],
    [
     2,
     -4,
     3
    ],
    [
     3,
     -4,
     3
    ],
    [
     4,
     -4,
     3
    ],
    [
     5,
     -4,
     3
    ],
    [
     6,
     -4,
     3
    ],
    [
     7,
     -4,
     3
    ],
    [
     8,
     -4,
     3
    ],
    [
     9,
     -4,
     3
    ],
    [
     10,
     -4,
     3
    ],
    [
     11,
     -4,
     3
    ],
    [
     12,
     -4,
     3
    ],
    [
     13,
     -4,
     3
    ],
    [
     14,
     -4,
     3
    ],
    [
     15,
     -4,
     3
    ],
    [
     16,
     -4,
     3
    ],
    [
     17,
     -4,
     3
    ],
    [
     18,
     -4,
     3
    ],
    [
     19,
     -4,
     3
    ],
    [
     20,
     -4,
     3
    ],
    [
     21,
     -4,
     3
    ],
    [
     22,
     -4,
     3
    ],
    [
     23,
     -4,
     3
    ],
    [
     0,
     3,
     3
    ],
    [
     1,
     3,
     3
    ],
    [
     2,
     3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     4,
     3,
     3
    ],
    [
     5,
     3,
     3
    ],
    [
     6,
     3,
     3
    ],
    [
     7,
     3,
     3
    ],
    [
     8,
     3,
     3
    ],
    [
     9,
     3,
     3
    ],
    [
     10,
     3,
     3
    ],
    [
     11,
     3,
     3
    ],
    [
     12,
     3,
     3
    ],
    [
     13,
     3,
     3
    ],
    [
     14,
     3,
     3
    ],
    [
     15,
     3,
     3
    ],
    [
     16,
     3,
     3
    ],
    [
     17,
     3,
     3
    ],
    [
     18,
     3,
     3
    ],
    [
     19,
     3,
     3
    ],
    [
     20,
     3,
     3
    ],
    [
     21,
     3,
     3
    ],
    [
     22,
     3,
     3
    ],
    [
     23,
     3,
     3
    ],
    [
     1,
     -3,
     3
    ],
    [
     8,
     -3,
     3
    ],
    [
     15,
     -3,
     3
    ],
    [
     22,
     -3,
     3
    ],
    [
     1,
     -2,
     3
    ],
    [
     8,
     -2,
     3
    ],
    [
     15,
     -2,
     3
    ],
    [
     22,
     -2,
     3
    ],
    [
     1,
     -1,
     3
    ],
    [
     8,
     -1,
     3
    ],
    [
     15,
     -1,
     3
    ],
    [
     22,
     -1,
     3
    ],
    [
     1,
     0,
     3
    ],
    [
     8,
     0,
     3
    ],
    [
     15,
     0,
     3
    ],
    [
     22,
     0,
     3
    ],
    [
     1,
     1,
     3
    ],
    [
     8,
     1,
     3
    ],
    [
     15,
     1,
     3
    ],
    [
     22,
     1,
     3
    ],
    [
     1,
     2,
     3
    ],
    [
     8,
     2,
     3
    ],
    [
     15,
     2,
     3
    ],
    [
     22,
     2,
     3
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 72
  },
  "axles": {
   "cells": [
    [
     4,
     -5,
     2
    ],
    [
     20,
     -5,
     2
    ],
    [
     4,
     -4,
     2
    ],
    [
     20,
     -4,
     2
    ],
    [
     4,
     -3,
     2
    ],
    [
     20,
     -3,
     2
    ],
    [
     4,
     -2,
     2
    ],
    [
     20,
     -2,
     2
    ],
    [
     4,
     -1,
     2
    ],
    [
     20,
     -1,
     2
    ],
    [
     4,
     0,
     2
    ],
    [
     20,
     0,
     2
    ],
    [
     4,
     1,
     2
    ],
    [
     20,
     1,
     2
    ],
    [
     4,
     2,
     2
    ],
    [
     20,
     2,
     2
    ],
    [
     4,
     3,
     2
    ],
    [
     20,
     3,
     2
    ],
    [
     4,
     4,
     2
    ],
    [
     20,
     4,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 20,
   "block_shape": [
    0.35,
    1,
    0.35
   ]
  },
  "wheel_rl": {
   "cells": [
    [
     2,
     -6,
     2
    ],
    [
     3,
     -6,
     1
    ],
    [
     3,
     -6,
     2
    ],
    [
     3,
     -6,
     3
    ],
    [
     4,
     -6,
     0
    ],
    [
     4,
     -6,
     1
    ],
    [
     4,
     -6,
     2
    ],
    [
     4,
     -6,
     3
    ],
    [
     4,
     -6,
     4
    ],
    [
     5,
     -6,
     1
    ],
    [
     5,
     -6,
     2
    ],
    [
     5,
     -6,
     3
    ],
    [
     6,
     -6,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "rubber",
   "expect_blocks": 13
  },
  "hub_rl": {
   "cells": [
    [
     4,
     -7,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 1
  },
  "wheel_rr": {
   "cells": [
    [
     2,
     5,
     2
    ],
    [
     3,
     5,
     1
    ],
    [
     3,
     5,
     2
    ],
    [
     3,
     5,
     3
    ],
    [
     4,
     5,
     0
    ],
    [
     4,
     5,
     1
    ],
    [
     4,
     5,
     2
    ],
    [
     4,
     5,
     3
    ],
    [
     4,
     5,
     4
    ],
    [
     5,
     5,
     1
    ],
    [
     5,
     5,
     2
    ],
    [
     5,
     5,
     3
    ],
    [
     6,
     5,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "rubber",
   "expect_blocks": 13
  },
  "hub_rr": {
   "cells": [
    [
     4,
     6,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 1
  },
  "wheel_fl": {
   "cells": [
    [
     18,
     -6,
     2
    ],
    [
     19,
     -6,
     1
    ],
    [
     19,
     -6,
     2
    ],
    [
     19,
     -6,
     3
    ],
    [
     20,
     -6,
     0
    ],
    [
     20,
     -6,
     1
    ],
    [
     20,
     -6,
     2
    ],
    [
     20,
     -6,
     3
    ],
    [
     20,
     -6,
     4
    ],
    [
     21,
     -6,
     1
    ],
    [
     21,
     -6,
     2
    ],
    [
     21,
     -6,
     3
    ],
    [
     22,
     -6,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "rubber",
   "expect_blocks": 13
  },
  "hub_fl": {
   "cells": [
    [
     20,
     -7,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 1
  },
  "wheel_fr": {
   "cells": [
    [
     18,
     5,
     2
    ],
    [
     19,
     5,
     1
    ],
    [
     19,
     5,
     2
    ],
    [
     19,
     5,
     3
    ],
    [
     20,
     5,
     0
    ],
    [
     20,
     5,
     1
    ],
    [
     20,
     5,
     2
    ],
    [
     20,
     5,
     3
    ],
    [
     20,
     5,
     4
    ],
    [
     21,
     5,
     1
    ],
    [
     21,
     5,
     2
    ],
    [
     21,
     5,
     3
    ],
    [
     22,
     5,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "rubber",
   "expect_blocks": 13
  },
  "hub_fr": {
   "cells": [
    [
     20,
     6,
     2
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 1
  },
  "floor": {
   "cells": [
    [
     0,
     -5,
     4
    ],
    [
     1,
     -5,
     4
    ],
    [
     2,
     -5,
     4
    ],
    [
     3,
     -5,
     4
    ],
    [
     4,
     -5,
     4
    ],
    [
     5,
     -5,
     4
    ],
    [
     6,
     -5,
     4
    ],
    [
     7,
     -5,
     4
    ],
    [
     8,
     -5,
     4
    ],
    [
     9,
     -5,
     4
    ],
    [
     10,
     -5,
     4
    ],
    [
     11,
     -5,
     4
    ],
    [
     12,
     -5,
     4
    ],
    [
     13,
     -5,
     4
    ],
    [
     14,
     -5,
     4
    ],
    [
     0,
     -4,
     4
    ],
    [
     1,
     -4,
     4
    ],
    [
     2,
     -4,
     4
    ],
    [
     3,
     -4,
     4
    ],
    [
     4,
     -4,
     4
    ],
    [
     5,
     -4,
     4
    ],
    [
     6,
     -4,
     4
    ],
    [
     7,
     -4,
     4
    ],
    [
     8,
     -4,
     4
    ],
    [
     9,
     -4,
     4
    ],
    [
     10,
     -4,
     4
    ],
    [
     11,
     -4,
     4
    ],
    [
     12,
     -4,
     4
    ],
    [
     13,
     -4,
     4
    ],
    [
     14,
     -4,
     4
    ],
    [
     0,
     -3,
     4
    ],
    [
     1,
     -3,
     4
    ],
    [
     2,
     -3,
     4
    ],
    [
     3,
     -3,
     4
    ],
    [
     4,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     6,
     -3,
     4
    ],
    [
     7,
     -3,
     4
    ],
    [
     8,
     -3,
     4
    ],
    [
     9,
     -3,
     4
    ],
    [
     10,
     -3,
     4
    ],
    [
     11,
     -3,
     4
    ],
    [
     12,
     -3,
     4
    ],
    [
     13,
     -3,
     4
    ],
    [
     14,
     -3,
     4
    ],
    [
     0,
     -2,
     4
    ],
    [
     1,
     -2,
     4
    ],
    [
     2,
     -2,
     4
    ],
    [
     3,
     -2,
     4
    ],
    [
     4,
     -2,
     4
    ],
    [
     5,
     -2,
     4
    ],
    [
     6,
     -2,
     4
    ],
    [
     7,
     -2,
     4
    ],
    [
     8,
     -2,
     4
    ],
    [
     9,
     -2,
     4
    ],
    [
     10,
     -2,
     4
    ],
    [
     11,
     -2,
     4
    ],
    [
     12,
     -2,
     4
    ],
    [
     13,
     -2,
     4
    ],
    [
     14,
     -2,
     4
    ],
    [
     0,
     -1,
     4
    ],
    [
     1,
     -1,
     4
    ],
    [
     2,
     -1,
     4
    ],
    [
     3,
     -1,
     4
    ],
    [
     4,
     -1,
     4
    ],
    [
     5,
     -1,
     4
    ],
    [
     6,
     -1,
     4
    ],
    [
     7,
     -1,
     4
    ],
    [
     8,
     -1,
     4
    ],
    [
     9,
     -1,
     4
    ],
    [
     10,
     -1,
     4
    ],
    [
     11,
     -1,
     4
    ],
    [
     12,
     -1,
     4
    ],
    [
     13,
     -1,
     4
    ],
    [
     14,
     -1,
     4
    ],
    [
     0,
     0,
     4
    ],
    [
     1,
     0,
     4
    ],
    [
     2,
     0,
     4
    ],
    [
     3,
     0,
     4
    ],
    [
     4,
     0,
     4
    ],
    [
     5,
     0,
     4
    ],
    [
     6,
     0,
     4
    ],
    [
     7,
     0,
     4
    ],
    [
     8,
     0,
     4
    ],
    [
     9,
     0,
     4
    ],
    [
     10,
     0,
     4
    ],
    [
     11,
     0,
     4
    ],
    [
     12,
     0,
     4
    ],
    [
     13,
     0,
     4
    ],
    [
     14,
     0,
     4
    ],
    [
     0,
     1,
     4
    ],
    [
     1,
     1,
     4
    ],
    [
     2,
     1,
     4
    ],
    [
     3,
     1,
     4
    ],
    [
     4,
     1,
     4
    ],
    [
     5,
     1,
     4
    ],
    [
     6,
     1,
     4
    ],
    [
     7,
     1,
     4
    ],
    [
     8,
     1,
     4
    ],
    [
     9,
     1,
     4
    ],
    [
     10,
     1,
     4
    ],
    [
     11,
     1,
     4
    ],
    [
     12,
     1,
     4
    ],
    [
     13,
     1,
     4
    ],
    [
     14,
     1,
     4
    ],
    [
     0,
     2,
     4
    ],
    [
     1,
     2,
     4
    ],
    [
     2,
     2,
     4
    ],
    [
     3,
     2,
     4
    ],
    [
     4,
     2,
     4
    ],
    [
     5,
     2,
     4
    ],
    [
     6,
     2,
     4
    ],
    [
     7,
     2,
     4
    ],
    [
     8,
     2,
     4
    ],
    [
     9,
     2,
     4
    ],
    [
     10,
     2,
     4
    ],
    [
     11,
     2,
     4
    ],
    [
     12,
     2,
     4
    ],
    [
     13,
     2,
     4
    ],
    [
     14,
     2,
     4
    ],
    [
     0,
     3,
     4
    ],
    [
     1,
     3,
     4
    ],
    [
     2,
     3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     4,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     6,
     3,
     4
    ],
    [
     7,
     3,
     4
    ],
    [
     8,
     3,
     4
    ],
    [
     9,
     3,
     4
    ],
    [
     10,
     3,
     4
    ],
    [
     11,
     3,
     4
    ],
    [
     12,
     3,
     4
    ],
    [
     13,
     3,
     4
    ],
    [
     14,
     3,
     4
    ],
    [
     0,
     4,
     4
    ],
    [
     1,
     4,
     4
    ],
    [
     2,
     4,
     4
    ],
    [
     3,
     4,
     4
    ],
    [
     4,
     4,
     4
    ],
    [
     5,
     4,
     4
    ],
    [
     6,
     4,
     4
    ],
    [
     7,
     4,
     4
    ],
    [
     8,
     4,
     4
    ],
    [
     9,
     4,
     4
    ],
    [
     10,
     4,
     4
    ],
    [
     11,
     4,
     4
    ],
    [
     12,
     4,
     4
    ],
    [
     13,
     4,
     4
    ],
    [
     14,
     4,
     4
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 150
  },
  "seat": {
   "cells": [
    [
     5,
     -3,
     5
    ],
    [
     6,
     -3,
     5
    ],
    [
     7,
     -3,
     5
    ],
    [
     8,
     -3,
     5
    ],
    [
     5,
     -2,
     5
    ],
    [
     6,
     -2,
     5
    ],
    [
     7,
     -2,
     5
    ],
    [
     8,
     -2,
     5
    ],
    [
     5,
     -1,
     5
    ],
    [
     6,
     -1,
     5
    ],
    [
     7,
     -1,
     5
    ],
    [
     8,
     -1,
     5
    ],
    [
     5,
     0,
     5
    ],
    [
     6,
     0,
     5
    ],
    [
     7,
     0,
     5
    ],
    [
     8,
     0,
     5
    ],
    [
     5,
     1,
     5
    ],
    [
     6,
     1,
     5
    ],
    [
     7,
     1,
     5
    ],
    [
     8,
     1,
     5
    ],
    [
     5,
     2,
     5
    ],
    [
     6,
     2,
     5
    ],
    [
     7,
     2,
     5
    ],
    [
     8,
     2,
     5
    ],
    [
     4,
     -3,
     6
    ],
    [
     4,
     -2,
     6
    ],
    [
     4,
     -1,
     6
    ],
    [
     4,
     0,
     6
    ],
    [
     4,
     1,
     6
    ],
    [
     4,
     2,
     6
    ],
    [
     4,
     -3,
     7
    ],
    [
     4,
     -2,
     7
    ],
    [
     4,
     -1,
     7
    ],
    [
     4,
     0,
     7
    ],
    [
     4,
     1,
     7
    ],
    [
     4,
     2,
     7
    ],
    [
     4,
     -3,
     8
    ],
    [
     4,
     -2,
     8
    ],
    [
     4,
     -1,
     8
    ],
    [
     4,
     0,
     8
    ],
    [
     4,
     1,
     8
    ],
    [
     4,
     2,
     8
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 42
  },
  "sides": {
   "cells": [
    [
     0,
     -5,
     5
    ],
    [
     1,
     -5,
     5
    ],
    [
     2,
     -5,
     5
    ],
    [
     3,
     -5,
     5
    ],
    [
     4,
     -5,
     5
    ],
    [
     5,
     -5,
     5
    ],
    [
     6,
     -5,
     5
    ],
    [
     7,
     -5,
     5
    ],
    [
     8,
     -5,
     5
    ],
    [
     9,
     -5,
     5
    ],
    [
     10,
     -5,
     5
    ],
    [
     11,
     -5,
     5
    ],
    [
     12,
     -5,
     5
    ],
    [
     13,
     -5,
     5
    ],
    [
     14,
     -5,
     5
    ],
    [
     0,
     4,
     5
    ],
    [
     1,
     4,
     5
    ],
    [
     2,
     4,
     5
    ],
    [
     3,
     4,
     5
    ],
    [
     4,
     4,
     5
    ],
    [
     5,
     4,
     5
    ],
    [
     6,
     4,
     5
    ],
    [
     7,
     4,
     5
    ],
    [
     8,
     4,
     5
    ],
    [
     9,
     4,
     5
    ],
    [
     10,
     4,
     5
    ],
    [
     11,
     4,
     5
    ],
    [
     12,
     4,
     5
    ],
    [
     13,
     4,
     5
    ],
    [
     14,
     4,
     5
    ],
    [
     0,
     -5,
     6
    ],
    [
     1,
     -5,
     6
    ],
    [
     2,
     -5,
     6
    ],
    [
     3,
     -5,
     6
    ],
    [
     4,
     -5,
     6
    ],
    [
     5,
     -5,
     6
    ],
    [
     6,
     -5,
     6
    ],
    [
     7,
     -5,
     6
    ],
    [
     8,
     -5,
     6
    ],
    [
     9,
     -5,
     6
    ],
    [
     10,
     -5,
     6
    ],
    [
     11,
     -5,
     6
    ],
    [
     12,
     -5,
     6
    ],
    [
     13,
     -5,
     6
    ],
    [
     14,
     -5,
     6
    ],
    [
     0,
     4,
     6
    ],
    [
     1,
     4,
     6
    ],
    [
     2,
     4,
     6
    ],
    [
     3,
     4,
     6
    ],
    [
     4,
     4,
     6
    ],
    [
     5,
     4,
     6
    ],
    [
     6,
     4,
     6
    ],
    [
     7,
     4,
     6
    ],
    [
     8,
     4,
     6
    ],
    [
     9,
     4,
     6
    ],
    [
     10,
     4,
     6
    ],
    [
     11,
     4,
     6
    ],
    [
     12,
     4,
     6
    ],
    [
     13,
     4,
     6
    ],
    [
     14,
     4,
     6
    ],
    [
     0,
     -4,
     5
    ],
    [
     0,
     -3,
     5
    ],
    [
     0,
     -2,
     5
    ],
    [
     0,
     -1,
     5
    ],
    [
     0,
     0,
     5
    ],
    [
     0,
     1,
     5
    ],
    [
     0,
     2,
     5
    ],
    [
     0,
     3,
     5
    ],
    [
     0,
     -4,
     6
    ],
    [
     0,
     -3,
     6
    ],
    [
     0,
     -2,
     6
    ],
    [
     0,
     -1,
     6
    ],
    [
     0,
     0,
     6
    ],
    [
     0,
     1,
     6
    ],
    [
     0,
     2,
     6
    ],
    [
     0,
     3,
     6
    ],
    [
     0,
     -4,
     7
    ],
    [
     0,
     -3,
     7
    ],
    [
     0,
     -2,
     7
    ],
    [
     0,
     -1,
     7
    ],
    [
     0,
     0,
     7
    ],
    [
     0,
     1,
     7
    ],
    [
     0,
     2,
     7
    ],
    [
     0,
     3,
     7
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 84
  },
  "dash": {
   "cells": [
    [
     12,
     -4,
     5
    ],
    [
     13,
     -4,
     5
    ],
    [
     12,
     -3,
     5
    ],
    [
     13,
     -3,
     5
    ],
    [
     12,
     -2,
     5
    ],
    [
     13,
     -2,
     5
    ],
    [
     12,
     -1,
     5
    ],
    [
     13,
     -1,
     5
    ],
    [
     12,
     0,
     5
    ],
    [
     13,
     0,
     5
    ],
    [
     12,
     1,
     5
    ],
    [
     13,
     1,
     5
    ],
    [
     12,
     2,
     5
    ],
    [
     13,
     2,
     5
    ],
    [
     12,
     3,
     5
    ],
    [
     13,
     3,
     5
    ],
    [
     12,
     -4,
     6
    ],
    [
     13,
     -4,
     6
    ],
    [
     12,
     -3,
     6
    ],
    [
     13,
     -3,
     6
    ],
    [
     12,
     -2,
     6
    ],
    [
     13,
     -2,
     6
    ],
    [
     12,
     -1,
     6
    ],
    [
     13,
     -1,
     6
    ],
    [
     12,
     0,
     6
    ],
    [
     13,
     0,
     6
    ],
    [
     12,
     1,
     6
    ],
    [
     13,
     1,
     6
    ],
    [
     12,
     2,
     6
    ],
    [
     13,
     2,
     6
    ],
    [
     12,
     3,
     6
    ],
    [
     13,
     3,
     6
    ],
    [
     12,
     -4,
     7
    ],
    [
     13,
     -4,
     7
    ],
    [
     12,
     -3,
     7
    ],
    [
     13,
     -3,
     7
    ],
    [
     12,
     -2,
     7
    ],
    [
     13,
     -2,
     7
    ],
    [
     12,
     -1,
     7
    ],
    [
     13,
     -1,
     7
    ],
    [
     12,
     0,
     7
    ],
    [
     13,
     0,
     7
    ],
    [
     12,
     1,
     7
    ],
    [
     13,
     1,
     7
    ],
    [
     12,
     2,
     7
    ],
    [
     13,
     2,
     7
    ],
    [
     12,
     3,
     7
    ],
    [
     13,
     3,
     7
    ],
    [
     10,
     -1,
     6
    ],
    [
     10,
     -1,
     7
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 50
  },
  "steer": {
   "cells": [
    [
     10,
     -2,
     8
    ],
    [
     10,
     -1,
     8
    ],
    [
     10,
     0,
     8
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "rubber",
   "expect_blocks": 3
  },
  "glass": {
   "cells": [
    [
     13,
     -4,
     8
    ],
    [
     13,
     -3,
     8
    ],
    [
     13,
     -2,
     8
    ],
    [
     13,
     -1,
     8
    ],
    [
     13,
     0,
     8
    ],
    [
     13,
     1,
     8
    ],
    [
     13,
     2,
     8
    ],
    [
     13,
     3,
     8
    ],
    [
     13,
     -4,
     9
    ],
    [
     13,
     -3,
     9
    ],
    [
     13,
     -2,
     9
    ],
    [
     13,
     -1,
     9
    ],
    [
     13,
     0,
     9
    ],
    [
     13,
     1,
     9
    ],
    [
     13,
     2,
     9
    ],
    [
     13,
     3,
     9
    ],
    [
     13,
     -4,
     10
    ],
    [
     13,
     -3,
     10
    ],
    [
     13,
     -2,
     10
    ],
    [
     13,
     -1,
     10
    ],
    [
     13,
     0,
     10
    ],
    [
     13,
     1,
     10
    ],
    [
     13,
     2,
     10
    ],
    [
     13,
     3,
     10
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 24,
   "block_shape": [
    0.3,
    1,
    1
   ]
  },
  "crankcase": {
   "cells": [
    [
     15,
     -4,
     4
    ],
    [
     16,
     -4,
     4
    ],
    [
     17,
     -4,
     4
    ],
    [
     18,
     -4,
     4
    ],
    [
     19,
     -4,
     4
    ],
    [
     20,
     -4,
     4
    ],
    [
     21,
     -4,
     4
    ],
    [
     22,
     -4,
     4
    ],
    [
     23,
     -4,
     4
    ],
    [
     15,
     -3,
     4
    ],
    [
     16,
     -3,
     4
    ],
    [
     17,
     -3,
     4
    ],
    [
     18,
     -3,
     4
    ],
    [
     19,
     -3,
     4
    ],
    [
     20,
     -3,
     4
    ],
    [
     21,
     -3,
     4
    ],
    [
     22,
     -3,
     4
    ],
    [
     23,
     -3,
     4
    ],
    [
     15,
     -2,
     4
    ],
    [
     16,
     -2,
     4
    ],
    [
     17,
     -2,
     4
    ],
    [
     18,
     -2,
     4
    ],
    [
     19,
     -2,
     4
    ],
    [
     20,
     -2,
     4
    ],
    [
     21,
     -2,
     4
    ],
    [
     22,
     -2,
     4
    ],
    [
     23,
     -2,
     4
    ],
    [
     15,
     -1,
     4
    ],
    [
     16,
     -1,
     4
    ],
    [
     17,
     -1,
     4
    ],
    [
     18,
     -1,
     4
    ],
    [
     19,
     -1,
     4
    ],
    [
     20,
     -1,
     4
    ],
    [
     21,
     -1,
     4
    ],
    [
     22,
     -1,
     4
    ],
    [
     23,
     -1,
     4
    ],
    [
     15,
     0,
     4
    ],
    [
     16,
     0,
     4
    ],
    [
     17,
     0,
     4
    ],
    [
     18,
     0,
     4
    ],
    [
     19,
     0,
     4
    ],
    [
     20,
     0,
     4
    ],
    [
     21,
     0,
     4
    ],
    [
     22,
     0,
     4
    ],
    [
     23,
     0,
     4
    ],
    [
     15,
     1,
     4
    ],
    [
     16,
     1,
     4
    ],
    [
     17,
     1,
     4
    ],
    [
     18,
     1,
     4
    ],
    [
     19,
     1,
     4
    ],
    [
     20,
     1,
     4
    ],
    [
     21,
     1,
     4
    ],
    [
     22,
     1,
     4
    ],
    [
     23,
     1,
     4
    ],
    [
     15,
     2,
     4
    ],
    [
     16,
     2,
     4
    ],
    [
     17,
     2,
     4
    ],
    [
     18,
     2,
     4
    ],
    [
     19,
     2,
     4
    ],
    [
     20,
     2,
     4
    ],
    [
     21,
     2,
     4
    ],
    [
     22,
     2,
     4
    ],
    [
     23,
     2,
     4
    ],
    [
     15,
     3,
     4
    ],
    [
     16,
     3,
     4
    ],
    [
     17,
     3,
     4
    ],
    [
     18,
     3,
     4
    ],
    [
     19,
     3,
     4
    ],
    [
     20,
     3,
     4
    ],
    [
     21,
     3,
     4
    ],
    [
     22,
     3,
     4
    ],
    [
     23,
     3,
     4
    ],
    [
     15,
     -4,
     5
    ],
    [
     16,
     -4,
     5
    ],
    [
     17,
     -4,
     5
    ],
    [
     18,
     -4,
     5
    ],
    [
     19,
     -4,
     5
    ],
    [
     20,
     -4,
     5
    ],
    [
     21,
     -4,
     5
    ],
    [
     22,
     -4,
     5
    ],
    [
     23,
     -4,
     5
    ],
    [
     15,
     -3,
     5
    ],
    [
     16,
     -3,
     5
    ],
    [
     17,
     -3,
     5
    ],
    [
     18,
     -3,
     5
    ],
    [
     19,
     -3,
     5
    ],
    [
     20,
     -3,
     5
    ],
    [
     21,
     -3,
     5
    ],
    [
     22,
     -3,
     5
    ],
    [
     23,
     -3,
     5
    ],
    [
     15,
     -2,
     5
    ],
    [
     16,
     -2,
     5
    ],
    [
     17,
     -2,
     5
    ],
    [
     18,
     -2,
     5
    ],
    [
     19,
     -2,
     5
    ],
    [
     20,
     -2,
     5
    ],
    [
     21,
     -2,
     5
    ],
    [
     22,
     -2,
     5
    ],
    [
     23,
     -2,
     5
    ],
    [
     15,
     -1,
     5
    ],
    [
     16,
     -1,
     5
    ],
    [
     17,
     -1,
     5
    ],
    [
     18,
     -1,
     5
    ],
    [
     19,
     -1,
     5
    ],
    [
     20,
     -1,
     5
    ],
    [
     21,
     -1,
     5
    ],
    [
     22,
     -1,
     5
    ],
    [
     23,
     -1,
     5
    ],
    [
     15,
     0,
     5
    ],
    [
     16,
     0,
     5
    ],
    [
     17,
     0,
     5
    ],
    [
     18,
     0,
     5
    ],
    [
     19,
     0,
     5
    ],
    [
     20,
     0,
     5
    ],
    [
     21,
     0,
     5
    ],
    [
     22,
     0,
     5
    ],
    [
     23,
     0,
     5
    ],
    [
     15,
     1,
     5
    ],
    [
     16,
     1,
     5
    ],
    [
     17,
     1,
     5
    ],
    [
     18,
     1,
     5
    ],
    [
     19,
     1,
     5
    ],
    [
     20,
     1,
     5
    ],
    [
     21,
     1,
     5
    ],
    [
     22,
     1,
     5
    ],
    [
     23,
     1,
     5
    ],
    [
     15,
     2,
     5
    ],
    [
     16,
     2,
     5
    ],
    [
     17,
     2,
     5
    ],
    [
     18,
     2,
     5
    ],
    [
     19,
     2,
     5
    ],
    [
     20,
     2,
     5
    ],
    [
     21,
     2,
     5
    ],
    [
     22,
     2,
     5
    ],
    [
     23,
     2,
     5
    ],
    [
     15,
     3,
     5
    ],
    [
     16,
     3,
     5
    ],
    [
     17,
     3,
     5
    ],
    [
     18,
     3,
     5
    ],
    [
     19,
     3,
     5
    ],
    [
     20,
     3,
     5
    ],
    [
     21,
     3,
     5
    ],
    [
     22,
     3,
     5
    ],
    [
     23,
     3,
     5
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 144
  },
  "crank": {
   "cells": [
    [
     18,
     -5,
     4
    ],
    [
     18,
     -5,
     5
    ],
    [
     18,
     -5,
     6
    ],
    [
     19,
     -5,
     4
    ],
    [
     19,
     -5,
     5
    ],
    [
     19,
     -5,
     6
    ],
    [
     20,
     -5,
     4
    ],
    [
     20,
     -5,
     5
    ],
    [
     20,
     -5,
     6
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 9
  },
  "crankpin": {
   "cells": [
    [
     20,
     -6,
     5
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 1
  },
  "walls": {
   "cells": [
    [
     16,
     -4,
     6
    ],
    [
     22,
     -4,
     6
    ],
    [
     16,
     -3,
     6
    ],
    [
     22,
     -3,
     6
    ],
    [
     16,
     -2,
     6
    ],
    [
     22,
     -2,
     6
    ],
    [
     16,
     -1,
     6
    ],
    [
     22,
     -1,
     6
    ],
    [
     16,
     0,
     6
    ],
    [
     22,
     0,
     6
    ],
    [
     16,
     1,
     6
    ],
    [
     22,
     1,
     6
    ],
    [
     16,
     2,
     6
    ],
    [
     22,
     2,
     6
    ],
    [
     16,
     3,
     6
    ],
    [
     22,
     3,
     6
    ],
    [
     16,
     -4,
     7
    ],
    [
     22,
     -4,
     7
    ],
    [
     16,
     -3,
     7
    ],
    [
     22,
     -3,
     7
    ],
    [
     16,
     -2,
     7
    ],
    [
     22,
     -2,
     7
    ],
    [
     16,
     -1,
     7
    ],
    [
     22,
     -1,
     7
    ],
    [
     16,
     0,
     7
    ],
    [
     22,
     0,
     7
    ],
    [
     16,
     1,
     7
    ],
    [
     22,
     1,
     7
    ],
    [
     16,
     2,
     7
    ],
    [
     22,
     2,
     7
    ],
    [
     16,
     3,
     7
    ],
    [
     22,
     3,
     7
    ],
    [
     16,
     -4,
     8
    ],
    [
     22,
     -4,
     8
    ],
    [
     16,
     -3,
     8
    ],
    [
     22,
     -3,
     8
    ],
    [
     16,
     -2,
     8
    ],
    [
     22,
     -2,
     8
    ],
    [
     16,
     -1,
     8
    ],
    [
     22,
     -1,
     8
    ],
    [
     16,
     0,
     8
    ],
    [
     22,
     0,
     8
    ],
    [
     16,
     1,
     8
    ],
    [
     22,
     1,
     8
    ],
    [
     16,
     2,
     8
    ],
    [
     22,
     2,
     8
    ],
    [
     16,
     3,
     8
    ],
    [
     22,
     3,
     8
    ],
    [
     16,
     -4,
     9
    ],
    [
     22,
     -4,
     9
    ],
    [
     16,
     -3,
     9
    ],
    [
     22,
     -3,
     9
    ],
    [
     16,
     -2,
     9
    ],
    [
     22,
     -2,
     9
    ],
    [
     16,
     -1,
     9
    ],
    [
     22,
     -1,
     9
    ],
    [
     16,
     0,
     9
    ],
    [
     22,
     0,
     9
    ],
    [
     16,
     1,
     9
    ],
    [
     22,
     1,
     9
    ],
    [
     16,
     2,
     9
    ],
    [
     22,
     2,
     9
    ],
    [
     16,
     3,
     9
    ],
    [
     22,
     3,
     9
    ],
    [
     16,
     -4,
     10
    ],
    [
     22,
     -4,
     10
    ],
    [
     16,
     -3,
     10
    ],
    [
     22,
     -3,
     10
    ],
    [
     16,
     -2,
     10
    ],
    [
     22,
     -2,
     10
    ],
    [
     16,
     -1,
     10
    ],
    [
     22,
     -1,
     10
    ],
    [
     16,
     0,
     10
    ],
    [
     22,
     0,
     10
    ],
    [
     16,
     1,
     10
    ],
    [
     22,
     1,
     10
    ],
    [
     16,
     2,
     10
    ],
    [
     22,
     2,
     10
    ],
    [
     16,
     3,
     10
    ],
    [
     22,
     3,
     10
    ],
    [
     16,
     -4,
     11
    ],
    [
     22,
     -4,
     11
    ],
    [
     16,
     -3,
     11
    ],
    [
     22,
     -3,
     11
    ],
    [
     16,
     -2,
     11
    ],
    [
     22,
     -2,
     11
    ],
    [
     16,
     -1,
     11
    ],
    [
     22,
     -1,
     11
    ],
    [
     16,
     0,
     11
    ],
    [
     22,
     0,
     11
    ],
    [
     16,
     1,
     11
    ],
    [
     22,
     1,
     11
    ],
    [
     16,
     2,
     11
    ],
    [
     22,
     2,
     11
    ],
    [
     16,
     3,
     11
    ],
    [
     22,
     3,
     11
    ],
    [
     17,
     3,
     6
    ],
    [
     18,
     3,
     6
    ],
    [
     19,
     3,
     6
    ],
    [
     20,
     3,
     6
    ],
    [
     21,
     3,
     6
    ],
    [
     17,
     3,
     7
    ],
    [
     18,
     3,
     7
    ],
    [
     19,
     3,
     7
    ],
    [
     20,
     3,
     7
    ],
    [
     21,
     3,
     7
    ],
    [
     17,
     3,
     8
    ],
    [
     18,
     3,
     8
    ],
    [
     19,
     3,
     8
    ],
    [
     20,
     3,
     8
    ],
    [
     21,
     3,
     8
    ],
    [
     17,
     3,
     9
    ],
    [
     18,
     3,
     9
    ],
    [
     19,
     3,
     9
    ],
    [
     20,
     3,
     9
    ],
    [
     21,
     3,
     9
    ],
    [
     17,
     3,
     10
    ],
    [
     18,
     3,
     10
    ],
    [
     19,
     3,
     10
    ],
    [
     20,
     3,
     10
    ],
    [
     21,
     3,
     10
    ],
    [
     17,
     3,
     11
    ],
    [
     18,
     3,
     11
    ],
    [
     19,
     3,
     11
    ],
    [
     20,
     3,
     11
    ],
    [
     21,
     3,
     11
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 126,
   "parts": {
    "hot": "z >= 7"
   }
  },
  "head": {
   "cells": [
    [
     16,
     -4,
     12
    ],
    [
     17,
     -4,
     12
    ],
    [
     18,
     -4,
     12
    ],
    [
     19,
     -4,
     12
    ],
    [
     20,
     -4,
     12
    ],
    [
     21,
     -4,
     12
    ],
    [
     22,
     -4,
     12
    ],
    [
     16,
     -3,
     12
    ],
    [
     17,
     -3,
     12
    ],
    [
     18,
     -3,
     12
    ],
    [
     19,
     -3,
     12
    ],
    [
     20,
     -3,
     12
    ],
    [
     21,
     -3,
     12
    ],
    [
     22,
     -3,
     12
    ],
    [
     16,
     -2,
     12
    ],
    [
     17,
     -2,
     12
    ],
    [
     18,
     -2,
     12
    ],
    [
     19,
     -2,
     12
    ],
    [
     20,
     -2,
     12
    ],
    [
     21,
     -2,
     12
    ],
    [
     22,
     -2,
     12
    ],
    [
     16,
     -1,
     12
    ],
    [
     17,
     -1,
     12
    ],
    [
     18,
     -1,
     12
    ],
    [
     20,
     -1,
     12
    ],
    [
     21,
     -1,
     12
    ],
    [
     22,
     -1,
     12
    ],
    [
     16,
     0,
     12
    ],
    [
     17,
     0,
     12
    ],
    [
     18,
     0,
     12
    ],
    [
     19,
     0,
     12
    ],
    [
     20,
     0,
     12
    ],
    [
     21,
     0,
     12
    ],
    [
     22,
     0,
     12
    ],
    [
     16,
     1,
     12
    ],
    [
     17,
     1,
     12
    ],
    [
     18,
     1,
     12
    ],
    [
     19,
     1,
     12
    ],
    [
     20,
     1,
     12
    ],
    [
     21,
     1,
     12
    ],
    [
     22,
     1,
     12
    ],
    [
     16,
     2,
     12
    ],
    [
     17,
     2,
     12
    ],
    [
     18,
     2,
     12
    ],
    [
     19,
     2,
     12
    ],
    [
     20,
     2,
     12
    ],
    [
     21,
     2,
     12
    ],
    [
     22,
     2,
     12
    ],
    [
     16,
     3,
     12
    ],
    [
     17,
     3,
     12
    ],
    [
     18,
     3,
     12
    ],
    [
     19,
     3,
     12
    ],
    [
     20,
     3,
     12
    ],
    [
     21,
     3,
     12
    ],
    [
     22,
     3,
     12
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 55
  },
  "plug": {
   "cells": [
    [
     19,
     -1,
     12
    ],
    [
     19,
     -1,
     13
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 2,
   "parts": {
    "all": "z >= 0"
   }
  },
  "intake": {
   "cells": [
    [
     17,
     -1,
     13
    ],
    [
     17,
     -1,
     14
    ],
    [
     16,
     -1,
     14
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 3,
   "block_shape": [
    0.5,
    0.5,
    1
   ]
  },
  "stack": {
   "cells": [
    [
     21,
     -1,
     13
    ],
    [
     21,
     -1,
     14
    ],
    [
     21,
     -1,
     15
    ],
    [
     21,
     -1,
     16
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 4,
   "block_shape": [
    0.6,
    0.6,
    1
   ]
  },
  "piston": {
   "cells": [
    [
     17,
     -4,
     10
    ],
    [
     18,
     -4,
     10
    ],
    [
     19,
     -4,
     10
    ],
    [
     20,
     -4,
     10
    ],
    [
     21,
     -4,
     10
    ],
    [
     17,
     -3,
     10
    ],
    [
     18,
     -3,
     10
    ],
    [
     19,
     -3,
     10
    ],
    [
     20,
     -3,
     10
    ],
    [
     21,
     -3,
     10
    ],
    [
     17,
     -2,
     10
    ],
    [
     18,
     -2,
     10
    ],
    [
     19,
     -2,
     10
    ],
    [
     20,
     -2,
     10
    ],
    [
     21,
     -2,
     10
    ],
    [
     17,
     -1,
     10
    ],
    [
     18,
     -1,
     10
    ],
    [
     19,
     -1,
     10
    ],
    [
     20,
     -1,
     10
    ],
    [
     21,
     -1,
     10
    ],
    [
     17,
     0,
     10
    ],
    [
     18,
     0,
     10
    ],
    [
     19,
     0,
     10
    ],
    [
     20,
     0,
     10
    ],
    [
     21,
     0,
     10
    ],
    [
     17,
     1,
     10
    ],
    [
     18,
     1,
     10
    ],
    [
     19,
     1,
     10
    ],
    [
     20,
     1,
     10
    ],
    [
     21,
     1,
     10
    ],
    [
     17,
     2,
     10
    ],
    [
     18,
     2,
     10
    ],
    [
     19,
     2,
     10
    ],
    [
     20,
     2,
     10
    ],
    [
     21,
     2,
     10
    ],
    [
     19,
     -1,
     9
    ]
   ],
   "at": [
    -1.3,
    0.1
   ],
   "centered": false,
   "material": "iron",
   "expect_blocks": 36
  },
  "mix": {
   "cells": [
    [
     -2,
     -2,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     0,
     1,
     9
    ],
    [
     -1,
     -3,
     9
    ],
    [
     2,
     0,
     10
    ],
    [
     -2,
     1,
     10
    ],
    [
     0,
     -2,
     10
    ],
    [
     1,
     2,
     11
    ],
    [
     -1,
     0,
     11
    ],
    [
     2,
     -3,
     11
    ]
   ],
   "at": [
    0.5999999999999999,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 10,
   "parts": {
    "all": "z >= 0"
   },
   "block_shape": [
    0.6,
    0.6,
    0.6
   ]
  },
  "smoke": {
   "cells": [
    [
     0,
     0,
     17
    ],
    [
     1,
     0,
     18
    ],
    [
     -1,
     1,
     18
    ],
    [
     0,
     -1,
     19
    ],
    [
     1,
     1,
     20
    ]
   ],
   "at": [
    0.8,
    0.0
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 5,
   "block_shape": [
    0.7,
    0.7,
    0.7
   ]
  }
 },
 "actions": [
  {
   "type": "face_camera",
   "when": 0.0,
   "until": 0.5
  },
  {
   "type": "build",
   "target": "chassis",
   "when": 0.5,
   "until": 2.1,
   "order": "x",
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "axles",
   "when": 2.1,
   "until": 2.5,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "wheel_rl",
   "when": 2.5,
   "until": 2.95,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "hub_rl",
   "when": 2.95,
   "until": 3.1,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "wheel_rr",
   "when": 3.1,
   "until": 3.55,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "hub_rr",
   "when": 3.55,
   "until": 3.7,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "floor",
   "when": 3.7,
   "until": 4.3,
   "order": "x",
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "seat",
   "when": 4.3,
   "until": 4.8,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sides",
   "when": 6.22,
   "until": 6.82,
   "order": "x",
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dash",
   "when": 6.82,
   "until": 7.22,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "steer",
   "when": 7.22,
   "until": 7.47,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "glass",
   "when": 7.47,
   "until": 7.82,
   "stand_at": [
    -1.75,
    -0.15,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    1.6,
    -0.35,
    0.0
   ],
   "when": 7.82,
   "until": 8.92,
   "via": [
    [
     -0.2,
     -1.2,
     0.0
    ],
    [
     1.4,
     -1.2,
     0.0
    ]
   ]
  },
  {
   "type": "build",
   "target": "wheel_fl",
   "when": 8.92,
   "until": 9.37,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "hub_fl",
   "when": 9.37,
   "until": 9.52,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "wheel_fr",
   "when": 9.52,
   "until": 9.97,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "hub_fr",
   "when": 9.97,
   "until": 10.12,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "crankcase",
   "when": 10.12,
   "until": 10.62,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "crank",
   "when": 10.62,
   "until": 10.92,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "crankpin",
   "when": 10.92,
   "until": 11.07,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "walls",
   "when": 11.07,
   "until": 11.88,
   "order": "z",
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "piston",
   "when": 11.88,
   "until": 12.22,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "head",
   "when": 12.22,
   "until": 12.57,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "plug",
   "when": 12.57,
   "until": 12.77,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "intake",
   "when": 12.77,
   "until": 13.02,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "stack",
   "when": 13.02,
   "until": 13.32,
   "stand_at": [
    1.6,
    -0.35,
    0.0
   ]
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 13.92,
   "until": 15.02,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 13.92,
   "until": 15.02,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 13.92,
   "until": 15.02,
   "ticks_per_turn": 0
  },
  {
   "type": "build",
   "target": "mix",
   "order": "z",
   "when": 13.92,
   "until": 15.02,
   "quiet": true
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 17.7,
   "until": 18.8,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 17.7,
   "until": 18.8,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 17.7,
   "until": 18.8,
   "ticks_per_turn": 0
  },
  {
   "type": "grow",
   "target": "mix",
   "factor": 0.25,
   "when": 17.7,
   "until": 18.8
  },
  {
   "type": "slide",
   "target": "mix",
   "by": [
    0.0,
    0,
    0.9
   ],
   "when": 17.7,
   "until": 18.8,
   "sound": "none"
  },
  {
   "type": "crack",
   "target": "plug",
   "part": "all",
   "when": 20.95,
   "until": 21.03,
   "sound": "spark",
   "cool_from": 21.2,
   "cool_until": 21.55
  },
  {
   "type": "crack",
   "target": "mix",
   "part": "all",
   "when": 21.4,
   "until": 21.5,
   "sound": "bang"
  },
  {
   "type": "crack",
   "target": "walls",
   "part": "hot",
   "when": 21.4,
   "until": 21.48,
   "sound": "none",
   "cool_from": 21.75,
   "cool_until": 22.6
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 21.45,
   "until": 22.15,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 21.45,
   "until": 22.15,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 21.45,
   "until": 22.15,
   "ticks_per_turn": 0
  },
  {
   "type": "grow",
   "target": "mix",
   "factor": 4.0,
   "when": 21.45,
   "until": 22.15
  },
  {
   "type": "slide",
   "target": "mix",
   "by": [
    0.0,
    0,
    -0.9
   ],
   "when": 21.45,
   "until": 22.15,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 25.25,
   "until": 26.35,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 25.25,
   "until": 26.35,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 25.25,
   "until": 26.35,
   "ticks_per_turn": 0
  },
  {
   "type": "remove",
   "target": "mix",
   "quiet": true,
   "when": 25.25,
   "until": 26.13
  },
  {
   "type": "build",
   "target": "smoke",
   "quiet": true,
   "order": "z",
   "when": 25.55,
   "until": 26.35
  },
  {
   "type": "slide",
   "target": "smoke",
   "by": [
    0.0,
    0,
    0.5
   ],
   "when": 26.35,
   "until": 27.75,
   "sound": "none"
  },
  {
   "type": "remove",
   "target": "smoke",
   "quiet": true,
   "when": 27.35,
   "until": 27.85
  },
  {
   "type": "point",
   "pos": [
    0.5999999999999999,
    -0.4,
    0.5
   ],
   "when": 27.63,
   "until": 30.03
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 28.03,
   "until": 28.45,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.03,
   "until": 28.45,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.03,
   "until": 28.45,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 28.45,
   "until": 28.87,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.45,
   "until": 28.87,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.45,
   "until": 28.87,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 28.87,
   "until": 29.29,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.87,
   "until": 29.29,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 28.87,
   "until": 29.29,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 29.29,
   "until": 29.71,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 29.29,
   "until": 29.71,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 29.29,
   "until": 29.71,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 29.71,
   "until": 30.13,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 29.71,
   "until": 30.13,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 29.71,
   "until": 30.13,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 30.13,
   "until": 30.55,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 30.13,
   "until": 30.55,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 30.13,
   "until": 30.55,
   "ticks_per_turn": 0
  },
  {
   "type": "move",
   "to": [
    1.6,
    -1.0,
    0.0
   ],
   "when": 31.84,
   "until": 32.54
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 32.54,
   "until": 32.68,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.54,
   "until": 32.68,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.54,
   "until": 32.68,
   "ticks_per_turn": 0
  },
  {
   "type": "face_camera",
   "when": 32.54,
   "until": "end"
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 32.68,
   "until": 32.82,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.68,
   "until": 32.82,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.68,
   "until": 32.82,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 32.82,
   "until": 32.96,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.82,
   "until": 32.96,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.82,
   "until": 32.96,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 32.96,
   "until": 33.1,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.96,
   "until": 33.1,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 32.96,
   "until": 33.1,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 33.1,
   "until": 33.24,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.1,
   "until": 33.24,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.1,
   "until": 33.24,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "chassis",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "axles",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "wheel_rl",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "hub_rl",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "wheel_rr",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "hub_rr",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "wheel_fl",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "hub_fl",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "wheel_fr",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "hub_fr",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "floor",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "seat",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sides",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "dash",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "steer",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "glass",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "crankcase",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "crank",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "crankpin",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "walls",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "head",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "plug",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "intake",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "stack",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "smoke",
   "by": [
    1.1,
    0,
    0
   ],
   "when": 33.14,
   "until": 35.34,
   "sound": "none"
  },
  {
   "type": "spin",
   "target": "wheel_rl",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    4,
    -6,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "hub_rl",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    4,
    -6,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "wheel_rr",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    4,
    5,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "hub_rr",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    4,
    5,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "wheel_fl",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    20,
    -6,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "hub_fl",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    20,
    -6,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "wheel_fr",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    20,
    5,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "hub_fr",
   "axis": "y",
   "degrees": -252.10164279871026,
   "pivot": [
    20,
    5,
    2
   ],
   "when": 33.14,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 33.24,
   "until": 33.38,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.24,
   "until": 33.38,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.24,
   "until": 33.38,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 33.38,
   "until": 33.52,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.38,
   "until": 33.52,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.38,
   "until": 33.52,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 33.52,
   "until": 33.66,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.52,
   "until": 33.66,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.52,
   "until": 33.66,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 33.66,
   "until": 33.8,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.66,
   "until": 33.8,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.66,
   "until": 33.8,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 33.8,
   "until": 33.94,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.8,
   "until": 33.94,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.8,
   "until": 33.94,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 33.94,
   "until": 34.08,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.94,
   "until": 34.08,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 33.94,
   "until": 34.08,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 34.08,
   "until": 34.22,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.08,
   "until": 34.22,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.08,
   "until": 34.22,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 34.22,
   "until": 34.36,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.22,
   "until": 34.36,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.22,
   "until": 34.36,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 34.36,
   "until": 34.5,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.36,
   "until": 34.5,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.36,
   "until": 34.5,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 34.5,
   "until": 34.64,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.5,
   "until": 34.64,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.5,
   "until": 34.64,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 34.64,
   "until": 34.78,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.64,
   "until": 34.78,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.64,
   "until": 34.78,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 34.78,
   "until": 34.92,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.78,
   "until": 34.92,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.78,
   "until": 34.92,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 34.92,
   "until": 35.06,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.92,
   "until": 35.06,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 34.92,
   "until": 35.06,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 35.06,
   "until": 35.2,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.06,
   "until": 35.2,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.06,
   "until": 35.2,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 35.2,
   "until": 35.34,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.2,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.2,
   "until": 35.34,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    -0.3
   ],
   "when": 35.34,
   "until": 35.48,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.34,
   "until": 35.48,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.34,
   "until": 35.48,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "piston",
   "by": [
    0.0,
    0,
    0.3
   ],
   "when": 35.48,
   "until": 35.62,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "crank",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.48,
   "until": 35.62,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "crankpin",
   "axis": "y",
   "degrees": 180,
   "pivot": [
    19,
    -5,
    5
   ],
   "when": 35.48,
   "until": 35.62,
   "ticks_per_turn": 0
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    -1.4,
    -0.2,
    0.7
   ],
   "az": -75,
   "el": 8,
   "dist": 1.9
  },
  {
   "at": 3.0,
   "target": [
    -0.2,
    0.0,
    0.55
   ],
   "az": -82,
   "el": 14,
   "dist": 3.1
  },
  {
   "at": 9.22,
   "target": [
    0.4,
    0.0,
    0.6
   ],
   "az": -92,
   "el": 12,
   "dist": 3.0
  },
  {
   "at": 13.32,
   "target": [
    0.5999999999999999,
    -0.1,
    0.95
   ],
   "az": -90,
   "el": 6,
   "dist": 1.75
  },
  {
   "at": 25.05,
   "target": [
    0.6999999999999998,
    -0.1,
    1.25
   ],
   "az": -90,
   "el": 6,
   "dist": 2.1
  },
  {
   "at": 27.43,
   "target": [
    0.6999999999999998,
    -0.1,
    1.3
   ],
   "az": -90,
   "el": 6,
   "dist": 2.15
  },
  {
   "at": 27.93,
   "target": [
    0.2999999999999999,
    -0.1,
    0.75
   ],
   "az": -92,
   "el": 8,
   "dist": 2.2
  },
  {
   "at": 31.84,
   "target": [
    0.4,
    0.0,
    0.6
   ],
   "az": -90,
   "el": 12,
   "dist": 3.3
  },
  {
   "at": "end",
   "target": [
    1.0,
    0.0,
    0.6
   ],
   "az": -92,
   "el": 12,
   "dist": 3.5
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "THOUSANDS OF",
   "color": "white",
   "y": 470,
   "size": 92,
   "when": 0.3,
   "until": 6.22
  },
  {
   "style": "title",
   "text": "EXPLOSIONS!",
   "color": "amber",
   "y": 590,
   "size": 104,
   "when": 2.0,
   "until": 6.22
  },
  {
   "style": "title",
   "text": "1. SUCK",
   "color": "cyan",
   "y": 470,
   "size": 110,
   "when": 13.62,
   "until": 17.1
  },
  {
   "style": "title",
   "text": "2. SQUEEZE",
   "color": "green",
   "y": 470,
   "size": 110,
   "when": 17.4,
   "until": 20.4
  },
  {
   "style": "impact",
   "text": "3. BANG!",
   "color": "red",
   "y": 480,
   "size": 140,
   "when": 21.4,
   "until": 24.75
  },
  {
   "style": "title",
   "text": "4. BLOW",
   "color": "white",
   "y": 470,
   "size": 110,
   "when": 25.05,
   "until": 27.63
  },
  {
   "style": "panel",
   "y": 340,
   "border": "amber",
   "when": 27.83,
   "until": 32.14,
   "rows": [
    {
     "text": "PUSH -> CRANK -> WHEELS",
     "color": "amber",
     "at": 27.93,
     "size": 48
    }
   ]
  },
  {
   "style": "panel",
   "y": 340,
   "border": "green",
   "when": 32.24,
   "until": 36.84,
   "rows": [
    {
     "text": "4 cylinders x 3,000 rpm",
     "color": "white",
     "at": 32.34,
     "size": 46
    },
    {
     "text": "= 6,000 bangs a minute",
     "color": "green",
     "at": 33.04,
     "size": 50
    }
   ]
  },
  {
   "style": "end",
   "title1": "SUCK, SQUEEZE,",
   "title2": "BANG, BLOW",
   "sub": "A four-stroke engine: 2 turns of the crank for every bang",
   "cta": "Follow for more Block Lab",
   "when": 36.84
  }
 ]
}

In [ ]:
%%writefile stories/trebuchet.json
{
 "title": "How a trebuchet works",
 "slug": "trebuchet",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "loop": true,
 "start": 1.756,
 "tail": 0.3732,
 "world": {
  "ground": "grass",
  "time": "morning",
  "weather": "clear",
  "animals": [
   "cat"
  ]
 },
 "lines": [
  {
   "id": "whoa",
   "text": "Whoa! Where did that come from?... Ah, I know. A trebuchet! Let me build one.",
   "pause": 0.12
  },
  {
   "id": "frame",
   "text": "A strong frame... and a long arm, on a pivot. That's a lever!",
   "pause": 0.12
  },
  {
   "id": "weight",
   "text": "On the short end, a heavy box of rocks.",
   "pause": 0.12
  },
  {
   "id": "load",
   "text": "Pull the long end down... and load the stone.",
   "pause": 0.12
  },
  {
   "id": "lesson",
   "text": "When the weight drops, the long end swings four times faster... and flings the stone!",
   "pause": 0.138
  },
  {
   "id": "history",
   "text": "That's how castles got knocked down in the Middle Ages.",
   "pause": 0.25
  },
  {
   "id": "fire",
   "text": "Ready... fire!",
   "pause": 0.12
  },
  {
   "id": "end",
   "text": "...and that's where it came from!",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   -1.15,
   -0.8,
   0.0
  ],
  "outfit": {
   "hat": "cap",
   "shirt": "rust",
   "extras": [
    "hammer"
   ]
  }
 },
 "structures": {
  "base": {
   "cells": [
    [
     -2,
     -3,
     0
    ],
    [
     -1,
     -3,
     0
    ],
    [
     0,
     -3,
     0
    ],
    [
     1,
     -3,
     0
    ],
    [
     2,
     -3,
     0
    ],
    [
     3,
     -3,
     0
    ],
    [
     4,
     -3,
     0
    ],
    [
     5,
     -3,
     0
    ],
    [
     6,
     -3,
     0
    ],
    [
     7,
     -3,
     0
    ],
    [
     8,
     -3,
     0
    ],
    [
     9,
     -3,
     0
    ],
    [
     10,
     -3,
     0
    ],
    [
     -2,
     3,
     0
    ],
    [
     -1,
     3,
     0
    ],
    [
     0,
     3,
     0
    ],
    [
     1,
     3,
     0
    ],
    [
     2,
     3,
     0
    ],
    [
     3,
     3,
     0
    ],
    [
     4,
     3,
     0
    ],
    [
     5,
     3,
     0
    ],
    [
     6,
     3,
     0
    ],
    [
     7,
     3,
     0
    ],
    [
     8,
     3,
     0
    ],
    [
     9,
     3,
     0
    ],
    [
     10,
     3,
     0
    ],
    [
     -2,
     -2,
     0
    ],
    [
     4,
     -2,
     0
    ],
    [
     10,
     -2,
     0
    ],
    [
     -2,
     -1,
     0
    ],
    [
     4,
     -1,
     0
    ],
    [
     10,
     -1,
     0
    ],
    [
     -2,
     0,
     0
    ],
    [
     4,
     0,
     0
    ],
    [
     10,
     0,
     0
    ],
    [
     -2,
     1,
     0
    ],
    [
     4,
     1,
     0
    ],
    [
     10,
     1,
     0
    ],
    [
     -2,
     2,
     0
    ],
    [
     4,
     2,
     0
    ],
    [
     10,
     2,
     0
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 41
  },
  "uprights": {
   "cells": [
    [
     3,
     -3,
     1
    ],
    [
     5,
     -3,
     1
    ],
    [
     3,
     3,
     1
    ],
    [
     5,
     3,
     1
    ],
    [
     3,
     -3,
     2
    ],
    [
     5,
     -3,
     2
    ],
    [
     3,
     3,
     2
    ],
    [
     5,
     3,
     2
    ],
    [
     3,
     -3,
     3
    ],
    [
     5,
     -3,
     3
    ],
    [
     3,
     3,
     3
    ],
    [
     5,
     3,
     3
    ],
    [
     3,
     -3,
     4
    ],
    [
     5,
     -3,
     4
    ],
    [
     3,
     3,
     4
    ],
    [
     5,
     3,
     4
    ],
    [
     3,
     -3,
     5
    ],
    [
     5,
     -3,
     5
    ],
    [
     3,
     3,
     5
    ],
    [
     5,
     3,
     5
    ],
    [
     3,
     -3,
     6
    ],
    [
     5,
     -3,
     6
    ],
    [
     3,
     3,
     6
    ],
    [
     5,
     3,
     6
    ],
    [
     3,
     -3,
     7
    ],
    [
     5,
     -3,
     7
    ],
    [
     3,
     3,
     7
    ],
    [
     5,
     3,
     7
    ],
    [
     3,
     -3,
     8
    ],
    [
     5,
     -3,
     8
    ],
    [
     3,
     3,
     8
    ],
    [
     5,
     3,
     8
    ],
    [
     3,
     -3,
     9
    ],
    [
     5,
     -3,
     9
    ],
    [
     3,
     3,
     9
    ],
    [
     5,
     3,
     9
    ],
    [
     3,
     -3,
     10
    ],
    [
     5,
     -3,
     10
    ],
    [
     3,
     3,
     10
    ],
    [
     5,
     3,
     10
    ],
    [
     3,
     -3,
     11
    ],
    [
     5,
     -3,
     11
    ],
    [
     3,
     3,
     11
    ],
    [
     5,
     3,
     11
    ],
    [
     3,
     -3,
     12
    ],
    [
     5,
     -3,
     12
    ],
    [
     3,
     3,
     12
    ],
    [
     5,
     3,
     12
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 48
  },
  "axle": {
   "cells": [
    [
     4,
     -3,
     12
    ],
    [
     4,
     -2,
     12
    ],
    [
     4,
     -1,
     12
    ],
    [
     4,
     1,
     12
    ],
    [
     4,
     2,
     12
    ],
    [
     4,
     3,
     12
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 6,
   "block_shape": [
    1,
    0.4,
    0.4
   ]
  },
  "arm": {
   "cells": [
    [
     1,
     0,
     12
    ],
    [
     2,
     0,
     12
    ],
    [
     3,
     0,
     12
    ],
    [
     4,
     0,
     12
    ],
    [
     5,
     0,
     12
    ],
    [
     6,
     0,
     12
    ],
    [
     7,
     0,
     12
    ],
    [
     8,
     0,
     12
    ],
    [
     9,
     0,
     12
    ],
    [
     10,
     0,
     12
    ],
    [
     11,
     0,
     12
    ],
    [
     12,
     0,
     12
    ],
    [
     13,
     0,
     12
    ],
    [
     14,
     0,
     12
    ],
    [
     15,
     0,
     12
    ],
    [
     16,
     0,
     12
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 16
  },
  "weight": {
   "cells": [
    [
     1,
     0,
     11
    ],
    [
     0,
     -1,
     8
    ],
    [
     1,
     -1,
     8
    ],
    [
     2,
     -1,
     8
    ],
    [
     0,
     0,
     8
    ],
    [
     1,
     0,
     8
    ],
    [
     2,
     0,
     8
    ],
    [
     0,
     1,
     8
    ],
    [
     1,
     1,
     8
    ],
    [
     2,
     1,
     8
    ],
    [
     0,
     -1,
     9
    ],
    [
     1,
     -1,
     9
    ],
    [
     2,
     -1,
     9
    ],
    [
     0,
     0,
     9
    ],
    [
     1,
     0,
     9
    ],
    [
     2,
     0,
     9
    ],
    [
     0,
     1,
     9
    ],
    [
     1,
     1,
     9
    ],
    [
     2,
     1,
     9
    ],
    [
     0,
     -1,
     10
    ],
    [
     1,
     -1,
     10
    ],
    [
     2,
     -1,
     10
    ],
    [
     0,
     0,
     10
    ],
    [
     1,
     0,
     10
    ],
    [
     2,
     0,
     10
    ],
    [
     0,
     1,
     10
    ],
    [
     1,
     1,
     10
    ],
    [
     2,
     1,
     10
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "brick",
   "expect_blocks": 28
  },
  "stone_b": {
   "cells": [
    [
     15,
     -1,
     13
    ],
    [
     16,
     -1,
     13
    ],
    [
     15,
     0,
     13
    ],
    [
     16,
     0,
     13
    ],
    [
     15,
     -1,
     14
    ],
    [
     16,
     -1,
     14
    ],
    [
     15,
     0,
     14
    ],
    [
     16,
     0,
     14
    ]
   ],
   "at": [
    0.0,
    0.0
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 8
  },
  "stone_a": {
   "cells": [
    [
     0,
     0,
     0
    ],
    [
     1,
     0,
     0
    ],
    [
     0,
     1,
     0
    ],
    [
     1,
     1,
     0
    ],
    [
     0,
     0,
     1
    ],
    [
     1,
     0,
     1
    ],
    [
     0,
     1,
     1
    ],
    [
     1,
     1,
     1
    ]
   ],
   "at": [
    -0.8775802815176158,
    -0.1
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 8,
   "prebuilt": true
  }
 },
 "actions": [
  {
   "type": "fly",
   "target": "stone_a",
   "v0": [
    -0.6077686218342562,
    0.0,
    3.446827135542728
   ],
   "g": 2.0,
   "tau0": 2.325871,
   "land": true,
   "origin": [
    1.45572207696461,
    -0.1,
    1.5060989761521477
   ],
   "anchor": [
    -0.8775802815176158,
    -0.1,
    0.0
   ],
   "when": 0.0,
   "until": 1.513259
  },
  {
   "type": "shrug",
   "when": 1.61,
   "until": 2.81
  },
  {
   "type": "face_camera",
   "when": 2.81,
   "until": 6.06
  },
  {
   "type": "move",
   "to": [
    -0.75,
    -0.75,
    0.0
   ],
   "when": 4.26,
   "until": 4.86
  },
  {
   "type": "build",
   "target": "base",
   "when": 4.86,
   "until": 6.06,
   "stand_at": [
    -0.75,
    -0.75,
    0.0
   ],
   "order": "x"
  },
  {
   "type": "build",
   "target": "uprights",
   "when": 6.06,
   "until": 7.26,
   "stand_at": [
    -0.75,
    -0.75,
    0.0
   ],
   "order": "z"
  },
  {
   "type": "build",
   "target": "axle",
   "when": 7.26,
   "until": 7.56,
   "stand_at": [
    -0.75,
    -0.75,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "arm",
   "when": 7.56,
   "until": 8.46,
   "stand_at": [
    -0.75,
    -0.75,
    0.0
   ],
   "order": "x"
  },
  {
   "type": "build",
   "target": "weight",
   "when": 10.49,
   "until": 11.49,
   "stand_at": [
    -0.75,
    -0.75,
    0.0
   ],
   "order": "z"
  },
  {
   "type": "spin",
   "target": "arm",
   "axis": "y",
   "degrees": 50.0,
   "pivot": [
    4,
    0,
    12
   ],
   "when": 13.637,
   "until": 14.837,
   "ease": true,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "stone_b",
   "axis": "y",
   "degrees": 50.0,
   "pivot": [
    4,
    0,
    12
   ],
   "when": 13.637,
   "until": 14.837,
   "ease": true,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "weight",
   "by": [
    0.107,
    0,
    0.23
   ],
   "when": 13.64,
   "until": 14.84,
   "sound": "none"
  },
  {
   "type": "remove",
   "target": "stone_a",
   "when": 14.84,
   "until": 15.14,
   "stand_at": [
    -1.13,
    -0.55,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    1.42,
    -0.6,
    0.0
   ],
   "when": 15.19,
   "until": 16.19
  },
  {
   "type": "build",
   "target": "stone_b",
   "when": 16.19,
   "until": 16.49,
   "stand_at": [
    1.42,
    -0.6,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    -1.25,
    -0.95,
    0.0
   ],
   "when": 16.78,
   "until": 18.08
  },
  {
   "type": "slide",
   "target": "weight",
   "by": [
    0.043,
    0,
    -0.49
   ],
   "when": 26.94,
   "until": 27.49,
   "sound": "chug"
  },
  {
   "type": "spin",
   "target": "arm",
   "axis": "y",
   "degrees": -110.0,
   "pivot": [
    4,
    0,
    12
   ],
   "when": 26.941,
   "until": 27.491,
   "ease": false,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "stone_b",
   "axis": "y",
   "degrees": -60.0,
   "pivot": [
    4,
    0,
    12
   ],
   "when": 26.941,
   "until": 27.241,
   "ease": false,
   "ticks_per_turn": 0
  },
  {
   "type": "fly",
   "target": "stone_b",
   "v0": [
    -0.6077686218342562,
    0.0,
    3.446827135542728
   ],
   "g": 2.0,
   "origin": [
    0,
    0,
    0
   ],
   "anchor": [
    0,
    0,
    0
   ],
   "when": 27.240796,
   "until": 30.541
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    -2.256,
    -2.293,
    3.019
   ],
   "az": -136.47,
   "el": -70.0,
   "dist": 3.0
  },
  {
   "at": 1.86,
   "target": [
    -0.7,
    -0.4,
    0.55
   ],
   "az": -75,
   "el": 10,
   "dist": 2.3
  },
  {
   "at": 4.26,
   "target": [
    0.1,
    0.0,
    0.8
   ],
   "az": -88,
   "el": 12,
   "dist": 3.4
  },
  {
   "at": 13.44,
   "target": [
    0.6,
    0.0,
    0.6
   ],
   "az": -95,
   "el": 12,
   "dist": 3.0
  },
  {
   "at": 17.28,
   "target": [
    0.2,
    0.0,
    0.9
   ],
   "az": -100,
   "el": 10,
   "dist": 3.6
  },
  {
   "at": 26.94,
   "target": [
    0.1,
    0.0,
    1.0
   ],
   "az": -96,
   "el": 10,
   "dist": 3.7
  },
  {
   "at": 27.49,
   "target": [
    0.0,
    0.0,
    1.6
   ],
   "az": -110,
   "el": -10,
   "dist": 3.3
  },
  {
   "at": 29.041,
   "target": [
    -2.256,
    -2.293,
    3.019
   ],
   "az": -136.47,
   "el": -70.0,
   "dist": 3.0
  },
  {
   "at": "end",
   "target": [
    -2.256,
    -2.293,
    3.019
   ],
   "az": -136.47,
   "el": -70.0,
   "dist": 3.0
  }
 ],
 "cards": [
  {
   "style": "impact",
   "text": "WHOA!",
   "color": "amber",
   "y": 560,
   "size": 150,
   "when": 1.51,
   "until": 2.91
  },
  {
   "style": "title",
   "text": "TREBUCHET",
   "color": "white",
   "y": 470,
   "size": 100,
   "when": 4.06,
   "until": 6.06
  },
  {
   "style": "title",
   "text": "A GIANT LEVER",
   "color": "cyan",
   "y": 470,
   "size": 96,
   "when": 8.26,
   "until": 10.79
  },
  {
   "style": "title",
   "text": "HEAVY WEIGHT",
   "color": "amber",
   "y": 470,
   "size": 96,
   "when": 10.79,
   "until": 13.44
  },
  {
   "style": "panel",
   "y": 340,
   "border": "cyan",
   "when": 16.98,
   "until": 22.47,
   "rows": [
    {
     "text": "SHORT END 1 : LONG END 4",
     "color": "white",
     "at": 17.08,
     "size": 46
    },
    {
     "text": "= 4x FASTER!",
     "color": "green",
     "at": 18.38,
     "size": 56
    }
   ]
  },
  {
   "style": "title",
   "text": "CASTLE BUSTER",
   "color": "red",
   "y": 470,
   "size": 96,
   "when": 22.77,
   "until": 26.04
  },
  {
   "style": "impact",
   "text": "FIRE!",
   "color": "red",
   "y": 560,
   "size": 150,
   "when": 26.94,
   "until": 27.94
  }
 ]
}

In [ ]:
%%writefile stories/lock_key.json
{
 "title": "How a lock and key work",
 "slug": "lock_key",
 "voice": "am_santa",
 "speed": 0.95,
 "channel_tag": "GRANDPA'S BLOCK LAB",
 "loop": true,
 "start": 0.3,
 "tail": 0.194,
 "world": {
  "ground": "grass",
  "time": "day",
  "weather": "clear",
  "animals": []
 },
 "lines": [
  {
   "id": "hook",
   "text": "Every key has a secret code... cut right into its teeth. Let me show you!",
   "pause": 0.3
  },
  {
   "id": "build",
   "text": "Here's the inside of a lock. A brass plug that turns... inside a case that doesn't.",
   "pause": 0.2
  },
  {
   "id": "pins",
   "text": "Little pins, in two parts, pushed down by springs.",
   "pause": 0.518
  },
  {
   "id": "blocked",
   "text": "With no key, the top pins block the gap... so it can't turn. Locked!",
   "pause": 0.12
  },
  {
   "id": "key",
   "text": "The right key lifts every pin to exactly the right height...",
   "pause": 0.15
  },
  {
   "id": "turn",
   "text": "...the gaps all line up... and it turns! Click.",
   "pause": 1.091
  },
  {
   "id": "end",
   "text": "So next time you unlock your door... remember:",
   "pause": 0.0
  }
 ],
 "grandpa": {
  "start": [
   -0.5000000000000001,
   -0.55,
   0.0
  ],
  "outfit": {
   "hat": "cap",
   "shirt": "blue",
   "extras": [
    "hammer"
   ]
  }
 },
 "structures": {
  "base": {
   "cells": [
    [
     -6,
     0,
     0
    ],
    [
     -6,
     1,
     0
    ],
    [
     -5,
     0,
     0
    ],
    [
     -5,
     1,
     0
    ],
    [
     -4,
     0,
     0
    ],
    [
     -4,
     1,
     0
    ],
    [
     -3,
     0,
     0
    ],
    [
     -3,
     1,
     0
    ],
    [
     -2,
     0,
     0
    ],
    [
     -2,
     1,
     0
    ],
    [
     -1,
     0,
     0
    ],
    [
     -1,
     1,
     0
    ],
    [
     0,
     0,
     0
    ],
    [
     0,
     1,
     0
    ],
    [
     1,
     0,
     0
    ],
    [
     1,
     1,
     0
    ],
    [
     2,
     0,
     0
    ],
    [
     2,
     1,
     0
    ],
    [
     3,
     0,
     0
    ],
    [
     3,
     1,
     0
    ],
    [
     4,
     0,
     0
    ],
    [
     4,
     1,
     0
    ],
    [
     5,
     0,
     0
    ],
    [
     5,
     1,
     0
    ],
    [
     6,
     0,
     0
    ],
    [
     6,
     1,
     0
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "stone",
   "expect_blocks": 26
  },
  "plug": {
   "cells": [
    [
     -5,
     0,
     1
    ],
    [
     -5,
     0,
     3
    ],
    [
     -5,
     0,
     4
    ],
    [
     -5,
     0,
     5
    ],
    [
     -5,
     0,
     6
    ],
    [
     -4,
     0,
     1
    ],
    [
     -3,
     0,
     1
    ],
    [
     -3,
     0,
     3
    ],
    [
     -3,
     0,
     4
    ],
    [
     -3,
     0,
     5
    ],
    [
     -3,
     0,
     6
    ],
    [
     -2,
     0,
     1
    ],
    [
     -1,
     0,
     1
    ],
    [
     -1,
     0,
     3
    ],
    [
     -1,
     0,
     4
    ],
    [
     -1,
     0,
     5
    ],
    [
     -1,
     0,
     6
    ],
    [
     0,
     0,
     1
    ],
    [
     1,
     0,
     1
    ],
    [
     1,
     0,
     3
    ],
    [
     1,
     0,
     4
    ],
    [
     1,
     0,
     5
    ],
    [
     1,
     0,
     6
    ],
    [
     2,
     0,
     1
    ],
    [
     3,
     0,
     1
    ],
    [
     3,
     0,
     3
    ],
    [
     3,
     0,
     4
    ],
    [
     3,
     0,
     5
    ],
    [
     3,
     0,
     6
    ],
    [
     4,
     0,
     1
    ],
    [
     5,
     0,
     1
    ],
    [
     5,
     0,
     2
    ],
    [
     5,
     0,
     3
    ],
    [
     5,
     0,
     4
    ],
    [
     5,
     0,
     5
    ],
    [
     5,
     0,
     6
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "gold",
   "expect_blocks": 36
  },
  "case": {
   "cells": [
    [
     -6,
     0,
     1
    ],
    [
     -6,
     0,
     2
    ],
    [
     -6,
     0,
     3
    ],
    [
     -6,
     0,
     4
    ],
    [
     -6,
     0,
     5
    ],
    [
     -6,
     0,
     6
    ],
    [
     -6,
     0,
     7
    ],
    [
     -6,
     0,
     8
    ],
    [
     -6,
     0,
     9
    ],
    [
     -6,
     0,
     10
    ],
    [
     -6,
     0,
     11
    ],
    [
     -6,
     0,
     12
    ],
    [
     -6,
     0,
     13
    ],
    [
     -6,
     1,
     1
    ],
    [
     -6,
     1,
     2
    ],
    [
     -6,
     1,
     3
    ],
    [
     -6,
     1,
     4
    ],
    [
     -6,
     1,
     5
    ],
    [
     -6,
     1,
     6
    ],
    [
     -6,
     1,
     7
    ],
    [
     -6,
     1,
     8
    ],
    [
     -6,
     1,
     9
    ],
    [
     -6,
     1,
     10
    ],
    [
     -6,
     1,
     11
    ],
    [
     -6,
     1,
     12
    ],
    [
     -6,
     1,
     13
    ],
    [
     -5,
     0,
     7
    ],
    [
     -5,
     0,
     8
    ],
    [
     -5,
     0,
     9
    ],
    [
     -5,
     0,
     10
    ],
    [
     -5,
     0,
     11
    ],
    [
     -5,
     0,
     12
    ],
    [
     -5,
     0,
     13
    ],
    [
     -5,
     1,
     7
    ],
    [
     -5,
     1,
     8
    ],
    [
     -5,
     1,
     9
    ],
    [
     -5,
     1,
     10
    ],
    [
     -5,
     1,
     11
    ],
    [
     -5,
     1,
     12
    ],
    [
     -5,
     1,
     13
    ],
    [
     -4,
     0,
     13
    ],
    [
     -4,
     1,
     7
    ],
    [
     -4,
     1,
     8
    ],
    [
     -4,
     1,
     9
    ],
    [
     -4,
     1,
     10
    ],
    [
     -4,
     1,
     11
    ],
    [
     -4,
     1,
     12
    ],
    [
     -4,
     1,
     13
    ],
    [
     -3,
     0,
     7
    ],
    [
     -3,
     0,
     8
    ],
    [
     -3,
     0,
     9
    ],
    [
     -3,
     0,
     10
    ],
    [
     -3,
     0,
     11
    ],
    [
     -3,
     0,
     12
    ],
    [
     -3,
     0,
     13
    ],
    [
     -3,
     1,
     7
    ],
    [
     -3,
     1,
     8
    ],
    [
     -3,
     1,
     9
    ],
    [
     -3,
     1,
     10
    ],
    [
     -3,
     1,
     11
    ],
    [
     -3,
     1,
     12
    ],
    [
     -3,
     1,
     13
    ],
    [
     -2,
     0,
     13
    ],
    [
     -2,
     1,
     7
    ],
    [
     -2,
     1,
     8
    ],
    [
     -2,
     1,
     9
    ],
    [
     -2,
     1,
     10
    ],
    [
     -2,
     1,
     11
    ],
    [
     -2,
     1,
     12
    ],
    [
     -2,
     1,
     13
    ],
    [
     -1,
     0,
     7
    ],
    [
     -1,
     0,
     8
    ],
    [
     -1,
     0,
     9
    ],
    [
     -1,
     0,
     10
    ],
    [
     -1,
     0,
     11
    ],
    [
     -1,
     0,
     12
    ],
    [
     -1,
     0,
     13
    ],
    [
     -1,
     1,
     7
    ],
    [
     -1,
     1,
     8
    ],
    [
     -1,
     1,
     9
    ],
    [
     -1,
     1,
     10
    ],
    [
     -1,
     1,
     11
    ],
    [
     -1,
     1,
     12
    ],
    [
     -1,
     1,
     13
    ],
    [
     0,
     0,
     13
    ],
    [
     0,
     1,
     7
    ],
    [
     0,
     1,
     8
    ],
    [
     0,
     1,
     9
    ],
    [
     0,
     1,
     10
    ],
    [
     0,
     1,
     11
    ],
    [
     0,
     1,
     12
    ],
    [
     0,
     1,
     13
    ],
    [
     1,
     0,
     7
    ],
    [
     1,
     0,
     8
    ],
    [
     1,
     0,
     9
    ],
    [
     1,
     0,
     10
    ],
    [
     1,
     0,
     11
    ],
    [
     1,
     0,
     12
    ],
    [
     1,
     0,
     13
    ],
    [
     1,
     1,
     7
    ],
    [
     1,
     1,
     8
    ],
    [
     1,
     1,
     9
    ],
    [
     1,
     1,
     10
    ],
    [
     1,
     1,
     11
    ],
    [
     1,
     1,
     12
    ],
    [
     1,
     1,
     13
    ],
    [
     2,
     0,
     13
    ],
    [
     2,
     1,
     7
    ],
    [
     2,
     1,
     8
    ],
    [
     2,
     1,
     9
    ],
    [
     2,
     1,
     10
    ],
    [
     2,
     1,
     11
    ],
    [
     2,
     1,
     12
    ],
    [
     2,
     1,
     13
    ],
    [
     3,
     0,
     7
    ],
    [
     3,
     0,
     8
    ],
    [
     3,
     0,
     9
    ],
    [
     3,
     0,
     10
    ],
    [
     3,
     0,
     11
    ],
    [
     3,
     0,
     12
    ],
    [
     3,
     0,
     13
    ],
    [
     3,
     1,
     7
    ],
    [
     3,
     1,
     8
    ],
    [
     3,
     1,
     9
    ],
    [
     3,
     1,
     10
    ],
    [
     3,
     1,
     11
    ],
    [
     3,
     1,
     12
    ],
    [
     3,
     1,
     13
    ],
    [
     4,
     0,
     13
    ],
    [
     4,
     1,
     7
    ],
    [
     4,
     1,
     8
    ],
    [
     4,
     1,
     9
    ],
    [
     4,
     1,
     10
    ],
    [
     4,
     1,
     11
    ],
    [
     4,
     1,
     12
    ],
    [
     4,
     1,
     13
    ],
    [
     5,
     0,
     7
    ],
    [
     5,
     0,
     8
    ],
    [
     5,
     0,
     9
    ],
    [
     5,
     0,
     10
    ],
    [
     5,
     0,
     11
    ],
    [
     5,
     0,
     12
    ],
    [
     5,
     0,
     13
    ],
    [
     5,
     1,
     7
    ],
    [
     5,
     1,
     8
    ],
    [
     5,
     1,
     9
    ],
    [
     5,
     1,
     10
    ],
    [
     5,
     1,
     11
    ],
    [
     5,
     1,
     12
    ],
    [
     5,
     1,
     13
    ],
    [
     6,
     0,
     1
    ],
    [
     6,
     0,
     2
    ],
    [
     6,
     0,
     3
    ],
    [
     6,
     0,
     4
    ],
    [
     6,
     0,
     5
    ],
    [
     6,
     0,
     6
    ],
    [
     6,
     0,
     7
    ],
    [
     6,
     0,
     8
    ],
    [
     6,
     0,
     9
    ],
    [
     6,
     0,
     10
    ],
    [
     6,
     0,
     11
    ],
    [
     6,
     0,
     12
    ],
    [
     6,
     0,
     13
    ],
    [
     6,
     1,
     1
    ],
    [
     6,
     1,
     2
    ],
    [
     6,
     1,
     3
    ],
    [
     6,
     1,
     4
    ],
    [
     6,
     1,
     5
    ],
    [
     6,
     1,
     6
    ],
    [
     6,
     1,
     7
    ],
    [
     6,
     1,
     8
    ],
    [
     6,
     1,
     9
    ],
    [
     6,
     1,
     10
    ],
    [
     6,
     1,
     11
    ],
    [
     6,
     1,
     12
    ],
    [
     6,
     1,
     13
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "wood",
   "expect_blocks": 176
  },
  "kp0": {
   "cells": [
    [
     -4,
     0,
     3
    ],
    [
     -4,
     0,
     4
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 2
  },
  "dp0": {
   "cells": [
    [
     -4,
     0,
     5
    ],
    [
     -4,
     0,
     6
    ],
    [
     -4,
     0,
     7
    ],
    [
     -4,
     0,
     8
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 4
  },
  "sp0": {
   "cells": [
    [
     0,
     0,
     9
    ],
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ]
   ],
   "at": [
    0.19999999999999996,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 4,
   "block_shape": [
    0.5,
    0.5,
    0.8
   ]
  },
  "kp1": {
   "cells": [
    [
     -2,
     0,
     3
    ],
    [
     -2,
     0,
     4
    ],
    [
     -2,
     0,
     5
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 3
  },
  "dp1": {
   "cells": [
    [
     -2,
     0,
     6
    ],
    [
     -2,
     0,
     7
    ],
    [
     -2,
     0,
     8
    ],
    [
     -2,
     0,
     9
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 4
  },
  "sp1": {
   "cells": [
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ]
   ],
   "at": [
    0.39999999999999997,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 3,
   "block_shape": [
    0.5,
    0.5,
    0.8
   ]
  },
  "kp2": {
   "cells": [
    [
     0,
     0,
     3
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 1
  },
  "dp2": {
   "cells": [
    [
     0,
     0,
     4
    ],
    [
     0,
     0,
     5
    ],
    [
     0,
     0,
     6
    ],
    [
     0,
     0,
     7
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 4
  },
  "sp2": {
   "cells": [
    [
     0,
     0,
     8
    ],
    [
     0,
     0,
     9
    ],
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 5,
   "block_shape": [
    0.5,
    0.5,
    0.8
   ]
  },
  "kp3": {
   "cells": [
    [
     2,
     0,
     3
    ],
    [
     2,
     0,
     4
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 2
  },
  "dp3": {
   "cells": [
    [
     2,
     0,
     5
    ],
    [
     2,
     0,
     6
    ],
    [
     2,
     0,
     7
    ],
    [
     2,
     0,
     8
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 4
  },
  "sp3": {
   "cells": [
    [
     0,
     0,
     9
    ],
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ]
   ],
   "at": [
    0.8,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 4,
   "block_shape": [
    0.5,
    0.5,
    0.8
   ]
  },
  "kp4": {
   "cells": [
    [
     4,
     0,
     3
    ],
    [
     4,
     0,
     4
    ],
    [
     4,
     0,
     5
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "red",
   "expect_blocks": 3
  },
  "dp4": {
   "cells": [
    [
     4,
     0,
     6
    ],
    [
     4,
     0,
     7
    ],
    [
     4,
     0,
     8
    ],
    [
     4,
     0,
     9
    ]
   ],
   "at": [
    0.6,
    0.0
   ],
   "centered": false,
   "material": "blue",
   "expect_blocks": 4
  },
  "sp4": {
   "cells": [
    [
     0,
     0,
     10
    ],
    [
     0,
     0,
     11
    ],
    [
     0,
     0,
     12
    ]
   ],
   "at": [
    1.0,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 3,
   "block_shape": [
    0.5,
    0.5,
    0.8
   ]
  },
  "key": {
   "cells": [
    [
     -9,
     -1,
     1
    ],
    [
     -9,
     -1,
     2
    ],
    [
     -9,
     -1,
     3
    ],
    [
     -9,
     -1,
     4
    ],
    [
     -8,
     -1,
     1
    ],
    [
     -8,
     -1,
     4
    ],
    [
     -7,
     -1,
     1
    ],
    [
     -7,
     -1,
     2
    ],
    [
     -7,
     -1,
     3
    ],
    [
     -7,
     -1,
     4
    ],
    [
     -6,
     -1,
     2
    ],
    [
     -5,
     -1,
     2
    ],
    [
     -4,
     -1,
     2
    ],
    [
     -4,
     -1,
     3
    ],
    [
     -4,
     -1,
     4
    ],
    [
     -3,
     -1,
     2
    ],
    [
     -2,
     -1,
     2
    ],
    [
     -2,
     -1,
     3
    ],
    [
     -1,
     -1,
     2
    ],
    [
     0,
     -1,
     2
    ],
    [
     0,
     -1,
     3
    ],
    [
     0,
     -1,
     4
    ],
    [
     0,
     -1,
     5
    ],
    [
     1,
     -1,
     2
    ],
    [
     2,
     -1,
     2
    ],
    [
     2,
     -1,
     3
    ],
    [
     2,
     -1,
     4
    ],
    [
     3,
     -1,
     2
    ],
    [
     4,
     -1,
     2
    ],
    [
     4,
     -1,
     3
    ],
    [
     5,
     -1,
     2
    ]
   ],
   "at": [
    -0.4,
    0.0
   ],
   "centered": false,
   "material": "steel",
   "expect_blocks": 31
  }
 },
 "actions": [
  {
   "type": "move",
   "to": [
    -0.25,
    -0.45,
    0.0
   ],
   "when": 4.291,
   "until": 4.891
  },
  {
   "type": "build",
   "target": "base",
   "when": 4.891,
   "until": 5.391,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ],
   "order": "x"
  },
  {
   "type": "build",
   "target": "plug",
   "when": 5.391,
   "until": 6.391,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ],
   "order": "x"
  },
  {
   "type": "build",
   "target": "case",
   "when": 8.738,
   "until": 10.538,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ],
   "order": "z"
  },
  {
   "type": "build",
   "target": "kp0",
   "when": 10.99,
   "until": 11.21,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "kp1",
   "when": 11.21,
   "until": 11.43,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "kp2",
   "when": 11.43,
   "until": 11.65,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "kp3",
   "when": 11.65,
   "until": 11.87,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "kp4",
   "when": 11.87,
   "until": 12.09,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dp0",
   "when": 12.09,
   "until": 12.31,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dp1",
   "when": 12.31,
   "until": 12.53,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dp2",
   "when": 12.53,
   "until": 12.75,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dp3",
   "when": 12.75,
   "until": 12.97,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "dp4",
   "when": 12.97,
   "until": 13.19,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sp0",
   "when": 13.19,
   "until": 13.39,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sp1",
   "when": 13.39,
   "until": 13.59,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sp2",
   "when": 13.59,
   "until": 13.79,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sp3",
   "when": 13.79,
   "until": 13.99,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "build",
   "target": "sp4",
   "when": 13.99,
   "until": 14.19,
   "stand_at": [
    -0.25,
    -0.45,
    0.0
   ]
  },
  {
   "type": "move",
   "to": [
    -0.5000000000000001,
    -0.55,
    0.0
   ],
   "when": 14.19,
   "until": 14.99
  },
  {
   "type": "point",
   "target": "case",
   "pos": [
    0.6,
    -0.05,
    0.7
   ],
   "when": 15.19,
   "until": 17.595
  },
  {
   "type": "spin",
   "target": "plug",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp0",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp1",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp2",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp3",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp4",
   "axis": "x",
   "degrees": -6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.595,
   "until": 17.735,
   "ticks_per_turn": 0
  },
  {
   "type": "slide",
   "target": "plug",
   "by": [
    0,
    0,
    0
   ],
   "when": 17.635,
   "until": 17.735,
   "sound": "clunk"
  },
  {
   "type": "spin",
   "target": "plug",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp0",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp1",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp2",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp3",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp4",
   "axis": "x",
   "degrees": 6,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 17.815,
   "until": 18.035,
   "ticks_per_turn": 0
  },
  {
   "type": "move",
   "to": [
    -1.45,
    -0.32,
    0.0
   ],
   "when": 18.095,
   "until": 19.195
  },
  {
   "type": "build",
   "target": "key",
   "when": 19.627,
   "until": 20.527,
   "stand_at": [
    -1.45,
    -0.32,
    0.0
   ],
   "order": "x"
  },
  {
   "type": "slide",
   "target": "key",
   "by": [
    1.0,
    0,
    0
   ],
   "when": 20.627,
   "until": 21.727,
   "sound": "none"
  },
  {
   "type": "move",
   "to": [
    -0.5000000000000001,
    -0.4,
    0.0
   ],
   "when": 20.627,
   "until": 21.727
  },
  {
   "type": "slide",
   "target": "kp0",
   "by": [
    0,
    0,
    0.2
   ],
   "when": 21.307,
   "until": 21.567,
   "sound": "click",
   "local": true
  },
  {
   "type": "slide",
   "target": "dp0",
   "by": [
    0,
    0,
    0.2
   ],
   "when": 21.307,
   "until": 21.567,
   "sound": "none"
  },
  {
   "type": "grow",
   "target": "sp0",
   "factor": 0.5,
   "when": 21.307,
   "until": 21.567,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sp0",
   "by": [
    0,
    0,
    0.65
   ],
   "when": 21.307,
   "until": 21.567,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "kp1",
   "by": [
    0,
    0,
    0.1
   ],
   "when": 21.377,
   "until": 21.637,
   "sound": "click",
   "local": true
  },
  {
   "type": "slide",
   "target": "dp1",
   "by": [
    0,
    0,
    0.1
   ],
   "when": 21.377,
   "until": 21.637,
   "sound": "none"
  },
  {
   "type": "grow",
   "target": "sp1",
   "factor": 0.666667,
   "when": 21.377,
   "until": 21.637,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sp1",
   "by": [
    0,
    0,
    0.4333
   ],
   "when": 21.377,
   "until": 21.637,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "kp2",
   "by": [
    0,
    0,
    0.3
   ],
   "when": 21.447,
   "until": 21.707,
   "sound": "click",
   "local": true
  },
  {
   "type": "slide",
   "target": "dp2",
   "by": [
    0,
    0,
    0.3
   ],
   "when": 21.447,
   "until": 21.707,
   "sound": "none"
  },
  {
   "type": "grow",
   "target": "sp2",
   "factor": 0.4,
   "when": 21.447,
   "until": 21.707,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sp2",
   "by": [
    0,
    0,
    0.78
   ],
   "when": 21.447,
   "until": 21.707,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "kp3",
   "by": [
    0,
    0,
    0.2
   ],
   "when": 21.517,
   "until": 21.777,
   "sound": "click",
   "local": true
  },
  {
   "type": "slide",
   "target": "dp3",
   "by": [
    0,
    0,
    0.2
   ],
   "when": 21.517,
   "until": 21.777,
   "sound": "none"
  },
  {
   "type": "grow",
   "target": "sp3",
   "factor": 0.5,
   "when": 21.517,
   "until": 21.777,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sp3",
   "by": [
    0,
    0,
    0.65
   ],
   "when": 21.517,
   "until": 21.777,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "kp4",
   "by": [
    0,
    0,
    0.1
   ],
   "when": 21.587,
   "until": 21.847,
   "sound": "click",
   "local": true
  },
  {
   "type": "slide",
   "target": "dp4",
   "by": [
    0,
    0,
    0.1
   ],
   "when": 21.587,
   "until": 21.847,
   "sound": "none"
  },
  {
   "type": "grow",
   "target": "sp4",
   "factor": 0.666667,
   "when": 21.587,
   "until": 21.847,
   "sound": "none"
  },
  {
   "type": "slide",
   "target": "sp4",
   "by": [
    0,
    0,
    0.4333
   ],
   "when": 21.587,
   "until": 21.847,
   "sound": "none"
  },
  {
   "type": "point",
   "target": "key",
   "pos": [
    -0.20000000000000007,
    -0.1,
    0.3
   ],
   "when": 25.656,
   "until": 26.906
  },
  {
   "type": "spin",
   "target": "plug",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 12
  },
  {
   "type": "spin",
   "target": "key",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp0",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp1",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp2",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp3",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "spin",
   "target": "kp4",
   "axis": "x",
   "degrees": -90,
   "pivot": [
    0,
    0,
    3.5
   ],
   "when": 25.956,
   "until": 26.706,
   "ticks_per_turn": 0
  },
  {
   "type": "move",
   "to": [
    -0.5000000000000001,
    -0.55,
    0.0
   ],
   "when": 27.006,
   "until": 27.706
  },
  {
   "type": "remove",
   "target": "key",
   "when": 27.706,
   "until": 28.006,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "sp0",
   "when": 28.006,
   "until": 28.106,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "sp1",
   "when": 28.106,
   "until": 28.206,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "sp2",
   "when": 28.206,
   "until": 28.306,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "sp3",
   "when": 28.306,
   "until": 28.406,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "sp4",
   "when": 28.406,
   "until": 28.506,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "dp0",
   "when": 28.506,
   "until": 28.606,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "dp1",
   "when": 28.606,
   "until": 28.706,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "dp2",
   "when": 28.706,
   "until": 28.806,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "dp3",
   "when": 28.806,
   "until": 28.906,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "dp4",
   "when": 28.906,
   "until": 29.006,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "kp0",
   "when": 29.006,
   "until": 29.086,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "kp1",
   "when": 29.086,
   "until": 29.166,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "kp2",
   "when": 29.166,
   "until": 29.246,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "kp3",
   "when": 29.246,
   "until": 29.326,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "kp4",
   "when": 29.326,
   "until": 29.406,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "case",
   "when": 29.406,
   "until": 30.156,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "plug",
   "when": 30.156,
   "until": 30.506,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  },
  {
   "type": "remove",
   "target": "base",
   "when": 30.506,
   "until": 30.756,
   "stand_at": [
    -0.5000000000000001,
    -0.55,
    0.0
   ]
  }
 ],
 "camera": [
  {
   "at": 0.0,
   "target": [
    -0.15000000000000002,
    0.0,
    0.65
   ],
   "az": -90,
   "el": 8,
   "dist": 2.5
  },
  {
   "at": 4.491,
   "target": [
    -0.15000000000000002,
    0.0,
    0.65
   ],
   "az": -90,
   "el": 8,
   "dist": 2.5
  },
  {
   "at": 5.791,
   "target": [
    0.35,
    0.0,
    0.75
   ],
   "az": -86,
   "el": 9,
   "dist": 2.8
  },
  {
   "at": 11.49,
   "target": [
    0.4,
    0.0,
    0.8
   ],
   "az": -92,
   "el": 7,
   "dist": 2.45
  },
  {
   "at": 15.29,
   "target": [
    0.25,
    0.0,
    0.7
   ],
   "az": -90,
   "el": 6,
   "dist": 2.5
  },
  {
   "at": 18.095,
   "target": [
    0.25,
    0.0,
    0.7
   ],
   "az": -90,
   "el": 6,
   "dist": 2.5
  },
  {
   "at": 19.195,
   "target": [
    -0.55,
    0.0,
    0.55
   ],
   "az": -90,
   "el": 9,
   "dist": 3.2
  },
  {
   "at": 20.627,
   "target": [
    -0.55,
    0.0,
    0.55
   ],
   "az": -90,
   "el": 9,
   "dist": 3.2
  },
  {
   "at": 21.927,
   "target": [
    0.2,
    0.0,
    0.6
   ],
   "az": -88,
   "el": 13,
   "dist": 2.7
  },
  {
   "at": 26.906,
   "target": [
    0.2,
    0.0,
    0.62
   ],
   "az": -89,
   "el": 11,
   "dist": 2.75
  },
  {
   "at": 30.706,
   "target": [
    -0.15000000000000002,
    0.0,
    0.65
   ],
   "az": -90,
   "el": 8,
   "dist": 2.5
  },
  {
   "at": "end",
   "target": [
    -0.15000000000000002,
    0.0,
    0.65
   ],
   "az": -90,
   "el": 8,
   "dist": 2.5
  }
 ],
 "cards": [
  {
   "style": "title",
   "text": "SECRET CODE",
   "color": "amber",
   "y": 470,
   "size": 104,
   "when": 0.9,
   "until": 5.39
  },
  {
   "style": "title",
   "text": "PIN TUMBLER LOCK",
   "color": "white",
   "y": 470,
   "size": 88,
   "when": 5.69,
   "until": 10.99
  },
  {
   "style": "panel",
   "y": 340,
   "border": "cyan",
   "when": 11.19,
   "until": 14.99,
   "rows": [
    {
     "text": "KEY PINS",
     "color": "red",
     "at": 11.29,
     "size": 52
    },
    {
     "text": "DRIVER PINS",
     "color": "cyan",
     "at": 12.29,
     "size": 52
    },
    {
     "text": "SPRINGS",
     "color": "white",
     "at": 13.39,
     "size": 52
    }
   ]
  },
  {
   "style": "title",
   "text": "SHEAR LINE",
   "color": "cyan",
   "y": 470,
   "size": 96,
   "when": 15.39,
   "until": 17.7
  },
  {
   "style": "impact",
   "text": "LOCKED!",
   "color": "red",
   "y": 560,
   "size": 150,
   "when": 17.7,
   "until": 19.0
  },
  {
   "style": "panel",
   "y": 340,
   "border": "amber",
   "when": 20.15,
   "until": 24.0,
   "rows": [
    {
     "text": "5 CUTS = 5 HEIGHTS",
     "color": "white",
     "at": 20.25,
     "size": 50
    },
    {
     "text": "THE SECRET CODE!",
     "color": "amber",
     "at": 21.85,
     "size": 56
    }
   ]
  },
  {
   "style": "impact",
   "text": "CLICK!",
   "color": "green",
   "y": 560,
   "size": 150,
   "when": 26.66,
   "until": 27.81
  }
 ]
}

## 2. ⬆️ Upload your story
Click **Choose files** and pick your `.json` story file. If you upload nothing (or cancel), the `EXAMPLE` story
from the settings is used. The story's `slug` becomes the video's file name.

In [ ]:
import json, shutil
from google.colab import files
STORY = "stories/%s.json" % EXAMPLE
try:
    up = files.upload()
except Exception as e:
    up = {}
    print("upload skipped:", e)
for fn, data in up.items():
    if fn.lower().endswith(".json"):
        dst = os.path.join("stories", os.path.basename(fn))
        open(dst, "wb").write(data)
        STORY = dst
story = json.load(open(STORY))                    # fails here (with the line number) if the JSON is malformed
SLUG = story.get("slug", os.path.splitext(os.path.basename(STORY))[0])
print("story:", STORY, "|", story.get("title"), "|", len(story["lines"]), "lines,", len(story["structures"]), "structures")
shutil.rmtree("build", ignore_errors=True)        # fresh build folder for this story
FRAMES_DIR = ("/content/drive/MyDrive/block_lab/frames_" + SLUG) if SAVE_FRAMES_TO_DRIVE else "/content/frames_" + SLUG
if QUICK_DRAFT:
    FRAMES_DIR = WORK + "/build/draft"
os.makedirs(FRAMES_DIR, exist_ok=True)

## 3. Install Blender 5.2 and Kokoro TTS (first run only)

In [ ]:
import glob, re, subprocess, urllib.request
BLENDER_SERIES = "5.2"
base = "https://download.blender.org/release/Blender%s/" % BLENDER_SERIES
if not glob.glob("/content/blender-*/blender"):
    req = urllib.request.Request(base, headers={"User-Agent": "Mozilla/5.0"})   # blender.org refuses Python's default agent
    html = urllib.request.urlopen(req).read().decode()
    builds = sorted(set(re.findall(r"blender-%s\.(\d+)-linux-x64\.tar\.xz" % re.escape(BLENDER_SERIES), html)), key=int)
    tarball = "blender-%s.%s-linux-x64.tar.xz" % (BLENDER_SERIES, builds[-1])
    print("downloading", base + tarball)
    !wget -q --user-agent="Mozilla/5.0" {base + tarball} -O /content/blender.tar.xz
    assert os.path.getsize("/content/blender.tar.xz") > 100e6, "Blender download failed - check " + base
    !tar -xf /content/blender.tar.xz -C /content && rm /content/blender.tar.xz
!apt-get -qq install -y libxrender1 libxi6 libxkbcommon0 libsm6 libgl1 libxxf86vm1 libxfixes3 > /dev/null 2>&1
BLENDER = sorted(glob.glob("/content/blender-*/blender"))[-1]
BPY = sorted(glob.glob(os.path.join(os.path.dirname(BLENDER), "*", "python", "bin", "python3*")))[0]
!{BPY} -m ensurepip -q > /dev/null 2>&1
!{BPY} -m pip install -q scipy
!{BLENDER} --version | head -1
print("BLENDER =", BLENDER)

In [ ]:
!pip -q install kokoro-onnx soundfile
KOKORO_DIR = "/content/kokoro"
os.makedirs(KOKORO_DIR, exist_ok=True)
rel = "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/"
for fn in ("kokoro-v1.0.onnx", "voices-v1.0.bin"):
    if not os.path.exists(os.path.join(KOKORO_DIR, fn)):
        !wget -q {rel + fn} -O {KOKORO_DIR}/{fn}
os.environ["KOKORO_MODEL"] = KOKORO_DIR + "/kokoro-v1.0.onnx"
os.environ["KOKORO_VOICES"] = KOKORO_DIR + "/voices-v1.0.bin"

## 4. Grandpa's voice + the plan (the whole edit follows his timing)

In [ ]:
!python bl_voice.py --story {STORY}
!python bl_common.py
from IPython.display import Audio
Audio("build/audio/voice.wav")

## 5. Build the scene + preview stills with the lesson graphics

In [ ]:
plan = json.load(open("build/plan.json"))
FR = [int(plan["nf"] * k / 8) for k in range(8)]
!{BLENDER} -b --factory-startup -P bl_scene.py -- --preview {" ".join(map(str, FR))} 2>&1 | grep -E "Error|Traceback|Saved as"
os.makedirs("build/stillsrc", exist_ok=True)
for f in FR:
    shutil.copy("build/previews/%04d.png" % f, "build/stillsrc/%04d.png" % f)
!python bl_overlay.py --frames build/stillsrc --still {" ".join(map(str, FR))}
from PIL import Image
from IPython.display import display
row = Image.new("RGB", (270 * len(FR), 480))
for i, f in enumerate(FR):
    row.paste(Image.open("build/stills/%04d.png" % f).resize((270, 480)), (270 * i, 0))
display(row)

## 6. Sound + mix

In [ ]:
!python bl_audio.py
Audio("build/audio/mix.wav")

## 7. Render all frames
This step takes the most time. It's resumable: re-running it skips frames that already exist.

In [ ]:
gpu = "1" if USE_GPU else "0"
draft = "--draft" if QUICK_DRAFT else ""
!GPU={gpu} FRAMES_DIR={FRAMES_DIR} {BLENDER} -b -P render.py -- {draft} 2>&1 | grep -E --line-buffered "Rendering on|No CUDA|RENDERED|RENDER DONE|Error"
print(len(os.listdir(FRAMES_DIR)), "of", plan["nf"], "frames rendered")

## 8. Lesson graphics, captions + encode

In [ ]:
!python bl_overlay.py --frames {FRAMES_DIR}
!ffprobe -v error -show_entries stream=codec_name,width,height,pix_fmt,r_frame_rate:format=duration -of compact {SLUG}.mp4

## 9. Verify

In [ ]:
!{BLENDER} -b -P bl_verify.py 2>&1 | grep -E "PASS|FAIL|OVERALL"

## 10. Watch / download

In [ ]:
from IPython.display import Video
display(Video(SLUG + ".mp4", embed=True, width=360))
files.download(SLUG + ".mp4")

---
## 📄 How a story file works (quick reference)

Every time in a story is either seconds (`1.3`), a moment inside one of grandpa's lines
(`["legs", 0.62]` means 62% of the way through the line called `legs`), a line's start or end plus/minus seconds
(`["giant", "end", 0.05]`), or `"end"`.

| Section | What goes in it |
|---|---|
| `title`, `slug`, `voice`, `speed`, `format` | The name, the file name, the Kokoro voice (`am_santa`), the speaking speed, and `"format": "wide"` for a 1920x1080 video (leave it out for a vertical Short). `"loop": true` makes a seamless looping Short: leave out the end card, make the first and last frames the same shot; the sound past the end is folded onto the start, and the checks compare the first and last frames. |
| `lines` | Grandpa's script, as `{"id", "text", "pause"}`. Write it as hook, then value, then reward. |
| `structures` | What can be built. Each one takes **cells** (`[[x,y,z], …]`), **copy_of** + **scale**, **box** `[nx,ny,nz]`, **gnomon** `k` (L-shape) or **pyramid** `[9,7,5,…]`. Options: `at` `[x,y]` in metres, `material` (sandstone, stone, wood, brick, gold, grass, blue, iron, rubber, red; each one also has its own block sound), `eyes`, `stand` (right, left, behind or front), `parts` (named subsets such as `{"legs": "z < 2"}`), `centered`, `expect_blocks`, `block_shape` `[fx,fy,fz]` (thin blocks: `[1,1,0.35]` planks, `[0.3,0.3,1]` posts or rails) |
| `actions` | `build` (with `order` x, y or z, and optional `stand_at` [x, y] for where grandpa stands), `point` (`pos`, or `target` + `offset`), `highlight` (`target` + `part`), `grow` (`factor`, `to`), `crack` (`part`; optional `cool_from` / `cool_until` to let the glow fade again, and `sound`: `sizzle`, `spark`, `bang` or `none`), `collapse`, `shrug`, `move` (`to` `[x,y]` or `[x,y,height]`, optional `via` waypoints and `face` `[x,y]`: going up or down in height plays the ladder climb), `remove` (takes a built structure apart, top first), `face_camera`. `build` / `remove` with `"quiet": true` happen without grandpa (e.g. a crate filling). Positions such as `stand_at` can include a height to stand on scaffolding. Moving structures: `spin` (`axis` x/y/z, `degrees`, `pivot` block [x,y,z], optional `ease`: false, `ticks_per_turn`), `swing` (`axis`, `degrees`, `period`, `pivot`), `slide` (`by` [dx,dy,dz] metres; optional `sound`: `chug` or `none`), `fly` (a thrown object on a true arc: `v0` [vx,vy,vz] m/s, `g`, optional `tau0`, `origin`, `anchor`, `land`: true for a thud). Several `grow`s on one structure multiply. `melt`: the structure drops into a flat slab over its own footprint (pair it with a `crack` on a part covering everything for the molten glow). Each one takes `when` / `until`. |
| `camera` | Keys `{at, target, az, el, dist}`, smoothly eased between |
| `cards` | Styles `title`, `bubble` (live counts with `{count:STRUCT}`, or `{count:tower*}` to add up every structure whose name starts with `tower`), `tag`, `panel` (timed `rows`), `impact` and `end`. Each one takes `when` / `until`. |
| `world` | The setting (leave it out for the sunny desert): `ground` (desert, grass, snow, beach, stone, paris), `time` (day, morning, sunset, night), `weather` (clear, clouds, rain, snow), `animals` (any of birds, seagulls, pigeons, cat). Background sound follows the world: rain, waves, crickets at night, birdsong, gulls. |
| `grandpa` | `start` `[x,y]`, and `outfit`: `hat` (cap, hardhat, santa, beret, straw, beanie, none), `shirt` (rust, red, blue, green, yellow, purple, white, striped), `extras` (any of goggles, scarf, vest, hammer) |

Open `stories/giants.json` and `stories/odd_squares.json` above as templates. Ask Claude for a new story file
for any lesson.